# Cahpter IV

In [1]:
%matplotlib inline
import re
import sys
from pathlib import Path

import hyperspy.api as hs
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib import gridspec, patches, ticker
from matplotlib.colorbar import Colorbar
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, to_hex, to_rgba
from matplotlib.lines import Line2D
from matplotlib.transforms import Bbox
from mpl_toolkits.mplot3d import Axes3D
from scipy import ndimage


In [ ]:
# Ensure custom module Path is set before import
sys.path.append(r"D:\CHENG\OneDrive - UAB\ICMAB-Python\Figure")
from colors import main  # type: ignore

main()

In [2]:
# Ensure custom module Path is set before import
sys.path.append(r"D:\CHENG\OneDrive - UAB\ICMAB-Python\Figure")
from colors import tol_cmap, tol_cset  # type: ignore

# 画图的初始设置
plt.style.use(r"D:\CHENG\OneDrive - UAB\ICMAB-Python\Figure\liuchzzyy.mplstyle")
# print(plt.style.available)

# xarray setting
xr.set_options(
    cmap_sequential="viridis",
    cmap_divergent="viridis",
    display_width=150,
)  # viridis, gray

# 颜色设定
colors = tol_cset("vibrant")
if colors is not None:
    colors = list(colors)
    colors_opt = ["#b0a3d1", "#8bd0d5", "#a8e0ee", "#c5e1a3", "#ffe48b", "#f5a37d", "#e88db1"]
    colors_opt2 = list(tol_cset("bright"))
else:
    # Fallback colors in case tol_cset returns None
    colors = ["#0077BB", "#33BBEE", "#009988", "#EE7733", "#CC3311", "#EE3377", "#BBBBBB"]
if r"sunset" not in plt.colormaps():
    cmap = tol_cmap("sunset")
    if isinstance(cmap, LinearSegmentedColormap):
        plt.colormaps.register(cmap)
if r"rainbow_PuRd" not in plt.colormaps():
    cmap = tol_cmap("rainbow_PuRd")
    if isinstance(cmap, LinearSegmentedColormap):
        plt.colormaps.register(cmap)  # 备用 plasma

# 输出的文件夹
path_out = Path(r"C:\Users\chengliu\Desktop\Figure")

# Set math font
mpl.rcParams["mathtext.fontset"] = "custom"
mpl.rcParams["mathtext.rm"] = "Arial"
mpl.rcParams["mathtext.it"] = "Arial:italic"
mpl.rcParams["mathtext.bf"] = "Arial:bold"
mpl.rcParams["mathtext.sf"] = "Arial"
mpl.rcParams["mathtext.tt"] = "Arial"
mpl.rcParams["mathtext.cal"] = "Arial"
mpl.rcParams["mathtext.default"] = "regular"

## TOC

In [ ]:
# 读取一个常规的 αMnO2 的数据
echem = []
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\1M ZnSO4 + 0.2M MnSO4").glob(r"LC*.xlsx"))
# 读取电化学数据
for path_file in path_filelist[0:1]:
    df = pd.read_excel(path_file, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echem.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echem)):
    echem[i] = echem[i][echem[i]["Cycle"].isin([0, 1, 2])]
    echem[i] = echem[i][echem[i]["Voltage/V"] >= 0]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(2, 1, width_ratios=None, height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.3)

ax.set_axis_off()
labels = [
    [None, None],
]
for _, data in enumerate(echem):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        if j == 0:
            ax.plot(
                temp.loc[idx_min:, "SpeCap/mAh/g"],
                temp.loc[idx_min:, "Voltage/V"],
                ls="-",
                lw=1.0,
                c=colors[j],
                zorder=0,
            )  # 充电

# 图 B
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.3)

ax.set_axis_off()
labels = [
    [None, None],
]
for i, data in enumerate(echem):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        if j == 1:
            ax.plot(
                temp.loc[:idx_min, "SpeCap/mAh/g"],
                temp.loc[:idx_min, "Voltage/V"],
                ls="-",
                lw=1.0,
                c=colors[j],
                label=labels[i][j],
                zorder=0,
            )  # 放电


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TOC_00_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TOC_00_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TOC_00_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TOC_00_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

## XAS

### Figrue operando EXAFS: Mn

In [ ]:
# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\Echem").glob(r"**\*.txt"))
# 读取电化学数据
echem_all = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem_all.append(df)

# 合并所有电化学数据为一个二维表格
echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 读取 reference + operando 数据
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")

data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_1.chir2_mag"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_2.chir2_mag"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
exafs_Mn = pd.concat([data1, data2.iloc[:, 4:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, 2, 3, *list(range(13, exafs_Mn.shape[1], 1))]
exafs_Mn = exafs_Mn.iloc[:, charge_index]  # noqa: N816
pdf_Mn = exafs_Mn.iloc[:, 0:4]  # noqa: N816
pdf_Mn.columns = [
    # r"Energy",
    r"0.2M_MnSO4(aq.)",
    r"alpha_MnO2_Electrode",
    r"alpha_MnO2_Powder",
    r"FC1st",
]
opData_Mn = exafs_Mn.iloc[:, list(range(4, exafs_Mn.shape[1]))]  # noqa: N816
opData_Mn.columns = list(range(0, opData_Mn.shape[1], 1))

# # Zn
# data1 = pd.read_csv(
#     Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_1.chir2_mag"),
#     sep=r"\s+",
#     index_col=0,
#     header=None,
#     comment="#",
# )
# data2 = pd.read_csv(
#     Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_2.chir2_mag"),
#     sep=r"\s+",
#     index_col=0,
#     header=None,
#     comment="#",
# )
# exafs_Zn = pd.concat([data1, data2.iloc[:, 2:]], axis=1, ignore_index=True)  # noqa: N816
# charge_index = [0, 1, *list(range(12, exafs_Zn.shape[1], 1))]
# exafs_Zn = exafs_Zn.iloc[:, charge_index]  # noqa: N816

# pdf_Zn = exafs_Zn.iloc[:, 0:2]  # noqa: N816
# pdf_Zn.columns = [
#     # r"Energy",
#     r"0.5M_ZnSO4(aq.)",
#     r"ZSH"]
# opData_Zn = exafs_Zn.iloc[:, list(range(2, exafs_Zn.shape[1]))]  # noqa: N816
# opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))

In [ ]:
# 电化学时间对齐
echem_all["Time"] = (df["time/s"] - df["time/s"].iloc[0]).dt.total_seconds() / 3600

# 剔除不好的谱线， Mn
opData_Mn = opData_Mn.drop(columns=opData_Mn.columns[1], axis=1)  # noqa: N816

echem_index_low = echem_all[echem_all["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high = echem_all[echem_all["cycle number"] == 1]["Ewe/V"].idxmax()

In [ ]:
cutoff_idx = np.where(opData_Mn.index < 6.0)[0]
if cutoff_idx.size == 0:
    raise ValueError("No R-space points below 6.0 Å found.")
opData_Mn = opData_Mn.iloc[: cutoff_idx[-1] + 1, :]  # noqa: N816

In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 5))
gs = gridspec.GridSpec(2, 2, width_ratios=[0.5, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1.0, 1.0))
ax.set_box_aspect(2.0)
time = echem_all["Time"].iloc[echem_index_low:echem_index_high] - echem_all["Time"].iloc[echem_index_low]
ax.plot(echem_all["Ewe/V"].iloc[echem_index_low:echem_index_high], time, c=colors[0], ls="-", lw=1.0)


# 设置刻度线等格式
ax.set_ylabel(r"Duration Hours (h)", fontsize=9)
ax.set_ylim(-0.3, 19)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1))

ax.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}}\!)$", fontsize=9)
ax.set_xlim(0.7, 1.9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=-0.1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))

ax.tick_params(axis="both", which="both", bottom=True, left=True, labelbottom=True, labelleft=True)
ax.text(0.03, 0.15, r"C/10", ha="left", va="top", rotation=90, transform=ax.transAxes, fontsize=9, c="k")

# ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

# ax.text(
#     0.01,
#     0.15,
#     r"0.5 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[0],  # type: ignore
# )
# ax.text(
#     0.01,
#     0.23,
#     r"1 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[1],  # type: ignore
# )

ax2 = ax.twiny()
ax2.plot(echem_all["<I>/mA"].iloc[echem_index_low:echem_index_high] * 1000, time, c=colors[1], ls="--", lw=1.0)

# 设置刻度线等格式
ax2.set_xlabel(r"$\mathrm{Current \ (\mu A)}$", fontsize=9)
ax2.set_xlim(-80, 80)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=20, offset=0))

ax2.tick_params(axis="both", which="both", top=True, left=True, labeltop=True, labelleft=True)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.3, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.0)

ax.imshow(
    opData_Mn.T,
    aspect="auto",
    extent=[0.0, 6.0, 0, opData_Mn.shape[1]],
    cmap="coolwarm",
    origin="lower",  # vmin=0, vmax=1.2,
)

# ax.set_xlim(0.0, 6.0)
ax.set_xlabel(r"$\mathrm{R \ space \ (\AA)}$", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_yticks([])

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.05, 0.1, 0.10, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.ax.text(
    1.045,
    0.96,
    r"High",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
colorbar.ax.text(
    1.055,
    0.08,
    r"Low",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
# colorbar.outline.set_visible(False)  # type: ignore

# # 图 C
# subfig = fig.add_subfigure(gs[1, 0])
# ax = subfig.add_axes((0, -0.2, 1.0, 1.0))
# ax.set_box_aspect(2.0)
# time = echem_all['Time'].iloc[echem_index_low:echem_index_high] - echem_all['Time'].iloc[echem_index_low]
# ax.plot(echem_all['Ewe/V'].iloc[echem_index_low:echem_index_high], time, c=colors[0], ls='-', lw=1.0)


# # 设置刻度线等格式
# ax.set_ylabel(r"Duration Hours (h)", fontsize=9)
# ax.set_ylim(-0.3, 19)
# ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
# ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1))

# ax.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}}\!)$", fontsize=9)
# ax.set_xlim(0.7, 1.9)
# ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=-0.1))
# ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))

# ax.tick_params(axis="both", which="both", bottom=True, left=True, labelbottom=True, labelleft=True)
# ax.text(0.03, 0.15, r"C/10", ha="left", va="top", rotation=90, transform=ax.transAxes, fontsize=9, c="k")

# # ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

# # ax.text(
# #     0.01,
# #     0.15,
# #     r"0.5 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
# #     ha="left",
# #     va="top",
# #     transform=ax.transAxes,
# #     fontsize=9,
# #     c=colors[0],  # type: ignore
# # )
# # ax.text(
# #     0.01,
# #     0.23,
# #     r"1 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
# #     ha="left",
# #     va="top",
# #     transform=ax.transAxes,
# #     fontsize=9,
# #     c=colors[1],  # type: ignore
# # )

# ax2 = ax.twiny()
# ax2.plot(echem_all['<I>/mA'].iloc[echem_index_low:echem_index_high]*1000, time, c=colors[1], ls='--', lw=1.0)

# # 设置刻度线等格式
# ax2.set_xlabel(r"$\mathrm{Current \ (\mu A)}$", fontsize=9)
# ax2.set_xlim(-80, 80)
# ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
# ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=20, offset=0))

# ax2.tick_params(axis="both", which="both", top=True, left=True, labeltop=True, labelleft=True)

# # 图 D
# subfig = fig.add_subfigure(gs[1, 1], zorder=0)
# ax = subfig.add_subplot()
# ax.set_position((0.15, -0.2, 1.0, 1.0))
# ax.set_box_aspect(1.0)

# ax.imshow(
#     opData_Zn.T, aspect='auto',
#     extent=[0.0, 6.0, 0, opData_Zn.shape[1]],
#     cmap='coolwarm', origin='lower', vmin=0, vmax=2.4,
#     )

# ax.set_xlim(0.0, 6.0)
# ax.set_xlabel(r"Energy (eV)", fontsize=11)
# ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1, offset=0))
# ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

# ax.set_yticks([])

# # Add colorbar and adjust ticks afterwards
# colorbar = Colorbar(
#     ax=ax.inset_axes((1.08, 0.1, 0.10, 0.8)),
#     location="right",
#     orientation="vertical",
#     cmap='coolwarm',
#     ticklocation="right",
#     spacing="proportional",
#     drawedges=False,
# )
# colorbar.set_ticks([])
# colorbar.ax.text(
#     1.06,
#     0.98,
#     r"High",
#     transform=ax.transAxes,
#     fontsize=9,
#     va="top",
#     ha="left",
#     fontfamily="Arial",
# )
# colorbar.ax.text(
#     1.08,
#     0.08,
#     r"Low",
#     transform=ax.transAxes,
#     fontsize=9,
#     va="top",
#     ha="left",
#     fontfamily="Arial",
# )
# # colorbar.outline.set_visible(False)  # type: ignore


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_EXAFS_00_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_EXAFS_00_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_EXAFS_00_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_EXAFS_00_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure EdgeJump -V1

In [ ]:
# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\Echem").glob(r"**\*.txt"))
# 读取电化学数据
echem_all = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem_all.append(df)

# 合并所有电化学数据为一个二维表格
echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

ej_path = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\EdgeJump")
edgejump = pd.read_excel(ej_path.joinpath(r"Cell2_Edge Jump.xlsx"), sheet_name=r"EJ_Cheng", header=0, comment="#")

In [ ]:
# 电化学时间对齐
echem_all["Time"] = (echem_all["time/s"] - echem_all["time/s"].iloc[0]).dt.total_seconds() / 3600

echem_index_low = echem_all[echem_all["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high = echem_all[echem_all["cycle number"] == 1]["Ewe/V"].idxmax()

In [ ]:
edgejump_mn2 = edgejump.iloc[9:, [2, 3]].copy().reset_index(drop=True)
edgejump_mn4 = edgejump.iloc[9:, [4, 5, 8, 9]].copy().reset_index(drop=True)

edgejump_zn2 = edgejump.iloc[9:, [12, 13]].copy().reset_index(drop=True)
edgejump_zsh = edgejump.iloc[9:, [14, 15, 18, 19]].copy().reset_index(drop=True)

In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 6))
gs = gridspec.GridSpec(3, 2, width_ratios=[1, 1], height_ratios=[1, 1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.5)

time = echem_all["Time"].iloc[echem_index_low:echem_index_high] - echem_all["Time"].iloc[echem_index_low]
ax.plot(time, echem_all["Ewe/V"].iloc[echem_index_low:echem_index_high], c=colors[0], ls="-", lw=1.0, label=r"Voltage")

# 设置刻度线等格式
# ax.set_xlabel(r"Duration Hours (h)", fontsize=9)
ax.set_xlim(-0.3, 19)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}}\!)$", fontsize=9)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))

ax.tick_params(axis="both", which="both", bottom=False, labelbottom=False, left=True, labelleft=True)
ax.text(0.03, 0.10, r"C/10", ha="left", va="top", rotation=0, transform=ax.transAxes, fontsize=9, c="k")

ax.legend(loc="upper right", bbox_to_anchor=(0.95, 1.05), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.spines[["bottom", "top"]].set_visible(False)
# ax.text(
#     0.01,
#     0.15,
#     r"0.5 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[0],  # type: ignore
# )
# ax.text(
#     0.01,
#     0.23,
#     r"1 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[1],  # type: ignore
# )

# 插入图片
ax2 = ax.twinx()
ax2.set_position((0, 0, 1.0, 1.0))
ax2.set_box_aspect(0.5)
ax2.plot(time, echem_all["<I>/mA"].iloc[echem_index_low:echem_index_high] * 1000, c=colors[1], ls="--", lw=1.0, label=r"Current")

# 设置刻度线等格式
ax2.set_ylabel(r"$\mathrm{Current \ (\mu A)}$", fontsize=9)
ax2.set_ylim(-80, 80)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=40, offset=0))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=20, offset=0))

ax2.tick_params(axis="both", which="both", bottom=False, labelbottom=False, right=True, labelright=True)
ax2.legend(loc="upper right", bbox_to_anchor=(0.7, 1.05), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax2.spines[["bottom", "top"]].set_visible(False)

# 图 B
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.5)

timea = np.linspace(0, time.iat[-1], num=edgejump_mn2.shape[0])
ax.errorbar(timea, edgejump_mn2.iloc[:, 0], edgejump_mn2.iloc[:, 1], c=colors[0], ls="-", lw=1.0, label=r"$\mathrm{Liquid \ Mn^{2+}}$", marker=r"o")
ax.spines[["bottom", "top"]].set_visible(False)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1.15), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.tick_params(axis="both", which="both", bottom=False, labelbottom=False, left=True, labelleft=True)

# 设置刻度线等格式
# ax.set_xlabel(r"Duration Hours (h)", fontsize=9)
ax.set_xlim(-0.3, 19)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1))
ax.set_ylabel(r"Mn in $\mathrm{Edge \ Jump \ (arb.u.)}$", fontsize=9)
ax.set_ylim(0.05, 0.17)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.03, offset=-0.01))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.015, offset=-0.01))

# 插入图片
ax2 = ax.twinx()
ax2.set_position((0.0, 0.0, 1.0, 1.0))
ax2.set_box_aspect(0.5)
ax2.errorbar(timea, edgejump_zn2.iloc[:, 0], edgejump_zn2.iloc[:, 1], c=colors[1], ls="--", lw=1.0, label=r"$\mathrm{Liquid \ Zn^{2+}}$", marker=r"s")
ax2.spines[["bottom", "top"]].set_visible(False)
ax2.legend(loc="upper right", bbox_to_anchor=(0.7, 1.15), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax2.tick_params(axis="both", which="both", bottom=False, labelbottom=False, right=True, labelright=True)

# 刻度线
ax2.set_ylabel(r"Zn in $\mathrm{Edge \ Jump \ (arb.u.)}$", fontsize=9)
ax2.set_ylim(0.15, 0.30)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.05, offset=0))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.025, offset=0))

# 图 C
subfig = fig.add_subfigure(gs[2, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.5)

timea = np.linspace(0, time.iat[-1], num=edgejump_mn4.shape[0])
ax.errorbar(timea, edgejump_mn4.iloc[:, 0], edgejump_mn4.iloc[:, 1], c=colors[0], ls="-", lw=1.0, label=r"$\mathrm{Solid \ Mn}$", marker=r"o")
ax.spines[["top"]].set_visible(False)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1.15), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.tick_params(axis="both", which="both", bottom=True, labelbottom=True, left=True, labelleft=True)

# 设置刻度线等格式
ax.set_xlabel(r"Duration Hours (h)", fontsize=9)
ax.set_xlim(-0.3, 19)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1))
ax.set_ylabel(r"Mn in $\mathrm{Edge \ Jump \ (arb.u.)}$", fontsize=9)
ax.set_ylim(0.32, 0.44)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.03, offset=-0.01))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.015, offset=-0.01))

# 插入图片
ax2 = ax.twinx()
ax2.set_position((0.0, 0.0, 1.0, 1.0))
ax2.set_box_aspect(0.5)
ax2.errorbar(timea, edgejump_zsh.iloc[:, 0], edgejump_zsh.iloc[:, 1], c=colors[1], ls="--", lw=1.0, label=r"$\mathrm{Solid \ Zn}$", marker=r"s")
ax2.spines[["top"]].set_visible(False)
ax2.legend(loc="upper right", bbox_to_anchor=(0.7, 1.15), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax2.tick_params(axis="both", which="both", bottom=True, labelbottom=True, right=True, labelright=True)

# 刻度线
ax2.set_ylabel(r"Zn in $\mathrm{Edge \ Jump \ (arb.u.)}$", fontsize=9)
ax2.set_ylim(0.33, 0.48)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.05, offset=0))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.025, offset=0))

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_06_V1_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_06_V1_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_06_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_06_V1_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure EdgeJump

In [ ]:
# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\Echem").glob(r"**\*.txt"))
# 读取电化学数据
echem_all = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem_all.append(df)

# 合并所有电化学数据为一个二维表格
echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

ej_path = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\EdgeJump")
edgejump = pd.read_excel(ej_path.joinpath(r"Cell2_Edge Jump.xlsx"), sheet_name=r"ZSH_MnO2", header=0, comment="#")

In [ ]:
# 电化学时间对齐
echem_all["Time"] = (echem_all["time/s"] - echem_all["time/s"].iloc[0]).dt.total_seconds() / 3600

echem_index_low = echem_all[echem_all["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high = echem_all[echem_all["cycle number"] == 1]["Ewe/V"].idxmax()

In [ ]:
edgejump.iloc[12, 0] = np.nan

In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 5))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.5)

time = echem_all["Time"].iloc[echem_index_low:echem_index_high] - echem_all["Time"].iloc[echem_index_low]
ax.plot(time, echem_all["Ewe/V"].iloc[echem_index_low:echem_index_high], c=colors[0], ls="-", lw=1.0, label=r"Voltage")

# 设置刻度线等格式
# ax.set_xlabel(r"Duration Hours (h)", fontsize=9)
ax.set_xlim(-0.3, 19)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}}\!)$", fontsize=9)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))

ax.tick_params(axis="both", which="both", bottom=False, labelbottom=False, left=True, labelleft=True)
ax.text(0.03, 0.10, r"C/10", ha="left", va="top", rotation=0, transform=ax.transAxes, fontsize=9, c="k")

ax.legend(loc="upper right", bbox_to_anchor=(0.95, 1.05), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.spines[["bottom", "top"]].set_visible(False)
# ax.text(
#     0.01,
#     0.15,
#     r"0.5 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[0],  # type: ignore
# )
# ax.text(
#     0.01,
#     0.23,
#     r"1 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[1],  # type: ignore
# )

# 插入图片
ax2 = ax.twinx()
ax2.set_position((0, 0, 1.0, 1.0))
ax2.set_box_aspect(0.5)
ax2.plot(time, echem_all["<I>/mA"].iloc[echem_index_low:echem_index_high] * 1000, c=colors[1], ls="--", lw=1.0, label=r"Current")

# 设置刻度线等格式
ax2.set_ylabel(r"$\mathrm{Current \ (\mu A)}$", fontsize=9)
ax2.set_ylim(-80, 80)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=40, offset=0))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=20, offset=0))

ax2.tick_params(axis="both", which="both", bottom=False, labelbottom=False, right=True, labelright=True)
ax2.legend(loc="upper right", bbox_to_anchor=(0.7, 1.05), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax2.spines[["bottom", "top"]].set_visible(False)

# 图 B
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.20, 1.0, 1.0))
ax.set_box_aspect(0.5)

timea = np.linspace(0, time.iat[-1], num=edgejump.shape[0])
ax.plot(timea, edgejump.iloc[:, 0], c=colors[0], ls="-", lw=1.0, label=r"Solid Mn", marker=r"o")
ax.spines[["top"]].set_visible(False)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1.15), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

# 设置刻度线等格式
ax.set_xlabel(r"Duration Hours (h)", fontsize=9)
ax.set_xlim(-0.3, 19)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1))
ax.set_ylabel(r"Mn in $\mathrm{solid \ Mn \ (\mu moL)}$", fontsize=9)
ax.set_ylim(0, 10)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))

# 插入图片
ax2 = ax.twinx()
ax2.set_position((0.0, 0.20, 1.0, 1.0))
ax2.set_box_aspect(0.5)
ax2.plot(timea, edgejump.iloc[:, 1], c=colors[1], ls="--", lw=1.0, label=r"Solid Zn", marker=r"s")
ax2.spines[["top"]].set_visible(False)
ax2.legend(loc="upper right", bbox_to_anchor=(0.7, 1.15), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

# 刻度线
ax2.set_ylabel(r"Zn in $\mathrm{solid \ Zn \ (\mu moL)}$", fontsize=9)
ax2.set_ylim(0, 12)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=3, offset=0))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=1.5, offset=0))

# 图 C
subfig = fig.add_subfigure(gs[0:2, 1])
ax = subfig.add_axes((0.35, 0.18, 0.75, 0.75))
ax.set_box_aspect(1.5)

for i, idx in enumerate([[0, 12], [12, 23], [23, 37]]):
    temp = edgejump.iloc[idx[0] : idx[1], :]
    ax.plot(
        temp.iloc[:, 0],
        temp.iloc[:, 1],
        label=r"Edge Jump",
        c=colors[i],
        ls="-",
        marker="*",
        markeredgecolor=colors[i],
        markerfacecolor=colors[i],
        ms=10,
    )
# ax.plot(edgejump.iloc[:, 0], transform_thin(edgejump.iloc[:, 0]), label=None, c=colors[1], ls="--")

ax.set_xlabel(r"$\mathrm{Mn \ in \ solid \ Mn \ (\mu moL)}$", fontsize=9)
ax.set_xlim(10.0, 0.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2.0, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1.0, offset=0))

ax.set_ylabel(r"$\mathrm{Zn \ in \ solid \ Zn \ (\mu moL)}$", fontsize=9)
ax.set_ylim(0, 12)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=3))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1.5))
ax.tick_params(axis="both", which="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)
# ax.legend(loc='upper left', bbox_to_anchor=(0.0, 1), ncols=1, frameon=False, labelcolor='linecolor', fontsize=9) # noqa: E501, ERA001
# ax.text(0.05, 0.80, f'Slope: {abs(transform_thin[1]):.2f}', transform=ax.transAxes, fontsize=9, va='top', ha='left', fontfamily='Arial', )  # noqa: E501, ERA001

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_06_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_06_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_06_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_06_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure 挣扎, 第三个物相

In [ ]:
# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\Echem").glob(r"**\*.txt"))
# 读取电化学数据
echem_all = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem_all.append(df)

# 合并所有电化学数据为一个二维表格
echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 读取 reference + operando 数据
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")

data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
xas_Mn = pd.concat([data1, data2.iloc[:, 5:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, 2, 3, 4, *list(range(14, xas_Mn.shape[1], 1))]
xas_Mn = xas_Mn.iloc[:, charge_index]  # noqa: N816
pdf_Mn = xas_Mn.iloc[:, 0:5]  # noqa: N816
pdf_Mn.columns = [
    r"Energy",
    r"0.2M_MnSO4(aq.)",
    r"alpha_MnO2_Electrode",
    r"alpha_MnO2_Powder",
    r"FC1st",
]
opData_Mn = xas_Mn.iloc[:, list(range(5, xas_Mn.shape[1]))]  # noqa: N816
opData_Mn.columns = list(range(0, opData_Mn.shape[1], 1))

# Zn
data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
xas_Zn = pd.concat([data1, data2.iloc[:, 3:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, 2, *list(range(13, xas_Zn.shape[1], 1))]
xas_Zn = xas_Zn.iloc[:, charge_index]  # noqa: N816

pdf_Zn = xas_Zn.iloc[:, 0:3]  # noqa: N816
pdf_Zn.columns = [r"Energy", r"0.5M_ZnSO4(aq.)", r"ZSH"]
opData_Zn = xas_Zn.iloc[:, list(range(3, xas_Zn.shape[1]))]  # noqa: N816
opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))

In [ ]:
# 电化学时间对齐
echem_all["Time"] = (df["time/s"] - df["time/s"].iloc[0]).dt.total_seconds() / 3600

echem_index_low = echem_all[echem_all["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high = echem_all[echem_all["cycle number"] == 1]["Ewe/V"].idxmax()

# Mn
opData_Mn = opData_Mn[(pdf_Mn[r"Energy"] >= 6530) & (pdf_Mn[r"Energy"] <= 6630)].reset_index(drop=True)  # noqa: N816
pdf_Mn = pdf_Mn[(pdf_Mn[r"Energy"] >= 6530) & (pdf_Mn[r"Energy"] <= 6630)].reset_index(drop=True)  # noqa: N816

x = pdf_Mn.iloc[:, 0]
opData_Mn_1st = opData_Mn.apply(lambda col: np.gradient(col, x))  # noqa: N816

# Zn
opData_Zn = opData_Zn[(pdf_Zn[r"Energy"] >= 9630) & (pdf_Zn[r"Energy"] <= 9740)].reset_index(drop=True)  # noqa: N816
pdf_Zn = pdf_Zn[(pdf_Zn[r"Energy"] >= 9630) & (pdf_Zn[r"Energy"] <= 9740)].reset_index(drop=True)  # noqa: N816

x = pdf_Zn.iloc[:, 0]
opData_Zn_1st = opData_Zn.apply(lambda col: np.gradient(col, x))  # noqa: N816

In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 5))
gs = gridspec.GridSpec(1, 3, width_ratios=[0.3, 1, 1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1.0, 1.0))
ax.set_box_aspect(2.7)
time = echem_all["Time"].iloc[echem_index_low:echem_index_high] - echem_all["Time"].iloc[echem_index_low]
ax.plot(echem_all["Ewe/V"].iloc[echem_index_low:echem_index_high], time, c=colors[0], ls="-", lw=1.0)


# 设置刻度线等格式
ax.set_ylabel(r"Duration Hours (h)", fontsize=9)
ax.set_ylim(-0.3, 19)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1))

ax.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}}\!)$", fontsize=9)
ax.set_xlim(0.7, 1.9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=-0.1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))

ax.tick_params(axis="both", which="both", bottom=True, left=True, labelbottom=True, labelleft=True)
ax.text(0.03, 0.15, r"C/10", ha="left", va="top", rotation=90, transform=ax.transAxes, fontsize=9, c="k")

# ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

# ax.text(
#     0.01,
#     0.15,
#     r"0.5 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[0],  # type: ignore
# )
# ax.text(
#     0.01,
#     0.23,
#     r"1 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[1],  # type: ignore
# )

ax2 = ax.twiny()
ax2.plot(echem_all["<I>/mA"].iloc[echem_index_low:echem_index_high] * 1000, time, c=colors[1], ls="--", lw=1.0)

# 设置刻度线等格式
ax2.set_xlabel(r"$\mathrm{Current \ (\mu A)}$", fontsize=9)
ax2.set_xlim(-80, 80)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=20, offset=0))

ax2.tick_params(axis="both", which="both", top=True, left=True, labeltop=True, labelleft=True)

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_subplot()
ax.set_position((0.15, 0.104, 0.8, 0.8))
ax.set_box_aspect(1.0)

ax.imshow(
    opData_Mn_1st.T,
    aspect="auto",
    cmap="rainbow_PuRd",
    origin="lower",
    extent=[pdf_Mn[r"Energy"].min(), pdf_Mn[r"Energy"].max(), 0, opData_Mn_1st.shape[1]],
    # vmin=0, vmax=0.5,
)
# 设置刻度线等格式
ax.set_xlim(6530, 6610)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_yticks([])
ax.text(
    0.9,
    0.95,
    r"Mn",
    transform=ax.transAxes,
    fontsize=13,
    va="top",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)
# 图 C
subfig = fig.add_subfigure(gs[0, 2])
ax = subfig.add_subplot()
ax.set_position((0.1, 0.104, 0.8, 0.8))
ax.set_box_aspect(1.0)

ax.imshow(
    opData_Zn_1st.T,
    aspect="auto",
    cmap="rainbow_PuRd",
    origin="lower",
    extent=[pdf_Zn[r"Energy"].min(), pdf_Zn[r"Energy"].max(), 0, opData_Zn_1st.shape[1]],
    # vmin=0, vmax=0.5,
)
# 设置刻度线等格式
ax.set_xlim(9650, 9730)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_yticks([])
ax.text(
    0.9,
    0.95,
    r"Zn",
    transform=ax.transAxes,
    fontsize=13,
    va="top",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_05_V0_300.tif"),
    transparent=False,
    pad_inches=0.2,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_05_V0_600.tif"),
    transparent=False,
    pad_inches=0.2,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_05_V0_600.png"),
    pad_inches=0.2,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_05_V0_900.svg"),
    transparent=False,
    pad_inches=0.2,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure MCR-ALS Failed

In [ ]:
# 读取数据
mcr_path = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\FC1st-FD2nd\Oct2022_cell3_P2b\XANES\Mn\MCR")

Mn_MCR_2 = xr.open_dataset(mcr_path.joinpath(r"P2b_Mn_MCR_2_stFixed.NETCDF4"), engine="h5netcdf")
Mn_MCR_3 = xr.open_dataset(mcr_path.joinpath(r"P2b_Mn_MCR_3_stFixed.NETCDF4"), engine="h5netcdf")

In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7, 5.0))
gs = gridspec.GridSpec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

xascolors = [[colors[0], colors[1], colors[3]], [colors[4], colors[1]]]
xas_colors = ListedColormap(
    mpl.colormaps["coolwarm"](np.linspace(1.0, 0.0, Mn_MCR_2["MCR_residual"].shape[0])),
    name="xas_colors",
)
labels = [
    [r"${\alpha}$-MnO$\mathrm{_2}$", r"0.2 M Mn$\mathrm{^{2\!+}}$", r"FC1st"],
    [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"],
]

for i in range(Mn_MCR_2[r"MCR_C"].shape[1]):
    ax.plot(
        Mn_MCR_2[r"spectrum_number"],
        Mn_MCR_2[r"MCR_C"][:, i],
        marker="o",
        markerfacecolor=xascolors[0][i],
        markeredgecolor=xascolors[0][i],
        markersize=4,
        c=xascolors[0][i],
        label=labels[0][i],
        ls="-",
    )

#  刻度线
ax.set_xlim(-0.5, Mn_MCR_2[r"MCR_C"].shape[0])
ax.set_xlabel(r"Spectrum Number", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(0.0, 1.2)
ax.set_ylabel(r"Mn Molar Fraction (arb.u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    left=True,
    labelbottom=True,
    labelleft=True,
)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.0, 1.02),
    ncols=3,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.1,
)

ax.text(
    -0.23,
    1.0,
    r"A",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.35, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

Ref_XANES = Mn_MCR_2["Ref_XANES"].isel(mcr_number=[1, 0, 2])
for i in range(Ref_XANES.shape[0]):
    ax.plot(
        Ref_XANES["pca_energy"],
        Ref_XANES[i],
        c="b",
        ls="--",
        lw=1.0,
        label=None,
        zorder=5,
        alpha=0.5,
    )

for i in range(Mn_MCR_2["MCR_St"].shape[0]):
    ax.plot(Mn_MCR_2["MCR_St"]["pca_energy"], Mn_MCR_2["MCR_St"][i, :], c=xascolors[0][i], label=labels[0][i], ls="-", lw=1.0)

ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_ylim(0, 2.0)
ax.set_ylabel(r"Absorption (arb.u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

ax.legend(loc="upper left", bbox_to_anchor=(0.5, 1.0), ncols=1, handlelength=1, labelcolor="linecolor", fontsize=9)
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    direction="out",
    bottom=True,
    left=True,
    labelbottom=True,
    labelleft=True,
)
ax.text(
    -0.23,
    1.0,
    r"B",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

for i in range(Mn_MCR_2["MCR_residual"].shape[0]):
    ax.plot(Mn_MCR_2["MCR_recon_Data"]["pca_energy"], Mn_MCR_2["MCR_recon_Data"][i, :], c=xas_colors.colors[i], label=None, ls="-", lw=1.0)
    ax.plot(Mn_MCR_2["MCR_residual"]["pca_energy"], Mn_MCR_2["MCR_residual"][i, :] - 0.2, c=xas_colors.colors[i], label=None, ls="-", lw=1.0)

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((0.5, 0.15, 0.42, 0.05)),
    location="bottom",
    orientation="horizontal",
    cmap=xas_colors,
    ticklocation="top",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])

ax.text(
    0.49,
    0.28,
    r"Charge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.84,
    0.36,
    r"Discharge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="right",
    fontfamily="Arial",
)
ax.text(
    0.92,
    0.28,
    r"Charge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="right",
    fontfamily="Arial",
)
colorbar.outline.set_visible(False)  # type: ignore

ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_ylim(-0.4, 1.6)
ax.set_ylabel(r"Absorption (arb.u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    direction="out",
    bottom=True,
    left=True,
    labelbottom=True,
    labelleft=True,
)

ax.text(
    -0.23,
    1.0,
    r"C",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 D
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

xascolors = [[colors[0], colors[1], colors[3]], [colors[4], colors[1]]]
xas_colors = ListedColormap(
    mpl.colormaps["coolwarm"](np.linspace(1.0, 0.0, Mn_MCR_2["MCR_residual"].shape[0])),
    name="xas_colors",
)
labels = [
    [r"${\alpha}$-MnO$\mathrm{_2}$", r"0.2 M Mn$\mathrm{^{2\!+}}$", r"FC1st"],
    [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"],
]

for i in range(Mn_MCR_3[r"MCR_C"].shape[1]):
    ax.plot(
        Mn_MCR_3[r"spectrum_number"],
        Mn_MCR_3[r"MCR_C"][:, i],
        marker="o",
        markerfacecolor=xascolors[0][i],
        markeredgecolor=xascolors[0][i],
        markersize=5,
        c=xascolors[0][i],
        label=labels[0][i],
        ls="-",
    )

#  刻度线
ax.set_xlim(-0.5, Mn_MCR_3[r"MCR_C"].shape[0])
ax.set_xlabel(r"Spectrum Number", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(0.0, 0.8)
ax.set_ylabel(r"Mn Molar Fraction (arb.u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    left=True,
    labelbottom=True,
    labelleft=True,
)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.0, 1.02),
    ncols=3,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.1,
)

ax.text(
    -0.23,
    1.0,
    r"D",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 E
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.35, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)
Ref_XANES = Mn_MCR_3["Ref_XANES"].isel(mcr_number=[1, 0, 2])
for i in range(Ref_XANES.shape[0]):
    ax.plot(
        Ref_XANES["pca_energy"],
        Ref_XANES[i],
        c="b",
        ls="--",
        lw=1.0,
        label=None,
        zorder=5,
        alpha=0.5,
    )
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    direction="out",
    bottom=True,
    left=True,
    labelbottom=True,
    labelleft=True,
)
for i in range(Mn_MCR_3["MCR_St"].shape[0]):
    ax.plot(Mn_MCR_3["MCR_St"]["pca_energy"], Mn_MCR_3["MCR_St"][i, :], c=xascolors[0][i], label=labels[0][i], ls="-", lw=1.0)

ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_ylim(0, 2.0)
ax.set_ylabel(r"Absorption (arb.u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

ax.legend(loc="upper left", bbox_to_anchor=(0.5, 1.0), ncols=1, handlelength=1, labelcolor="linecolor", fontsize=9)

ax.text(
    -0.23,
    1.0,
    r"E",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 F
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

for i in range(Mn_MCR_3["MCR_residual"].shape[0]):
    ax.plot(Mn_MCR_3["MCR_recon_Data"]["pca_energy"], Mn_MCR_3["MCR_recon_Data"][i, :], c=xas_colors.colors[i], label=None, ls="-", lw=1.0)
    ax.plot(Mn_MCR_3["MCR_residual"]["pca_energy"], Mn_MCR_3["MCR_residual"][i, :] - 0.2, c=xas_colors.colors[i], label=None, ls="-", lw=1.0)

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((0.5, 0.15, 0.42, 0.05)),
    location="bottom",
    orientation="horizontal",
    cmap=xas_colors,
    ticklocation="top",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])

ax.text(
    0.49,
    0.28,
    r"Charge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.84,
    0.36,
    r"Discharge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="right",
    fontfamily="Arial",
)
ax.text(
    0.92,
    0.28,
    r"Charge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="right",
    fontfamily="Arial",
)
colorbar.outline.set_visible(False)  # type: ignore

ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_ylim(-0.4, 1.6)
ax.set_ylabel(r"Absorption (arb.u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    direction="out",
    bottom=True,
    left=True,
    labelbottom=True,
    labelleft=True,
)

ax.text(
    -0.23,
    1.0,
    r"F",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_04_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_04_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_04_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_04_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure XANES MCR-ALS: Third Phase

In [ ]:
path_ref = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")
path_dino = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\Dino")

ref = pd.read_csv(
    path_ref.joinpath(r"Overview", r"cell2_opXAS_P1_Mn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
).iloc[:, [0, 2, 4]]

ref = ref[(ref.iloc[:, 0] >= 6530.0) & (ref.iloc[:, 0] <= 6630.0)].copy().reset_index(drop=True)


dino = pd.read_csv(Path.joinpath(path_dino, r"FC1st_3rd_phase.txt"), sep=r"\s+", header=None, index_col=None)
dino = dino[(dino.iloc[:, 0] >= 6530.0) & (dino.iloc[:, 0] <= 6630.0)].copy()

In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(3.3, 2.5))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(ref.iloc[:, 0], ref.iloc[:, 1], c=colors[0], ls="-", lw=1.0, label=r"$\mathrm{\alpha-MnO_2}$")
ax.plot(ref.iloc[:, 0], ref.iloc[:, 2], c=colors[1], ls="-", lw=1.0, label=r"FC#C")
ax.plot(dino.iloc[:, 0], dino.iloc[:, 3], c=colors[3], ls="--", lw=1.0, label=r"$\mathrm{3^{rd} \ Phase}$")

# 设置刻度线等格式
ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_ylim(0, 1.6)
ax.set_ylabel(r"Absorption (arb.u.)", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

ax.tick_params(axis="both", which="both", bottom=True, left=True, labelbottom=True, labelleft=True)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.55, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=10,
    handlelength=3,
    handletextpad=0.2,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_03_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_03_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_03_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_03_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure XANES and PCA on Cathode P2b - V3

In [ ]:
# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\Echem").glob(r"**\*.txt"))
# 读取电化学数据
echem_all = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem_all.append(df)

# 合并所有电化学数据为一个二维表格
echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 读取 reference + operando 数据
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")

data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
xas_Mn = pd.concat([data1, data2.iloc[:, 4:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, 2, 3, *list(range(13, xas_Mn.shape[1], 1))]
xas_Mn = xas_Mn.iloc[:, charge_index]  # noqa: N816
pdf_Mn = xas_Mn.iloc[:, 0:4]  # noqa: N816
pdf_Mn.columns = [
    # r"Energy",
    r"0.2M_MnSO4(aq.)",
    r"alpha_MnO2_Electrode",
    r"alpha_MnO2_Powder",
    r"FC1st",
]
opData_Mn = xas_Mn.iloc[:, list(range(4, xas_Mn.shape[1]))]  # noqa: N816
opData_Mn.columns = list(range(0, opData_Mn.shape[1], 1))

# Zn
data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
xas_Zn = pd.concat([data1, data2.iloc[:, 2:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, *list(range(12, xas_Zn.shape[1], 1))]
xas_Zn = xas_Zn.iloc[:, charge_index]  # noqa: N816

pdf_Zn = xas_Zn.iloc[:, 0:2]  # noqa: N816
pdf_Zn.columns = [
    # r"Energy",
    r"0.5M_ZnSO4(aq.)",
    r"ZSH",
]
opData_Zn = xas_Zn.iloc[:, list(range(2, xas_Zn.shape[1]))]  # noqa: N816
opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))

# 读取 PCA 数据
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\FC1st-FD2nd\Oct2022_cell3_P2b\XANES")
# Mn
pca_Mn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"cell2_1stFC_opXAS_P2b_Mn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Mn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)
# Zn
pca_Zn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"cell2_1stFC_opXAS_P2b_Zn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Zn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)

In [ ]:
# 电化学时间对齐
echem_all["Time"] = (df["time/s"] - df["time/s"].iloc[0]).dt.total_seconds() / 3600

In [ ]:
echem_index_low = echem_all[echem_all["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high = echem_all[echem_all["cycle number"] == 1]["Ewe/V"].idxmax()

In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 5))
gs = gridspec.GridSpec(2, 4, width_ratios=[0.5, 1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1.0, 1.0))
ax.set_box_aspect(2.0)
time = echem_all["Time"].iloc[echem_index_low:echem_index_high] - echem_all["Time"].iloc[echem_index_low]
ax.plot(echem_all["Ewe/V"].iloc[echem_index_low:echem_index_high], time, c=colors[0], ls="-", lw=1.0)


# 设置刻度线等格式
ax.set_ylabel(r"Duration Hours (h)", fontsize=9)
ax.set_ylim(-0.3, 19)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1))

ax.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}}\!)$", fontsize=9)
ax.set_xlim(0.7, 1.9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=-0.1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))

ax.tick_params(axis="both", which="both", bottom=True, left=True, labelbottom=True, labelleft=True)
ax.text(0.03, 0.15, r"C/10", ha="left", va="top", rotation=90, transform=ax.transAxes, fontsize=9, c="k")

# ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

# ax.text(
#     0.01,
#     0.15,
#     r"0.5 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[0],  # type: ignore
# )
# ax.text(
#     0.01,
#     0.23,
#     r"1 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[1],  # type: ignore
# )

ax2 = ax.twiny()
ax2.plot(echem_all["<I>/mA"].iloc[echem_index_low:echem_index_high] * 1000, time, c=colors[1], ls="--", lw=1.0)

# 设置刻度线等格式
ax2.set_xlabel(r"$\mathrm{Current \ (\mu A)}$", fontsize=9)
ax2.set_xlim(-80, 80)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=20, offset=0))

ax2.tick_params(axis="both", which="both", top=True, left=True, labeltop=True, labelleft=True)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.15, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.0)

ax.imshow(
    opData_Mn.T,
    aspect="auto",
    extent=[6530, 6630, 0, opData_Mn.shape[1]],
    cmap="coolwarm",
    origin="lower",
    vmin=0,
    vmax=1.6,
)

ax.set_xlim(6535, 6575)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=5))

ax.set_yticks([])

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.08, 0.1, 0.10, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.ax.text(
    1.06,
    0.98,
    r"High",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
colorbar.ax.text(
    1.08,
    0.08,
    r"Low",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
# colorbar.outline.set_visible(False)  # type: ignore

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, 0.0, 1.2, 1.0))
ax.set_box_aspect(0.8)

# xascolors = [[colors[0], colors[4]], [colors[2], colors[1]]]
xas_colors = ListedColormap(
    mpl.colormaps["sunset"](np.linspace(1.0, 0.0, opData_Mn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]
line = []
for i in range(pca_Mn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Mn.iloc[:, 0], pca_Mn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-10, 2)
ax.set_ylabel(r"PCA Intensity", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1]],
    labels=[r"Component#1", r"Component#2"],
    loc="upper left",
    bbox_to_anchor=(0.4, 0.3),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)

# 图 E
subfig = fig.add_subfigure(gs[0, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((1.3, 0.0, 1.2, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Mn2.shape[0], 1), np.log10(pca_Mn2.iloc[:, 1]), s=50, c=colors[0], edgecolors="face")

ax.set_xlim(-0.5, pca_Mn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-8.0, 0.5)
ax.set_ylabel(r"log of Variance", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

# 图 F
subfig = fig.add_subfigure(gs[1, 0])
ax = subfig.add_axes((0, -0.2, 1.0, 1.0))
ax.set_box_aspect(2.0)
time = echem_all["Time"].iloc[echem_index_low:echem_index_high] - echem_all["Time"].iloc[echem_index_low]
ax.plot(echem_all["Ewe/V"].iloc[echem_index_low:echem_index_high], time, c=colors[0], ls="-", lw=1.0)


# 设置刻度线等格式
ax.set_ylabel(r"Duration Hours (h)", fontsize=9)
ax.set_ylim(-0.3, 19)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1))

ax.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}}\!)$", fontsize=9)
ax.set_xlim(0.7, 1.9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=-0.1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))

ax.tick_params(axis="both", which="both", bottom=True, left=True, labelbottom=True, labelleft=True)
ax.text(0.03, 0.15, r"C/10", ha="left", va="top", rotation=90, transform=ax.transAxes, fontsize=9, c="k")

# ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

# ax.text(
#     0.01,
#     0.15,
#     r"0.5 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[0],  # type: ignore
# )
# ax.text(
#     0.01,
#     0.23,
#     r"1 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[1],  # type: ignore
# )

ax2 = ax.twiny()
ax2.plot(echem_all["<I>/mA"].iloc[echem_index_low:echem_index_high] * 1000, time, c=colors[1], ls="--", lw=1.0)

# 设置刻度线等格式
ax2.set_xlabel(r"$\mathrm{Current \ (\mu A)}$", fontsize=9)
ax2.set_xlim(-80, 80)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=20, offset=0))

ax2.tick_params(axis="both", which="both", top=True, left=True, labeltop=True, labelleft=True)

# 图 G
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.15, -0.2, 1.0, 1.0))
ax.set_box_aspect(1.0)

ax.imshow(
    opData_Zn.T,
    aspect="auto",
    extent=[9650, 9750, 0, opData_Zn.shape[1]],
    cmap="coolwarm",
    origin="lower",
    vmin=0,
    vmax=2.4,
)

ax.set_xlim(9655, 9685)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=5))

ax.set_yticks([])

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.08, 0.1, 0.10, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.ax.text(
    1.06,
    0.98,
    r"High",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
colorbar.ax.text(
    1.08,
    0.08,
    r"Low",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
# colorbar.outline.set_visible(False)  # type: ignore

# 图 H
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, -0.2, 1.2, 1.0))
ax.set_box_aspect(0.8)
xas_colors = ListedColormap(
    mpl.colormaps["sunset"](np.linspace(1.0, 0.0, opData_Mn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]
line = []
for i in range(pca_Zn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Zn.iloc[:, 0], pca_Zn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(9650, 9750)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-15.0, 1)
ax.set_ylabel(r"PCA Intensity", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1]],
    labels=[r"Component#1", r"Component#2"],
    loc="upper left",
    bbox_to_anchor=(0.4, 0.3),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)

# 图 I
subfig = fig.add_subfigure(gs[1, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((1.3, -0.2, 1.2, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Zn2.shape[0], 1), np.log10(pca_Zn2.iloc[:, 1]), s=50, c=colors[1], edgecolors="face")

ax.set_xlim(-0.5, pca_Zn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-8, 0.5)
ax.set_ylabel(r"log of Variance", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_01_V3_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_01_V3_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_01_V3_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_01_V3_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure XANES and PCA on Cathode P2b - V2

In [ ]:
# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\Echem").glob(r"**\*.txt"))
# 读取电化学数据
echem_all = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem_all.append(df)

# 合并所有电化学数据为一个二维表格
echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 读取 reference + operando 数据
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")

data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
xas_Mn = pd.concat([data1, data2.iloc[:, 4:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, 2, 3, *list(range(13, xas_Mn.shape[1], 1))]
xas_Mn = xas_Mn.iloc[:, charge_index]  # noqa: N816
pdf_Mn = xas_Mn.iloc[:, 0:4]  # noqa: N816
pdf_Mn.columns = [
    # r"Energy",
    r"0.2M_MnSO4(aq.)",
    r"alpha_MnO2_Electrode",
    r"alpha_MnO2_Powder",
    r"FC1st",
]
opData_Mn = xas_Mn.iloc[:, list(range(4, xas_Mn.shape[1]))]  # noqa: N816
opData_Mn.columns = list(range(0, opData_Mn.shape[1], 1))

# Zn
data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
xas_Zn = pd.concat([data1, data2.iloc[:, 2:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, *list(range(12, xas_Zn.shape[1], 1))]
xas_Zn = xas_Zn.iloc[:, charge_index]  # noqa: N816

pdf_Zn = xas_Zn.iloc[:, 0:2]  # noqa: N816
pdf_Zn.columns = [
    # r"Energy",
    r"0.5M_ZnSO4(aq.)",
    r"ZSH",
]
opData_Zn = xas_Zn.iloc[:, list(range(2, xas_Zn.shape[1]))]  # noqa: N816
opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))

# 读取 PCA 数据
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\FC1st-FD2nd\Oct2022_cell3_P2b\XANES")
# Mn
pca_Mn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"cell2_1stFC_opXAS_P2b_Mn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Mn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)
# Zn
pca_Zn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"cell2_1stFC_opXAS_P2b_Zn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Zn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)

In [ ]:
# 电化学时间对齐
echem_all["Time"] = (df["time/s"] - df["time/s"].iloc[0]).dt.total_seconds() / 3600

echem_index_low = echem_all[echem_all["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high = echem_all[echem_all["cycle number"] == 1]["Ewe/V"].idxmax()

In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 5))
gs = gridspec.GridSpec(2, 4, width_ratios=[0.5, 1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0:2])
ax = subfig.add_subplot(projection="3d")
ax.set_position((0.0, 0.0, 1.2, 1.2))
ax.set_box_aspect([2, 1, 1])

Energy, Spectrum_number = np.meshgrid(opData_Mn.index.values, np.arange(opData_Mn.shape[1]))

ax.plot_surface(
    Energy,
    Spectrum_number,
    opData_Mn.T,
    rstride=1,
    cstride=2,
    antialiased=True,
    linewidth=0.2,
    edgecolor="k",
    alpha=1.0,
    cmap="jet",
    vmin=0,
    vmax=1.6,
    shade=False,
    axlim_clip=True,
)

ax.view_init(elev=25, azim=250)

# 设置白色底面
for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    axis.pane.set_facecolor("white")
    axis.pane.set_edgecolor("white")
    axis._axinfo["grid"]["linewidth"] = 0.3
    axis._axinfo["grid"]["color"] = (0.85, 0.85, 0.85, 1)  # 淡灰

for label in ax.get_xticklabels():
    label.set_rotation(10)  # X轴刻度标签旋转 10°
    label.set_horizontalalignment("left")
    label.set_verticalalignment("top")
for label in ax.get_yticklabels():
    label.set_rotation(0)  # Y轴刻度标签旋转 0°
    label.set_horizontalalignment("left")
    label.set_verticalalignment("center")
for label in ax.get_zticklabels():
    label.set_rotation(0)  # Z轴保持竖直
    label.set_horizontalalignment("center")
    label.set_verticalalignment("center")

ax.tick_params(axis="x", direction="out", labelsize=9, pad=-5, colors="black")
ax.tick_params(axis="y", direction="out", labelsize=9, pad=0, colors="black")
ax.tick_params(axis="z", direction="out", labelsize=9, pad=0, colors="black")

ax.set_xlim(6530, 6610)
ax.set_xlabel(r"Energy (eV)", fontsize=8, labelpad=-2)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))

ax.set_ylim(0, opData_Mn.shape[1])
ax.set_ylabel(r"Spectrum", fontsize=8, labelpad=-7)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))

ax.set_zlim(-0.1, 1.6)
ax.zaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.zaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))
ax.set_zlabel("Absorption (arb.u.)", fontsize=8, labelpad=-5)

ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
    grid_linewidth=0.1,
)

# 图 B
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.55, 0.1, 1.0, 1.0))
ax.set_box_aspect(0.8)

xas_colors = ListedColormap(
    mpl.colormaps["sunset"](np.linspace(1.0, 0.0, opData_Mn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]
line = []
for i in range(pca_Mn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Mn.iloc[:, 0], pca_Mn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(6530, 6610)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-10, 2)
ax.set_ylabel(r"PCA Intensity", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=2))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1]],
    labels=[r"Component#1", r"Component#2"],
    loc="upper left",
    bbox_to_anchor=(0.42, 0.3),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)

# 图 C
subfig = fig.add_subfigure(gs[0, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.9, 0.1, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Mn2.shape[0], 1), np.log10(pca_Mn2.iloc[:, 1]), s=50, c=colors[0], edgecolors="face")

ax.set_xlim(-0.5, pca_Mn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-8.0, 0.5)
ax.set_ylabel(r"log of Variance", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

# 图 D
subfig = fig.add_subfigure(gs[1, 0:2])
ax = subfig.add_subplot(projection="3d")
ax.set_position((0.0, -0.1, 1.2, 1.2))
ax.set_box_aspect([2, 1, 1])

Energy, Spectrum_number = np.meshgrid(opData_Zn.index.values, np.arange(opData_Zn.shape[1]))

ax.plot_surface(
    Energy,
    Spectrum_number,
    opData_Zn.T,
    rstride=1,
    cstride=2,
    antialiased=True,
    linewidth=0.2,
    edgecolor="k",
    alpha=1.0,
    cmap="jet",
    vmin=0,
    vmax=2.0,
    shade=False,
    axlim_clip=True,
)

ax.view_init(elev=27, azim=250)

# 设置白色底面
for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    axis.pane.set_facecolor("white")
    axis.pane.set_edgecolor("white")
    axis._axinfo["grid"]["linewidth"] = 0.3
    axis._axinfo["grid"]["color"] = (0.85, 0.85, 0.85, 1)  # 淡灰

for label in ax.get_xticklabels():
    label.set_rotation(10)  # X轴刻度标签旋转 10°
    label.set_horizontalalignment("left")
    label.set_verticalalignment("top")
for label in ax.get_yticklabels():
    label.set_rotation(0)  # Y轴刻度标签旋转 0°
    label.set_horizontalalignment("left")
    label.set_verticalalignment("center")
for label in ax.get_zticklabels():
    label.set_rotation(0)  # Z轴保持竖直
    label.set_horizontalalignment("center")
    label.set_verticalalignment("center")

ax.tick_params(axis="x", direction="out", labelsize=9, pad=-5, colors="black")
ax.tick_params(axis="y", direction="out", labelsize=9, pad=0, colors="black")
ax.tick_params(axis="z", direction="out", labelsize=9, pad=0, colors="black")

ax.set_xlim(9650, 9730)
ax.set_xlabel(r"Energy (eV)", fontsize=8, labelpad=-2)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))

ax.set_ylim(0, opData_Zn.shape[1])
ax.set_ylabel(r"Spectrum", fontsize=8, labelpad=-7)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))

ax.set_zlim(-0.1, 2.4)
ax.zaxis.set_major_locator(ticker.MultipleLocator(base=0.6))
ax.zaxis.set_minor_locator(ticker.MultipleLocator(base=0.3))
ax.set_zlabel("Absorption (arb.u.)", fontsize=8, labelpad=-5)

ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
    grid_linewidth=0.1,
)

# 图 E
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.55, 0.1, 1.0, 1.0))
ax.set_box_aspect(0.8)

xas_colors = ListedColormap(
    mpl.colormaps["sunset"](np.linspace(1.0, 0.0, opData_Mn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]
line = []
for i in range(pca_Zn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Zn.iloc[:, 0], pca_Zn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(9650, 9730)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-15, 1)
ax.set_ylabel(r"PCA Intensity", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1]],
    labels=[r"Component#1", r"Component#2"],
    loc="upper left",
    bbox_to_anchor=(0.35, 0.3),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)

# 图 E
subfig = fig.add_subfigure(gs[1, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.9, 0.1, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Zn2.shape[0], 1), np.log10(pca_Zn2.iloc[:, 1]), s=50, c=colors[1], edgecolors="face")

ax.set_xlim(-0.5, pca_Zn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-8.0, 0.5)
ax.set_ylabel(r"log of Variance", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_01_V2_300.tif"),
    transparent=False,
    pad_inches=0.2,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_01_V2_600.tif"),
    transparent=False,
    pad_inches=0.2,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_01_V2_600.png"),
    pad_inches=0.2,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_01_V2_900.svg"),
    transparent=False,
    pad_inches=0.2,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure XANES and PCA on Cathode P2b - V1

In [ ]:
# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\Echem").glob(r"**\*.txt"))
# 读取电化学数据
echem_all = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem_all.append(df)

# 合并所有电化学数据为一个二维表格
echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 读取 reference + operando 数据
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")

data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
xas_Mn = pd.concat([data1, data2.iloc[:, 4:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, 2, 3, *list(range(13, xas_Mn.shape[1], 1))]
xas_Mn = xas_Mn.iloc[:, charge_index]  # noqa: N816
pdf_Mn = xas_Mn.iloc[:, 0:4]  # noqa: N816
pdf_Mn.columns = [
    # r"Energy",
    r"0.2M_MnSO4(aq.)",
    r"alpha_MnO2_Electrode",
    r"alpha_MnO2_Powder",
    r"FC1st",
]
opData_Mn = xas_Mn.iloc[:, list(range(4, xas_Mn.shape[1]))]  # noqa: N816
opData_Mn.columns = list(range(0, opData_Mn.shape[1], 1))

# Zn
data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
xas_Zn = pd.concat([data1, data2.iloc[:, 2:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, *list(range(12, xas_Zn.shape[1], 1))]
xas_Zn = xas_Zn.iloc[:, charge_index]  # noqa: N816

pdf_Zn = xas_Zn.iloc[:, 0:2]  # noqa: N816
pdf_Zn.columns = [
    # r"Energy",
    r"0.5M_ZnSO4(aq.)",
    r"ZSH",
]
opData_Zn = xas_Zn.iloc[:, list(range(2, xas_Zn.shape[1]))]  # noqa: N816
opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))

# 读取 PCA 数据
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\FC1st-FD2nd\Oct2022_cell3_P2b\XANES")
# Mn
pca_Mn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"cell2_1stFC_opXAS_P2b_Mn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Mn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)
# Zn
pca_Zn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"cell2_1stFC_opXAS_P2b_Zn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Zn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)

In [ ]:
# 电化学时间对齐
echem_all["Time"] = (df["time/s"] - df["time/s"].iloc[0]).dt.total_seconds() / 3600

In [ ]:
echem_index_low = echem_all[echem_all["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high = echem_all[echem_all["cycle number"] == 1]["Ewe/V"].idxmax()

In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 5))
gs = gridspec.GridSpec(2, 4, width_ratios=[0.5, 1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1.0, 1.0))
ax.set_box_aspect(2.0)
time = echem_all["Time"].iloc[echem_index_low:echem_index_high] - echem_all["Time"].iloc[echem_index_low]
ax.plot(echem_all["Ewe/V"].iloc[echem_index_low:echem_index_high], time, c=colors[0], ls="-", lw=1.0)


# 设置刻度线等格式
ax.set_ylabel(r"Duration Hours (h)", fontsize=9)
ax.set_ylim(-0.3, 19)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1))

ax.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}}\!)$", fontsize=9)
ax.set_xlim(0.7, 1.9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=-0.1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))

ax.tick_params(axis="both", which="both", bottom=True, left=True, labelbottom=True, labelleft=True)
ax.text(0.03, 0.15, r"C/10", ha="left", va="top", rotation=90, transform=ax.transAxes, fontsize=9, c="k")

# ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

# ax.text(
#     0.01,
#     0.15,
#     r"0.5 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[0],  # type: ignore
# )
# ax.text(
#     0.01,
#     0.23,
#     r"1 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[1],  # type: ignore
# )

ax2 = ax.twiny()
ax2.plot(echem_all["<I>/mA"].iloc[echem_index_low:echem_index_high] * 1000, time, c=colors[1], ls="--", lw=1.0)

# 设置刻度线等格式
ax2.set_xlabel(r"$\mathrm{Current \ (\mu A)}$", fontsize=9)
ax2.set_xlim(-80, 80)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=20, offset=0))

ax2.tick_params(axis="both", which="both", top=True, left=True, labeltop=True, labelleft=True)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.15, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.0)

ax.imshow(
    opData_Mn.T,
    aspect="auto",
    extent=[6530, 6630, 0, opData_Mn.shape[1]],
    cmap="coolwarm",
    origin="lower",
    vmin=0,
    vmax=1.6,
)

ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))

ax.set_yticks([])

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.08, 0.1, 0.10, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.ax.text(
    1.06,
    0.98,
    r"High",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
colorbar.ax.text(
    1.08,
    0.08,
    r"Low",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
# colorbar.outline.set_visible(False)  # type: ignore

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, 0.0, 1.2, 1.0))
ax.set_box_aspect(0.8)

# xascolors = [[colors[0], colors[4]], [colors[2], colors[1]]]
xas_colors = ListedColormap(
    mpl.colormaps["sunset"](np.linspace(1.0, 0.0, opData_Mn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]
line = []
for i in range(pca_Mn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Mn.iloc[:, 0], pca_Mn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-10, 2)
ax.set_ylabel(r"PCA Intensity", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1]],
    labels=[r"Component#1", r"Component#2"],
    loc="upper left",
    bbox_to_anchor=(0.4, 0.3),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)

# 图 E
subfig = fig.add_subfigure(gs[0, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((1.3, 0.0, 1.2, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Mn2.shape[0], 1), np.log10(pca_Mn2.iloc[:, 1]), s=50, c=colors[0], edgecolors="face")

ax.set_xlim(-0.5, pca_Mn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-8.0, 0.5)
ax.set_ylabel(r"log of Variance", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

# 图 F
subfig = fig.add_subfigure(gs[1, 0])
ax = subfig.add_axes((0, -0.2, 1.0, 1.0))
ax.set_box_aspect(2.0)
time = echem_all["Time"].iloc[echem_index_low:echem_index_high] - echem_all["Time"].iloc[echem_index_low]
ax.plot(echem_all["Ewe/V"].iloc[echem_index_low:echem_index_high], time, c=colors[0], ls="-", lw=1.0)


# 设置刻度线等格式
ax.set_ylabel(r"Duration Hours (h)", fontsize=9)
ax.set_ylim(-0.3, 19)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1))

ax.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}}\!)$", fontsize=9)
ax.set_xlim(0.7, 1.9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=-0.1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))

ax.tick_params(axis="both", which="both", bottom=True, left=True, labelbottom=True, labelleft=True)
ax.text(0.03, 0.15, r"C/10", ha="left", va="top", rotation=90, transform=ax.transAxes, fontsize=9, c="k")

# ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

# ax.text(
#     0.01,
#     0.15,
#     r"0.5 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[0],  # type: ignore
# )
# ax.text(
#     0.01,
#     0.23,
#     r"1 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[1],  # type: ignore
# )

ax2 = ax.twiny()
ax2.plot(echem_all["<I>/mA"].iloc[echem_index_low:echem_index_high] * 1000, time, c=colors[1], ls="--", lw=1.0)

# 设置刻度线等格式
ax2.set_xlabel(r"$\mathrm{Current \ (\mu A)}$", fontsize=9)
ax2.set_xlim(-80, 80)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=20, offset=0))

ax2.tick_params(axis="both", which="both", top=True, left=True, labeltop=True, labelleft=True)

# 图 G
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.15, -0.2, 1.0, 1.0))
ax.set_box_aspect(1.0)

ax.imshow(
    opData_Zn.T,
    aspect="auto",
    extent=[9650, 9750, 0, opData_Zn.shape[1]],
    cmap="coolwarm",
    origin="lower",
    vmin=0,
    vmax=2.4,
)

ax.set_xlim(9650, 9750)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))

ax.set_yticks([])

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.08, 0.1, 0.10, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.ax.text(
    1.06,
    0.98,
    r"High",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
colorbar.ax.text(
    1.08,
    0.08,
    r"Low",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
# colorbar.outline.set_visible(False)  # type: ignore

# 图 H
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, -0.2, 1.2, 1.0))
ax.set_box_aspect(0.8)
xas_colors = ListedColormap(
    mpl.colormaps["sunset"](np.linspace(1.0, 0.0, opData_Mn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]
line = []
for i in range(pca_Zn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Zn.iloc[:, 0], pca_Zn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(9650, 9750)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-15.0, 1)
ax.set_ylabel(r"PCA Intensity", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1]],
    labels=[r"Component#1", r"Component#2"],
    loc="upper left",
    bbox_to_anchor=(0.4, 0.3),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)

# 图 I
subfig = fig.add_subfigure(gs[1, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((1.3, -0.2, 1.2, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Zn2.shape[0], 1), np.log10(pca_Zn2.iloc[:, 1]), s=50, c=colors[1], edgecolors="face")

ax.set_xlim(-0.5, pca_Zn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-8, 0.5)
ax.set_ylabel(r"log of Variance", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_01_V1_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_01_V1_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_01_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_01_V1_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure XANES and PCA on Cathode P2b

In [ ]:
# 读取 reference + operando 数据
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")

data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
xas_Mn = pd.concat([data1, data2.iloc[:, 5:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, 2, 3, 4, *list(range(14, xas_Mn.shape[1], 1))]
xas_Mn = xas_Mn.iloc[:, charge_index]  # noqa: N816
pdf_Mn = xas_Mn.iloc[:, 0:5]  # noqa: N816
pdf_Mn.columns = [
    r"Energy",
    r"0.2M_MnSO4(aq.)",
    r"alpha_MnO2_Electrode",
    r"alpha_MnO2_Powder",
    r"FC1st",
]
opData_Mn = xas_Mn.iloc[:, [0, *list(range(5, xas_Mn.shape[1]))]]  # noqa: N816
opData_Mn.columns = list(range(0, opData_Mn.shape[1], 1))

# Zn
data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
xas_Zn = pd.concat([data1, data2.iloc[:, 3:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, 2, *list(range(12, xas_Zn.shape[1], 1))]
xas_Zn = xas_Zn.iloc[:, charge_index]  # noqa: N816

pdf_Zn = xas_Zn.iloc[:, 0:3]  # noqa: N816
pdf_Zn.columns = [r"Energy", r"0.5M_ZnSO4(aq.)", r"ZHS"]
opData_Zn = xas_Zn.iloc[:, [0, *list(range(3, xas_Zn.shape[1]))]]  # noqa: N816
opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))

# 读取 PCA 数据
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\FC1st-FD2nd\Oct2022_cell3_P2b\XANES")
# Mn
pca_Mn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"cell2_1stFC_opXAS_P2b_Mn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Mn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)
# Zn
pca_Zn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"cell2_1stFC_opXAS_P2b_Zn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Zn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)

In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 5))
gs = gridspec.GridSpec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

xascolors = [[colors[0], colors[4]], [colors[2], colors[1]]]
xas_colors = ListedColormap(
    mpl.colormaps["sunset"](np.linspace(1.0, 0.0, opData_Mn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

for i in range(pdf_Mn.shape[1] - 3):
    ax.plot(pdf_Mn.iloc[:, 0], pdf_Mn.iloc[:, i + 1], c=xascolors[0][i], ls="--", lw=1.0, label=labels[0][i], zorder=5)
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
for i in range(opData_Mn.shape[1] - 1):
    ax.plot(opData_Mn.iloc[:, 0], opData_Mn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0)  # type: ignore

ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_ylim(0, 2.4)
ax.set_ylabel(r"Absorption (arb.u.)", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((0.5, 0.05, 0.42, 0.05)),
    location="bottom",
    orientation="horizontal",
    cmap=xas_colors,
    ticklocation="top",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.42, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=10,
    handlelength=3,
    handletextpad=0.2,
)
ax.text(
    0.49,
    0.18,
    r"Charge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.84,
    0.26,
    r"Discharge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="right",
    fontfamily="Arial",
)
ax.text(
    0.92,
    0.18,
    r"Charge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="right",
    fontfamily="Arial",
)
colorbar.outline.set_visible(False)  # type: ignore

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.35, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

line = []
for i in range(pca_Mn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Mn.iloc[:, 0], pca_Mn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-10, 2.0)
ax.set_ylabel(r"PCA Intensity", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=3, offset=-1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1.5, offset=-1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1], line[2]],
    labels=[r"Component#1", r"Component#2", r"Component#3"],
    loc="upper left",
    bbox_to_anchor=(0.4, 0.33),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Mn2.shape[0], 1), np.log10(pca_Mn2.iloc[:, 1]), s=50, c=colors[0], edgecolors="face")

ax.set_xlim(-0.5, pca_Mn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-8.0, 0.5)
ax.set_ylabel(r"log of Variance", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

# 图 D
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

xascolors = [[colors[0], colors[4]], [colors[2], colors[1]]]
xas_colors = ListedColormap(
    mpl.colormaps["coolwarm"](np.linspace(0.0, 1.0, opData_Zn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]

for i in range(pdf_Zn.shape[1] - 1):
    ax.plot(pdf_Zn.iloc[:, 0], pdf_Zn.iloc[:, i + 1], c=xascolors[1][i], ls="--", lw=1.0, label=labels[1][i], zorder=5)
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
for i in range(opData_Zn.shape[1] - 1):
    ax.plot(opData_Zn.iloc[:, 0], opData_Zn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0)  # type: ignore

ax.set_xlim(9650, 9750)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_ylim(0, 2.4)
ax.set_ylabel(r"Absorption (arb.u.)", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((0.5, 0.05, 0.42, 0.05)),
    location="bottom",
    orientation="horizontal",
    cmap=xas_colors,
    ticklocation="top",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.42, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=10,
    handlelength=3,
    handletextpad=0.2,
)
ax.text(
    0.49,
    0.18,
    r"Charge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.84,
    0.26,
    r"Discharge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="right",
    fontfamily="Arial",
)
ax.text(
    0.92,
    0.18,
    r"Charge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="right",
    fontfamily="Arial",
)
colorbar.outline.set_visible(False)  # type: ignore

# 图 E
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.35, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

line = []
for i in range(pca_Zn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Zn.iloc[:, 0], pca_Zn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(9650, 9750)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-15, 1)
ax.set_ylabel(r"PCA Intensity", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1], line[2]],
    labels=[r"Component#1", r"Component#2", r"Component#3"],
    loc="upper left",
    bbox_to_anchor=(0.4, 0.35),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)

# 图 F
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Zn2.shape[0], 1), np.log10(pca_Zn2.iloc[:, 1]), s=50, c=colors[1], edgecolors="face")

ax.set_xlim(-0.5, pca_Zn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-8, 0.5)
ax.set_ylabel(r"log of Variance", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_01_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_01_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_01_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_01_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure XANES and PCA on electrolyte - V3

In [ ]:
# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\Echem").glob(r"**\*.txt"))
# 读取电化学数据
echem_all = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem_all.append(df)

# 合并所有电化学数据为一个二维表格
echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)


# 读取 reference + operando 数据 # 放电数据
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")
# Mn
data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Mn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Mn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
xas_Mn = pd.concat([data1, data2.iloc[:, 4:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, 2, 3, *list(range(13, xas_Mn.shape[1], 1))]
xas_Mn = xas_Mn.iloc[:, charge_index]  # noqa: N816
pdf_Mn = xas_Mn.iloc[:, 0:4]  # noqa: N816
pdf_Mn.columns = [
    # r"Energy",
    r"0.2M_MnSO4(aq.)",
    r"alpha_MnO2_Electrode",
    r"alpha_MnO2_Powder",
    r"FC1st",
]
opData_Mn = xas_Mn.iloc[:, list(range(4, xas_Mn.shape[1]))]  # noqa: N816
opData_Mn.columns = list(range(0, opData_Mn.shape[1], 1))


# Zn
data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Zn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Zn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
xas_Zn = pd.concat([data1, data2.iloc[:, 2:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, *list(range(11, xas_Zn.shape[1], 1))]
xas_Zn = xas_Zn.iloc[:, charge_index]  # noqa: N816

pdf_Zn = xas_Zn.iloc[:, 0:2]  # noqa: N816
pdf_Zn.columns = [
    # r"Energy",
    r"0.5M_ZnSO4(aq.)",
    r"ZSH",
]
opData_Zn = xas_Zn.iloc[:, list(range(2, xas_Zn.shape[1]))]  # noqa: N816
opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))

# 读取 PCA 数据
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\FC1st-FD2nd\Oct2022_cell2_P1\XANES")
# Mn
pca_Mn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"cell2_1stFC_opXAS_P1_Mn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Mn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)
# Zn
pca_Zn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"cell2_1stFC_opXAS_P1_Zn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Zn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)

In [ ]:
# 剔除不好的谱线， Zn
opData_Zn = opData_Zn.drop(columns=opData_Zn.columns[13], axis=1)  # noqa: N816
# 电化学时间对齐
echem_all["Time"] = (df["time/s"] - df["time/s"].iloc[0]).dt.total_seconds() / 3600

In [ ]:
echem_index_low = echem_all[echem_all["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high = echem_all[echem_all["cycle number"] == 1]["Ewe/V"].idxmax()

In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 5))
gs = gridspec.GridSpec(2, 4, width_ratios=[0.5, 1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1.0, 1.0))
ax.set_box_aspect(2.0)
time = echem_all["Time"].iloc[echem_index_low:echem_index_high] - echem_all["Time"].iloc[echem_index_low]
ax.plot(echem_all["Ewe/V"].iloc[echem_index_low:echem_index_high], time, c=colors[0], ls="-", lw=1.0)


# 设置刻度线等格式
ax.set_ylabel(r"Duration Hours (h)", fontsize=9)
ax.set_ylim(-0.3, 19)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1))

ax.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}}\!)$", fontsize=9)
ax.set_xlim(0.7, 1.9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=-0.1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))

ax.tick_params(axis="both", which="both", bottom=True, left=True, labelbottom=True, labelleft=True)
ax.text(0.03, 0.15, r"C/10", ha="left", va="top", rotation=90, transform=ax.transAxes, fontsize=9, c="k")

# ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

# ax.text(
#     0.01,
#     0.15,
#     r"0.5 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[0],  # type: ignore
# )
# ax.text(
#     0.01,
#     0.23,
#     r"1 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[1],  # type: ignore
# )

ax2 = ax.twiny()
ax2.plot(echem_all["<I>/mA"].iloc[echem_index_low:echem_index_high] * 1000, time, c=colors[1], ls="--", lw=1.0)

# 设置刻度线等格式
ax2.set_xlabel(r"$\mathrm{Current \ (\mu A)}$", fontsize=9)
ax2.set_xlim(-80, 80)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=20, offset=0))

ax2.tick_params(axis="both", which="both", top=True, left=True, labeltop=True, labelleft=True)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.15, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.0)

ax.imshow(
    opData_Mn.T,
    aspect="auto",
    extent=[6530, 6630, 0, opData_Mn.shape[1]],
    cmap="coolwarm",
    origin="lower",
    vmin=0,
    vmax=2.4,
)

ax.set_xlim(6535, 6575)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=5))

ax.set_yticks([])

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.08, 0.1, 0.10, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.ax.text(
    1.06,
    0.98,
    r"High",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
colorbar.ax.text(
    1.08,
    0.08,
    r"Low",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
# colorbar.outline.set_visible(False)  # type: ignore

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, 0.0, 1.2, 1.0))
ax.set_box_aspect(0.8)

# xascolors = [[colors[0], colors[4]], [colors[2], colors[1]]]
xas_colors = ListedColormap(
    mpl.colormaps["sunset"](np.linspace(1.0, 0.0, opData_Mn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]
line = []
for i in range(pca_Mn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Mn.iloc[:, 0], pca_Mn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-15, 1)
ax.set_ylabel(r"PCA Intensity", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1]],
    labels=[r"Component#1", r"Component#2"],
    loc="upper left",
    bbox_to_anchor=(0.4, 0.3),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)

# 图 E
subfig = fig.add_subfigure(gs[0, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((1.3, 0.0, 1.2, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Mn2.shape[0], 1), np.log10(pca_Mn2.iloc[:, 1]), s=50, c=colors[0], edgecolors="face")

ax.set_xlim(-0.5, pca_Mn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-6.0, 0.5)
ax.set_ylabel(r"log of Variance", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

# 图 F
subfig = fig.add_subfigure(gs[1, 0])
ax = subfig.add_axes((0, -0.2, 1.0, 1.0))
ax.set_box_aspect(2.0)
time = echem_all["Time"].iloc[echem_index_low:echem_index_high] - echem_all["Time"].iloc[echem_index_low]
ax.plot(echem_all["Ewe/V"].iloc[echem_index_low:echem_index_high], time, c=colors[0], ls="-", lw=1.0)


# 设置刻度线等格式
ax.set_ylabel(r"Duration Hours (h)", fontsize=9)
ax.set_ylim(-0.3, 19)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1))

ax.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}}\!)$", fontsize=9)
ax.set_xlim(0.7, 1.9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=-0.1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))

ax.tick_params(axis="both", which="both", bottom=True, left=True, labelbottom=True, labelleft=True)
ax.text(0.03, 0.15, r"C/10", ha="left", va="top", rotation=90, transform=ax.transAxes, fontsize=9, c="k")

# ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

# ax.text(
#     0.01,
#     0.15,
#     r"0.5 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[0],  # type: ignore
# )
# ax.text(
#     0.01,
#     0.23,
#     r"1 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[1],  # type: ignore
# )

ax2 = ax.twiny()
ax2.plot(echem_all["<I>/mA"].iloc[echem_index_low:echem_index_high] * 1000, time, c=colors[1], ls="--", lw=1.0)

# 设置刻度线等格式
ax2.set_xlabel(r"$\mathrm{Current \ (\mu A)}$", fontsize=9)
ax2.set_xlim(-80, 80)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=20, offset=0))

ax2.tick_params(axis="both", which="both", top=True, left=True, labeltop=True, labelleft=True)

# 图 G
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.15, -0.2, 1.0, 1.0))
ax.set_box_aspect(1.0)

ax.imshow(
    opData_Zn.T,
    aspect="auto",
    extent=[9650, 9750, 0, opData_Zn.shape[1]],
    cmap="coolwarm",
    origin="lower",
    vmin=0,
    vmax=2.4,
)

ax.set_xlim(9655, 9690)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=5))

ax.set_yticks([])

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.08, 0.1, 0.10, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.ax.text(
    1.06,
    0.98,
    r"High",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
colorbar.ax.text(
    1.08,
    0.08,
    r"Low",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
# colorbar.outline.set_visible(False)  # type: ignore

# 图 H
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, -0.2, 1.2, 1.0))
ax.set_box_aspect(0.8)
xas_colors = ListedColormap(
    mpl.colormaps["sunset"](np.linspace(1.0, 0.0, opData_Mn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]
line = []
for i in range(pca_Zn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Zn.iloc[:, 0], pca_Zn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(9650, 9750)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-15.0, 1)
ax.set_ylabel(r"PCA Intensity", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1]],
    labels=[r"Component#1", r"Component#2"],
    loc="upper left",
    bbox_to_anchor=(0.4, 0.3),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)

# 图 I
subfig = fig.add_subfigure(gs[1, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((1.3, -0.2, 1.2, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Zn2.shape[0], 1), np.log10(pca_Zn2.iloc[:, 1]), s=50, c=colors[1], edgecolors="face")

ax.set_xlim(-0.5, pca_Zn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-8, 0.5)
ax.set_ylabel(r"log of Variance", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_00_V3_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_00_V3_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_00_V3_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_00_V3_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure XANES and PCA on electrolyte - V2

In [ ]:
# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\Echem").glob(r"**\*.txt"))
# 读取电化学数据
echem_all = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem_all.append(df)

# 合并所有电化学数据为一个二维表格
echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)


# 读取 reference + operando 数据 # 放电数据
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")
# Mn
data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Mn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Mn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
xas_Mn = pd.concat([data1, data2.iloc[:, 4:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, 2, 3, *list(range(13, xas_Mn.shape[1], 1))]
xas_Mn = xas_Mn.iloc[:, charge_index]  # noqa: N816
pdf_Mn = xas_Mn.iloc[:, 0:4]  # noqa: N816
pdf_Mn.columns = [
    # r"Energy",
    r"0.2M_MnSO4(aq.)",
    r"alpha_MnO2_Electrode",
    r"alpha_MnO2_Powder",
    r"FC1st",
]
opData_Mn = xas_Mn.iloc[:, list(range(4, xas_Mn.shape[1]))]  # noqa: N816
opData_Mn.columns = list(range(0, opData_Mn.shape[1], 1))


# Zn
data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Zn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Zn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
xas_Zn = pd.concat([data1, data2.iloc[:, 2:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, *list(range(11, xas_Zn.shape[1], 1))]
xas_Zn = xas_Zn.iloc[:, charge_index]  # noqa: N816

pdf_Zn = xas_Zn.iloc[:, 0:2]  # noqa: N816
pdf_Zn.columns = [
    # r"Energy",
    r"0.5M_ZnSO4(aq.)",
    r"ZSH",
]
opData_Zn = xas_Zn.iloc[:, list(range(2, xas_Zn.shape[1]))]  # noqa: N816
opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))

# 读取 PCA 数据
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\FC1st-FD2nd\Oct2022_cell2_P1\XANES")
# Mn
pca_Mn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"cell2_1stFC_opXAS_P1_Mn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Mn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)
# Zn
pca_Zn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"cell2_1stFC_opXAS_P1_Zn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Zn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)

In [ ]:
# 剔除不好的谱线， Zn
opData_Zn = opData_Zn.drop(columns=opData_Zn.columns[13], axis=1)  # noqa: N816
# 电化学时间对齐
echem_all["Time"] = (df["time/s"] - df["time/s"].iloc[0]).dt.total_seconds() / 3600

echem_index_low = echem_all[echem_all["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high = echem_all[echem_all["cycle number"] == 1]["Ewe/V"].idxmax()

In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 5))
gs = gridspec.GridSpec(2, 4, width_ratios=[0.5, 1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0:2])
ax = subfig.add_subplot(projection="3d")
ax.set_position((0.0, 0.0, 1.2, 1.2))
ax.set_box_aspect([2, 1, 1])

Energy, Spectrum_number = np.meshgrid(opData_Mn.index.values, np.arange(opData_Mn.shape[1]))

ax.plot_surface(
    Energy,
    Spectrum_number,
    opData_Mn.T,
    rstride=1,
    cstride=2,
    antialiased=True,
    linewidth=0.2,
    edgecolor="k",
    alpha=1.0,
    cmap="jet",
    vmin=0,
    vmax=2.4,
    shade=False,
    axlim_clip=True,
)

ax.view_init(elev=25, azim=250)

# 设置白色底面
for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    axis.pane.set_facecolor("white")
    axis.pane.set_edgecolor("white")
    axis._axinfo["grid"]["linewidth"] = 0.3
    axis._axinfo["grid"]["color"] = (0.85, 0.85, 0.85, 1)  # 淡灰

for label in ax.get_xticklabels():
    label.set_rotation(10)  # X轴刻度标签旋转 10°
    label.set_horizontalalignment("left")
    label.set_verticalalignment("top")
for label in ax.get_yticklabels():
    label.set_rotation(0)  # Y轴刻度标签旋转 0°
    label.set_horizontalalignment("left")
    label.set_verticalalignment("center")
for label in ax.get_zticklabels():
    label.set_rotation(0)  # Z轴保持竖直
    label.set_horizontalalignment("center")
    label.set_verticalalignment("center")

ax.tick_params(axis="x", direction="out", labelsize=9, pad=-5, colors="black")
ax.tick_params(axis="y", direction="out", labelsize=9, pad=0, colors="black")
ax.tick_params(axis="z", direction="out", labelsize=9, pad=0, colors="black")

ax.set_xlim(6530, 6610)
ax.set_xlabel(r"Energy (eV)", fontsize=8, labelpad=-2)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))

ax.set_ylim(0, opData_Mn.shape[1])
ax.set_ylabel(r"Spectrum", fontsize=8, labelpad=-7)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))

ax.set_zlim(-0.1, 2.4)
ax.zaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.zaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))
ax.set_zlabel("Absorption (arb.u.)", fontsize=8, labelpad=-5)

ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
    grid_linewidth=0.1,
)

# 图 B
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.55, 0.1, 1.0, 1.0))
ax.set_box_aspect(0.8)

xas_colors = ListedColormap(
    mpl.colormaps["sunset"](np.linspace(1.0, 0.0, opData_Mn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]
line = []
for i in range(pca_Mn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Mn.iloc[:, 0], pca_Mn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(6530, 6610)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-15, 1)
ax.set_ylabel(r"PCA Intensity", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1]],
    labels=[r"Component#1", r"Component#2"],
    loc="upper left",
    bbox_to_anchor=(0.42, 0.3),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)

# 图 C
subfig = fig.add_subfigure(gs[0, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.9, 0.1, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Mn2.shape[0], 1), np.log10(pca_Mn2.iloc[:, 1]), s=50, c=colors[0], edgecolors="face")

ax.set_xlim(-0.5, pca_Mn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-8.0, 0.5)
ax.set_ylabel(r"log of Variance", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

# 图 D
subfig = fig.add_subfigure(gs[1, 0:2])
ax = subfig.add_subplot(projection="3d")
ax.set_position((0.0, -0.1, 1.2, 1.2))
ax.set_box_aspect([2, 1, 1])

Energy, Spectrum_number = np.meshgrid(opData_Zn.index.values, np.arange(opData_Zn.shape[1]))

ax.plot_surface(
    Energy,
    Spectrum_number,
    opData_Zn.T,
    rstride=1,
    cstride=2,
    antialiased=True,
    linewidth=0.2,
    edgecolor="k",
    alpha=1.0,
    cmap="jet",
    vmin=0,
    vmax=2.4,
    shade=False,
    axlim_clip=True,
)

ax.view_init(elev=27, azim=250)

# 设置白色底面
for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    axis.pane.set_facecolor("white")
    axis.pane.set_edgecolor("white")
    axis._axinfo["grid"]["linewidth"] = 0.3
    axis._axinfo["grid"]["color"] = (0.85, 0.85, 0.85, 1)  # 淡灰

for label in ax.get_xticklabels():
    label.set_rotation(10)  # X轴刻度标签旋转 10°
    label.set_horizontalalignment("left")
    label.set_verticalalignment("top")
for label in ax.get_yticklabels():
    label.set_rotation(0)  # Y轴刻度标签旋转 0°
    label.set_horizontalalignment("left")
    label.set_verticalalignment("center")
for label in ax.get_zticklabels():
    label.set_rotation(0)  # Z轴保持竖直
    label.set_horizontalalignment("center")
    label.set_verticalalignment("center")

ax.tick_params(axis="x", direction="out", labelsize=9, pad=-5, colors="black")
ax.tick_params(axis="y", direction="out", labelsize=9, pad=0, colors="black")
ax.tick_params(axis="z", direction="out", labelsize=9, pad=0, colors="black")

ax.set_xlim(9650, 9730)
ax.set_xlabel(r"Energy (eV)", fontsize=8, labelpad=-2)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))

ax.set_ylim(0, opData_Zn.shape[1])
ax.set_ylabel(r"Spectrum", fontsize=8, labelpad=-7)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))

ax.set_zlim(-0.1, 2.4)
ax.zaxis.set_major_locator(ticker.MultipleLocator(base=0.6))
ax.zaxis.set_minor_locator(ticker.MultipleLocator(base=0.3))
ax.set_zlabel("Absorption (arb.u.)", fontsize=8, labelpad=-5)

ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
    grid_linewidth=0.1,
)

# 图 E
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.55, 0.1, 1.0, 1.0))
ax.set_box_aspect(0.8)

xas_colors = ListedColormap(
    mpl.colormaps["sunset"](np.linspace(1.0, 0.0, opData_Mn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]
line = []
for i in range(pca_Zn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Zn.iloc[:, 0], pca_Zn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(9650, 9730)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-15, 1)
ax.set_ylabel(r"PCA Intensity", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1]],
    labels=[r"Component#1", r"Component#2"],
    loc="upper left",
    bbox_to_anchor=(0.35, 0.3),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)

# 图 E
subfig = fig.add_subfigure(gs[1, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.9, 0.1, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Zn2.shape[0], 1), np.log10(pca_Zn2.iloc[:, 1]), s=50, c=colors[1], edgecolors="face")

ax.set_xlim(-0.5, pca_Zn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-8.0, 0.5)
ax.set_ylabel(r"log of Variance", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_00_V2_300.tif"),
    transparent=False,
    pad_inches=0.2,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_00_V2_600.tif"),
    transparent=False,
    pad_inches=0.2,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_00_V2_600.png"),
    pad_inches=0.2,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_00_V2_900.svg"),
    transparent=False,
    pad_inches=0.2,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure XANES and PCA on electrolyte - V1

In [ ]:
# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\Echem").glob(r"**\*.txt"))
# 读取电化学数据
echem_all = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem_all.append(df)

# 合并所有电化学数据为一个二维表格
echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)


# 读取 reference + operando 数据 # 放电数据
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")
# Mn
data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Mn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Mn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
xas_Mn = pd.concat([data1, data2.iloc[:, 4:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, 2, 3, *list(range(13, xas_Mn.shape[1], 1))]
xas_Mn = xas_Mn.iloc[:, charge_index]  # noqa: N816
pdf_Mn = xas_Mn.iloc[:, 0:4]  # noqa: N816
pdf_Mn.columns = [
    # r"Energy",
    r"0.2M_MnSO4(aq.)",
    r"alpha_MnO2_Electrode",
    r"alpha_MnO2_Powder",
    r"FC1st",
]
opData_Mn = xas_Mn.iloc[:, list(range(4, xas_Mn.shape[1]))]  # noqa: N816
opData_Mn.columns = list(range(0, opData_Mn.shape[1], 1))


# Zn
data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Zn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Zn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
xas_Zn = pd.concat([data1, data2.iloc[:, 2:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, *list(range(11, xas_Zn.shape[1], 1))]
xas_Zn = xas_Zn.iloc[:, charge_index]  # noqa: N816

pdf_Zn = xas_Zn.iloc[:, 0:2]  # noqa: N816
pdf_Zn.columns = [
    # r"Energy",
    r"0.5M_ZnSO4(aq.)",
    r"ZSH",
]
opData_Zn = xas_Zn.iloc[:, list(range(2, xas_Zn.shape[1]))]  # noqa: N816
opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))

# 读取 PCA 数据
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\FC1st-FD2nd\Oct2022_cell2_P1\XANES")
# Mn
pca_Mn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"cell2_1stFC_opXAS_P1_Mn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Mn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)
# Zn
pca_Zn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"cell2_1stFC_opXAS_P1_Zn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Zn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)

In [ ]:
# 剔除不好的谱线， Zn
opData_Zn = opData_Zn.drop(columns=opData_Zn.columns[13], axis=1)  # noqa: N816
# 电化学时间对齐
echem_all["Time"] = (df["time/s"] - df["time/s"].iloc[0]).dt.total_seconds() / 3600

In [ ]:
echem_index_low = echem_all[echem_all["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high = echem_all[echem_all["cycle number"] == 1]["Ewe/V"].idxmax()

In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 5))
gs = gridspec.GridSpec(2, 4, width_ratios=[0.5, 1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1.0, 1.0))
ax.set_box_aspect(2.0)
time = echem_all["Time"].iloc[echem_index_low:echem_index_high] - echem_all["Time"].iloc[echem_index_low]
ax.plot(echem_all["Ewe/V"].iloc[echem_index_low:echem_index_high], time, c=colors[0], ls="-", lw=1.0)


# 设置刻度线等格式
ax.set_ylabel(r"Duration Hours (h)", fontsize=9)
ax.set_ylim(-0.3, 19)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1))

ax.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}}\!)$", fontsize=9)
ax.set_xlim(0.7, 1.9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=-0.1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))

ax.tick_params(axis="both", which="both", bottom=True, left=True, labelbottom=True, labelleft=True)
ax.text(0.03, 0.15, r"C/10", ha="left", va="top", rotation=90, transform=ax.transAxes, fontsize=9, c="k")

# ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

# ax.text(
#     0.01,
#     0.15,
#     r"0.5 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[0],  # type: ignore
# )
# ax.text(
#     0.01,
#     0.23,
#     r"1 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[1],  # type: ignore
# )

ax2 = ax.twiny()
ax2.plot(echem_all["<I>/mA"].iloc[echem_index_low:echem_index_high] * 1000, time, c=colors[1], ls="--", lw=1.0)

# 设置刻度线等格式
ax2.set_xlabel(r"$\mathrm{Current \ (\mu A)}$", fontsize=9)
ax2.set_xlim(-80, 80)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=20, offset=0))

ax2.tick_params(axis="both", which="both", top=True, left=True, labeltop=True, labelleft=True)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.15, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.0)

ax.imshow(
    opData_Mn.T,
    aspect="auto",
    extent=[6530, 6630, 0, opData_Mn.shape[1]],
    cmap="coolwarm",
    origin="lower",
    vmin=0,
    vmax=2.4,
)

ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))

ax.set_yticks([])

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.08, 0.1, 0.10, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.ax.text(
    1.06,
    0.98,
    r"High",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
colorbar.ax.text(
    1.08,
    0.08,
    r"Low",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
# colorbar.outline.set_visible(False)  # type: ignore

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, 0.0, 1.2, 1.0))
ax.set_box_aspect(0.8)

# xascolors = [[colors[0], colors[4]], [colors[2], colors[1]]]
xas_colors = ListedColormap(
    mpl.colormaps["sunset"](np.linspace(1.0, 0.0, opData_Mn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]
line = []
for i in range(pca_Mn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Mn.iloc[:, 0], pca_Mn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-15, 1)
ax.set_ylabel(r"PCA Intensity", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1]],
    labels=[r"Component#1", r"Component#2"],
    loc="upper left",
    bbox_to_anchor=(0.4, 0.3),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)

# 图 E
subfig = fig.add_subfigure(gs[0, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((1.3, 0.0, 1.2, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Mn2.shape[0], 1), np.log10(pca_Mn2.iloc[:, 1]), s=50, c=colors[0], edgecolors="face")

ax.set_xlim(-0.5, pca_Mn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-6.0, 0.5)
ax.set_ylabel(r"log of Variance", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

# 图 F
subfig = fig.add_subfigure(gs[1, 0])
ax = subfig.add_axes((0, -0.2, 1.0, 1.0))
ax.set_box_aspect(2.0)
time = echem_all["Time"].iloc[echem_index_low:echem_index_high] - echem_all["Time"].iloc[echem_index_low]
ax.plot(echem_all["Ewe/V"].iloc[echem_index_low:echem_index_high], time, c=colors[0], ls="-", lw=1.0)


# 设置刻度线等格式
ax.set_ylabel(r"Duration Hours (h)", fontsize=9)
ax.set_ylim(-0.3, 19)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1))

ax.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}}\!)$", fontsize=9)
ax.set_xlim(0.7, 1.9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=-0.1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))

ax.tick_params(axis="both", which="both", bottom=True, left=True, labelbottom=True, labelleft=True)
ax.text(0.03, 0.15, r"C/10", ha="left", va="top", rotation=90, transform=ax.transAxes, fontsize=9, c="k")

# ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

# ax.text(
#     0.01,
#     0.15,
#     r"0.5 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[0],  # type: ignore
# )
# ax.text(
#     0.01,
#     0.23,
#     r"1 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c=colors[1],  # type: ignore
# )

ax2 = ax.twiny()
ax2.plot(echem_all["<I>/mA"].iloc[echem_index_low:echem_index_high] * 1000, time, c=colors[1], ls="--", lw=1.0)

# 设置刻度线等格式
ax2.set_xlabel(r"$\mathrm{Current \ (\mu A)}$", fontsize=9)
ax2.set_xlim(-80, 80)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=20, offset=0))

ax2.tick_params(axis="both", which="both", top=True, left=True, labeltop=True, labelleft=True)

# 图 G
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.15, -0.2, 1.0, 1.0))
ax.set_box_aspect(1.0)

ax.imshow(
    opData_Zn.T,
    aspect="auto",
    extent=[9650, 9750, 0, opData_Zn.shape[1]],
    cmap="coolwarm",
    origin="lower",
    vmin=0,
    vmax=2.4,
)

ax.set_xlim(9650, 9750)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))

ax.set_yticks([])

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.08, 0.1, 0.10, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.ax.text(
    1.06,
    0.98,
    r"High",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
colorbar.ax.text(
    1.08,
    0.08,
    r"Low",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
# colorbar.outline.set_visible(False)  # type: ignore

# 图 H
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, -0.2, 1.2, 1.0))
ax.set_box_aspect(0.8)
xas_colors = ListedColormap(
    mpl.colormaps["sunset"](np.linspace(1.0, 0.0, opData_Mn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]
line = []
for i in range(pca_Zn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Zn.iloc[:, 0], pca_Zn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(9650, 9750)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-15.0, 1)
ax.set_ylabel(r"PCA Intensity", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1]],
    labels=[r"Component#1", r"Component#2"],
    loc="upper left",
    bbox_to_anchor=(0.4, 0.3),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)

# 图 I
subfig = fig.add_subfigure(gs[1, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((1.3, -0.2, 1.2, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Zn2.shape[0], 1), np.log10(pca_Zn2.iloc[:, 1]), s=50, c=colors[1], edgecolors="face")

ax.set_xlim(-0.5, pca_Zn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-8, 0.5)
ax.set_ylabel(r"log of Variance", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_00_V1_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
# plt.savefig(
#     Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_00_V1_600.tif"),
#     transparent=False,
#     pad_inches=0.05,
#     bbox_inches="tight",
#     dpi=600,
#     pil_kwargs={"compression": "tiff_lzw"},
# )
# plt.savefig(
#     Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_00_V1_600.png"),
#     pad_inches=0.05,
#     bbox_inches="tight",
#     dpi=600,
#     transparent=False,
# )
# plt.savefig(
#     Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_00_V1_900.svg"),
#     transparent=False,
#     pad_inches=0.05,
#     bbox_inches="tight",
#     dpi=900,
# )

plt.gcf().set_facecolor("white")
plt.show()

### Figure XANES and PCA on electrolyte

In [ ]:
# 读取 reference + operando 数据 # 放电数据
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")
# Mn
data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Mn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Mn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
xas_Mn = pd.concat([data1, data2.iloc[:, 5:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, 2, 3, 4, *list(range(14, xas_Mn.shape[1], 1))]
xas_Mn = xas_Mn.iloc[:, charge_index]  # noqa: N816
pdf_Mn = xas_Mn.iloc[:, 0:5]  # noqa: N816
pdf_Mn.columns = [
    r"Energy",
    r"0.2M_MnSO4(aq.)",
    r"alpha_MnO2_Electrode",
    r"alpha_MnO2_Powder",
    r"FC1st",
]
opData_Mn = xas_Mn.iloc[:, [0, *list(range(5, xas_Mn.shape[1]))]]  # noqa: N816
opData_Mn.columns = list(range(0, opData_Mn.shape[1], 1))


# Zn
data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Zn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Zn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
xas_Zn = pd.concat([data1, data2.iloc[:, 3:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, 2, *list(range(12, xas_Zn.shape[1], 1))]
xas_Zn = xas_Zn.iloc[:, charge_index]  # noqa: N816

pdf_Zn = xas_Zn.iloc[:, 0:3]  # noqa: N816
pdf_Zn.columns = [r"Energy", r"0.5M_ZnSO4(aq.)", r"ZSH"]
opData_Zn = xas_Zn.iloc[:, [0, *list(range(3, xas_Zn.shape[1]))]]  # noqa: N816
opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))

# 读取 PCA 数据
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\FC1st-FD2nd\Oct2022_cell2_P1\XANES")
# Mn
pca_Mn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"cell2_1stFC_opXAS_P1_Mn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Mn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)
# Zn
pca_Zn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"cell2_1stFC_opXAS_P1_Zn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Zn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)

In [ ]:
# 剔除不好的谱线， Zn
opData_Zn = opData_Zn.drop(columns=opData_Zn.columns[14], axis=1)  # noqa: N816


In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 5))
gs = gridspec.GridSpec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

xascolors = [[colors[0], colors[4]], [colors[2], colors[1]]]
xas_colors = ListedColormap(
    mpl.colormaps["sunset"](np.linspace(1.0, 0.0, opData_Mn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

for i in range(pdf_Mn.shape[1] - 3):
    ax.plot(pdf_Mn.iloc[:, 0], pdf_Mn.iloc[:, i + 1], c=xascolors[0][i], ls="--", lw=1.0, label=labels[0][i], zorder=5)
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
for i in range(opData_Mn.shape[1] - 1):
    ax.plot(opData_Mn.iloc[:, 0], opData_Mn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0)  # type: ignore

ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_ylim(0, 2.4)
ax.set_ylabel(r"Absorption (arb.u.)", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((0.5, 0.05, 0.42, 0.05)),
    location="bottom",
    orientation="horizontal",
    cmap=xas_colors,
    ticklocation="top",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.45, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=10,
    handlelength=3,
    handletextpad=0.2,
)
ax.text(
    0.49,
    0.18,
    r"Charge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.84,
    0.26,
    r"Discharge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="right",
    fontfamily="Arial",
)
ax.text(
    0.92,
    0.18,
    r"Charge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="right",
    fontfamily="Arial",
)
colorbar.outline.set_visible(False)  # type: ignore

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.35, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

line = []
for i in range(pca_Mn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Mn.iloc[:, 0], pca_Mn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-15, 1)
ax.set_ylabel(r"PCA Intensity", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1]],
    labels=[r"Component#1", r"Component#2"],
    loc="upper left",
    bbox_to_anchor=(0.4, 0.3),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Mn2.shape[0], 1), np.log10(pca_Mn2.iloc[:, 1]), s=50, c=colors[0], edgecolors="face")

ax.set_xlim(-0.5, pca_Mn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-6.0, 0.5)
ax.set_ylabel(r"log of Variance", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

# 图 D
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

xascolors = [[colors[0], colors[4]], [colors[2], colors[1]]]
xas_colors = ListedColormap(
    mpl.colormaps["coolwarm"](np.linspace(0.0, 1.0, opData_Zn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]

for i in range(pdf_Zn.shape[1] - 1):
    ax.plot(pdf_Zn.iloc[:, 0], pdf_Zn.iloc[:, i + 1], c=xascolors[1][i], ls="--", lw=1.0, label=labels[1][i], zorder=5)
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
for i in range(opData_Zn.shape[1] - 1):
    ax.plot(opData_Zn.iloc[:, 0], opData_Zn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0)  # type: ignore

ax.set_xlim(9650, 9750)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_ylim(0, 2.4)
ax.set_ylabel(r"Absorption (arb.u.)", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((0.5, 0.05, 0.42, 0.05)),
    location="bottom",
    orientation="horizontal",
    cmap=xas_colors,
    ticklocation="top",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.45, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=10,
    handlelength=3,
    handletextpad=0.2,
)
ax.text(
    0.49,
    0.18,
    r"Charge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.84,
    0.26,
    r"Discharge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="right",
    fontfamily="Arial",
)
ax.text(
    0.92,
    0.18,
    r"Charge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="right",
    fontfamily="Arial",
)
colorbar.outline.set_visible(False)  # type: ignore

# 图 E
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.35, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

line = []
for i in range(pca_Zn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Zn.iloc[:, 0], pca_Zn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(9650, 9750)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-15.0, 1)
ax.set_ylabel(r"PCA Intensity", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1]],
    labels=[r"Component#1", r"Component#2"],
    loc="upper left",
    bbox_to_anchor=(0.4, 0.3),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)

# 图 F
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Zn2.shape[0], 1), np.log10(pca_Zn2.iloc[:, 1]), s=50, c=colors[1], edgecolors="face")

ax.set_xlim(-0.5, pca_Zn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-8, 0.5)
ax.set_ylabel(r"log of Variance", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_00_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_00_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_00_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XAS_XANES_00_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

## TEM-HRTEM

### 2nd0.9V TEM

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
path_tem = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\TEM\Results\01\TEM0.9V")
HAADF = hs.load(path_tem.joinpath(r"0025-Ceta_265_kx_Camera_Ceta.dm4"))

TEM_1 = hs.load([path_tem.joinpath(r"FFT of 0025-Ceta_265_kx_Camera_Ceta.dm4"), path_tem.joinpath(r"IFFT of Untitled.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled.dm4")])
TEM_2 = hs.load([path_tem.joinpath(r"FFT of 0025-Ceta_265_kx_Camera_Ceta_1.dm4"), path_tem.joinpath(r"IFFT of Untitled_1.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled_1.dm4")])
TEM_3 = hs.load([path_tem.joinpath(r"FFT of 0025-Ceta_265_kx_Camera_Ceta_2.dm4"), path_tem.joinpath(r"IFFT of Untitled_2.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled_2.dm4")])

HAADF = convert_units(HAADF)[0]
TEM_1 = convert_units([TEM_1])[0]
TEM_2 = convert_units([TEM_2])[0]
TEM_3 = convert_units([TEM_3])[0]

path_tem2 = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\TEM\Results\02\TEM0.9V-2")
HAADF2 = hs.load(path_tem2.joinpath(r"0016-Ceta_420_kx_Camera_Ceta.dm4"))

TEM2_1 = hs.load([path_tem2.joinpath(r"FFT of 0016-Ceta_420_kx_Camera_Ceta.dm4"), path_tem2.joinpath(r"IFFT of Untitled.dm4"), path_tem2.joinpath(r"Live Profile Of IFFT of Untitled.dm4")])

HAADF2 = convert_units(HAADF2)[0]
TEM2_1 = convert_units([TEM2_1])[0]

In [ ]:
# 画图
fig = plt.figure(figsize=(7.0, 7.5))
gs = gridspec.GridSpec(6, 4, width_ratios=[1, 1, 1, 1], height_ratios=[1, 1, 1, 1, 1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0:2, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

vmin, vmax = np.percentile(HAADF.data, [1.0, 99.0])
ax.imshow(HAADF.data, cmap="gray", vmin=vmin, vmax=vmax)

scalebar = add_sizebar(
    ax,
    size=10,
    bardata=HAADF,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()

ax.text(
    0.23,
    0.95,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=11,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.1,
    1.0,
    r"A",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, 0.0, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_1[2].axes_manager[-1].axis,
    y1=TEM_1[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[1],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 4.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-150, 200)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.12,
    1.0,
    r"B",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B1
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_axes((-0.15, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_1[1].data, [1.0, 99.0])
ax.imshow(TEM_1[1].data, cmap="gray", vmin=vmin, vmax=vmax)
add_sizebar(ax, 1, TEM_1[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 B2
subfig = fig.add_subfigure(gs[1, 3], zorder=0)
ax = subfig.add_axes((-0.05, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_1[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 C
subfig = fig.add_subfigure(gs[2, 0:2], zorder=0)
ax = subfig.add_axes((0.00, -0.15, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_2[2].axes_manager[-1].axis,
    y1=TEM_2[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[0],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 6.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-40, 40)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.1,
    1.0,
    r"C",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C1
subfig = fig.add_subfigure(gs[3, 0], zorder=0)
ax = subfig.add_axes((0.07, -0.16, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_2[1].data, [1.0, 99.0])
ax.imshow(TEM_2[1].data, cmap="gray", vmin=vmin, vmax=vmax)
add_sizebar(ax, 1, TEM_2[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 C2
subfig = fig.add_subfigure(gs[3, 1], zorder=0)
ax = subfig.add_axes((0.15, -0.16, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_2[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 D
subfig = fig.add_subfigure(gs[2, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, -0.15, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_3[2].axes_manager[-1].axis,
    y1=TEM_3[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[4],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 3.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-15, 20)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.1,
    1.0,
    r"D",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 D1
subfig = fig.add_subfigure(gs[3, 2], zorder=0)
ax = subfig.add_axes((-0.15, -0.15, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_3[1].data, [1.0, 99.0])
ax.imshow(TEM_3[1].data, cmap="gray", vmin=vmin, vmax=vmax)
add_sizebar(ax, 1, TEM_3[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 D2
subfig = fig.add_subfigure(gs[3, 3], zorder=0)
ax = subfig.add_axes((-0.07, -0.15, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_3[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 E
subfig = fig.add_subfigure(gs[4:6, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, -0.2, 1.0, 1.0))
# ax.set_box_aspect(1.0)

vmin, vmax = np.percentile(HAADF2.data, [1.0, 99.0])
ax.imshow(HAADF2.data, cmap="gray", vmin=vmin, vmax=vmax)

scalebar = add_sizebar(
    ax,
    size=5,
    bardata=HAADF2,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()

ax.text(
    0.23,
    0.95,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=11,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.1,
    1.0,
    r"E",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 F
subfig = fig.add_subfigure(gs[4, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, -0.4, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM2_1[2].axes_manager[-1].axis,
    y1=TEM2_1[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[3],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 3.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-90, 110)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.12,
    1.0,
    r"F",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 F1
subfig = fig.add_subfigure(gs[5, 2], zorder=0)
ax = subfig.add_axes((-0.15, -0.4, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM2_1[1].data, [1.0, 99.0])
ax.imshow(TEM2_1[1].data, cmap="gray", vmin=vmin, vmax=vmax)
add_sizebar(ax, 1, TEM2_1[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 F2
subfig = fig.add_subfigure(gs[5, 3], zorder=0)
ax = subfig.add_axes((-0.05, -0.4, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM2_1[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_04_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_04_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_04_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_04_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### 2nd13.0V TEM

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
path_tem = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\TEM\Results\01\TEM130V")
HAADF = hs.load(path_tem.joinpath(r"0023-Ceta_420_kx_Camera_Ceta.dm4"))

TEM_1 = hs.load([path_tem.joinpath(r"FFT of 0023-Ceta_420_kx_Camera_Ceta.dm4"), path_tem.joinpath(r"IFFT of Untitled.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled.dm4")])
TEM_2 = hs.load([path_tem.joinpath(r"FFT of 0023-Ceta_420_kx_Camera_Ceta_1.dm4"), path_tem.joinpath(r"IFFT of Untitled_1.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled_1.dm4")])
TEM_3 = hs.load([path_tem.joinpath(r"FFT of 0023-Ceta_420_kx_Camera_Ceta_2.dm4"), path_tem.joinpath(r"IFFT of Untitled_2.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled_2.dm4")])
TEM_4 = hs.load([path_tem.joinpath(r"FFT of 0023-Ceta_420_kx_Camera_Ceta_3.dm4"), path_tem.joinpath(r"IFFT of Untitled_3.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled_3.dm4")])

HAADF = convert_units(HAADF)[0]
TEM_1 = convert_units([TEM_1])[0]
TEM_2 = convert_units([TEM_2])[0]
TEM_3 = convert_units([TEM_3])[0]
TEM_4 = convert_units([TEM_4])[0]

path_tem2 = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\TEM\Results\02\TEM130V-2")
HAADF2 = hs.load(path_tem2.joinpath(r"0026-Ceta_800_kx_Camera_Ceta.dm4"))

TEM2_1 = hs.load([path_tem2.joinpath(r"FFT of 0026-Ceta_800_kx_Camera_Ceta.dm4"), path_tem2.joinpath(r"IFFT of Untitled.dm4"), path_tem2.joinpath(r"Live Profile Of IFFT of Untitled.dm4")])

HAADF2 = convert_units(HAADF2)[0]
TEM2_1 = convert_units([TEM2_1])[0]

In [ ]:
# 画图
fig = plt.figure(figsize=(7.0, 7.5))
gs = gridspec.GridSpec(6, 4, width_ratios=[1, 1, 1, 1], height_ratios=[1, 1, 1, 1, 1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0:2, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

vmin, vmax = np.percentile(HAADF.data, [1.0, 99.0])
ax.imshow(HAADF.data, cmap="gray", vmin=vmin, vmax=vmax)

scalebar = add_sizebar(
    ax,
    size=10,
    bardata=HAADF,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()

ax.text(
    0.23,
    0.95,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=11,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.1,
    1.0,
    r"A",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, 0.0, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_1[2].axes_manager[-1].axis,
    y1=TEM_1[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[1],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 3.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-50, 70)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.12,
    1.0,
    r"B",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B1
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_axes((-0.15, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_1[1].data, [1.0, 99.0])
ax.imshow(TEM_1[1].data, cmap="gray", vmin=vmin, vmax=vmax)
add_sizebar(ax, 1, TEM_1[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 B2
subfig = fig.add_subfigure(gs[1, 3], zorder=0)
ax = subfig.add_axes((-0.05, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_1[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 C
subfig = fig.add_subfigure(gs[2, 0:2], zorder=0)
ax = subfig.add_axes((0.00, -0.15, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_2[2].axes_manager[-1].axis,
    y1=TEM_2[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[0],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 2.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-150, 200)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.1,
    1.0,
    r"C",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C1
subfig = fig.add_subfigure(gs[3, 0], zorder=0)
ax = subfig.add_axes((0.07, -0.16, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_2[1].data, [1.0, 99.0])
ax.imshow(TEM_2[1].data, cmap="gray", vmin=vmin, vmax=vmax)
add_sizebar(ax, 1, TEM_2[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 C2
subfig = fig.add_subfigure(gs[3, 1], zorder=0)
ax = subfig.add_axes((0.15, -0.16, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_2[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 D
subfig = fig.add_subfigure(gs[2, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, -0.15, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_4[2].axes_manager[-1].axis,
    y1=TEM_4[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[4],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 3.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-150, 200)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.1,
    1.0,
    r"D",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 D1
subfig = fig.add_subfigure(gs[3, 2], zorder=0)
ax = subfig.add_axes((-0.15, -0.15, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_4[1].data, [1.0, 99.0])
ax.imshow(TEM_4[1].data, cmap="gray", vmin=vmin, vmax=vmax)
add_sizebar(ax, 1, TEM_4[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 D2
subfig = fig.add_subfigure(gs[3, 3], zorder=0)
ax = subfig.add_axes((-0.07, -0.15, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_4[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 E
subfig = fig.add_subfigure(gs[4:6, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, -0.2, 1.0, 1.0))
# ax.set_box_aspect(1.0)

vmin, vmax = np.percentile(HAADF2.data, [1.0, 99.0])
ax.imshow(HAADF2.data, cmap="gray", vmin=vmin, vmax=vmax)

scalebar = add_sizebar(
    ax,
    size=5,
    bardata=HAADF2,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()

ax.text(
    0.23,
    0.95,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=11,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.1,
    1.0,
    r"E",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 F
subfig = fig.add_subfigure(gs[4, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, -0.4, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM2_1[2].axes_manager[-1].axis,
    y1=TEM2_1[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[3],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 2.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-250, 300)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.12,
    1.0,
    r"F",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 F1
subfig = fig.add_subfigure(gs[5, 2], zorder=0)
ax = subfig.add_axes((-0.15, -0.4, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM2_1[1].data, [1.0, 99.0])
ax.imshow(TEM2_1[1].data, cmap="gray", vmin=vmin, vmax=vmax)
add_sizebar(ax, 1, TEM2_1[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 F2
subfig = fig.add_subfigure(gs[5, 3], zorder=0)
ax = subfig.add_axes((-0.05, -0.4, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM2_1[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_03_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_03_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_03_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_03_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### 1st1.80V TEM

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
path_tem = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\TEM\Results\01\Workspace 1")
HAADF = hs.load(path_tem.joinpath(r"0024 - 0947 Pristine MnO2 HAADF 2.60 Mx.emd.conv_HAADF.dm4"))

TEM_1 = hs.load([
    path_tem.joinpath(r"FFT of 0024 - 0947 Pristine MnO2 HAADF 2.60 Mx.emd.conv_HAADF.dm4"),
    path_tem.joinpath(r"IFFT of Untitled.dm4"),
    path_tem.joinpath(r"Live Profile Of IFFT of Untitled.dm4"),
])
TEM_2 = hs.load([
    path_tem.joinpath(r"FFT of 0024 - 0947 Pristine MnO2 HAADF 2.60 Mx.emd.conv_HAADF_1.dm4"),
    path_tem.joinpath(r"IFFT of Untitled_1.dm4"),
    path_tem.joinpath(r"Live Profile Of IFFT of Untitled_1.dm4"),
])
TEM_3 = hs.load([
    path_tem.joinpath(r"FFT of 0024 - 0947 Pristine MnO2 HAADF 2.60 Mx.emd.conv_HAADF_2.dm4"),
    path_tem.joinpath(r"IFFT of Untitled_2.dm4"),
    path_tem.joinpath(r"Live Profile Of IFFT of Untitled_2.dm4"),
])

HAADF = convert_units(HAADF)[0]
TEM_1 = convert_units([TEM_1])[0]
TEM_2 = convert_units([TEM_2])[0]
TEM_3 = convert_units([TEM_3])[0]

In [ ]:
# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(4, 4, width_ratios=[1, 1, 1, 1], height_ratios=[1, 1, 1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0:2, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(1.0)

vmin, vmax = np.percentile(HAADF.data, [1.0, 99.0])
ax.imshow(HAADF.data, cmap="gray", vmin=vmin, vmax=vmax)

scalebar = add_sizebar(
    ax,
    size=5,
    bardata=HAADF,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()

ax.text(
    0.23,
    0.95,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=11,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.1,
    1.0,
    r"A",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, 0.0, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_1[2].axes_manager[-1].axis,
    y1=TEM_1[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[1],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 3.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-2800, 3500)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.12,
    1.0,
    r"B",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B1
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_axes((-0.15, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_1[1].data, [1.0, 99.0])
ax.imshow(TEM_1[1].data, cmap="gray", vmin=vmin, vmax=vmax)
add_sizebar(ax, 1, TEM_1[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 B2
subfig = fig.add_subfigure(gs[1, 3], zorder=0)
ax = subfig.add_axes((-0.05, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_1[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 C
subfig = fig.add_subfigure(gs[2, 0:2], zorder=0)
ax = subfig.add_axes((0.00, -0.15, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_2[2].axes_manager[-1].axis,
    y1=TEM_2[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[0],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 2.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-1500, 2000)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.1,
    1.0,
    r"C",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C1
subfig = fig.add_subfigure(gs[3, 0], zorder=0)
ax = subfig.add_axes((0.07, -0.16, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_2[1].data, [1.0, 99.0])
ax.imshow(TEM_2[1].data, cmap="gray", vmin=vmin, vmax=vmax)
add_sizebar(ax, 1, TEM_2[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 C2
subfig = fig.add_subfigure(gs[3, 1], zorder=0)
ax = subfig.add_axes((0.15, -0.16, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_2[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 D
subfig = fig.add_subfigure(gs[2, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, -0.15, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_3[2].axes_manager[-1].axis,
    y1=TEM_3[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[4],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 3.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-4000, 5000)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.1,
    1.0,
    r"D",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 D1
subfig = fig.add_subfigure(gs[3, 2], zorder=0)
ax = subfig.add_axes((-0.15, -0.15, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_3[1].data, [1.0, 99.0])
ax.imshow(TEM_3[1].data, cmap="gray", vmin=vmin, vmax=vmax)
add_sizebar(ax, 1, TEM_3[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 D2
subfig = fig.add_subfigure(gs[3, 3], zorder=0)
ax = subfig.add_axes((-0.07, -0.15, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_3[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_02_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_02_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_02_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_02_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### 1st1.63V TEM

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
path_tem = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\TEM\Results\01\TEM163V")
HAADF = hs.load(path_tem.joinpath(r"0004-20241211_Ceta_210_kx_Ceta.dm4"))

TEM_1 = hs.load([path_tem.joinpath(r"FFT of 0004-20241211_Ceta_210_kx_Ceta.dm4"), path_tem.joinpath(r"IFFT of Untitled.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled.dm4")])

HAADF = convert_units(HAADF)[0]
TEM_1 = convert_units([TEM_1])[0]

In [ ]:
# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(4, 4, width_ratios=[1, 1, 1, 1], height_ratios=[1, 1, 1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0:2, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

vmin, vmax = np.percentile(HAADF.data, [1.0, 99.0])
ax.imshow(HAADF.data, cmap="gray", vmin=vmin, vmax=vmax)

scalebar = add_sizebar(
    ax,
    size=20,
    bardata=HAADF,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()

ax.text(
    0.23,
    0.95,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=11,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.05,
    1.0,
    r"A",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, 0.0, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_1[2].axes_manager[-1].axis,
    y1=TEM_1[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[1],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 5.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-20, 30)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.12,
    1.0,
    r"B",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B1
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_axes((-0.15, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_1[1].data, [1.0, 99.0])
ax.imshow(TEM_1[1].data, cmap="gray", vmin=vmin, vmax=vmax)
add_sizebar(ax, 1, TEM_1[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 B2
subfig = fig.add_subfigure(gs[1, 3], zorder=0)
ax = subfig.add_axes((-0.05, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_1[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_01_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_01_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_01_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_01_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### 1st1.53V TEM

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
path_tem = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\TEM\Results\01\TEM1.53V")
HAADF = hs.load(path_tem.joinpath(r"0010-20241211_Ceta_340_kx_Ceta.dm4"))

TEM_1 = hs.load([path_tem.joinpath(r"FFT of 0010-20241211_Ceta_340_kx_Ceta.dm4"), path_tem.joinpath(r"IFFT of Untitled.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled.dm4")])
TEM_2 = hs.load([path_tem.joinpath(r"FFT of 0010-20241211_Ceta_340_kx_Ceta_1.dm4"), path_tem.joinpath(r"IFFT of Untitled.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled.dm4")])
TEM_3 = hs.load([path_tem.joinpath(r"FFT of 0010-20241211_Ceta_340_kx_Ceta_2.dm4"), path_tem.joinpath(r"IFFT of Untitled.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled.dm4")])

HAADF = convert_units(HAADF)[0]
TEM_1 = convert_units([TEM_1])[0]
TEM_2 = convert_units([TEM_2])[0]
TEM_3 = convert_units([TEM_3])[0]

In [ ]:
# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(4, 4, width_ratios=[1, 1, 1, 1], height_ratios=[1, 1, 1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0:2, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(1.0)

vmin, vmax = np.percentile(HAADF.data, [1.0, 99.0])
ax.imshow(HAADF.data, cmap="gray", vmin=vmin, vmax=vmax)

scalebar = add_sizebar(
    ax,
    size=10,
    bardata=HAADF,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()

ax.text(
    0.23,
    0.95,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=11,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.1,
    1.0,
    r"A",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, 0.0, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_1[2].axes_manager[-1].axis,
    y1=TEM_1[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[1],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 5.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-30, 50)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.12,
    1.0,
    r"B",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B1
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_axes((-0.15, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_1[1].data, [1.0, 99.0])
ax.imshow(TEM_1[1].data, cmap="gray", vmin=vmin, vmax=vmax)
add_sizebar(ax, 1, TEM_1[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 B2
subfig = fig.add_subfigure(gs[1, 3], zorder=0)
ax = subfig.add_axes((-0.05, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_1[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 C
subfig = fig.add_subfigure(gs[2, 0:2], zorder=0)
ax = subfig.add_axes((0.00, -0.15, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_2[2].axes_manager[-1].axis,
    y1=TEM_2[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[0],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 5.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-30, 45)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.1,
    1.0,
    r"C",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C1
subfig = fig.add_subfigure(gs[3, 0], zorder=0)
ax = subfig.add_axes((0.07, -0.16, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_2[1].data, [1.0, 99.0])
ax.imshow(TEM_2[1].data, cmap="gray", vmin=vmin, vmax=vmax)
add_sizebar(ax, 1, TEM_2[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 C2
subfig = fig.add_subfigure(gs[3, 1], zorder=0)
ax = subfig.add_axes((0.15, -0.16, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_2[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 D
subfig = fig.add_subfigure(gs[2, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, -0.15, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_3[2].axes_manager[-1].axis,
    y1=TEM_3[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[4],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 5.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-30, 45)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.1,
    1.0,
    r"D",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 D1
subfig = fig.add_subfigure(gs[3, 2], zorder=0)
ax = subfig.add_axes((-0.15, -0.15, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_3[1].data, [1.0, 99.0])
ax.imshow(TEM_3[1].data, cmap="gray", vmin=vmin, vmax=vmax)
add_sizebar(ax, 1, TEM_3[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 D2
subfig = fig.add_subfigure(gs[3, 3], zorder=0)
ax = subfig.add_axes((-0.07, -0.15, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_3[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_00_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_00_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_00_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_TEM_00_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

## TEM-EELS

### SETM-EELS Clustering

#### FD2 2ndFullDischarge

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def cluster(signal_list, key_word=(r"labels", r"ADF"), crop_box=None):
    cluster_data = {}

    for s in signal_list:
        try:
            title = str(s.metadata["General"]["title"])
        except Exception:
            title = ""

        match = re.search(r"\b(" + "|".join(key_word) + r")\b", title, re.IGNORECASE)
        # print(title, match)

        # 判断是否包含关键词
        if match:
            try:
                if crop_box:
                    s_crop = s.isig[crop_box].copy() if s.axes_manager.signal_dimension == 2 else s.inav[crop_box].copy()
            except Exception:
                s_crop = s.copy()
            cluster_data[f"{match.group(0)}"] = s_crop
        else:
            cluster_data["others"] = s

    return cluster_data

In [ ]:
# 读取 Clustering_Mn 的数据
path_HD2_clustering = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\EELS\Results\SI1_100pA_10ms_1.5nm"
)
HD2_clustering_haadf = hs.load(path_HD2_clustering.joinpath(r"Corrected SI", r"STEM SI processed.dm4"))
HD2_clustering_clusterings_total_mn = hs.load(path_HD2_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_Mn_all_N3_*.hspy"))
HD2_clustering_clusterings_total_o = hs.load(path_HD2_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_O_all_N3_*.hspy"))

In [ ]:
HD2_clustering_haadf = convert_units(HD2_clustering_haadf[1])
HD2_clustering_clusterings_total_mn = convert_units(HD2_clustering_clusterings_total_mn)
HD2_clustering_clusterings_total_o = convert_units(HD2_clustering_clusterings_total_o)

HD2_clustering_haadf = cluster(HD2_clustering_haadf, crop_box=(slice(None), slice(None)))
HD2_clustering_clusterings_total_mn = cluster(HD2_clustering_clusterings_total_mn, crop_box=(slice(None), slice(None)))
HD2_clustering_clusterings_total_o = cluster(HD2_clustering_clusterings_total_o, crop_box=(slice(None), slice(None)))

In [ ]:
energy_shift = 3.3  # eV
HD2_clustering_clusterings_total_mn["others"].axes_manager.signal_axes[0].offset += energy_shift
energy_shift = 2.5  # eV
HD2_clustering_clusterings_total_o["others"].axes_manager.signal_axes[0].offset += energy_shift

In [ ]:
# 画图
%matplotlib inline
plt.close("all")
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

ax.imshow(HD2_clustering_haadf["ADF"].data, cmap="gray", aspect=1.0)

scalebar = add_sizebar(
    ax,
    size=20,
    bardata=HD2_clustering_haadf["ADF"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(0.03, 0.90, r"HAADF", transform=ax.transAxes, fontsize=11, va="center", ha="left", fontfamily="Arial", color="w")
# ax.text(-0.03, 1.0, r'A', transform=ax.transAxes, fontsize=13, va='center', ha='right', fontfamily='Arial', fontweight='bold')

# 图 B
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.54, 0.25, 0.5, 0.5))

hs.plot.plot_images(
    [HD2_clustering_clusterings_total_mn["labels"].inav[i] for i in range(HD2_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[colors_opt[2], colors_opt[4], "k"],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HD2_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HD2_clustering_clusterings_total_mn['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"Mn", transform=ax.transAxes, color="k", fontsize=11, va="center", ha="left", fontfamily="Arial")

# 插入图 B1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [HD2_clustering_clusterings_total_mn["others"].inav[1], HD2_clustering_clusterings_total_mn["others"].inav[2]],
    ax=axins,
    style="overlap",
    color=[colors_opt[2], colors_opt[4]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=9, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=9, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(630, 660)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=9)

# 图 C
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.05, 0.25, 0.5, 0.5))
# ax.set_box_aspect(1.0)

hs.plot.plot_images(
    [HD2_clustering_clusterings_total_o["labels"].inav[i] for i in range(HD2_clustering_clusterings_total_o["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[colors_opt[2], colors_opt[4], r"k"],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HD2_clustering_clusterings_total_o["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HD2_clustering_clusterings_total_o['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"O", transform=ax.transAxes, color="k", fontsize=11, va="center", ha="left", fontfamily="Arial")

# 插入图 C1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [HD2_clustering_clusterings_total_o["others"].inav[0], HD2_clustering_clusterings_total_o["others"].inav[1]],
    ax=axins,
    style="overlap",
    color=[colors_opt[2], colors_opt[4]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=9, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=9, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(520, 550)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=9)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_08_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_08_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_08_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_08_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

#### HD1 2ndHalfDischarge -V1

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def cluster(signal_list, key_word=(r"labels", r"ADF"), crop_box=None):
    cluster_data = {}

    for s in signal_list:
        try:
            title = str(s.metadata["General"]["title"])
        except Exception:
            title = ""

        match = re.search(r"\b(" + "|".join(key_word) + r")\b", title, re.IGNORECASE)
        # print(title, match)

        # 判断是否包含关键词
        if match:
            try:
                if crop_box:
                    s_crop = s.isig[crop_box].copy() if s.axes_manager.signal_dimension == 2 else s.inav[crop_box].copy()
            except Exception:
                s_crop = s.copy()
            cluster_data[f"{match.group(0)}"] = s_crop
        else:
            cluster_data["others"] = s

    return cluster_data

In [ ]:
# 读取 Clustering_Mn 的数据
path_HD1_clustering = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\EELS\Results\SI3_CL320mm_0.9eV_25ms_1.5nm step_120pA"
)
HD1_clustering_haadf = hs.load(path_HD1_clustering.joinpath(r"Corrected SI", r"STEM SI processed.dm4"))
HD1_clustering_clusterings_total_mn = hs.load(path_HD1_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_Mn_all_N3_*.hspy"))
HD1_clustering_clusterings_total_o = hs.load(path_HD1_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_O_all_N3_*.hspy"))

In [ ]:
HD1_clustering_haadf = convert_units(HD1_clustering_haadf[1])
HD1_clustering_clusterings_total_mn = convert_units(HD1_clustering_clusterings_total_mn)
HD1_clustering_clusterings_total_o = convert_units(HD1_clustering_clusterings_total_o)

HD1_clustering_haadf = cluster(HD1_clustering_haadf, crop_box=(slice(None), slice(None)))
HD1_clustering_clusterings_total_mn = cluster(HD1_clustering_clusterings_total_mn, crop_box=(slice(None), slice(None)))
HD1_clustering_clusterings_total_o = cluster(HD1_clustering_clusterings_total_o, crop_box=(slice(None), slice(None)))

In [ ]:
energy_shift = 3.3  # eV
HD1_clustering_clusterings_total_mn["others"].axes_manager.signal_axes[0].offset += energy_shift
energy_shift = 2.5  # eV
HD1_clustering_clusterings_total_o["others"].axes_manager.signal_axes[0].offset += energy_shift

In [ ]:
# 画图
%matplotlib inline
plt.close("all")
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

ax.imshow(HD1_clustering_haadf["ADF"].data, cmap="gray", aspect=1.0)

scalebar = add_sizebar(
    ax,
    size=20,
    bardata=HD1_clustering_haadf["ADF"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(0.03, 0.92, r"HAADF", transform=ax.transAxes, fontsize=11, va="center", ha="left", fontfamily="Arial", color="w")
# ax.text(-0.03, 1.0, r'A', transform=ax.transAxes, fontsize=13, va='center', ha='right', fontfamily='Arial', fontweight='bold')

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.1, 0.38, 0.6, 0.6))

hs.plot.plot_images(
    [HD1_clustering_clusterings_total_mn["labels"].inav[i] for i in range(HD1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[r"k", colors_opt[4], colors_opt[2]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HD1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HD1_clustering_clusterings_total_mn['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"Mn", transform=ax.transAxes, color="k", fontsize=11, va="center", ha="left", fontfamily="Arial")

# 插入图 B1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [HD1_clustering_clusterings_total_mn["others"].inav[1], HD1_clustering_clusterings_total_mn["others"].inav[2]],
    ax=axins,
    style="overlap",
    color=[colors_opt[4], colors_opt[2]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=9, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=9, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(630, 660)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=9)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.20, 0.38, 0.6, 0.6))
# ax.set_box_aspect(1.0)

hs.plot.plot_images(
    [HD1_clustering_clusterings_total_o["labels"].inav[i] for i in range(HD1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[r"k", colors_opt[4], colors_opt[2]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HD1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HD1_clustering_clusterings_total_o['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"O", transform=ax.transAxes, color="k", fontsize=11, va="center", ha="left", fontfamily="Arial")

# 插入图 C1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [HD1_clustering_clusterings_total_o["others"].inav[1], HD1_clustering_clusterings_total_o["others"].inav[2]],
    ax=axins,
    style="overlap",
    color=[colors_opt[4], colors_opt[2]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=9, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=9, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(520, 550)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=9)


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_07_V1_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_07_V1_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_07_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_07_V1_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

#### HD1 2ndHalfDischarge

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def cluster(signal_list, key_word=(r"labels", r"ADF"), crop_box=None):
    cluster_data = {}

    for s in signal_list:
        try:
            title = str(s.metadata["General"]["title"])
        except Exception:
            title = ""

        match = re.search(r"\b(" + "|".join(key_word) + r")\b", title, re.IGNORECASE)
        # print(title, match)

        # 判断是否包含关键词
        if match:
            try:
                if crop_box:
                    s_crop = s.isig[crop_box].copy() if s.axes_manager.signal_dimension == 2 else s.inav[crop_box].copy()
            except Exception:
                s_crop = s.copy()
            cluster_data[f"{match.group(0)}"] = s_crop
        else:
            cluster_data["others"] = s

    return cluster_data

In [ ]:
# 读取 Clustering_Mn 的数据
path_HD1_clustering = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\EELS\Results\SI3_CL320mm_0.9eV_25ms_1.5nm step_120pA"
)
HD1_clustering_haadf = hs.load(path_HD1_clustering.joinpath(r"Corrected SI", r"STEM SI processed.dm4"))
HD1_clustering_clusterings_total_mn = hs.load(path_HD1_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_Mn_all_N3_*.hspy"))
HD1_clustering_clusterings_total_o = hs.load(path_HD1_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_O_all_N3_*.hspy"))

In [ ]:
HD1_clustering_haadf = convert_units(HD1_clustering_haadf[1])
HD1_clustering_clusterings_total_mn = convert_units(HD1_clustering_clusterings_total_mn)
HD1_clustering_clusterings_total_o = convert_units(HD1_clustering_clusterings_total_o)

HD1_clustering_haadf = cluster(HD1_clustering_haadf, crop_box=(slice(None), slice(None)))
HD1_clustering_clusterings_total_mn = cluster(HD1_clustering_clusterings_total_mn, crop_box=(slice(None), slice(None)))
HD1_clustering_clusterings_total_o = cluster(HD1_clustering_clusterings_total_o, crop_box=(slice(None), slice(None)))

In [ ]:
energy_shift = 3.5  # eV
HD1_clustering_clusterings_total_mn["others"].axes_manager.signal_axes[0].offset += energy_shift
energy_shift = 2.5  # eV
HD1_clustering_clusterings_total_o["others"].axes_manager.signal_axes[0].offset += energy_shift

In [ ]:
# 画图
%matplotlib inline
plt.close("all")
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

ax.imshow(HD1_clustering_haadf["ADF"].data, cmap="gray", aspect=1.0)

scalebar = add_sizebar(
    ax,
    size=20,
    bardata=HD1_clustering_haadf["ADF"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(0.03, 0.90, r"HAADF", transform=ax.transAxes, fontsize=11, va="center", ha="left", fontfamily="Arial", color="w")
# ax.text(-0.03, 1.0, r'A', transform=ax.transAxes, fontsize=13, va='center', ha='right', fontfamily='Arial', fontweight='bold')

# 图 B
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.63, 0.25, 0.6, 0.6))

hs.plot.plot_images(
    [HD1_clustering_clusterings_total_mn["labels"].inav[i] for i in range(HD1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[r"k", colors_opt[4], colors_opt[2]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HD1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HD1_clustering_clusterings_total_mn['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"Mn", transform=ax.transAxes, color="k", fontsize=11, va="center", ha="left", fontfamily="Arial")

# 插入图 B1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [HD1_clustering_clusterings_total_mn["others"].inav[1], HD1_clustering_clusterings_total_mn["others"].inav[2]],
    ax=axins,
    style="overlap",
    color=[colors_opt[4], colors_opt[2]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=9, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=9, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(630, 660)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=9)

# 图 C
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.05, 0.25, 0.6, 0.6))
# ax.set_box_aspect(1.0)

hs.plot.plot_images(
    [HD1_clustering_clusterings_total_o["labels"].inav[i] for i in range(HD1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[r"k", colors_opt[4], colors_opt[2]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HD1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HD1_clustering_clusterings_total_o['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"O", transform=ax.transAxes, color="k", fontsize=11, va="center", ha="left", fontfamily="Arial")

# 插入图 C1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [HD1_clustering_clusterings_total_o["others"].inav[1], HD1_clustering_clusterings_total_o["others"].inav[2]],
    ax=axins,
    style="overlap",
    color=[colors_opt[4], colors_opt[2]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=9, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=9, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(520, 550)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=9)


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_07_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_07_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_07_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_07_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

#### FC1 1stCharge

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def cluster(signal_list, key_word=(r"labels", r"ADF"), crop_box=None):
    cluster_data = {}

    for s in signal_list:
        try:
            title = str(s.metadata["General"]["title"])
        except Exception:
            title = ""

        match = re.search(r"\b(" + "|".join(key_word) + r")\b", title, re.IGNORECASE)
        # print(title, match)

        # 判断是否包含关键词
        if match:
            try:
                if crop_box:
                    s_crop = s.isig[crop_box].copy() if s.axes_manager.signal_dimension == 2 else s.inav[crop_box].copy()
            except Exception:
                s_crop = s.copy()
            cluster_data[f"{match.group(0)}"] = s_crop
        else:
            cluster_data["others"] = s

    return cluster_data

In [ ]:
# 读取 Clustering_Mn 的数据
path_FC1_clustering = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\EELS\Results\SI2_EFTEM400m_5mm_50ms_200pA_5nm"
)
FC1_clustering_haadf = hs.load(path_FC1_clustering.joinpath(r"Corrected SI", r"STEM SI processed.dm4"))
FC1_clustering_clusterings_total_mn = hs.load(path_FC1_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_Mn_all_N3_*.hspy"))
FC1_clustering_clusterings_total_o = hs.load(path_FC1_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_O_all_N3_*.hspy"))

In [ ]:
FC1_clustering_haadf = convert_units(FC1_clustering_haadf[1])
FC1_clustering_clusterings_total_mn = convert_units(FC1_clustering_clusterings_total_mn)
FC1_clustering_clusterings_total_o = convert_units(FC1_clustering_clusterings_total_o)

FC1_clustering_haadf = cluster(FC1_clustering_haadf, crop_box=(slice(None), slice(0, 70)))
FC1_clustering_clusterings_total_mn = cluster(FC1_clustering_clusterings_total_mn, crop_box=(slice(None), slice(0, 70)))
FC1_clustering_clusterings_total_o = cluster(FC1_clustering_clusterings_total_o, crop_box=(slice(None), slice(0, 70)))

In [ ]:
# 画图
%matplotlib inline
plt.close("all")
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

ax.imshow(FC1_clustering_haadf["ADF"].data.T, cmap="gray", aspect=1.0)

scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FC1_clustering_haadf["ADF"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(0.03, 0.95, r"HAADF", transform=ax.transAxes, fontsize=11, va="center", ha="left", fontfamily="Arial", color="w")
# ax.text(-0.03, 1.0, r'A', transform=ax.transAxes, fontsize=13, va='center', ha='right', fontfamily='Arial', fontweight='bold')

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.05, 0.42, 0.7, 0.7))
# ax.set_box_aspect(1.0)

hs.plot.plot_images(
    [FC1_clustering_clusterings_total_mn["labels"].inav[i] for i in range(FC1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[colors_opt[0], r"k", colors_opt[2]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * FC1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=FC1_clustering_clusterings_total_mn['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"Mn", transform=ax.transAxes, color="k", fontsize=11, va="center", ha="left", fontfamily="Arial")

# 插入图 B1
axins = ax.inset_axes((0.0, -1.18, 1.0, 1.1), zorder=1)

hs.plot.plot_spectra(
    [FC1_clustering_clusterings_total_mn["others"].inav[0], FC1_clustering_clusterings_total_mn["others"].inav[2]],
    ax=axins,
    style="overlap",
    color=[colors_opt[0], colors_opt[2]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=9, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=9, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(630, 660)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=9)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.2, 0.42, 0.7, 0.7))
# ax.set_box_aspect(1.0)

hs.plot.plot_images(
    [FC1_clustering_clusterings_total_o["labels"].inav[i] for i in range(FC1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[r"k", colors_opt[2], colors_opt[0]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * FC1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=FC1_clustering_clusterings_total_o['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"O", transform=ax.transAxes, color="k", fontsize=11, va="center", ha="left", fontfamily="Arial")

# 插入图 C1
axins = ax.inset_axes((0.0, -1.18, 1.0, 1.1), zorder=1)

hs.plot.plot_spectra(
    [FC1_clustering_clusterings_total_o["others"].inav[1], FC1_clustering_clusterings_total_o["others"].inav[2]],
    ax=axins,
    style="overlap",
    color=[colors_opt[2], colors_opt[0]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=9, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=9, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(520, 550)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=9)


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_06_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_06_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_06_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_06_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

#### HC#2 1sthalfcharge

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def cluster(signal_list, key_word=(r"labels", r"ADF"), crop_box=None):
    cluster_data = {}

    for s in signal_list:
        try:
            title = str(s.metadata["General"]["title"])
        except Exception:
            title = ""

        match = re.search(r"\b(" + "|".join(key_word) + r")\b", title, re.IGNORECASE)
        # print(title, match)

        # 判断是否包含关键词
        if match:
            try:
                if crop_box:
                    s_crop = s.isig[crop_box].copy() if s.axes_manager.signal_dimension == 2 else s.inav[crop_box].copy()
            except Exception:
                s_crop = s.copy()
            cluster_data[f"{match.group(0)}"] = s_crop
        else:
            cluster_data["others"] = s

    return cluster_data

In [ ]:
# 读取 Clustering_Mn 的数据
path_1sthalfcharge2_clustering = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EELS\Results\SI 1_630mm_5mm_0.9eVCh_3nm_15ms_150pA"
)
HC2_clustering_haadf = hs.load(path_1sthalfcharge2_clustering.joinpath(r"Corrected SI", r"STEM SI processed.dm4"))
HC2_clustering_clusterings_total_mn = hs.load(path_1sthalfcharge2_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_Mn_all_N3_*.hspy"))
HC2_clustering_clusterings_total_o = hs.load(path_1sthalfcharge2_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_O_all_N3_*.hspy"))

In [ ]:
HC2_clustering_haadf = convert_units(HC2_clustering_haadf[1])
HC2_clustering_clusterings_total_mn = convert_units(HC2_clustering_clusterings_total_mn)
HC2_clustering_clusterings_total_o = convert_units(HC2_clustering_clusterings_total_o)

HC2_clustering_haadf = cluster(HC2_clustering_haadf, crop_box=(slice(None), slice(None)))
HC2_clustering_clusterings_total_mn = cluster(HC2_clustering_clusterings_total_mn, crop_box=(slice(None), slice(None)))
HC2_clustering_clusterings_total_o = cluster(HC2_clustering_clusterings_total_o, crop_box=(slice(None), slice(None)))

In [ ]:
# 画图
%matplotlib inline
plt.close("all")
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

ax.imshow(HC2_clustering_haadf["ADF"].data, cmap="gray", aspect=1.0)

scalebar = add_sizebar(
    ax,
    size=50,
    bardata=HC2_clustering_haadf["ADF"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.02, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(0.03, 0.90, r"HAADF", transform=ax.transAxes, fontsize=11, va="center", ha="left", fontfamily="Arial", color="w")
# ax.text(-0.03, 1.0, r'A', transform=ax.transAxes, fontsize=13, va='center', ha='right', fontfamily='Arial', fontweight='bold')

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.45, 0.45, 0.55, 0.55))

hs.plot.plot_images(
    [HC2_clustering_clusterings_total_mn["labels"].inav[i] for i in range(HC2_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[r"k", colors_opt[2], colors_opt[4]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HC2_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HC2_clustering_clusterings_total_mn['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.05, r"Mn", transform=ax.transAxes, color="k", fontsize=11, va="center", ha="left", fontfamily="Arial")

# 插入图 B1
axins = ax.inset_axes((0.0, -0.8, 1.0, 0.75), zorder=1)

hs.plot.plot_spectra(
    [HC2_clustering_clusterings_total_mn["others"].inav[1], HC2_clustering_clusterings_total_mn["others"].inav[2]],
    ax=axins,
    style="overlap",
    color=[colors_opt[2], colors_opt[4]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=9, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=9, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(630, 660)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=9)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.45, 0.55, 0.55))
# ax.set_box_aspect(1.0)

hs.plot.plot_images(
    [HC2_clustering_clusterings_total_o["labels"].inav[i] for i in range(HC2_clustering_clusterings_total_o["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[colors_opt[4], r"k", colors_opt[2]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HC2_clustering_clusterings_total_o["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HC2_clustering_clusterings_total_o['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.05, r"O", transform=ax.transAxes, color="k", fontsize=11, va="center", ha="left", fontfamily="Arial")

# 插入图 C1
axins = ax.inset_axes((0.0, -0.8, 1.0, 0.75), zorder=1)

hs.plot.plot_spectra(
    [HC2_clustering_clusterings_total_o["others"].inav[0], HC2_clustering_clusterings_total_o["others"].inav[2]],
    ax=axins,
    style="overlap",
    color=[colors_opt[4], colors_opt[2]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=9, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=9, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(520, 550)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=9)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_05_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_05_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_05_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_05_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

#### HC#1 1sthalfcharge -V1

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def cluster(signal_list, key_word=(r"labels", r"ADF"), crop_box=None):
    cluster_data = {}

    for s in signal_list:
        try:
            title = str(s.metadata["General"]["title"])
        except Exception:
            title = ""

        match = re.search(r"\b(" + "|".join(key_word) + r")\b", title, re.IGNORECASE)
        # print(title, match)

        # 判断是否包含关键词
        if match:
            try:
                if crop_box:
                    s_crop = s.isig[crop_box].copy() if s.axes_manager.signal_dimension == 2 else s.inav[crop_box].copy()
            except Exception:
                s_crop = s.copy()
            cluster_data[f"{match.group(0)}"] = s_crop
        else:
            cluster_data["others"] = s

    return cluster_data

In [ ]:
# 读取 Clustering_Mn 的数据
path_1sthalfcharge_clustering = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EELS\Results\SI 2_630mm_5mm_0.9eVCh_0.9nm_15ms_150pA"
)
HC1_clustering_haadf = hs.load(path_1sthalfcharge_clustering.joinpath(r"Corrected SI", r"STEM SI processed.dm4"))
HC1_clustering_clusterings_total_mn = hs.load(path_1sthalfcharge_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_Mn_all_N4_*.hspy"))
HC1_clustering_clusterings_total_o = hs.load(path_1sthalfcharge_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_O_all_N4_*.hspy"))

In [ ]:
HC1_clustering_haadf = convert_units(HC1_clustering_haadf[1])
HC1_clustering_clusterings_total_mn = convert_units(HC1_clustering_clusterings_total_mn)
HC1_clustering_clusterings_total_o = convert_units(HC1_clustering_clusterings_total_o)

HC1_clustering_haadf = cluster(HC1_clustering_haadf, crop_box=(slice(None), slice(None)))
HC1_clustering_clusterings_total_mn = cluster(HC1_clustering_clusterings_total_mn, crop_box=(slice(None), slice(None)))
HC1_clustering_clusterings_total_o = cluster(HC1_clustering_clusterings_total_o, crop_box=(slice(None), slice(None)))

In [ ]:
# 画图
%matplotlib inline
plt.close("all")
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

ax.imshow(HC1_clustering_haadf["ADF"].data, cmap="gray", aspect=1.0)

scalebar = add_sizebar(
    ax,
    size=20,
    bardata=HC1_clustering_haadf["ADF"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(0.03, 0.90, r"HAADF", transform=ax.transAxes, fontsize=11, va="center", ha="left", fontfamily="Arial", color="w")
# ax.text(-0.03, 1.0, r'A', transform=ax.transAxes, fontsize=13, va='center', ha='right', fontfamily='Arial', fontweight='bold')

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.05, 0.41, 0.5, 0.5))

hs.plot.plot_images(
    [HC1_clustering_clusterings_total_mn["labels"].inav[i] for i in range(HC1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[colors_opt[2], colors_opt[4], r"k", colors_opt[6]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HC1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HC1_clustering_clusterings_total_mn['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"Mn", transform=ax.transAxes, color="k", fontsize=11, va="center", ha="left", fontfamily="Arial")

# 插入图 B1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [HC1_clustering_clusterings_total_mn["others"].inav[0], HC1_clustering_clusterings_total_mn["others"].inav[1], HC1_clustering_clusterings_total_mn["others"].inav[3]],
    ax=axins,
    style="overlap",
    color=[colors_opt[2], colors_opt[4], colors_opt[6]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=9, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=9, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(630, 660)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=9)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.35, 0.41, 0.5, 0.5))
# ax.set_box_aspect(1.0)

hs.plot.plot_images(
    [HC1_clustering_clusterings_total_o["labels"].inav[i] for i in range(HC1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[colors_opt[2], colors_opt[4], r"k", colors_opt[6]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HC1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HC1_clustering_clusterings_total_o['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"O", transform=ax.transAxes, color="b", fontsize=11, va="center", ha="left", fontfamily="Arial")

# 插入图 C1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [HC1_clustering_clusterings_total_o["others"].inav[0], HC1_clustering_clusterings_total_o["others"].inav[1], HC1_clustering_clusterings_total_o["others"].inav[3]],
    ax=axins,
    style="overlap",
    color=[colors_opt[2], colors_opt[4], colors_opt[6]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=9, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=9, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(520, 550)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=9)


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_04_V1_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_04_V1_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_04_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_04_V1_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

#### HC#1 1sthalfcharge

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def cluster(signal_list, key_word=(r"labels", r"ADF"), crop_box=None):
    cluster_data = {}

    for s in signal_list:
        try:
            title = str(s.metadata["General"]["title"])
        except Exception:
            title = ""

        match = re.search(r"\b(" + "|".join(key_word) + r")\b", title, re.IGNORECASE)
        # print(title, match)

        # 判断是否包含关键词
        if match:
            try:
                if crop_box:
                    s_crop = s.isig[crop_box].copy() if s.axes_manager.signal_dimension == 2 else s.inav[crop_box].copy()
            except Exception:
                s_crop = s.copy()
            cluster_data[f"{match.group(0)}"] = s_crop
        else:
            cluster_data["others"] = s

    return cluster_data

In [ ]:
# 读取 Clustering_Mn 的数据
path_1sthalfcharge_clustering = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EELS\Results\SI 2_630mm_5mm_0.9eVCh_0.9nm_15ms_150pA"
)
HC1_clustering_haadf = hs.load(path_1sthalfcharge_clustering.joinpath(r"Corrected SI", r"STEM SI processed.dm4"))
HC1_clustering_clusterings_total_mn = hs.load(path_1sthalfcharge_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_Mn_all_N4_*.hspy"))
HC1_clustering_clusterings_total_o = hs.load(path_1sthalfcharge_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_O_all_N4_*.hspy"))

In [ ]:
HC1_clustering_haadf = convert_units(HC1_clustering_haadf[1])
HC1_clustering_clusterings_total_mn = convert_units(HC1_clustering_clusterings_total_mn)
HC1_clustering_clusterings_total_o = convert_units(HC1_clustering_clusterings_total_o)

HC1_clustering_haadf = cluster(HC1_clustering_haadf, crop_box=(slice(None), slice(None)))
HC1_clustering_clusterings_total_mn = cluster(HC1_clustering_clusterings_total_mn, crop_box=(slice(None), slice(None)))
HC1_clustering_clusterings_total_o = cluster(HC1_clustering_clusterings_total_o, crop_box=(slice(None), slice(None)))

In [ ]:
# 画图
%matplotlib inline
plt.close("all")
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

ax.imshow(HC1_clustering_haadf["ADF"].data, cmap="gray", aspect=1.0)

scalebar = add_sizebar(
    ax,
    size=20,
    bardata=HC1_clustering_haadf["ADF"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(0.03, 0.90, r"HAADF", transform=ax.transAxes, fontsize=11, va="center", ha="left", fontfamily="Arial", color="w")
# ax.text(-0.03, 1.0, r'A', transform=ax.transAxes, fontsize=13, va='center', ha='right', fontfamily='Arial', fontweight='bold')

# 图 B
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.62, 0.28, 0.5, 0.5))

hs.plot.plot_images(
    [HC1_clustering_clusterings_total_mn["labels"].inav[i] for i in range(HC1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[colors_opt[2], colors_opt[4], r"k", colors_opt[6]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HC1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HC1_clustering_clusterings_total_mn['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"Mn", transform=ax.transAxes, color="k", fontsize=11, va="center", ha="left", fontfamily="Arial")

# 插入图 B1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [HC1_clustering_clusterings_total_mn["others"].inav[0], HC1_clustering_clusterings_total_mn["others"].inav[1], HC1_clustering_clusterings_total_mn["others"].inav[3]],
    ax=axins,
    style="overlap",
    color=[colors_opt[2], colors_opt[4], colors_opt[6]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=9, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=9, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(630, 660)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=9)

# 图 C
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.05, 0.28, 0.5, 0.5))
# ax.set_box_aspect(1.0)

hs.plot.plot_images(
    [HC1_clustering_clusterings_total_o["labels"].inav[i] for i in range(HC1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[colors_opt[2], colors_opt[4], r"k", colors_opt[6]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HC1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HC1_clustering_clusterings_total_o['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"O", transform=ax.transAxes, color="b", fontsize=11, va="center", ha="left", fontfamily="Arial")

# 插入图 C1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [HC1_clustering_clusterings_total_o["others"].inav[0], HC1_clustering_clusterings_total_o["others"].inav[1], HC1_clustering_clusterings_total_o["others"].inav[3]],
    ax=axins,
    style="overlap",
    color=[colors_opt[2], colors_opt[4], colors_opt[6]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=9, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=9, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(520, 550)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=9)


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_04_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_04_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_04_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_04_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

#### FD1 1stDischarge

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def cluster(signal_list, key_word=(r"labels", r"ADF"), crop_box=None):
    cluster_data = {}

    for s in signal_list:
        try:
            title = str(s.metadata["General"]["title"])
        except Exception:
            title = ""

        match = re.search(r"\b(" + "|".join(key_word) + r")\b", title, re.IGNORECASE)
        # print(title, match)

        # 判断是否包含关键词
        if match:
            try:
                if crop_box:
                    s_crop = s.isig[crop_box].copy() if s.axes_manager.signal_dimension == 2 else s.inav[crop_box].copy()
            except Exception:
                s_crop = s.copy()
            cluster_data[f"{match.group(0)}"] = s_crop
        else:
            cluster_data["others"] = s

    return cluster_data

In [ ]:
# 读取 Clustering_Mn 的数据
path_1stDischarge_clustering = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\EELS\Results\SI2_80pA_10ms"
)
FD1_clustering_haadf = hs.load(path_1stDischarge_clustering.joinpath(r"Corrected SI", r"STEM SI processed.dm4"))
FD1_clustering_clusterings_total_mn = hs.load(path_1stDischarge_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_Mn_all_N4_*.hspy"))
FD1_clustering_clusterings_total_o = hs.load(path_1stDischarge_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_O_all_N4_*.hspy"))

In [ ]:
FD1_clustering_haadf = convert_units(FD1_clustering_haadf[1])
FD1_clustering_clusterings_total_mn = convert_units(FD1_clustering_clusterings_total_mn)
FD1_clustering_clusterings_total_o = convert_units(FD1_clustering_clusterings_total_o)

FD1_clustering_haadf = cluster(FD1_clustering_haadf, crop_box=(slice(None), slice(None)))
FD1_clustering_clusterings_total_mn = cluster(FD1_clustering_clusterings_total_mn, crop_box=(slice(None), slice(None)))
FD1_clustering_clusterings_total_o = cluster(FD1_clustering_clusterings_total_o, crop_box=(slice(None), slice(None)))

In [ ]:
# 画图
%matplotlib inline
plt.close("all")
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

ax.imshow(FD1_clustering_haadf["ADF"].data, cmap="gray", aspect=1.0)

scalebar = add_sizebar(
    ax,
    size=20,
    bardata=FD1_clustering_haadf["ADF"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(0.03, 0.90, r"HAADF", transform=ax.transAxes, fontsize=11, va="center", ha="left", fontfamily="Arial", color="w")
# ax.text(-0.03, 1.0, r'A', transform=ax.transAxes, fontsize=13, va='center', ha='right', fontfamily='Arial', fontweight='bold')

# 图 B
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.25, 0.25, 0.7, 0.7))

hs.plot.plot_images(
    [FD1_clustering_clusterings_total_mn["labels"].inav[i] for i in range(FD1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[r"k", colors_opt[2], colors_opt[4], colors_opt[6]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * FD1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=FD1_clustering_clusterings_total_mn['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"Mn", transform=ax.transAxes, color="k", fontsize=13, va="center", ha="left", fontfamily="Arial")

# 插入图 B1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [FD1_clustering_clusterings_total_mn["others"].inav[0], FD1_clustering_clusterings_total_mn["others"].inav[2], FD1_clustering_clusterings_total_mn["others"].inav[3]],
    ax=axins,
    style="overlap",
    color=[colors_opt[2], colors_opt[4], colors_opt[6]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=9, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=9, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(630, 660)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=9)

# 图 C
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.04, 0.25, 0.7, 0.7))

hs.plot.plot_images(
    [FD1_clustering_clusterings_total_o["labels"].inav[i] for i in range(FD1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[colors_opt[2], colors_opt[4], r"k", colors_opt[6]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * FD1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=FD1_clustering_clusterings_total_o['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"O", transform=ax.transAxes, color="k", fontsize=13, va="center", ha="left", fontfamily="Arial")

# 插入图 C1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [FD1_clustering_clusterings_total_o["others"].inav[0], FD1_clustering_clusterings_total_o["others"].inav[1], FD1_clustering_clusterings_total_o["others"].inav[3]],
    ax=axins,
    style="overlap",
    color=[colors_opt[2], colors_opt[4], colors_opt[6]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=9, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=9, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(520, 550)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=9)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_03_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_03_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_03_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_03_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### TEM-EELS 的各个状态 + 新相: 谱线

In [ ]:
# 读取 STEM-EELS 谱线的数据
path_eels = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\TEM-EDS-EELS-all")

TEM_EELS_Mn = pd.read_csv(path_eels.joinpath(r"TEM-EELS_Mn_selected.nor"), sep=r"\s+", header=None, index_col=None, comment=r"#")
TEM_EELS_O = pd.read_csv(path_eels.joinpath(r"TEM-EELS_O_selected.nor"), sep=r"\s+", header=None, index_col=None, comment=r"#")

TEM_EELS_Mn = {
    "energy": TEM_EELS_Mn.iloc[:, 0].copy(),
    "bulk": TEM_EELS_Mn.iloc[:, [1, 3, 7, 9, 12, 19, 21]].copy(),
    "surface": TEM_EELS_Mn.iloc[:, [2, 6, 8, 13, 18, 20]].copy(),
    "new_phase": TEM_EELS_Mn.iloc[:, [4, 5, 10, 11, 14, 15, 16, 17]].copy(),
}
TEM_EELS_O = {
    "energy": TEM_EELS_O.iloc[:, 0].copy(),
    "bulk": TEM_EELS_O.iloc[:, [1, 4, 8, 9, 13, 20, 22]].copy(),
    "surface": TEM_EELS_O.iloc[:, [3, 7, 10, 14, 19, 21]].copy(),
    "new_phase": TEM_EELS_O.iloc[:, [5, 6, 11, 12, 15, 16, 17, 18]].copy(),
    "ZSH": TEM_EELS_O.iloc[:, 2].copy(),
}

In [ ]:
# 计算 TEM_EELS_Mn 各相第一个峰值的1/10强度位置的index
import numpy as np
from scipy.signal import find_peaks


def find_first_peak_tenth_intensity_index(energy, intensity):  # noqa: ARG001
    """
    找到第一个峰值的1/10强度位置的index
    """
    # 找到峰值
    peaks, _ = find_peaks(intensity, height=intensity.max() * 0.1, prominence=intensity.max() * 0.05)

    if len(peaks) == 0:
        return None

    # 第一个峰值
    first_peak_idx = peaks[0]
    first_peak_intensity = intensity.iloc[first_peak_idx]

    # 计算1/10强度值
    tenth_intensity = first_peak_intensity / 10

    # 在第一个峰值左侧寻找1/10强度位置
    # 从峰值向左找到强度降到1/10的位置
    for i in range(first_peak_idx, -1, -1):
        if intensity.iloc[i] <= tenth_intensity:
            return i

    return 0  # 如果没找到，返回起始位置


# 对每个相的所有列计算第一个峰值的1/10强度位置
phases = ["bulk", "surface", "new_phase"]
results = {}

for phase in phases:
    phase_data = TEM_EELS_Mn[phase]
    energy = TEM_EELS_Mn["energy"]

    phase_results = {}
    for col_idx in range(phase_data.shape[1]):
        intensity = phase_data.iloc[:, col_idx]
        tenth_idx = find_first_peak_tenth_intensity_index(energy, intensity)
        phase_results[f"col_{col_idx}"] = {
            "index": tenth_idx,
            "energy": energy.iloc[tenth_idx] if tenth_idx is not None else None,
            "intensity": intensity.iloc[tenth_idx] if tenth_idx is not None else None,
        }

    results[phase] = phase_results

# 显示结果
print("TEM_EELS_Mn 各相第一个峰值的1/10强度位置:")
print("=" * 60)

for phase in phases:
    print(f"\n{phase.upper()}:")
    for col_key, result in results[phase].items():
        print(f"  {col_key}: index={result['index']}, energy={result['energy']:.2f} eV, intensity={result['intensity']:.4f}")

In [ ]:
# 从 TEM-EELS.ipynb 中移植 L2/L3 比例计算函数
from lmfit.models import LinearModel, StepModel
from scipy.integrate import simpson


def index_of(arr, threshold):
    """返回数组中第一个大于 threshold 的索引。"""
    return np.argmax(arr > threshold)


def auto_select_two_largest_peaks(data, distance=10):
    """自动选择两个主峰"""
    peaks, _ = find_peaks(data, distance=distance)
    if len(peaks) < 2:
        raise ValueError("找不到两个峰")
    peak_values = data[peaks]
    top_peaks = peaks[np.argsort(peak_values)[-2:]]
    return np.sort(top_peaks)


def find_onset_energy(energy, data, thread):  # noqa: ARG001
    """找到峰的起始能量"""
    imax = auto_select_two_largest_peaks(data)
    imin = find_peaks(-data, distance=10)[0]
    imin = [next((v for v in imin if v > peak), None) for peak in imax]

    if None in imin:
        raise ValueError("未能找到合适的峰后谷")

    ionset = [index_of(data[: imax[0]], thread * data[imax[0]]), index_of(data[imin[0] : imax[1]], data[imin[0]] + thread * data[imax[1]]) + imin[0]]

    return ionset, imin, imax


def baseline_fitting(energy, data, ionset, imax, imin):
    """进行基线拟合并计算L2/L3比例"""
    height = list(data[imin])
    xdat = np.concatenate((energy[: ionset[0] - 10], energy[imin[1] - 5 : imin[1] + 5]))
    ydat = np.concatenate((data[: ionset[0] - 10], data[imin[1] - 5 : imin[1] + 5]))

    line_mod = LinearModel(prefix="line_")
    step_mod1 = StepModel(form="arctan", prefix="step1_")
    step_mod2 = StepModel(form="arctan", prefix="step2_")

    params = line_mod.make_params(intercept=np.mean(data[: ionset[0] - 10]), slope={"value": 0})
    params.update(step_mod1.make_params(center={"value": energy[ionset[0]], "vary": False}, sigma={"value": 0.5, "expr": "1*step2_sigma"}, amplitude={"expr": "2*step2_amplitude"}))
    params.update(
        step_mod2.make_params(
            center={"value": energy[ionset[1]], "vary": False},
            sigma={"value": 0.5, "min": 0.1, "max": 1.0},
            amplitude={"value": height[1] / 3, "vary": True, "expr": f"({height[1]}-line_slope*step2_center-line_intercept)/3"},
        )
    )

    model = line_mod + step_mod1 + step_mod2
    result = model.fit(ydat, params, x=xdat)
    baseline = result.eval(result.params, x=energy)
    peaks = data - baseline

    # 计算L3和L2峰的积分面积
    area_L3 = energy_window_integral(energy, peaks, energy[imax[0]], width_ev=2.0)  # noqa: N806
    area_L2 = energy_window_integral(energy, peaks, energy[imax[1]], width_ev=2.0)  # noqa: N806
    peak_intensities = [area_L3, area_L2]

    # 计算L3/L2比例
    ratio = peak_intensities[0] / peak_intensities[1]

    return baseline, peaks, peak_intensities, ratio, result


def energy_window_integral(energy, data, peak, width_ev):
    """在指定能量窗口内进行积分"""
    mask = (energy >= peak - width_ev / 2) & (energy <= peak + width_ev / 2)
    if not np.any(mask):
        raise ValueError(f"在能量 {peak:.2f} 附近找不到足够点进行积分")
    return simpson(y=data[mask], x=energy[mask])


def calculate_L3_L2_ratio(energy, intensity, threshold=0.1):  # noqa: N802
    """计算单条谱线的L3/L2比例"""
    try:
        ionset, imin, imax = find_onset_energy(energy, intensity, threshold)
        baseline_fit, bkg_removed, peak_intensities, ratio, _ = baseline_fitting(energy, intensity, ionset, imax, imin)

        return {
            "success": True,
            "L3_intensity": peak_intensities[0],
            "L2_intensity": peak_intensities[1],
            "L3_L2_ratio": ratio,
            "L3_energy": energy[ionset[0]],
            "L2_energy": energy[ionset[1]],
            "baseline": baseline_fit,
            "bkg_removed": bkg_removed,
        }
    except Exception as e:
        return {"success": False, "error": str(e), "L3_intensity": np.nan, "L2_intensity": np.nan, "L3_L2_ratio": np.nan, "L3_energy": np.nan, "L2_energy": np.nan}

In [ ]:
# 计算 TEM_EELS_Mn 中每条谱线的 L3/L2 比例
print("计算 TEM_EELS_Mn 各相每条谱线的 L3/L2 比例...")
print("=" * 70)

# 存储所有结果
L3_L2_results = {}

for phase in phases:
    print(f"\n处理 {phase.upper()} 相:")
    phase_data = TEM_EELS_Mn[phase]
    energy = TEM_EELS_Mn["energy"].values  # 转换为numpy数组

    phase_results = {}

    for col_idx in range(phase_data.shape[1]):
        intensity = phase_data.iloc[:, col_idx].values  # 转换为numpy数组
        col_name = f"col_{col_idx}"

        # 计算L3/L2比例
        result = calculate_L3_L2_ratio(energy, intensity, threshold=0.1)
        phase_results[col_name] = result

        # 显示结果
        if result["success"]:
            print(f"  {col_name}: L3/L2 = {result['L3_L2_ratio']:.3f}, L3强度 = {result['L3_intensity']:.3f}, L2强度 = {result['L2_intensity']:.3f}")
        else:
            print(f"  {col_name}: 计算失败 - {result['error']}")

    L3_L2_results[phase] = phase_results

# 创建汇总表
print("\n" + "=" * 70)
print("L3/L2 比例汇总表:")
print("=" * 70)

summary_ratios = {}
for phase in phases:
    ratios = []
    for col_key, result in L3_L2_results[phase].items():
        if result["success"]:
            ratios.append(result["L3_L2_ratio"])
        else:
            ratios.append(np.nan)

    summary_ratios[phase] = ratios
    valid_ratios = [r for r in ratios if not np.isnan(r)]

    if valid_ratios:
        mean_ratio = np.mean(valid_ratios)
        std_ratio = np.std(valid_ratios)
        print(f"{phase.upper():10}: 平均值 = {mean_ratio:.3f} ± {std_ratio:.3f}, 个数 = {len(valid_ratios)}/{len(ratios)}")
        print(f"           所有值: {[f'{r:.3f}' if not np.isnan(r) else 'NaN' for r in ratios]}")
    else:
        print(f"{phase.upper():10}: 无有效数据")

# 存储结果供后续使用
TEM_EELS_Mn_L3L2_ratios = summary_ratios

In [ ]:
# 删除多余的 1stFD
TEM_EELS_Mn["bulk"].drop(columns=TEM_EELS_Mn["bulk"].columns[1], inplace=True)
TEM_EELS_Mn["surface"].drop(columns=TEM_EELS_Mn["surface"].columns[1], inplace=True)
TEM_EELS_O["bulk"].drop(columns=TEM_EELS_O["bulk"].columns[1], inplace=True)
TEM_EELS_O["surface"].drop(columns=TEM_EELS_O["surface"].columns[1], inplace=True)

In [ ]:
colors_opt2 = [colors_opt2[0], colors_opt2[2], colors_opt2[3], colors_opt2[4], colors_opt2[5], colors_opt2[6]]

In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(2.0)

labels = [r"Pristine", r"HC#A", r"HC#B", r"FC#C", r"HD#D", r"FD#E"]
# labels = [r'Pristine', r'FD1st', r'HC#A', r'HC#B', r'FC#C', r'HD#D', r'FD#E']
shift_energy = 3.3
for i in range(TEM_EELS_Mn["bulk"].shape[1]):
    if i < 4:
        ax.plot(TEM_EELS_Mn["energy"], TEM_EELS_Mn["bulk"].iloc[:, i] + 0.4 * i, label=labels[i], color=colors_opt2[i])
    else:
        ax.plot(TEM_EELS_Mn["energy"] + shift_energy, TEM_EELS_Mn["bulk"].iloc[:, i] + 0.4 * i, label=labels[i], color=colors_opt2[i])

for i in range(0, TEM_EELS_Mn["surface"].shape[1]):
    if i < 3:
        ax.plot(TEM_EELS_Mn["energy"], TEM_EELS_Mn["surface"].iloc[:, i] + 0.4 * i + 5, linestyle="-", label=None, color=colors_opt2[i + 1])
    else:
        ax.plot(TEM_EELS_Mn["energy"] + shift_energy, TEM_EELS_Mn["surface"].iloc[:, i] + 0.4 * i + 5, linestyle="-", label=None, color=colors_opt2[i + 1])

for i in range(0, TEM_EELS_Mn["new_phase"].shape[1] // 2):
    if i < 3:
        ax.plot(TEM_EELS_Mn["energy"], TEM_EELS_Mn["new_phase"].iloc[:, 2 * i] + 1.0 * i + 10, linestyle="-", label=None, color=colors_opt2[i + 2])
        ax.plot(TEM_EELS_Mn["energy"], TEM_EELS_Mn["new_phase"].iloc[:, 2 * i + 1] + 1.0 * i + 10, linestyle="-", label=None, color=colors_opt2[i + 2])
    else:
        ax.plot(TEM_EELS_Mn["energy"] + shift_energy, TEM_EELS_Mn["new_phase"].iloc[:, 2 * i] + 1.0 * i + 10, linestyle="-", label=None, color=colors_opt2[i + 2])
        ax.plot(TEM_EELS_Mn["energy"] + shift_energy, TEM_EELS_Mn["new_phase"].iloc[:, 2 * i + 1] + 1.0 * i + 10, linestyle="-", label=None, color=colors_opt2[i + 2])

# 刻度线
ax.set_xlim(630, 660)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.5, offset=0))
ax.set_ylim(-0.1, 18)
ax.set_ylabel(r"Relative Absorption (arb.u.)", fontsize=11)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    left=True,
    labelbottom=True,
    labelleft=True,
)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.0, 1.01),
    ncols=2,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    columnspacing=0.5,
)
ax.text(0.65, 0.25, r"Bulk", transform=ax.transAxes, fontsize=11, va="center", ha="left", fontfamily="Arial")
ax.text(0.65, 0.53, r"Surface", transform=ax.transAxes, fontsize=11, va="center", ha="left", fontfamily="Arial")
ax.text(0.65, 0.85, r"New Phase", transform=ax.transAxes, fontsize=11, va="center", ha="left", fontfamily="Arial")
ax.text(0.86, 0.95, r"Mn", transform=ax.transAxes, fontsize=13, va="center", ha="left", fontfamily="Arial", fontweight="bold")
ax.vlines(642.5, -0.1, 16, colors="gray", linestyles="dashed", linewidth=1.0, zorder=1)

ax.text(-0.10, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.1, 0, 1.0, 1.0))
ax.set_box_aspect(2.0)

labels = [r"Pristine", r"HC#A", r"HC#B", r"FC#C", r"HD#D", r"FD#E"]
# labels = [r'Pristine', r'FD1st', r'HC#A', r'HC#B', r'FC#C', r'HD#D', r'FD#E']
shift_energy = 2.5
for i in range(TEM_EELS_O["bulk"].shape[1]):
    if i < 4:
        ax.plot(TEM_EELS_O["energy"], TEM_EELS_O["bulk"].iloc[:, i] + 0.4 * i, label=labels[i], color=colors_opt2[i])
    else:
        ax.plot(TEM_EELS_O["energy"] + shift_energy, TEM_EELS_O["bulk"].iloc[:, i] + 0.4 * i, label=labels[i], color=colors_opt2[i])

ax.plot(TEM_EELS_O["energy"], TEM_EELS_O["ZSH"], label=r"ZSH", color="blue", linestyle="--")


for i in range(0, TEM_EELS_O["surface"].shape[1]):
    if i < 3:
        ax.plot(TEM_EELS_O["energy"], TEM_EELS_O["surface"].iloc[:, i] + 0.5 * i + 3.5, linestyle="-", label=None, color=colors_opt2[i + 1])
    else:
        ax.plot(TEM_EELS_O["energy"] + shift_energy, TEM_EELS_O["surface"].iloc[:, i] + 0.5 * i + 3.5, linestyle="-", label=None, color=colors_opt2[i + 1])

for i in range(0, TEM_EELS_O["new_phase"].shape[1] // 2):
    if i < 3:
        ax.plot(TEM_EELS_O["energy"], TEM_EELS_O["new_phase"].iloc[:, 2 * i] + 0.7 * i + 7, linestyle="-", label=None, color=colors_opt2[i + 2])
        ax.plot(TEM_EELS_O["energy"], TEM_EELS_O["new_phase"].iloc[:, 2 * i + 1] + 0.7 * i + 7, linestyle="-", label=None, color=colors_opt2[i + 2])
    else:
        ax.plot(TEM_EELS_O["energy"] + shift_energy, TEM_EELS_O["new_phase"].iloc[:, 2 * i] + 0.7 * i + 7, linestyle="-", label=None, color=colors_opt2[i + 2])
        ax.plot(TEM_EELS_O["energy"] + shift_energy, TEM_EELS_O["new_phase"].iloc[:, 2 * i + 1] + 0.7 * i + 7, linestyle="-", label=None, color=colors_opt2[i + 2])

# 刻度线
ax.set_xlim(520, 550)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.5, offset=0))
ax.set_ylim(-0.1, 12.5)
ax.set_ylabel(r"Relative Absorption (arb.u.)", fontsize=11)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    left=True,
    labelbottom=True,
    labelleft=True,
)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.0, 1.01),
    ncols=2,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    columnspacing=0.5,
)
ax.text(0.65, 0.30, r"Bulk", transform=ax.transAxes, fontsize=11, va="center", ha="left", fontfamily="Arial")
ax.text(0.65, 0.58, r"Surface", transform=ax.transAxes, fontsize=11, va="center", ha="left", fontfamily="Arial")
ax.text(0.65, 0.85, r"New Phase", transform=ax.transAxes, fontsize=11, va="center", ha="left", fontfamily="Arial")
ax.text(0.86, 0.95, r"O", transform=ax.transAxes, fontsize=13, va="center", ha="left", fontfamily="Arial", fontweight="bold")
ax.vlines(530.0, -0.1, 10, colors="gray", linestyles="dashed", linewidth=1.0, zorder=1)
ax.vlines(537, -0.1, 10.6, colors="gray", linestyles="dashed", linewidth=1.0, zorder=1)

ax.text(-0.10, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_01_V1_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_01_V1_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_01_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_01_V1_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### 常规电化学 + TEM-EELS 的各个状态 + 新相 -V1

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[0].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[0].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    for item in data:
        if isinstance(item, (list, tuple)):
            results.append([_process(s) for s in item])
        else:
            results.append(_process(item))
    return results


def element_maps(signal_list, elements=(r"K", r"Mn", r"Zn", r"O"), crop_box=None):
    element_maps = {}
    for signal in signal_list:
        if isinstance(signal, list):
            element_maps.setdefault("others", []).extend(signal)
            continue

        try:
            title = signal.metadata["General"]["title"]
        except (KeyError, AttributeError, TypeError):
            title = ""

        if not isinstance(title, str):
            title = str(title)

        match = re.search(r"\b([A-Z][a-z]?)\b\s+[^()]+", title) if title else None
        if not match:
            element_maps.setdefault("others", []).append(signal)
            continue

        element = match.group(1)
        if element in elements:
            if crop_box:
                if signal.axes_manager.navigation_shape:
                    signal_crop = signal.inav[crop_box].copy()
                elif signal.axes_manager.signal_shape:
                    signal_crop = signal.isig[crop_box].copy()
            else:
                signal_crop = signal.copy()

            element_maps[element] = signal_crop
    return element_maps


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def rois_plot(ax, roi_coords, data, textcolor=None, text_shift=None, roi_name=None, edgecolor=None, zorder=2) -> None:
    scale_x = data.axes_manager[0].scale
    scale_y = data.axes_manager[1].scale
    for i, (left, right, top, bottom) in enumerate(roi_coords):
        roi = hs.roi.RectangularROI(left=left, right=right, top=top, bottom=bottom)

        x = int(roi.x / scale_x)
        y = int(roi.y / scale_y)
        width = int(roi.width / scale_x)
        height = int(roi.height / scale_y)
        # 绘制矩形
        rect = patches.Rectangle(
            (x, y),
            width,
            height,
            linewidth=1,
            edgecolor=edgecolor[i] if edgecolor else "y",
            facecolor="none",
            transform=ax.transData,
            zorder=zorder,
        )
        ax.add_patch(rect)

        # 添加文本标签
        ax.text(
            x + text_shift[i][0],
            y - text_shift[i][1],
            f"{roi_name[i]}" if roi_name else f"ROI {i + 1}",
            fontsize=10,
            color=textcolor[i] if textcolor else "y",
            ha="center",
            va="center",
            transform=ax.transData,
            zorder=zorder + 1,
        )

In [ ]:
# 读取一个常规的 αMnO2 的数据
echem = []
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\1M ZnSO4 + 0.2M MnSO4").glob(r"LC*.xlsx"))
# 读取电化学数据
for path_file in path_filelist[0:1]:
    df = pd.read_excel(path_file, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echem.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echem)):
    echem[i] = echem[i][echem[i]["Cycle"].isin([0, 1, 2])]
    echem[i] = echem[i][echem[i]["Voltage/V"] >= 0]

# 读取 STEM-EELS 的数据
path_pristine = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Pristine\αMnO2\Powders\2023-EMCA\EELS\Results\SI1_37mm_5mm_09eV_2nm_50ms")
path_pristine_maps = list(path_pristine.joinpath(r"EELS Map").glob(r"*.dm4"))
pristine_maps = hs.load(path_pristine_maps)

path_FD1_maps = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\EELS\Results\SI2_80pA_10ms"
)
FD1_maps = hs.load(path_FD1_maps.joinpath(r"EELS Map").glob(r"*.dm4"))

path_HC1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EELS\Results\SI 2_630mm_5mm_0.9eVCh_0.9nm_15ms_150pA"
)
HC1_maps = hs.load(path_HC1.joinpath(r"EELS Map").glob(r"*.dm4"))

path_HC1_new = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EELS\Results\SI 1_630mm_5mm_0.9eVCh_4nm_15ms_150pA"
)
HC1_maps_new = hs.load(path_HC1_new.joinpath(r"EELS Map").glob(r"*.dm4"))

path_HC2 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EELS\Results\SI 1_630mm_5mm_0.9eVCh_3nm_15ms_150pA"
)
HC2_maps = hs.load(path_HC2.joinpath(r"EELS Map").glob(r"*.dm4"))

path_HC2_new = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EELS\Results\SI 2_630mm_5mm_0.9eVCh_4nm_15ms_150pA"
)
HC2_maps_new = hs.load(path_HC2_new.joinpath(r"EELS Map").glob(r"*.dm4"))

path_FC1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\EELS\Results\SI2_EFTEM400m_5mm_50ms_200pA_5nm"
)
FC1_maps = hs.load(path_FC1.joinpath(r"EELS Map").glob(r"*.dm4"))

path_FC1_new = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\EELS\Results\SI5_EFTEM400m_2nm_50ms_200pA_5nm"
)
FC1_maps_new = hs.load(path_FC1_new.joinpath(r"EELS Map").glob(r"*.dm4"))

path_HD1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\EELS\Results\SI3_CL320mm_0.9eV_25ms_1.5nm step_120pA"
)
HD1_maps = hs.load(path_HD1.joinpath(r"EELS Map").glob(r"*.dm4"))

path_HD1_new = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\EELS\Results\SI1_CL320mm_0.9eV_25ms_2.5nm step_120pA"
)
HD1_maps_new = hs.load(path_HD1_new.joinpath(r"EELS Map").glob(r"*.dm4"))

path_FD2 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\EELS\Results\SI1_100pA_10ms_1.5nm"
)
FD2_maps = hs.load(path_FD2.joinpath(r"EELS Map").glob(r"*.dm4"))

# TEM-EELS 的比例
path_fc_ratio = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\TEM-EDS-EELS-all").joinpath(r"TEM-EELS_Bulk_Surface_NewPhase.xlsx")
fc_ratio = pd.read_excel(path_fc_ratio, sheet_name=1, header=0, index_col=None).dropna(axis=1, how="all")


In [ ]:
fc_ratio = fc_ratio[~fc_ratio["Samples"].isin(["FD1st"])]

In [ ]:
pristine_maps = convert_units(pristine_maps)
FD1_maps = convert_units(FD1_maps)
HC1_maps = convert_units(HC1_maps)
HC1_maps_new = convert_units(HC1_maps_new)
HC2_maps = convert_units(HC2_maps)
HC2_maps_new = convert_units(HC2_maps_new)
FC1_maps = convert_units(FC1_maps)
FC1_maps_new = convert_units(FC1_maps_new)
HD1_maps = convert_units(HD1_maps)
HD1_maps_new = convert_units(HD1_maps_new)
FD2_maps = convert_units(FD2_maps)

pristine_maps = element_maps(pristine_maps, elements=["Mn", "O"], crop_box=(slice(0, 300), slice(None)))
FD1_maps = element_maps(FD1_maps, elements=["O", "Mn", "Zn"], crop_box=(slice(0, 180), slice(None)))
HC1_maps = element_maps(HC1_maps, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
HC1_maps_new = element_maps(HC1_maps_new, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
HC2_maps = element_maps(HC2_maps, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
HC2_maps_new = element_maps(HC2_maps_new, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
FC1_maps = element_maps(FC1_maps, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(0, 70)))
FC1_maps_new = element_maps(FC1_maps_new, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
HD1_maps = element_maps(HD1_maps, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
HD1_maps_new = element_maps(HD1_maps_new, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
FD2_maps = element_maps(FD2_maps, elements=["O", "Mn", "Zn"], crop_box=(slice(0, 120), slice(None)))

In [ ]:
pristine_rois = [(159, 278, 43, 60), (354, 552, 56, 126)]
FD1_rois = [(11, 53, 43, 60), (55, 93, 73, 81), (89, 124, 10, 26)]
HC1_rois = [(53, 72, 25, 64), (13, 21, 25, 60)]
HC1_rois_new = [(184, 360, 144, 324), (20, 120, 224, 324)]
HC2_rois = [(120, 159, 228, 330), (33, 72, 18, 129)]
HC2_rois_new = [(300, 470, 100, 160), (75, 251, 331, 391)]
FC1_rois = [(34, 67, 24, 31), (16, 56, 45, 59)]
FC1_rois_new = [(62, 98, 74, 106), (14, 30, 92, 126)]
HD1_rois = [(49, 65, 50, 80), (13, 35, 30, 59)]
HD1_rois_new = [(39, 84, 7, 47), (136, 164, 12, 57)]
FD2_rois = [(13, 60, 40, 59), (47, 110, 80, 94)]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.3)

ax.set_axis_off()
labels = [
    [None, None],
]
for i, data in enumerate(echem):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        if j == 0:
            ax.plot(
                temp.loc[idx_min:, "SpeCap/mAh/g"] + temp.loc[idx_min - 1, "SpeCap/mAh/g"],
                temp.loc[idx_min:, "Voltage/V"],
                ls="-",
                lw=1.0,
                c=colors[j],
                zorder=0,
            )  # 充电
        if j == 1:
            ax.plot(
                temp.loc[:idx_min, "SpeCap/mAh/g"] + 690,
                temp.loc[:idx_min, "Voltage/V"],
                ls="-",
                lw=1.0,
                c=colors[j],
                label=labels[i][j],
                zorder=0,
            )  # 放电

# # 插入图 A
# axins = ax.inset_axes((-0.03, 0.0, 0.4, 0.4), zorder=1)
# hs.plot.plot_images(
#     [FD1_maps['Mn'], FD1_maps['Zn']],
#     ax=axins,
#     overlay=True,
#     colors=[eels_colors['Mn'], eels_colors['Zn']],
#     axes_decor='off',
#     label=None,
#     alphas=[1.0, 1.0, 1.0],
#     vmax='100th',
# )
# scalebar =add_sizebar(
#     axins,
#     size=20,
#     bardata=FD1_maps['Mn'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=axins.transAxes)
# axins.annotate(
#     r"FD#1",
#     (-0.08, 0.05),
#     xytext=(0.5, -0.14),
#     fontsize=13,
#     c=colors[0],
#     xycoords="axes fraction",
#     textcoords="axes fraction",
#     fontweight="normal",
#     bbox=None,
#     transform=ax.transAxes,
#     ha="right",
#     va="center",
#     arrowprops={
#         "arrowstyle": "->",
#         "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
#         "ls": "-",
#         "lw": 1.0,
#         "color": "grey",
#     },
# )  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# axins.text(
#     0.03,
#     1.20,
#     r"Mn",
#     c="k",
#     bbox={"ec": None, "fc": eels_colors["Mn"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
#     transform=axins.transAxes,
#     fontsize=9,
#     va="top",
#     ha="left",
#     fontfamily="Arial",
#     fontweight="bold",
# )
# axins.text(
#     0.21,
#     1.20,
#     r"Zn",
#     c="w",
#     bbox={"ec": None, "fc": eels_colors["Zn"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
#     transform=axins.transAxes,
#     fontsize=9,
#     va="top",
#     ha="left",
#     fontfamily="Arial",
#     fontweight="bold",
# )
# # 标记 Bulk 和 Surface
# rois_plot(
#     axins,
#     roi_coords=FD1_rois,
#     data=FD1_maps['Mn'],
#     textcolor=["w", "k", "w"],
#     text_shift=([10, 10], [45, 8], [-20, -2]),
#     roi_name=[r"Bulk", r"Surface", r"ZSH"],
#     edgecolor=["w", "w", "w"],
#     zorder=2,
# )

# 插入图 B
axins = ax.inset_axes((0.08, 0.0, 0.4, 0.4), zorder=1)
hs.plot.plot_images(
    [HC1_maps["Mn"], HC1_maps["Zn"]],  # type: ignore
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    axins,
    size=20,
    bardata=HC1_maps["Mn"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"HC#A",
    (0.5, 1.65),
    xytext=(0.4, 1.1),
    fontsize=13,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=90,angleB=0,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=HC1_rois,
    data=HC1_maps["Mn"],
    textcolor=["k", "w"],
    text_shift=([20, -55], [20, 10]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)

# 插入图 B1
axins = ax.inset_axes((-0.08, 0.0, 0.4, 0.4), zorder=1)
hs.plot.plot_images(
    [HC1_maps_new["Mn"], HC1_maps_new["Zn"]],  # type: ignore
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    axins,
    size=100,
    bardata=HC1_maps_new["Mn"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"New Phase",
    (0.1, 1.55),
    xytext=(1.0, 1.1),
    fontsize=13,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops=None,
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=HC1_rois_new,
    data=HC1_maps_new["Mn"],
    textcolor=["k", "k"],
    text_shift=([20, 10], [15, 10]),
    roi_name=[r"#1", r"#2"],
    edgecolor=["w", "w"],
    zorder=2,
)

# 插入图 C
axins = ax.inset_axes((0.38, 0.0, 0.4, 0.5), zorder=1)
hs.plot.plot_images(
    [HC2_maps["Mn"], HC2_maps["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    axins,
    size=100,
    bardata=HC2_maps["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"HC#B",
    (-0.6, 1.55),
    xytext=(0.5, 1.07),
    fontsize=13,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=90,angleB=0,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=HC2_rois,
    data=HC2_maps["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, 10], [50, -10]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)

# 插入图 C1
axins = ax.inset_axes((0.245, 0.0, 0.4, 0.4), zorder=1)
hs.plot.plot_images(
    [HC2_maps_new["Mn"], HC2_maps_new["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    axins,
    size=100,
    bardata=HC2_maps_new["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.5, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"New Phase",
    (0.0, 1.52),
    xytext=(1.0, 1.1),
    fontsize=13,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops=None,
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=HC2_rois_new,
    data=HC2_maps_new["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, 10], [30, 10]),
    roi_name=[r"#1", r"#2"],
    edgecolor=["w", "w"],
    zorder=2,
)

# 插入图 D
axins = ax.inset_axes((0.48, 0.6, 0.5, 0.5), zorder=1)
hs.plot.plot_images(
    [FC1_maps["Mn"], FC1_maps["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    axins,
    size=20,
    bardata=FC1_maps["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"FC#C",
    (-0.35, 0.7),
    xytext=(0.0, 0.8),
    fontsize=13,
    c=colors[0],
    rotation=90,
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=90,angleB=0,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=FC1_rois,
    data=FC1_maps["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, 6], [10, 6]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)

# 插入图 D1
axins = ax.inset_axes((0.46, 1.15, 0.5, 0.5), zorder=1)
hs.plot.plot_images(
    [FC1_maps_new["Mn"], FC1_maps_new["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    axins,
    size=20,
    bardata=FC1_maps_new["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.6, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"New Phase",
    (-0.35, 0.7),
    xytext=(0.0, 0.45),
    fontsize=13,
    c=colors[0],
    rotation=90,
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops=None,
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=FC1_rois_new,
    data=FC1_maps_new["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, 6], [10, 6]),
    roi_name=[r"#1", r"#2"],
    edgecolor=["w", "w"],
    zorder=2,
)


# 插入图 E
axins = ax.inset_axes((0.71, 0.7, 0.4, 0.4), zorder=1)

hs.plot.plot_images(
    [HD1_maps["Mn"], HD1_maps["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    axins,
    size=20,
    bardata=HD1_maps["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"HD#D",
    (-1.0, -0.65),
    xytext=(1.0, -0.15),
    fontsize=13,
    c=colors[1],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=90,angleB=0,rad=10",
        "ls": "--",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=HD1_rois,
    data=HD1_maps["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, 6], [10, 6]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)

# 插入图 E1
axins = ax.inset_axes((0.685, 1.25, 0.4, 0.4), zorder=1)

hs.plot.plot_images(
    [HD1_maps_new["Mn"], HD1_maps_new["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    axins,
    size=20,
    bardata=HD1_maps_new["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"New Phase",
    (-1.0, -0.65),
    xytext=(1.0, -0.15),
    fontsize=13,
    c=colors[1],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops=None,
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=HD1_rois_new,
    data=HD1_maps_new["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, -26], [10, -26]),
    roi_name=[r"#1", r"#2"],
    edgecolor=["w", "w"],
    zorder=2,
)

# 插入图 F
axins = ax.inset_axes((0.57, -0.05, 0.4, 0.4), zorder=1)
hs.plot.plot_images(
    [FD2_maps["Mn"], FD2_maps["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=20,
    bardata=FD2_maps["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.75, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"FD#E",
    (1.35, 0.10),
    xytext=(1.0, -0.15),
    fontsize=13,
    c=colors[1],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=FD2_rois,
    data=FD2_maps["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, 6], [15, 6]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)

# 插入图 G
axins = ax.inset_axes((0.05, 0.75, 0.35, 0.35), zorder=1)
hs.plot.plot_images(
    [
        pristine_maps["Mn"],
    ],
    ax=axins,
    overlay=True,
    colors=[
        eels_colors["Mn"],
    ],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=100,
    bardata=pristine_maps["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"Pristine",
    (0.3, 1.42),
    xytext=(0.25, -0.15),
    fontsize=13,
    c=colors[2],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops=None,
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
axins.text(
    0.02,
    1.24,
    r"Mn",
    c="k",
    bbox={"ec": None, "fc": eels_colors["Mn"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=axins.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
axins.text(
    0.13,
    1.24,
    r"Zn",
    c="w",
    bbox={"ec": None, "fc": eels_colors["Zn"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=axins.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=pristine_rois,
    data=pristine_maps["Mn"],
    textcolor=[None, "w"],
    text_shift=([10, 10], [10, 10]),
    roi_name=[None, r"Bulk"],
    edgecolor=[None, "w"],
    zorder=2,
)

ax.text(0.04, 1.5, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[1, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.07, 0.0, 0.86, 0.86))
ax.set_box_aspect(0.4)

fc_ratio["Samples"] = fc_ratio["Samples"].astype(str).str.strip()
fc_ratio["Position"] = fc_ratio["Position"].astype(str).str.strip()

# 类别顺序
order = ["Bulk", "Surface", "NewPhase"]
fc_ratio["Position"] = pd.Categorical(fc_ratio["Position"], categories=order, ordered=True)

# 数值列
for col in [r"Mn/O", r"Zn/O", r"Zn/Mn"]:
    fc_ratio[col] = pd.to_numeric(fc_ratio[col], errors="coerce")

markers = {"Bulk": "o", "Surface": "s", "NewPhase": "*"}
for pos in ["Bulk", "Surface"]:
    sub = fc_ratio[fc_ratio["Position"] == pos]
    if not sub.empty:
        ax.plot(sub["Samples"], sub["Mn/O"], ls="-", marker=markers[pos], ms=10, c=colors[0], mec=colors[0], mfc=colors[0], zorder=2)
        ax.plot(sub["Samples"], sub["Zn/O"], ls="-", marker=markers[pos], ms=10, c=colors[1], mec=colors[1], mfc=colors[1], zorder=2)


# NewPhase 散点
sub_np = fc_ratio[fc_ratio["Position"] == "NewPhase"]
if not sub_np.empty:
    ax.scatter(sub_np["Samples"], sub_np["Mn/O"], ls="-", marker=markers["NewPhase"], s=200, c=colors[0], linewidths=1.0, edgecolors="w", zorder=2)
    ax.scatter(sub_np["Samples"], sub_np["Zn/O"], ls="-", marker=markers["NewPhase"], s=200, c=colors[1], linewidths=1.0, edgecolors="w", zorder=10)


ax.set_ylabel(r"Relative Ratio (arb.u.)", fontsize=9)
ax.set_ylim(-0.05, 0.6)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))

# ax.set_xlabel("Sample")
ax.tick_params(axis="x", rotation=45)
ax.text(-0.1, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

ax2 = ax.twinx()
ax2.set_position((0.07, 0.0, 0.86, 0.86))
ax2.set_box_aspect(0.4)

ax.set_zorder(3)  # 左轴放在更上层
ax2.set_zorder(2)  # 右轴放在下层
ax.patch.set_visible(False)

markers = {"Bulk": "o", "Surface": "s", "NewPhase": "*"}
for pos in ["Bulk", "Surface"]:
    sub = fc_ratio[fc_ratio["Position"] == pos]
    if not sub.empty:
        ax2.plot(sub["Samples"], sub["Zn/Mn"], ls="--", marker=markers[pos], ms=10, c=colors[4], mec=colors[4], mfc=colors[4], zorder=2)

# NewPhase 散点
if not sub_np.empty:
    ax2.scatter(sub_np["Samples"], sub_np["Zn/Mn"], marker=markers["NewPhase"], s=200, ls="--", linewidths=1.0, c=colors[4], edgecolors="w", zorder=10)

ax2.set_ylabel("Relative Ratio (arb.u.)", fontsize=9)
ax2.set_ylim(-0.4, 4.0)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=0))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))


# 图例
custom_handles = [
    Line2D([0], [0], linestyle="-", marker=None, color=colors[0]),
    Line2D([0], [0], linestyle="-", marker=None, color=colors[1]),
    Line2D([0], [0], linestyle="--", marker=None, color=colors[4]),
    Line2D([0], [0], linestyle="None", marker=markers["Bulk"], color="grey"),
    Line2D([0], [0], linestyle="None", marker=markers["Surface"], color="grey"),
    Line2D([0], [0], linestyle="None", marker=markers["NewPhase"], color="grey"),
]
custom_labels = ["Mn/O", "Zn/O", "Zn/Mn", "Bulk", "Surface", "NewPhase"]

legend1 = ax.legend(custom_handles[0:3], custom_labels[0:3], loc=(0.0, 1.08), ncol=3, frameon=False, fontsize=9, borderpad=0.7, handletextpad=0.4)
legend2 = ax.legend(custom_handles[3:], custom_labels[3:], loc=(0.0, 1.0), ncol=3, frameon=False, fontsize=9, borderpad=0.7, handletextpad=0.4)

ax.add_artist(legend1)
ax.add_artist(legend2)
arrowprops = {"arrowstyle": "<-", "color": colors[4], "connectionstyle": "angle,angleA=0,angleB=90,rad=10", "ls": r"--"}  # type: ignore
ax2.annotate(r" ", xy=(0.8, 0.75), xycoords="axes fraction", xytext=(1.0, 0.65), textcoords="axes fraction", arrowprops=arrowprops)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_00_V1_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_00_V1_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_00_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_00_V1_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### 常规电化学 + TEM-EELS 的各个状态 + 新相

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[0].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[0].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    for item in data:
        if isinstance(item, (list, tuple)):
            results.append([_process(s) for s in item])
        else:
            results.append(_process(item))
    return results


def element_maps(signal_list, elements=(r"K", r"Mn", r"Zn", r"O"), crop_box=None):
    element_maps = {}
    for signal in signal_list:
        if isinstance(signal, list):
            element_maps.setdefault("others", []).extend(signal)
            continue

        try:
            title = signal.metadata["General"]["title"]
        except (KeyError, AttributeError, TypeError):
            title = ""

        if not isinstance(title, str):
            title = str(title)

        match = re.search(r"\b([A-Z][a-z]?)\b\s+[^()]+", title) if title else None
        if not match:
            element_maps.setdefault("others", []).append(signal)
            continue

        element = match.group(1)
        if element in elements:
            if crop_box:
                if signal.axes_manager.navigation_shape:
                    signal_crop = signal.inav[crop_box].copy()
                elif signal.axes_manager.signal_shape:
                    signal_crop = signal.isig[crop_box].copy()
            else:
                signal_crop = signal.copy()

            element_maps[element] = signal_crop
    return element_maps


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def rois_plot(ax, roi_coords, data, textcolor=None, text_shift=None, roi_name=None, edgecolor=None, zorder=2) -> None:
    scale_x = data.axes_manager[0].scale
    scale_y = data.axes_manager[1].scale
    for i, (left, right, top, bottom) in enumerate(roi_coords):
        roi = hs.roi.RectangularROI(left=left, right=right, top=top, bottom=bottom)

        x = int(roi.x / scale_x)
        y = int(roi.y / scale_y)
        width = int(roi.width / scale_x)
        height = int(roi.height / scale_y)
        # 绘制矩形
        rect = patches.Rectangle(
            (x, y),
            width,
            height,
            linewidth=1,
            edgecolor=edgecolor[i] if edgecolor else "y",
            facecolor="none",
            transform=ax.transData,
            zorder=zorder,
        )
        ax.add_patch(rect)

        # 添加文本标签
        ax.text(
            x + text_shift[i][0],
            y - text_shift[i][1],
            f"{roi_name[i]}" if roi_name else f"ROI {i + 1}",
            fontsize=10,
            color=textcolor[i] if textcolor else "y",
            ha="center",
            va="center",
            transform=ax.transData,
            zorder=zorder + 1,
        )

In [ ]:
# 读取一个常规的 αMnO2 的数据
echem = []
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\1M ZnSO4 + 0.2M MnSO4").glob(r"LC*.xlsx"))
# 读取电化学数据
for path_file in path_filelist[0:1]:
    df = pd.read_excel(path_file, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echem.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echem)):
    echem[i] = echem[i][echem[i]["Cycle"].isin([0, 1, 2])]
    echem[i] = echem[i][echem[i]["Voltage/V"] >= 0]

# 读取 STEM-EELS 的数据
path_pristine = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Pristine\αMnO2\Powders\2023-EMCA\EELS\Results\SI1_37mm_5mm_09eV_2nm_50ms")
path_pristine_maps = list(path_pristine.joinpath(r"EELS Map").glob(r"*.dm4"))
pristine_maps = hs.load(path_pristine_maps)

path_FD1_maps = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\EELS\Results\SI2_80pA_10ms"
)
FD1_maps = hs.load(path_FD1_maps.joinpath(r"EELS Map").glob(r"*.dm4"))

path_HC1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EELS\Results\SI 2_630mm_5mm_0.9eVCh_0.9nm_15ms_150pA"
)
HC1_maps = hs.load(path_HC1.joinpath(r"EELS Map").glob(r"*.dm4"))

path_HC1_new = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EELS\Results\SI 1_630mm_5mm_0.9eVCh_4nm_15ms_150pA"
)
HC1_maps_new = hs.load(path_HC1_new.joinpath(r"EELS Map").glob(r"*.dm4"))

path_HC2 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EELS\Results\SI 1_630mm_5mm_0.9eVCh_3nm_15ms_150pA"
)
HC2_maps = hs.load(path_HC2.joinpath(r"EELS Map").glob(r"*.dm4"))

path_HC2_new = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EELS\Results\SI 2_630mm_5mm_0.9eVCh_4nm_15ms_150pA"
)
HC2_maps_new = hs.load(path_HC2_new.joinpath(r"EELS Map").glob(r"*.dm4"))

path_FC1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\EELS\Results\SI2_EFTEM400m_5mm_50ms_200pA_5nm"
)
FC1_maps = hs.load(path_FC1.joinpath(r"EELS Map").glob(r"*.dm4"))

path_FC1_new = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\EELS\Results\SI5_EFTEM400m_2nm_50ms_200pA_5nm"
)
FC1_maps_new = hs.load(path_FC1_new.joinpath(r"EELS Map").glob(r"*.dm4"))

path_HD1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\EELS\Results\SI3_CL320mm_0.9eV_25ms_1.5nm step_120pA"
)
HD1_maps = hs.load(path_HD1.joinpath(r"EELS Map").glob(r"*.dm4"))

path_HD1_new = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\EELS\Results\SI1_CL320mm_0.9eV_25ms_2.5nm step_120pA"
)
HD1_maps_new = hs.load(path_HD1_new.joinpath(r"EELS Map").glob(r"*.dm4"))

path_FD2 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\EELS\Results\SI1_100pA_10ms_1.5nm"
)
FD2_maps = hs.load(path_FD2.joinpath(r"EELS Map").glob(r"*.dm4"))

# TEM-EELS 的比例
path_fc_ratio = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\TEM-EDS-EELS-all").joinpath(r"TEM-EELS_Bulk_Surface_NewPhase.xlsx")
fc_ratio = pd.read_excel(path_fc_ratio, sheet_name=1, header=0, index_col=None).dropna(axis=1, how="all")
# # 清洗数据
# bulk = fc_ratio[fc_ratio["Position"] == "Bulk"].copy(deep=True)
# surface = fc_ratio[fc_ratio["Position"] == "Surface"].copy(deep=True)

In [ ]:
pristine_maps = convert_units(pristine_maps)
FD1_maps = convert_units(FD1_maps)
HC1_maps = convert_units(HC1_maps)
HC1_maps_new = convert_units(HC1_maps_new)
HC2_maps = convert_units(HC2_maps)
HC2_maps_new = convert_units(HC2_maps_new)
FC1_maps = convert_units(FC1_maps)
FC1_maps_new = convert_units(FC1_maps_new)
HD1_maps = convert_units(HD1_maps)
HD1_maps_new = convert_units(HD1_maps_new)
FD2_maps = convert_units(FD2_maps)

pristine_maps = element_maps(pristine_maps, elements=["Mn", "O"], crop_box=(slice(0, 300), slice(None)))
FD1_maps = element_maps(FD1_maps, elements=["O", "Mn", "Zn"], crop_box=(slice(0, 180), slice(None)))
HC1_maps = element_maps(HC1_maps, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
HC1_maps_new = element_maps(HC1_maps_new, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
HC2_maps = element_maps(HC2_maps, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
HC2_maps_new = element_maps(HC2_maps_new, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
FC1_maps = element_maps(FC1_maps, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(0, 70)))
FC1_maps_new = element_maps(FC1_maps_new, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
HD1_maps = element_maps(HD1_maps, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
HD1_maps_new = element_maps(HD1_maps_new, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
FD2_maps = element_maps(FD2_maps, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))

In [ ]:
pristine_rois = [(159, 278, 43, 60), (354, 552, 56, 126)]
FD1_rois = [(11, 53, 43, 60), (55, 93, 73, 81), (89, 124, 10, 26)]
HC1_rois = [(53, 72, 25, 64), (13, 21, 25, 60)]
HC1_rois_new = [(184, 360, 144, 324), (20, 120, 224, 324)]
HC2_rois = [(120, 159, 228, 330), (33, 72, 18, 129)]
HC2_rois_new = [(300, 470, 100, 160), (75, 251, 331, 391)]
FC1_rois = [(34, 67, 24, 31), (16, 56, 45, 59)]
FC1_rois_new = [(62, 98, 74, 106), (14, 30, 92, 126)]
HD1_rois = [(49, 65, 50, 80), (13, 35, 30, 59)]
HD1_rois_new = [(39, 84, 7, 47), (136, 164, 12, 57)]
FD2_rois = [(13, 60, 40, 59), (47, 110, 80, 94)]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.3)

ax.set_axis_off()
labels = [
    [None, None],
]
for i, data in enumerate(echem):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        if j == 0:
            ax.plot(
                temp.loc[idx_min:, "SpeCap/mAh/g"] + temp.loc[idx_min - 1, "SpeCap/mAh/g"],
                temp.loc[idx_min:, "Voltage/V"],
                ls="-",
                lw=1.0,
                c=colors[j],
                zorder=0,
            )  # 充电
        if j == 1:
            ax.plot(
                temp.loc[:idx_min, "SpeCap/mAh/g"] + 690,
                temp.loc[:idx_min, "Voltage/V"],
                ls="-",
                lw=1.0,
                c=colors[j],
                label=labels[i][j],
                zorder=0,
            )  # 放电

# 插入图 A
axins = ax.inset_axes((-0.03, 0.0, 0.4, 0.4), zorder=1)
hs.plot.plot_images(
    [FD1_maps["Mn"], FD1_maps["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=20,
    bardata=FD1_maps["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=axins.transAxes)
axins.annotate(
    r"FD#1",
    (-0.08, 0.05),
    xytext=(0.5, -0.14),
    fontsize=13,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
axins.text(
    0.03,
    1.20,
    r"Mn",
    c="k",
    bbox={"ec": None, "fc": eels_colors["Mn"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=axins.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
axins.text(
    0.21,
    1.20,
    r"Zn",
    c="w",
    bbox={"ec": None, "fc": eels_colors["Zn"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=axins.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=FD1_rois,
    data=FD1_maps["Mn"],
    textcolor=["w", "k", "w"],
    text_shift=([10, 10], [45, 8], [-20, -2]),
    roi_name=[r"Bulk", r"Surface", r"ZSH"],
    edgecolor=["w", "w", "w"],
    zorder=2,
)

# 插入图 B
axins = ax.inset_axes((0.18, 0.05, 0.4, 0.4), zorder=1)
hs.plot.plot_images(
    [HC1_maps["Mn"], HC1_maps["Zn"]],  # type: ignore
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=20,
    bardata=HC1_maps["Mn"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"HC#1",
    (0.1, 1.55),
    xytext=(0.6, 1.1),
    fontsize=13,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=90,angleB=0,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=HC1_rois,
    data=HC1_maps["Mn"],
    textcolor=["k", "w"],
    text_shift=([20, -55], [20, 10]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)

# 插入图 B1
axins = ax.inset_axes((0.16, -0.5, 0.4, 0.4), zorder=1)
hs.plot.plot_images(
    [HC1_maps_new["Mn"], HC1_maps_new["Zn"]],  # type: ignore
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=100,
    bardata=HC1_maps_new["Mn"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"New Phase",
    (0.1, 1.55),
    xytext=(1.0, 1.1),
    fontsize=13,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops=None,
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=HC1_rois_new,
    data=HC1_maps_new["Mn"],
    textcolor=["k", "k"],
    text_shift=([20, 10], [15, 10]),
    roi_name=[r"#1", r"#2"],
    edgecolor=["w", "w"],
    zorder=2,
)


# 插入图 C
axins = ax.inset_axes((0.33, 0.05, 0.4, 0.5), zorder=1)
hs.plot.plot_images(
    [HC2_maps["Mn"], HC2_maps["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=100,
    bardata=HC2_maps["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"HC#2",
    (0.0, 1.52),
    xytext=(0.7, 1.07),
    fontsize=13,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=90,angleB=0,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=HC2_rois,
    data=HC2_maps["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, 10], [50, -10]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)

# 插入图 C1
axins = ax.inset_axes((0.33, -0.5, 0.4, 0.4), zorder=1)
hs.plot.plot_images(
    [HC2_maps_new["Mn"], HC2_maps_new["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=100,
    bardata=HC2_maps_new["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.5, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"New Phase",
    (0.0, 1.52),
    xytext=(1.0, 1.1),
    fontsize=13,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops=None,
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=HC2_rois_new,
    data=HC2_maps_new["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, 10], [30, 10]),
    roi_name=[r"#1", r"#2"],
    edgecolor=["w", "w"],
    zorder=2,
)


# 插入图 D
axins = ax.inset_axes((0.48, 0.6, 0.5, 0.5), zorder=1)
hs.plot.plot_images(
    [FC1_maps["Mn"], FC1_maps["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=20,
    bardata=FC1_maps["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"FC#1",
    (-0.35, 0.7),
    xytext=(0.0, 0.8),
    fontsize=13,
    c=colors[0],
    rotation=90,
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=90,angleB=0,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=FC1_rois,
    data=FC1_maps["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, 6], [10, 6]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)

# 插入图 D1
axins = ax.inset_axes((0.46, 1.15, 0.5, 0.5), zorder=1)
hs.plot.plot_images(
    [FC1_maps_new["Mn"], FC1_maps_new["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=20,
    bardata=FC1_maps_new["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.6, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"New Phase",
    (-0.35, 0.7),
    xytext=(0.0, 0.45),
    fontsize=13,
    c=colors[0],
    rotation=90,
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops=None,
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=FC1_rois_new,
    data=FC1_maps_new["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, 6], [10, 6]),
    roi_name=[r"#1", r"#2"],
    edgecolor=["w", "w"],
    zorder=2,
)


# 插入图 E
axins = ax.inset_axes((0.71, 0.7, 0.4, 0.4), zorder=1)

hs.plot.plot_images(
    [HD1_maps["Mn"], HD1_maps["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=20,
    bardata=HD1_maps["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"HD#1",
    (-1.0, -0.65),
    xytext=(1.0, -0.15),
    fontsize=13,
    c=colors[1],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=90,angleB=0,rad=10",
        "ls": "--",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=HD1_rois,
    data=HD1_maps["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, 6], [10, 6]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)

# 插入图 E1
axins = ax.inset_axes((0.685, 1.25, 0.4, 0.4), zorder=1)

hs.plot.plot_images(
    [HD1_maps_new["Mn"], HD1_maps_new["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=20,
    bardata=HD1_maps_new["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"New Phase",
    (-1.0, -0.65),
    xytext=(1.0, -0.15),
    fontsize=13,
    c=colors[1],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops=None,
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=HD1_rois_new,
    data=HD1_maps_new["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, -26], [10, -26]),
    roi_name=[r"#1", r"#2"],
    edgecolor=["w", "w"],
    zorder=2,
)


# 插入图 F
axins = ax.inset_axes((0.54, 0.0, 0.4, 0.4), zorder=1)
hs.plot.plot_images(
    [FD2_maps["Mn"], FD2_maps["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=20,
    bardata=FD2_maps["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.75, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"FD#2",
    (1.255, 0.10),
    xytext=(1.0, -0.15),
    fontsize=13,
    c=colors[1],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=FD2_rois,
    data=FD2_maps["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, 6], [15, 6]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)


# 插入图 G
axins = ax.inset_axes((0.05, 0.75, 0.35, 0.35), zorder=1)
hs.plot.plot_images(
    [
        pristine_maps["Mn"],
    ],
    ax=axins,
    overlay=True,
    colors=[
        eels_colors["Mn"],
    ],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=100,
    bardata=pristine_maps["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"Pristine",
    (0.3, 1.42),
    xytext=(0.25, -0.15),
    fontsize=13,
    c=colors[2],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops=None,
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=pristine_rois,
    data=pristine_maps["Mn"],
    textcolor=[None, "w"],
    text_shift=([10, 10], [10, 10]),
    roi_name=[None, r"Bulk"],
    edgecolor=[None, "w"],
    zorder=2,
)

ax.text(0.04, 1.5, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[1, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.07, -0.3, 0.86, 0.86))
ax.set_box_aspect(0.4)

fc_ratio["Samples"] = fc_ratio["Samples"].astype(str).str.strip()
fc_ratio["Position"] = fc_ratio["Position"].astype(str).str.strip()

# 类别顺序
order = ["Bulk", "Surface", "NewPhase"]
fc_ratio["Position"] = pd.Categorical(fc_ratio["Position"], categories=order, ordered=True)

# 数值列
for col in ["Mn/O", "Zn/O"]:
    fc_ratio[col] = pd.to_numeric(fc_ratio[col], errors="coerce")

markers = {"Bulk": "o", "Surface": "s", "NewPhase": "*"}
for pos in ["Bulk", "Surface"]:
    sub = fc_ratio[fc_ratio["Position"] == pos]
    if not sub.empty:
        ax.plot(sub["Samples"], sub["Mn/O"], ls="-", marker=markers[pos], ms=10, c=colors[0], mec=colors[0], mfc=colors[0], zorder=2)

# NewPhase 散点
sub_np = fc_ratio[fc_ratio["Position"] == "NewPhase"]
if not sub_np.empty:
    ax.scatter(sub_np["Samples"], sub_np["Mn/O"], marker=markers["NewPhase"], s=100, c=colors[0], linewidths=0.5, edgecolors="w", zorder=10)

ax.set_ylabel(r"Mn/O, Relative Ratio (arb.u.)", fontsize=9)
ax.set_ylim(-0.05, 0.6)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))

# ax.set_xlabel("Sample")
ax.tick_params(axis="x", rotation=45)
ax.text(-0.1, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

ax2 = ax.twinx()
ax2.set_position((0.07, -0.3, 0.86, 0.86))
ax2.set_box_aspect(0.4)

ax.set_zorder(3)  # 左轴放在更上层
ax2.set_zorder(2)  # 右轴放在下层
ax.patch.set_visible(False)

markers = {"Bulk": "o", "Surface": "s", "NewPhase": "*"}
for pos in ["Bulk", "Surface"]:
    sub = fc_ratio[fc_ratio["Position"] == pos]
    if not sub.empty:
        ax2.plot(sub["Samples"], sub["Zn/O"], ls="-", marker=markers[pos], ms=10, c=colors[1], mec=colors[1], mfc=colors[1], zorder=2)

# NewPhase 散点
if not sub_np.empty:
    ax2.scatter(sub_np["Samples"], sub_np["Zn/O"], marker=markers["NewPhase"], s=100, linewidths=0.5, c=colors[1], edgecolors="w", zorder=10)

ax2.set_ylabel("Zn/O, Relative Ratio (arb.u.)", fontsize=9)
ax2.set_ylim(-0.02, 0.32)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.05, offset=0))

# 图例
custom_handles = [
    Line2D([0], [0], linestyle="-", marker=None, color=colors[0]),
    Line2D([0], [0], linestyle="-", marker=None, color=colors[1]),
    Line2D([0], [0], linestyle="None", marker=markers["Bulk"], color="grey"),
    Line2D([0], [0], linestyle="None", marker=markers["Surface"], color="grey"),
    Line2D([0], [0], linestyle="None", marker=markers["NewPhase"], color="grey"),
]
custom_labels = ["Mn/O", "Zn/O", "Bulk", "Surface", "NewPhase"]

legend1 = ax.legend(custom_handles[0:2], custom_labels[0:2], loc=(0.68, 1.2), ncol=1, frameon=False, fontsize=9, borderpad=0.7, handletextpad=0.4)
legend2 = ax.legend(custom_handles[2:], custom_labels[2:], loc=(0.8, 1.12), ncol=1, frameon=False, fontsize=9, borderpad=0.7, handletextpad=0.4)

ax.add_artist(legend1)
ax.add_artist(legend2)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_00_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_00_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_00_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EELS_00_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

## TEM-EDS

### Figure 对应的 Surface and bulk EDS 的结果 -V2

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[0].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[0].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    results = []
    for file in data:
        if len(file.axes_manager.shape) >= 2:
            if len(file.axes_manager.navigation_shape) == 2:
                if file.axes_manager.navigation_axes[0].units == r"µm":
                    file.axes_manager.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(file.axes_manager.signal_shape) == 2:  # noqa: SIM102
                if file.axes_manager.signal_axes[0].units == r"µm":
                    file.axes_manager.convert_units(axes="signal", units="nm", same_units=True, factor=1000)

        if len(file.axes_manager.shape) == 3:
            if len(file.axes_manager.navigation_shape) == 2:
                for axis in file.axes_manager.navigation_axes:
                    axis.offset = 0
            elif len(file.axes_manager.signal_shape) == 2:
                for axis in file.axes_manager.signal_axes:
                    axis.offset = 0
        results.append(file)
    return results


def element_maps(signal_list, elements=(r"K", r"Mn", r"Zn"), crop_box=None):
    element_maps = {}
    for signal in signal_list:
        title = signal.metadata["General"]["title"]
        if title in elements:
            if crop_box:
                if signal.axes_manager.navigation_shape:
                    signal_crop = signal.inav[crop_box].copy()
                elif signal.axes_manager.signal_shape:
                    signal_crop = signal.isig[crop_box].copy()
            else:
                signal_crop = signal.copy()

            element_maps[title] = signal_crop
    return element_maps


eds_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}
# eds_colors = {
#     "C": "#FF0000",
#     "K": "#00ff00",
#     "Mn": "#FFFF00",
#     "Zn": "#FF8000",
#     "S": "#1f3cef",
#     "O": "#00ffff",
# }


def rois_plot(ax, roi_coords, data, textcolor=None, text_shift=([10, 5], [10, 5]), roi_name=None, edgecolor=None, zorder=2) -> None:
    scale_x = data.axes_manager[0].scale
    scale_y = data.axes_manager[1].scale
    for i, (left, right, top, bottom) in enumerate(roi_coords):
        roi = hs.roi.RectangularROI(left=left, right=right, top=top, bottom=bottom)

        x = int(roi.x / scale_x)
        y = int(roi.y / scale_y)
        width = int(roi.width / scale_x)
        height = int(roi.height / scale_y)

        # 绘制矩形
        rect = patches.Rectangle(
            (x, y),
            width,
            height,
            linewidth=1,
            edgecolor=edgecolor[i] if edgecolor else "y",
            facecolor="none",
            transform=ax.transData,
            zorder=zorder,
        )
        ax.add_patch(rect)

        # 添加文本标签
        ax.text(
            x + text_shift[i][0],
            y - text_shift[i][1],
            f"{roi_name[i]}" if roi_name else f"ROI {i + 1}",
            fontsize=10,
            color=textcolor[i] if textcolor else "y",
            ha="center",
            va="center",
            transform=ax.transData,
            zorder=zorder + 1,
        )

In [ ]:
# 读取 EDS 的数据

path_pristine = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Pristine\αMnO2 + CNTs\2024-EMCA\EDS\0002 - B8_HAADF_47000_x")
pristine = hs.load(path_pristine.joinpath(r"Data", r"0002 - B8_HAADF_47000_x.emd"))  # type: ignore

path_HC1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EDS"
)
HC1_bulk = hs.load(path_HC1.joinpath(r"Data", r"0004-20241211_HAADF_29000_x_EDS", r"0004-20241211_HAADF_29000_x_EDS.emd"))  # type: ignore

HC1_surface = hs.load(path_HC1.joinpath(r"Data", r"0007-20241211_HAADF_115_kx_EDS", r"0007-20241211_HAADF_115_kx_EDS.emd"))  # type: ignore

path_HC2 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EDS"
)
HC2 = hs.load(path_HC2.joinpath(r"Data", r"0012-20241211_HAADF_58000_x_EDS", r"0012-20241211_HAADF_58000_x_EDS.emd"))  # type: ignore

path_FC1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\EDS"
)
FC1 = hs.load(path_FC1.joinpath(r"Data", r"0020 - 1406 Pristine MnO2 HAADF 10000 x", r"0020 - 1406 Pristine MnO2 HAADF 10000 x.emd"))

path_HD1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\EDS"
)
HD1_bulk = hs.load(path_HD1.joinpath(r"Data", r"0015-HAADF_23500_x_EDS_SI", r"0015-HAADF_23500_x_EDS_SI.emd"))  # type: ignore

HD1_surface = hs.load(path_HD1.joinpath(r"Data", r"0013-HAADF_33000_x_EDS_SI", r"0013-HAADF_33000_x_EDS_SI.emd"))  # type: ignore

path_FD2 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\EDS"
)
FD2 = hs.load(path_FD2.joinpath(r"Data", r"0028-HAADF_67000_x_EDS_SI", r"0028-HAADF_67000_x_EDS_SI.emd"))  # type: ignore


# TEM-EDS 的比例
path_fc_ratio = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\TEM-EDS-EELS-all").joinpath(r"TEM-EDX_Bulk_Surface.xlsx")
fc_ratio = pd.read_excel(path_fc_ratio, sheet_name=r"TEM-EDX", header=0, index_col=None).dropna(axis=1, how="all")
# 清洗数据
bulk = fc_ratio[fc_ratio["Position"] == "Bulk"].copy(deep=True)
surface = fc_ratio[fc_ratio["Position"] == "Surface"].copy(deep=True)

In [ ]:
pristine = convert_units(pristine)
HC1_bulk = convert_units(HC1_bulk)
HC1_surface = convert_units(HC1_surface)
HC2 = convert_units(HC2)
FC1 = convert_units(FC1)
HD1_bulk = convert_units(HD1_bulk)
HD1_surface = convert_units(HD1_surface)
FD2 = convert_units(FD2)

pristine = element_maps(pristine, elements=["K", "Mn"], crop_box=(slice(0, 60), slice(None)))
HC1_bulk = element_maps(HC1_bulk, elements=["K", "Mn", "Zn"])
HC1_surface = element_maps(HC1_surface, elements=["K", "Mn", "Zn"])
HC2 = element_maps(HC2, elements=["K", "Mn", "Zn"])
FC1 = element_maps(FC1, elements=["K", "Mn", "Zn"])
HD1_bulk = element_maps(HD1_bulk, elements=["K", "Mn", "Zn"])
HD1_surface = element_maps(HD1_surface, elements=["K", "Mn", "Zn"])
FD2 = element_maps(FD2, elements=["K", "Mn", "Zn"])

In [ ]:
# 交换 HD1 中 X Y 的数据
for key, data in HD1_surface.items():
    HD1_surface[key].data = data.data.T
for key, data in HD1_bulk.items():
    HD1_bulk[key].data = data.data.T
for key, data in FD2.items():
    FD2[key].data = data.data.T

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 9.0))
gs = gridspec.GridSpec(3, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [pristine["K"], pristine["Mn"]],
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=100,
    bardata=pristine["K"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# ax.text(
#     0.02,
#     1.22,
#     r"K",
#     c="w",
#     bbox={"ec": None, "fc": eds_colors["K"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
#     transform=ax.transAxes,
#     fontsize=9,
#     va="top",
#     ha="left",
#     fontfamily="Arial",
#     fontweight="bold",
# )
ax.text(
    0.02,
    1.12,
    r"Mn",
    c="k",
    bbox={"ec": None, "fc": eds_colors["Mn"], "alpha": 0.9, "boxstyle": "Square, pad=0.3"},
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
ax.text(
    0.14,
    1.12,
    r"Zn",
    c="w",
    bbox={"ec": None, "fc": eds_colors["Zn"], "alpha": 0.9, "boxstyle": "Square, pad=0.3"},
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)

# # 标记 Bulk 和 Surface
# rois_plot(
#     ax,
#     roi_coords=[(-652, -462, -132, -82), (-570, -404, -74, -41)],
#     data=pristine['Mn'],
#     textcolor=["w", "w"],
#     text_shift=([15, 8], [25, -18]),
#     roi_name=[r"Bulk", r"Surface"],
#     edgecolor=["w", "w"],
#     zorder=2,
# )
ax.text(-0.04, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.15, -0.12, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [HC1_bulk["K"], HC1_bulk["Mn"], HC1_bulk["Zn"]],  # type: ignore
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=200,
    bardata=HC1_bulk["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[
        (870, 1200, 870, 1150),
    ],
    data=HC1_bulk["Mn"],
    textcolor=[
        "w",
    ],
    text_shift=([15, 10],),
    roi_name=[
        r"Bulk",
    ],
    edgecolor=[
        "w",
    ],
    zorder=2,
)
ax.text(-0.04, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B Surface
ax = subfig.add_subplot()
ax.set_position((1.2, -0.025, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [HC1_surface["K"], HC1_surface["Mn"], HC1_surface["Zn"]],  # type: ignore
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=100,
    bardata=HC1_surface["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[
        (80, 450, 50, 100),
    ],
    data=HC1_surface["Mn"],
    textcolor=[
        "w",
    ],
    text_shift=([25, 12],),
    roi_name=[
        r"Surface",
    ],
    edgecolor=[
        "w",
    ],
    zorder=2,
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((1.35, -0.22, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [HC2["K"], HC2["Mn"], HC2["Zn"]],
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=100,
    bardata=HC2["K"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[(350, 420, 600, 840), (170, 290, 50, 300)],
    data=HC2["Mn"],
    textcolor=["w", "w"],
    text_shift=([12, 8], [12, -85]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)
ax.text(-0.04, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.4, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [FC1["K"], FC1["Mn"], FC1["Zn"]],
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=500,
    bardata=FC1["K"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[(1000, 1700, 2500, 2900), (2300, 2700, 1350, 1650)],
    data=FC1["Mn"],
    textcolor=["w", "w"],
    text_shift=([25, 20], [-10, 15]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)
ax.text(-0.04, 1.0, r"D", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 E Bulk
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-1.0, -0.23, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [HD1_bulk["K"], HD1_bulk["Mn"], HD1_bulk["Zn"]],  # type: ignore
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=500,
    bardata=HD1_bulk["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[
        (2200, 2700, 730, 1000),
    ],
    data=HD1_bulk["Mn"],
    textcolor=[
        "w",
    ],
    text_shift=([14, 10],),
    roi_name=[
        r"Bulk",
    ],
    edgecolor=[
        "w",
    ],
    zorder=2,
)
ax.text(-0.04, 1.0, r"E", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 E Surface
ax = subfig.add_subplot()
ax.set_position((0.1, -0.35, 1.25, 1.25))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [HD1_surface["K"], HD1_surface["Mn"], HD1_surface["Zn"]],  # type: ignore
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=500,
    bardata=HD1_surface["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[
        (290, 460, 850, 1000),
    ],
    data=HD1_surface["Mn"],
    textcolor=[
        "w",
    ],
    text_shift=([25, 12],),
    roi_name=[
        r"Surface",
    ],
    edgecolor=[
        "w",
    ],
    zorder=2,
)

# 图 F
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.5, -0.2, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [FD2["K"], FD2["Mn"], FD2["Zn"]],
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=50,
    bardata=FD2["K"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[(220, 320, 100, 300), (430, 500, 100, 300)],
    data=FD2["Mn"],
    textcolor=["w", "w"],
    text_shift=([4, 4], [7, 4]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)
ax.text(-0.04, 1.0, r"F", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 G
subfig = fig.add_subfigure(gs[2, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.05, -0.1, 1.0, 1.0))
ax.set_box_aspect(0.7)

labels = [r"K/Mn", r"Mn/O", r"Zn/O", r"Zn/Mn"]
markers = [r"o", r"s", r"D", r"*"]
for i in range(bulk.shape[1] - 3):
    ax.plot(
        np.arange(bulk.shape[0]),
        bulk.loc[:, labels[i]],
        c=colors[i],
        lw=1,
        ls="-",
        marker=markers[i],
        ms=8,
        mec=colors[i],
        mfc=colors[i],
        label=labels[i],
        zorder=0,
    )

ax.set_ylabel(r"Relative Ratio (arb.u.)", fontsize=9)
ax.set_ylim(-0.05, 0.8)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))

ax.set_xticks(np.arange(bulk.shape[0]), labels=bulk["Samples"])
plt.setp(ax.get_xticklabels(), rotation=90, ha="right", rotation_mode="anchor", fontfamily="Arial", fontsize=9)
ax.tick_params(axis="y", which="both", direction="out", left=True, labelleft=True, right=False)

ax.text(
    -0.11,
    1.0,
    r"G",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.5, 1.0),
    ncols=2,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    labelspacing=1.0,
)
ax.text(
    0.05,
    0.95,
    r"Bulk",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="left",
    fontfamily="Arial",
)

# 图 H
subfig = fig.add_subfigure(gs[2, 1:3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, -0.1, 1.0, 1.0))
ax.set_box_aspect(0.7)

labels = [r"K/Mn", r"Mn/O", r"Zn/O"]
markers = [r"o", r"s", r"D"]
for i in range(surface.shape[1] - 4):
    ax.plot(
        np.arange(surface.shape[0]),
        surface.loc[:, labels[i]],
        c=colors[i],
        lw=1,
        ls="--",
        marker=markers[i],
        ms=8,
        mec=colors[i],
        mfc=colors[i],
        label=labels[i],
        zorder=0,
    )

ax.set_ylabel(r"Relative Ratio (arb.u.)", fontsize=9)
ax.set_ylim(-0.05, 1.0)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))

ax.set_xticks(np.arange(bulk.shape[0]), labels=bulk["Samples"])
plt.setp(ax.get_xticklabels(), rotation=90, ha="right", rotation_mode="anchor", fontfamily="Arial", fontsize=9)
ax.tick_params(axis="y", which="both", direction="out", left=True, labelleft=True, right=False)

ax.text(
    -0.11,
    1.0,
    r"H",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.5, 1.0),
    ncols=2,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    labelspacing=1.0,
)
ax.text(
    0.05,
    0.95,
    r"Surface",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="left",
    fontfamily="Arial",
)

ax2 = ax.twinx()
ax2.set_position((0.7, -0.1, 1.0, 1.0))
ax2.set_box_aspect(0.7)

labels = [r"Zn/Mn"]
markers = [r"*"]
for i in range(1):
    ax2.plot(
        np.arange(surface.shape[0]),
        surface.loc[:, labels[i]],
        c=colors[i + 3],
        lw=1,
        ls="--",
        marker=markers[i],
        ms=8,
        mec=colors[i + 3],
        mfc=colors[i + 3],
        label=labels[i],
        zorder=0,
    )
ax2.set_ylabel(r"Relative Ratio (arb.u.)", fontsize=9)
ax2.set_ylim(-0.05, 3.0)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.6, offset=0))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.3, offset=0))
ax2.legend(
    loc="upper left",
    bbox_to_anchor=(0.71, 0.92),
    ncols=2,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    labelspacing=1.0,
)
arrowprops = {"arrowstyle": "<-", "color": colors[4], "connectionstyle": "angle,angleA=0,angleB=90,rad=10"}  # type: ignore
ax2.annotate(r" ", xy=(0.8, 0.6), xycoords="axes fraction", xytext=(1.0, 0.7), textcoords="axes fraction", arrowprops=arrowprops)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_02_V2_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_02_V2_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_02_V2_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_02_V2_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure 对应的 Surface and bulk EDS 的结果 -V1

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[0].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[0].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    results = []
    for file in data:
        if len(file.axes_manager.shape) >= 2:
            if len(file.axes_manager.navigation_shape) == 2:
                if file.axes_manager.navigation_axes[0].units == r"µm":
                    file.axes_manager.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(file.axes_manager.signal_shape) == 2:  # noqa: SIM102
                if file.axes_manager.signal_axes[0].units == r"µm":
                    file.axes_manager.convert_units(axes="signal", units="nm", same_units=True, factor=1000)

        if len(file.axes_manager.shape) == 3:
            if len(file.axes_manager.navigation_shape) == 2:
                for axis in file.axes_manager.navigation_axes:
                    axis.offset = 0
            elif len(file.axes_manager.signal_shape) == 2:
                for axis in file.axes_manager.signal_axes:
                    axis.offset = 0
        results.append(file)
    return results


def element_maps(signal_list, elements=(r"K", r"Mn", r"Zn"), crop_box=None):
    element_maps = {}
    for signal in signal_list:
        title = signal.metadata["General"]["title"]
        if title in elements:
            if crop_box:
                if signal.axes_manager.navigation_shape:
                    signal_crop = signal.inav[crop_box].copy()
                elif signal.axes_manager.signal_shape:
                    signal_crop = signal.isig[crop_box].copy()
            else:
                signal_crop = signal.copy()

            element_maps[title] = signal_crop
    return element_maps


eds_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}
# eds_colors = {
#     "C": "#FF0000",
#     "K": "#00ff00",
#     "Mn": "#FFFF00",
#     "Zn": "#FF8000",
#     "S": "#1f3cef",
#     "O": "#00ffff",
# }


def rois_plot(ax, roi_coords, data, textcolor=None, text_shift=([10, 5], [10, 5]), roi_name=None, edgecolor=None, zorder=2) -> None:
    scale_x = data.axes_manager[0].scale
    scale_y = data.axes_manager[1].scale
    for i, (left, right, top, bottom) in enumerate(roi_coords):
        roi = hs.roi.RectangularROI(left=left, right=right, top=top, bottom=bottom)

        x = int(roi.x / scale_x)
        y = int(roi.y / scale_y)
        width = int(roi.width / scale_x)
        height = int(roi.height / scale_y)

        # 绘制矩形
        rect = patches.Rectangle(
            (x, y),
            width,
            height,
            linewidth=1,
            edgecolor=edgecolor[i] if edgecolor else "y",
            facecolor="none",
            transform=ax.transData,
            zorder=zorder,
        )
        ax.add_patch(rect)

        # 添加文本标签
        ax.text(
            x + text_shift[i][0],
            y - text_shift[i][1],
            f"{roi_name[i]}" if roi_name else f"ROI {i + 1}",
            fontsize=10,
            color=textcolor[i] if textcolor else "y",
            ha="center",
            va="center",
            transform=ax.transData,
            zorder=zorder + 1,
        )

In [ ]:
# 读取 EDS 的数据

path_FD1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\EDS"
)
FD1 = hs.load(path_FD1.joinpath(r"Data", r"0003 - B8_HAADF_67000_x", r"0003 - B8_HAADF_67000_x.emd"))  # type: ignore

path_HC1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EDS"
)
HC1_bulk = hs.load(path_HC1.joinpath(r"Data", r"0004-20241211_HAADF_29000_x_EDS", r"0004-20241211_HAADF_29000_x_EDS.emd"))  # type: ignore

HC1_surface = hs.load(path_HC1.joinpath(r"Data", r"0007-20241211_HAADF_115_kx_EDS", r"0007-20241211_HAADF_115_kx_EDS.emd"))  # type: ignore

path_HC2 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EDS"
)
HC2 = hs.load(path_HC2.joinpath(r"Data", r"0012-20241211_HAADF_58000_x_EDS", r"0012-20241211_HAADF_58000_x_EDS.emd"))  # type: ignore

path_FC1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\EDS"
)
FC1 = hs.load(path_FC1.joinpath(r"Data", r"0020 - 1406 Pristine MnO2 HAADF 10000 x", r"0020 - 1406 Pristine MnO2 HAADF 10000 x.emd"))

path_HD1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\EDS"
)
HD1_bulk = hs.load(path_HD1.joinpath(r"Data", r"0015-HAADF_23500_x_EDS_SI", r"0015-HAADF_23500_x_EDS_SI.emd"))  # type: ignore

HD1_surface = hs.load(path_HD1.joinpath(r"Data", r"0013-HAADF_33000_x_EDS_SI", r"0013-HAADF_33000_x_EDS_SI.emd"))  # type: ignore

path_FD2 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\EDS"
)
FD2 = hs.load(path_FD2.joinpath(r"Data", r"0028-HAADF_67000_x_EDS_SI", r"0028-HAADF_67000_x_EDS_SI.emd"))  # type: ignore


# TEM-EDS 的比例
path_fc_ratio = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\TEM-EDS-EELS-all").joinpath(r"TEM-EDX_Bulk_Surface.xlsx")
fc_ratio = pd.read_excel(path_fc_ratio, sheet_name=r"TEM-EDX", header=0, index_col=None).dropna(axis=1, how="all")
# 清洗数据
bulk = fc_ratio[fc_ratio["Position"] == "Bulk"].copy(deep=True)
surface = fc_ratio[fc_ratio["Position"] == "Surface"].copy(deep=True)

In [ ]:
FD1 = convert_units(FD1)
HC1_bulk = convert_units(HC1_bulk)
HC1_surface = convert_units(HC1_surface)
HC2 = convert_units(HC2)
FC1 = convert_units(FC1)
HD1_bulk = convert_units(HD1_bulk)
HD1_surface = convert_units(HD1_surface)
FD2 = convert_units(FD2)

FD1 = element_maps(FD1, elements=["K", "Mn", "Zn"])
HC1_bulk = element_maps(HC1_bulk, elements=["K", "Mn", "Zn"])
HC1_surface = element_maps(HC1_surface, elements=["K", "Mn", "Zn"])
HC2 = element_maps(HC2, elements=["K", "Mn", "Zn"])
FC1 = element_maps(FC1, elements=["K", "Mn", "Zn"])
HD1_bulk = element_maps(HD1_bulk, elements=["K", "Mn", "Zn"])
HD1_surface = element_maps(HD1_surface, elements=["K", "Mn", "Zn"])
FD2 = element_maps(FD2, elements=["K", "Mn", "Zn"])

In [ ]:
# 交换 HD1 中 X Y 的数据
for key, data in HD1_surface.items():
    HD1_surface[key].data = data.data.T
for key, data in HD1_bulk.items():
    HD1_bulk[key].data = data.data.T
for key, data in FD2.items():
    FD2[key].data = data.data.T

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 9.0))
gs = gridspec.GridSpec(3, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [FD1["K"], FD1["Mn"], FD1["Zn"]],
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=50,
    bardata=FD1["K"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)

# ax.text(
#     0.02,
#     1.22,
#     r"K",
#     c="w",
#     bbox={"ec": None, "fc": eds_colors["K"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
#     transform=ax.transAxes,
#     fontsize=9,
#     va="top",
#     ha="left",
#     fontfamily="Arial",
#     fontweight="bold",
# )
ax.text(
    0.02,
    1.21,
    r"Mn",
    c="k",
    bbox={"ec": None, "fc": eds_colors["Mn"], "alpha": 0.9, "boxstyle": "Square, pad=0.3"},
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
ax.text(
    0.14,
    1.21,
    r"Zn",
    c="w",
    bbox={"ec": None, "fc": eds_colors["Zn"], "alpha": 0.9, "boxstyle": "Square, pad=0.3"},
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[(280, 420, 110, 140), (250, 420, 180, 200)],
    data=FD1["Mn"],
    textcolor=["w", "w"],
    text_shift=([15, 8], [25, -18]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)
ax.text(-0.04, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")
# 图 B Bulk
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.15, -0.22, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [HC1_bulk["K"], HC1_bulk["Mn"], HC1_bulk["Zn"]],  # type: ignore
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=200,
    bardata=HC1_bulk["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[
        (870, 1200, 870, 1150),
    ],
    data=HC1_bulk["Mn"],
    textcolor=[
        "w",
    ],
    text_shift=([15, 10],),
    roi_name=[
        r"Bulk",
    ],
    edgecolor=[
        "w",
    ],
    zorder=2,
)
ax.text(-0.04, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B Surface
ax = subfig.add_subplot()
ax.set_position((1.2, -0.125, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [HC1_surface["K"], HC1_surface["Mn"], HC1_surface["Zn"]],  # type: ignore
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=100,
    bardata=HC1_surface["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[
        (80, 450, 50, 100),
    ],
    data=HC1_surface["Mn"],
    textcolor=[
        "w",
    ],
    text_shift=([25, 12],),
    roi_name=[
        r"Surface",
    ],
    edgecolor=[
        "w",
    ],
    zorder=2,
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((1.35, -0.32, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [HC2["K"], HC2["Mn"], HC2["Zn"]],
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=100,
    bardata=HC2["K"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[(350, 420, 600, 840), (170, 290, 50, 300)],
    data=HC2["Mn"],
    textcolor=["w", "w"],
    text_shift=([12, 8], [12, -85]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)
ax.text(-0.04, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.5, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [FC1["K"], FC1["Mn"], FC1["Zn"]],
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=500,
    bardata=FC1["K"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[(1000, 1700, 2500, 2900), (2300, 2700, 1350, 1650)],
    data=FC1["Mn"],
    textcolor=["w", "w"],
    text_shift=([25, 20], [-10, 15]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)
ax.text(-0.04, 1.0, r"D", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 E Bulk
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-1.0, -0.13, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [HD1_bulk["K"], HD1_bulk["Mn"], HD1_bulk["Zn"]],  # type: ignore
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=500,
    bardata=HD1_bulk["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[
        (2200, 2700, 730, 1000),
    ],
    data=HD1_bulk["Mn"],
    textcolor=[
        "w",
    ],
    text_shift=([14, 10],),
    roi_name=[
        r"Bulk",
    ],
    edgecolor=[
        "w",
    ],
    zorder=2,
)
ax.text(-0.04, 1.0, r"E", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 E Surface
ax = subfig.add_subplot()
ax.set_position((0.1, -0.25, 1.25, 1.25))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [HD1_surface["K"], HD1_surface["Mn"], HD1_surface["Zn"]],  # type: ignore
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=500,
    bardata=HD1_surface["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[
        (290, 460, 850, 1000),
    ],
    data=HD1_surface["Mn"],
    textcolor=[
        "w",
    ],
    text_shift=([25, 12],),
    roi_name=[
        r"Surface",
    ],
    edgecolor=[
        "w",
    ],
    zorder=2,
)

# 图 F
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.5, -0.1, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [FD2["K"], FD2["Mn"], FD2["Zn"]],
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=50,
    bardata=FD2["K"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[(220, 320, 100, 300), (430, 500, 100, 300)],
    data=FD2["Mn"],
    textcolor=["w", "w"],
    text_shift=([4, 4], [7, 4]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)
ax.text(-0.04, 1.0, r"F", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 G
subfig = fig.add_subfigure(gs[2, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.05, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.7)

labels = [r"K/Mn", r"Mn/O", r"Zn/O", r"Zn/Mn"]
markers = [r"o", r"s", r"D", r"*"]
for i in range(bulk.shape[1] - 3):
    ax.plot(
        np.arange(bulk.shape[0]),
        bulk.loc[:, labels[i]],
        c=colors[i],
        lw=1,
        ls="-",
        marker=markers[i],
        ms=8,
        mec=colors[i],
        mfc=colors[i],
        label=labels[i],
        zorder=0,
    )

ax.set_ylabel(r"Relative Ratio (arb.u.)", fontsize=9)
ax.set_ylim(-0.05, 0.8)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))

ax.set_xticks(np.arange(bulk.shape[0]), labels=bulk["Samples"])
plt.setp(ax.get_xticklabels(), rotation=90, ha="right", rotation_mode="anchor", fontfamily="Arial", fontsize=9)
ax.tick_params(axis="y", which="both", direction="out", left=True, labelleft=True, right=False)

ax.text(
    -0.11,
    1.0,
    r"G",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.5, 1.0),
    ncols=2,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    labelspacing=1.0,
)
ax.text(
    0.05,
    0.95,
    r"Bulk",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="left",
    fontfamily="Arial",
)

# 图 H
subfig = fig.add_subfigure(gs[2, 1:3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.7)

labels = [r"K/Mn", r"Mn/O", r"Zn/O"]
markers = [r"o", r"s", r"D"]
for i in range(surface.shape[1] - 4):
    ax.plot(
        np.arange(surface.shape[0]),
        surface.loc[:, labels[i]],
        c=colors[i],
        lw=1,
        ls="--",
        marker=markers[i],
        ms=8,
        mec=colors[i],
        mfc=colors[i],
        label=labels[i],
        zorder=0,
    )

ax.set_ylabel(r"Relative Ratio (arb.u.)", fontsize=9)
ax.set_ylim(-0.05, 1.0)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))

ax.set_xticks(np.arange(bulk.shape[0]), labels=bulk["Samples"])
plt.setp(ax.get_xticklabels(), rotation=90, ha="right", rotation_mode="anchor", fontfamily="Arial", fontsize=9)
ax.tick_params(axis="y", which="both", direction="out", left=True, labelleft=True, right=False)

ax.text(
    -0.11,
    1.0,
    r"H",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.5, 1.0),
    ncols=2,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    labelspacing=1.0,
)
ax.text(
    0.05,
    0.95,
    r"Surface",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="left",
    fontfamily="Arial",
)

ax2 = ax.twinx()
ax2.set_position((0.7, 0.0, 1.0, 1.0))
ax2.set_box_aspect(0.7)

labels = [r"Zn/Mn"]
markers = [r"*"]
for i in range(1):
    ax2.plot(
        np.arange(surface.shape[0]),
        surface.loc[:, labels[i]],
        c=colors[i + 3],
        lw=1,
        ls="--",
        marker=markers[i],
        ms=8,
        mec=colors[i + 3],
        mfc=colors[i + 3],
        label=labels[i],
        zorder=0,
    )
ax2.set_ylabel(r"Relative Ratio (arb.u.)", fontsize=9)
ax2.set_ylim(-0.05, 3.0)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.6, offset=0))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.3, offset=0))
ax2.legend(
    loc="upper left",
    bbox_to_anchor=(0.71, 0.92),
    ncols=2,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    labelspacing=1.0,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_02_V2_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_02_V2_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_02_V2_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_02_V2_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure 对应的 Surface and bulk EDS 的结果

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[0].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[0].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    results = []
    for file in data:
        if len(file.axes_manager.shape) >= 2:
            if len(file.axes_manager.navigation_shape) == 2:
                if file.axes_manager.navigation_axes[0].units == r"µm":
                    file.axes_manager.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(file.axes_manager.signal_shape) == 2:  # noqa: SIM102
                if file.axes_manager.signal_axes[0].units == r"µm":
                    file.axes_manager.convert_units(axes="signal", units="nm", same_units=True, factor=1000)

        if len(file.axes_manager.shape) == 3:
            if len(file.axes_manager.navigation_shape) == 2:
                for axis in file.axes_manager.navigation_axes:
                    axis.offset = 0
            elif len(file.axes_manager.signal_shape) == 2:
                for axis in file.axes_manager.signal_axes:
                    axis.offset = 0
        results.append(file)
    return results


def element_maps(signal_list, elements=(r"K", r"Mn", r"Zn"), crop_box=None):
    element_maps = {}
    for signal in signal_list:
        title = signal.metadata["General"]["title"]
        if title in elements:
            if crop_box:
                if signal.axes_manager.navigation_shape:
                    signal_crop = signal.inav[crop_box].copy()
                elif signal.axes_manager.signal_shape:
                    signal_crop = signal.isig[crop_box].copy()
            else:
                signal_crop = signal.copy()

            element_maps[title] = signal_crop
    return element_maps


eds_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def rois_plot(ax, roi_coords, data, textcolor=None, text_shift=([10, 5], [10, 5]), roi_name=None, edgecolor=None, zorder=2) -> None:
    scale_x = data.axes_manager[0].scale
    scale_y = data.axes_manager[1].scale
    for i, (left, right, top, bottom) in enumerate(roi_coords):
        roi = hs.roi.RectangularROI(left=left, right=right, top=top, bottom=bottom)

        x = int(roi.x / scale_x)
        y = int(roi.y / scale_y)
        width = int(roi.width / scale_x)
        height = int(roi.height / scale_y)

        # 绘制矩形
        rect = patches.Rectangle(
            (x, y),
            width,
            height,
            linewidth=1,
            edgecolor=edgecolor[i] if edgecolor else "y",
            facecolor="none",
            transform=ax.transData,
            zorder=zorder,
        )
        ax.add_patch(rect)

        # 添加文本标签
        ax.text(
            x + text_shift[i][0],
            y - text_shift[i][1],
            f"{roi_name[i]}" if roi_name else f"ROI {i + 1}",
            fontsize=10,
            color=textcolor[i] if textcolor else "y",
            ha="center",
            va="center",
            transform=ax.transData,
            zorder=zorder + 1,
        )

In [ ]:
# 读取 EDS 的数据

path_FD1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\EDS"
)
FD1 = hs.load(path_FD1.joinpath(r"Data", r"0003 - B8_HAADF_67000_x", r"0003 - B8_HAADF_67000_x.emd"))  # type: ignore

path_HC1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EDS"
)
HC1_bulk = hs.load(path_HC1.joinpath(r"Data", r"0004-20241211_HAADF_29000_x_EDS", r"0004-20241211_HAADF_29000_x_EDS.emd"))  # type: ignore

HC1_surface = hs.load(path_HC1.joinpath(r"Data", r"0007-20241211_HAADF_115_kx_EDS", r"0007-20241211_HAADF_115_kx_EDS.emd"))  # type: ignore

path_HC2 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EDS"
)
HC2 = hs.load(path_HC2.joinpath(r"Data", r"0012-20241211_HAADF_58000_x_EDS", r"0012-20241211_HAADF_58000_x_EDS.emd"))  # type: ignore

path_FC1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\EDS"
)
FC1 = hs.load(path_FC1.joinpath(r"Data", r"0020 - 1406 Pristine MnO2 HAADF 10000 x", r"0020 - 1406 Pristine MnO2 HAADF 10000 x.emd"))

path_HD1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\EDS"
)
HD1_bulk = hs.load(path_HD1.joinpath(r"Data", r"0015-HAADF_23500_x_EDS_SI", r"0015-HAADF_23500_x_EDS_SI.emd"))  # type: ignore

HD1_surface = hs.load(path_HD1.joinpath(r"Data", r"0013-HAADF_33000_x_EDS_SI", r"0013-HAADF_33000_x_EDS_SI.emd"))  # type: ignore

path_FD2 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\EDS"
)
FD2 = hs.load(path_FD2.joinpath(r"Data", r"0028-HAADF_67000_x_EDS_SI", r"0028-HAADF_67000_x_EDS_SI.emd"))  # type: ignore

In [ ]:
FD1 = convert_units(FD1)
HC1_bulk = convert_units(HC1_bulk)
HC1_surface = convert_units(HC1_surface)
HC2 = convert_units(HC2)
FC1 = convert_units(FC1)
HD1_bulk = convert_units(HD1_bulk)
HD1_surface = convert_units(HD1_surface)
FD2 = convert_units(FD2)

FD1 = element_maps(FD1, elements=["K", "Mn", "Zn"])
HC1_bulk = element_maps(HC1_bulk, elements=["K", "Mn", "Zn"])
HC1_surface = element_maps(HC1_surface, elements=["K", "Mn", "Zn"])
HC2 = element_maps(HC2, elements=["K", "Mn", "Zn"])
FC1 = element_maps(FC1, elements=["K", "Mn", "Zn"])
HD1_bulk = element_maps(HD1_bulk, elements=["K", "Mn", "Zn"])
HD1_surface = element_maps(HD1_surface, elements=["K", "Mn", "Zn"])
FD2 = element_maps(FD2, elements=["K", "Mn", "Zn"])

In [ ]:
# 交换 HD1 中 X Y 的数据
for key, data in HD1_surface.items():
    HD1_surface[key].data = data.data.T
for key, data in HD1_bulk.items():
    HD1_bulk[key].data = data.data.T
for key, data in FD2.items():
    FD2[key].data = data.data.T

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 6.0))
gs = gridspec.GridSpec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [FD1["K"], FD1["Mn"], FD1["Zn"]],
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    ax,
    size=50,
    bardata=FD1["K"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)

ax.text(
    0.02,
    1.22,
    r"K",
    c="w",
    bbox={"ec": None, "fc": eds_colors["K"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
ax.text(
    0.10,
    1.22,
    r"Mn",
    c="k",
    bbox={"ec": None, "fc": eds_colors["Mn"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
ax.text(
    0.22,
    1.22,
    r"Zn",
    c="w",
    bbox={"ec": None, "fc": eds_colors["Zn"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[(280, 420, 110, 140), (250, 420, 180, 200)],
    data=FD1["Mn"],
    textcolor=["w", "w"],
    text_shift=([15, 8], [25, -18]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)
ax.text(-0.04, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")
# 图 B Bulk
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.15, -0.22, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [HC1_bulk["K"], HC1_bulk["Mn"], HC1_bulk["Zn"]],  # type: ignore
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    ax,
    size=200,
    bardata=HC1_bulk["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[
        (870, 1200, 870, 1150),
    ],
    data=HC1_bulk["Mn"],
    textcolor=[
        "w",
    ],
    text_shift=([15, 10],),
    roi_name=[
        r"Bulk",
    ],
    edgecolor=[
        "w",
    ],
    zorder=2,
)
ax.text(-0.04, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B Surface
ax = subfig.add_subplot()
ax.set_position((1.2, -0.125, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [HC1_surface["K"], HC1_surface["Mn"], HC1_surface["Zn"]],  # type: ignore
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    ax,
    size=100,
    bardata=HC1_surface["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[
        (80, 450, 50, 100),
    ],
    data=HC1_surface["Mn"],
    textcolor=[
        "w",
    ],
    text_shift=([25, 12],),
    roi_name=[
        r"Surface",
    ],
    edgecolor=[
        "w",
    ],
    zorder=2,
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((1.35, -0.32, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [HC2["K"], HC2["Mn"], HC2["Zn"]],
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    ax,
    size=100,
    bardata=HC2["K"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[(350, 420, 600, 840), (170, 290, 50, 300)],
    data=HC2["Mn"],
    textcolor=["w", "w"],
    text_shift=([12, 8], [12, -85]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)
ax.text(-0.04, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.5, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [FC1["K"], FC1["Mn"], FC1["Zn"]],
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    ax,
    size=500,
    bardata=FC1["K"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[(1000, 1700, 2500, 2900), (2300, 2700, 1350, 1650)],
    data=FC1["Mn"],
    textcolor=["w", "w"],
    text_shift=([25, 20], [-10, 15]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)
ax.text(-0.04, 1.0, r"D", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 E Bulk
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-1.0, -0.13, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [HD1_bulk["K"], HD1_bulk["Mn"], HD1_bulk["Zn"]],  # type: ignore
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    ax,
    size=500,
    bardata=HD1_bulk["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[
        (2200, 2700, 730, 1000),
    ],
    data=HD1_bulk["Mn"],
    textcolor=[
        "w",
    ],
    text_shift=([14, 10],),
    roi_name=[
        r"Bulk",
    ],
    edgecolor=[
        "w",
    ],
    zorder=2,
)
ax.text(-0.04, 1.0, r"E", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 E Surface
ax = subfig.add_subplot()
ax.set_position((0.1, -0.25, 1.25, 1.25))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [HD1_surface["K"], HD1_surface["Mn"], HD1_surface["Zn"]],  # type: ignore
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    ax,
    size=500,
    bardata=HD1_surface["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[
        (290, 460, 850, 1000),
    ],
    data=HD1_surface["Mn"],
    textcolor=[
        "w",
    ],
    text_shift=([25, 12],),
    roi_name=[
        r"Surface",
    ],
    edgecolor=[
        "w",
    ],
    zorder=2,
)

# 图 F
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.5, -0.1, 1.0, 1.0))
# ax.set_box_aspect(0.5)
ax.set_axis_off()

hs.plot.plot_images(
    [FD2["K"], FD2["Mn"], FD2["Zn"]],
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    ax,
    size=50,
    bardata=FD2["K"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[(220, 320, 100, 300), (430, 500, 100, 300)],
    data=FD2["Mn"],
    textcolor=["w", "w"],
    text_shift=([4, 4], [7, 4]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)
ax.text(-0.04, 1.0, r"F", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_02_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_02_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_02_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_02_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure 常规电化学 + TEM-EDS 的各个状态 - V1 -舍弃

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[0].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[0].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    results = []
    for file in data:
        if len(file.axes_manager.shape) >= 2:
            if len(file.axes_manager.navigation_shape) == 2:
                if file.axes_manager.navigation_axes[0].units == r"µm":
                    file.axes_manager.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(file.axes_manager.signal_shape) == 2:  # noqa: SIM102
                if file.axes_manager.signal_axes[0].units == r"µm":
                    file.axes_manager.convert_units(axes="signal", units="nm", same_units=True, factor=1000)

        if len(file.axes_manager.shape) == 3:
            if len(file.axes_manager.navigation_shape) == 2:
                for axis in file.axes_manager.navigation_axes:
                    axis.offset = 0
            elif len(file.axes_manager.signal_shape) == 2:
                for axis in file.axes_manager.signal_axes:
                    axis.offset = 0
        results.append(file)
    return results


def element_maps(signal_list, elements=(r"K", r"Mn", r"Zn"), crop_box=None):
    element_maps = {}
    for signal in signal_list:
        title = signal.metadata["General"]["title"]
        if title in elements:
            if crop_box:
                if signal.axes_manager.navigation_shape:
                    signal_crop = signal.inav[crop_box].copy()
                elif signal.axes_manager.signal_shape:
                    signal_crop = signal.isig[crop_box].copy()
            else:
                signal_crop = signal.copy()

            element_maps[title] = signal_crop
    return element_maps


eds_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}

In [ ]:
# 读取一个常规的 αMnO2 的数据
echem = []
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\1M ZnSO4 + 0.2M MnSO4").glob(r"LC*.xlsx"))
# 读取电化学数据
for path_file in path_filelist[0:1]:
    df = pd.read_excel(path_file, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echem.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echem)):
    echem[i] = echem[i][echem[i]["Cycle"].isin([0, 1, 2])]
    echem[i] = echem[i][echem[i]["Voltage/V"] >= 0]

# 读取 EDS 的数据
path_pristine = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Pristine\αMnO2 + CNTs\2024-EMCA\EDS\0002 - B8_HAADF_47000_x")
pristine = hs.load(path_pristine.joinpath(r"Data", r"0002 - B8_HAADF_47000_x.emd"))  # type: ignore

path_FD1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\EDS\Data"
)
FD1 = hs.load(path_FD1.joinpath(r"0003 - B8_HAADF_67000_x", r"0003 - B8_HAADF_67000_x.emd"))  # type: ignore

path_HC1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EDS\Data"
)
HC1 = hs.load(path_HC1.joinpath(r"0005-20241211_HAADF_82000_x_EDS", r"0005-20241211_HAADF_82000_x_EDS.emd"))  # type: ignore

path_HC2 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EDS\Data"
)
HC2 = hs.load(path_HC2.joinpath(r"0009-20241211_HAADF_82000_x_EDS", r"0009-20241211_HAADF_82000_x_EDS.emd"))  # type: ignore

path_FC1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\EDS\Data"
)
FC1 = hs.load(path_FC1.joinpath(r"0022 - 1508 Pristine MnO2 HAADF 41000 x", r"0022 - 1508 Pristine MnO2 HAADF 41000 x.emd"))

path_HD1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\EDS\Data"
)
HD1 = hs.load(path_HD1.joinpath(r"0013-HAADF_33000_x_EDS_SI", r"0013-HAADF_33000_x_EDS_SI.emd"))  # type: ignore

path_FD2 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\EDS\Data"
)
FD2 = hs.load(path_FD2.joinpath(r"0028-HAADF_67000_x_EDS_SI", r"0028-HAADF_67000_x_EDS_SI.emd"))  # type: ignore

# TEM-EDS 的比例
path_fc_ratio = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\TEM-EDS-EELS-all").joinpath(r"TEM-EDX_Bulk_Surface.xlsx")
fc_ratio = pd.read_excel(path_fc_ratio, sheet_name=1, header=0, index_col=None).dropna(axis=1, how="all")
# 清洗数据
bulk = fc_ratio[fc_ratio["Position"] == "Bulk"].copy(deep=True)
surface = fc_ratio[fc_ratio["Position"] == "Surface"].copy(deep=True)

In [ ]:
pristine = convert_units(pristine)
FD1 = convert_units(FD1)
HC1 = convert_units(HC1)
HC2 = convert_units(HC2)
FC1 = convert_units(FC1)
HD1 = convert_units(HD1)
FD2 = convert_units(FD2)

pristine = element_maps(pristine, elements=["K", "Mn"], crop_box=(slice(0, 60), slice(None)))
FD1 = element_maps(FD1, elements=["K", "Mn", "Zn"], crop_box=(slice(90, 200), slice(None)))
HC1 = element_maps(HC1, elements=["K", "Mn", "Zn"], crop_box=(slice(None), slice(90, 200)))
HC2 = element_maps(HC2, elements=["K", "Mn", "Zn"])
FC1 = element_maps(FC1, elements=["K", "Mn", "Zn"])
HD1 = element_maps(HD1, elements=["K", "Mn", "Zn"])
FD2 = element_maps(FD2, elements=["K", "Mn", "Zn"])

# 交换 HD1， FD2 中 X Y 的数据
for key, data in HD1.items():
    HD1[key].data = data.data.T
for key, data in FD2.items():
    FD2[key].data = data.data.T

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.3)

ax.set_axis_off()
labels = [
    [None, None],
]
for i, data in enumerate(echem):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        if j == 0:
            ax.plot(
                temp.loc[idx_min:, "SpeCap/mAh/g"] + temp.loc[idx_min - 1, "SpeCap/mAh/g"],
                temp.loc[idx_min:, "Voltage/V"],
                ls="-",
                lw=1.0,
                c=colors[j],
                zorder=0,
            )  # 充电
        if j == 1:
            ax.plot(
                temp.loc[:idx_min, "SpeCap/mAh/g"] + 690,
                temp.loc[:idx_min, "Voltage/V"],
                ls="-",
                lw=1.0,
                c=colors[j],
                label=labels[i][j],
                zorder=0,
            )  # 放电

# 插入图 A
axins = ax.inset_axes((-0.06, 0.0, 0.4, 0.4), zorder=1)
hs.plot.plot_images(
    [FD1["K"], FD1["Mn"], FD1["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=50,
    bardata=FD1["K"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, -0.05, 0, 0), transform=axins.transAxes)
axins.annotate(
    r"FD#1",
    (-0.15, 0.05),
    xytext=(0.6, -0.15),
    fontsize=13,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
axins.text(
    0.05,
    1.20,
    r"K",
    c="w",
    bbox={"ec": None, "fc": eds_colors["K"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=axins.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
axins.text(
    0.22,
    1.20,
    r"Mn",
    c="k",
    bbox={"ec": None, "fc": eds_colors["Mn"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=axins.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
axins.text(
    0.47,
    1.20,
    r"Zn",
    c="w",
    bbox={"ec": None, "fc": eds_colors["Zn"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=axins.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)

# 插入图 B
axins = ax.inset_axes((0.16, 0.05, 0.3, 0.4), zorder=1)
hs.plot.plot_images(
    [HC1["K"], HC1["Mn"], HC1["Zn"]],  # type: ignore
    ax=axins,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=50,
    bardata=HC1["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"HC#1",
    (0.5, 1.55),
    xytext=(1.0, 1.1),
    fontsize=13,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=90,angleB=0,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])

# 插入图 C
axins = ax.inset_axes((0.28, 0.05, 0.4, 0.5), zorder=1)
hs.plot.plot_images(
    [HC2["K"], HC2["Mn"], HC2["Zn"]],  # type: ignore
    ax=axins,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=50,
    bardata=HC2["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"HC#2",
    (0.4, 1.52),
    xytext=(1.0, 1.07),
    fontsize=13,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=90,angleB=0,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])

# 插入图 D
axins = ax.inset_axes((0.5, 0.6, 0.5, 0.5), zorder=1)
hs.plot.plot_images(
    [FC1["K"], FC1["Mn"], FC1["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=50,
    bardata=FC1["K"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"FC#1",
    (-0.5, 0.7),
    xytext=(0.0, 0.8),
    fontsize=13,
    c=colors[0],
    rotation=90,
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=90,angleB=0,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])

# 插入图 E
axins = ax.inset_axes((0.48, 0.00, 0.35, 0.35), zorder=1)

hs.plot.plot_images(
    [HD1["K"], HD1["Mn"], HD1["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=500,
    bardata=HD1["K"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"HD#1",
    (0.8, 1.3),
    xytext=(0.5, 1.1),
    fontsize=13,
    c=colors[1],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])

# 插入图 F
axins = ax.inset_axes((0.68, 0.0, 0.3, 0.3), zorder=1)
hs.plot.plot_images(
    [FD2["K"], FD2["Mn"], FD2["Zn"]],  # type: ignore
    ax=axins,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=100,
    bardata=FD2["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"FD#2",
    (1.55, 0.15),
    xytext=(1.0, -0.15),
    fontsize=13,
    c=colors[1],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])

# 插入图 G
axins = ax.inset_axes((-0.05, 0.72, 0.4, 0.4), zorder=1)
hs.plot.plot_images(
    [pristine["K"], pristine["Mn"]],  # type: ignore
    ax=axins,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=100,
    bardata=pristine["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"Pristine",
    (0.3, 1.42),
    xytext=(0.46, -0.14),
    fontsize=13,
    c=colors[2],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops=None,
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])

ax.text(0.04, 1.10, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[1, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.07, 0.08, 0.86, 0.86))
ax.set_box_aspect(0.4)

labelsa = ["K/Mn", "Mn/O", "Zn/O"]

for i in range(bulk.shape[1] - 3):
    ax.plot(
        np.arange(bulk.shape[0]),
        bulk.iloc[:, i + 2],
        c=colors[i],
        lw=1,
        ls="-",
        marker="o",
        ms=5,
        mec=colors[i],
        mfc=colors[i],
        label=labelsa[i],
        zorder=0,
    )

ax.set_ylabel(r"Relative Ratio (arb.u.)", fontsize=9)
ax.set_ylim(-0.05, 0.8)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))

ax.set_xticks(np.arange(bulk.shape[0]), labels=bulk["Samples"])
plt.setp(ax.get_xticklabels(), rotation=90, ha="right", rotation_mode="anchor", fontfamily="Arial", fontsize=9)
ax.tick_params(axis="y", which="both", direction="out", left=True, labelleft=True, right=False)

ax.text(
    -0.10,
    1.0,
    r"B",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

ax2 = ax.twinx()
ax2.set_position((0.07, 0.08, 0.86, 0.86))
ax2.set_box_aspect(0.4)

labelsb = ["K/Mn", "Mn/O", "Zn/O"]

for i in range(surface.shape[1] - 3):
    ax2.plot(
        np.arange(surface.shape[0]),
        surface.iloc[:, i + 2],
        c=colors[i],
        lw=1,
        ls="-",
        marker="D",
        ms=5,
        mec=colors[i],
        mfc=colors[i],
        alpha=0.7,
    )

ax2.set_ylabel(r"Relative Ratio (arb.u.)", fontsize=9)
ax2.set_ylim(-0.05, 1.0)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))

from matplotlib.lines import Line2D  # noqa: E402

# 图例 A：颜色代表比例类型
color_legend = [
    Line2D([0], [0], color=colors[0], lw=1, label="K/Mn"),
    Line2D([0], [0], color=colors[1], lw=1, label="Mn/O"),
    Line2D([0], [0], color=colors[2], lw=1, label="Zn/O"),
]

# 图例 B：marker 代表数据来源
marker_legend = [
    Line2D([0], [0], color="grey", marker="o", linestyle="None", markersize=4, label="Bulk"),
    Line2D([0], [0], color="grey", marker="D", linestyle="None", markersize=4, label="Surface"),
]

# 添加图例 A
legend1 = subfig.legend(handles=color_legend, loc="upper right", bbox_to_anchor=(0.7, 0.95), ncol=3, fontsize=9, frameon=False)

# 添加图例 B
legend2 = subfig.legend(handles=marker_legend, loc="upper right", bbox_to_anchor=(0.85, 0.95), ncol=1, fontsize=9, frameon=False)

# 确保两个图例都显示
subfig.add_artist(legend1)
subfig.add_artist(legend2)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_01_V1_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_01_V1_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_01_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_01_V1_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure 常规电化学 + TEM-EDS 的各个状态 -舍弃

In [ ]:
def EdsMappings(data: list | tuple, colors_map: dict = None) -> np.ndarray:  # type: ignore # noqa: N802, RUF013
    """
    读取 EDS 中的 K, Mn, Zn 的数据
    """
    images: list[dict[str, np.ndarray]] = []
    if isinstance(data, (list, tuple)):
        for file in data:
            if getattr(file, "metadata", None) and file.metadata["General"]["title"] in [r"K", r"Mn", r"Zn"] and getattr(file.data, "ndim", 0) == 2:
                img: np.ndarray = file.data  # type: ignore
                img = (img - img.min()) / (img.max() - img.min() + 1e-8)
                images.append({"title": file.metadata["General"]["title"], "data": img})

    if colors_map is None:
        colors_map = {
            "K": np.array([26 / 255, 51 / 255, 203 / 255]),
            "Mn": np.array([236 / 255, 236 / 255, 0 / 255]),
            "Zn": np.array([252 / 255, 0, 252 / 255]),
        }
    # 创建 RGB 图像
    if len(images) == 0:
        raise ValueError("No valid EDS images found in data.")
    shape = images[0]["data"].shape
    eds_mapping: np.ndarray = np.zeros((*shape, 3))
    for img in images:  # type: ignore
        if img["title"] in colors_map:
            eds_mapping += img["data"][..., None] * colors_map[img["title"]]

    # 裁剪防止饱和值超过 1
    eds_mapping = np.clip(eds_mapping, 0, 1)
    return eds_mapping


from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar  # noqa: E402


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[0].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[0].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb

In [ ]:
# 读取一个常规的 αMnO2 的数据
echem = []
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\1M ZnSO4 + 0.2M MnSO4").glob(r"LC*.xlsx"))
# 读取电化学数据
for path_file in path_filelist[0:1]:
    df = pd.read_excel(path_file, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echem.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echem)):
    echem[i] = echem[i][echem[i]["Cycle"].isin([0, 1, 2])]
    echem[i] = echem[i][echem[i]["Voltage/V"] >= 0]

# 读取 EDS 的数据
path_pristine = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Pristine\αMnO2 + CNTs\2024-EMCA\EDS\0002 - B8_HAADF_47000_x")
pristine = hs.load(path_pristine.joinpath(r"Data", r"0002 - B8_HAADF_47000_x.emd"))  # type: ignore

path_FD1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\EDS\Data"
)
FD1 = hs.load(path_FD1.joinpath(r"0003 - B8_HAADF_67000_x", r"0003 - B8_HAADF_67000_x.emd"))  # type: ignore

path_HC1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EDS\Data"
)
HC1 = hs.load(path_HC1.joinpath(r"0005-20241211_HAADF_82000_x_EDS", r"0005-20241211_HAADF_82000_x_EDS.emd"))  # type: ignore

path_HC2 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EDS\Data"
)
HC2 = hs.load(path_HC2.joinpath(r"0009-20241211_HAADF_82000_x_EDS", r"0009-20241211_HAADF_82000_x_EDS.emd"))  # type: ignore

path_FC1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\EDS\Data"
)
FC1 = hs.load(path_FC1.joinpath(r"0022 - 1508 Pristine MnO2 HAADF 41000 x", r"0022 - 1508 Pristine MnO2 HAADF 41000 x.emd"))

path_HD1 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\EDS\Data"
)
HD1 = hs.load(path_HD1.joinpath(r"0013-HAADF_33000_x_EDS_SI", r"0013-HAADF_33000_x_EDS_SI.emd"))  # type: ignore

path_FD2 = Path(  # noqa: N816
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\EDS\Data"
)
FD2 = hs.load(path_FD2.joinpath(r"0028-HAADF_67000_x_EDS_SI", r"0028-HAADF_67000_x_EDS_SI.emd"))  # type: ignore

# TEM-EDS 的比例
path_fc_ratio = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\TEM-EDS-EELS-all").joinpath(r"TEM-EDX_Bulk_Surface.xlsx")
fc_ratio = pd.read_excel(path_fc_ratio, sheet_name=1, header=0, index_col=None).dropna(axis=1, how="all")
# 清洗数据
bulk = fc_ratio[fc_ratio["Position"] == "Bulk"].copy(deep=True)
surface = fc_ratio[fc_ratio["Position"] == "Surface"].copy(deep=True)

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.3)

ax.set_axis_off()

labels = [
    [None, None],
]
for i, data in enumerate(echem):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        if j == 0:
            ax.plot(
                temp.loc[idx_min:, "SpeCap/mAh/g"] + temp.loc[idx_min - 1, "SpeCap/mAh/g"],
                temp.loc[idx_min:, "Voltage/V"],
                ls="-",
                lw=1.0,
                c=colors[i],
                zorder=0,
            )  # 充电
        if j == 1:
            ax.plot(
                temp.loc[:idx_min, "SpeCap/mAh/g"] + 690,
                temp.loc[:idx_min, "Voltage/V"],
                ls="-",
                lw=1.0,
                c=colors[i],
                label=labels[i][j],
                zorder=0,
            )  # 放电

colors_map = {
    "K": np.array([26 / 255, 51 / 255, 203 / 255]),
    "Mn": np.array([255 / 255, 255 / 255, 0 / 255]),
    "Zn": np.array([252 / 255, 0, 252 / 255]),
}

# 插入图 A
axins = ax.inset_axes((-0.06, 0.05, 0.4, 0.4), zorder=1)

FD1A = EdsMappings(FD1, colors_map=colors_map)
FD1A = FD1A[:, 90:200, :]
axins.imshow(FD1A, alpha=1.0, zorder=2)
scalebar = add_sizebar(axins, 50, FD1[4], "w")
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()

axins.text(
    0.05,
    1.20,
    r"K",
    c="w",
    bbox={"ec": None, "fc": colors_map["K"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=axins.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
axins.text(
    0.22,
    1.20,
    r"Mn",
    c="k",
    bbox={"ec": None, "fc": colors_map["Mn"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=axins.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
axins.text(
    0.47,
    1.20,
    r"Zn",
    c="w",
    bbox={"ec": None, "fc": colors_map["Zn"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=axins.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
axins.annotate(
    r"FD#1",
    (-0.15, 0.0),
    xytext=(0.5, -0.15),
    fontsize=13,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={"arrowstyle": "->", "ls": "-", "lw": 1.0, "color": "grey"},
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])

# 插入图 B
axins = ax.inset_axes((0.12, 0.05, 0.3, 0.4), zorder=1)
# axins.set_box_aspect(0.8)

HC1A = EdsMappings(HC1, colors_map=colors_map)
HC1A = ndimage.rotate(HC1A, angle=90, axes=(0, 1), reshape=True)
np.clip(HC1A, 0, 1, out=HC1A)
HC1A = HC1A[:, 90:200, :]
axins.imshow(HC1A, alpha=1.0, zorder=2)
scalebar = add_sizebar(axins, 50, HC1[11], "w")
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"HC#1",
    (0.85, 1.55),
    xytext=(1.0, 1.1),
    fontsize=13,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={"arrowstyle": "->", "ls": "-", "lw": 1.0, "color": "grey"},
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])

# 插入图 C
axins = ax.inset_axes((0.2, 0.05, 0.4, 0.5), zorder=1)
# axins.set_box_aspect(0.8)

HC2A = EdsMappings(HC2, colors_map=colors_map)
# HC2A = ndimage.rotate(HC2A, angle=90, axes=(0, 1), reshape=True)
# np.clip(HC2A, 0, 1, out=HC2A)
# HC2A = HC2A[:, 90:200, :]
axins.imshow(HC2A, alpha=1.0, zorder=2)
scalebar = add_sizebar(axins, 50, HC2[7], "w")
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"HC#2",
    (1.0, 1.52),
    xytext=(1.0, 1.07),
    fontsize=13,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=90,angleB=0,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])

# 插入图 D
axins = ax.inset_axes((0.5, 0.6, 0.5, 0.5), zorder=1)
# axins.set_box_aspect(0.8)

FC1A = EdsMappings(FC1, colors_map=colors_map)
FC1A = ndimage.rotate(FC1A, angle=0, axes=(0, 1), reshape=True)
np.clip(FC1A, 0, 1, out=FC1A)
# FC1A = FC1A[100:300, :300, :]
axins.imshow(FC1A, alpha=1.0, zorder=2)
scalebar = add_sizebar(axins, 50, FC1[5], "w")
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"FC#1",
    (-0.5, 0.7),
    xytext=(0.0, 0.8),
    fontsize=13,
    c=colors[0],
    rotation=90,
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={"arrowstyle": "->", "ls": "-", "lw": 1.0, "color": "grey"},
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])

# 插入图 E
axins = ax.inset_axes((0.4, 0.05, 0.35, 0.35), zorder=1)
# axins.set_box_aspect(0.8)

HD1A = EdsMappings(HD1, colors_map=colors_map)
HD1A = ndimage.rotate(HD1A, angle=90, axes=(0, 1), reshape=True)
np.clip(HD1A, 0, 1, out=HD1A)
# HD1A = HD1A[200:400, :, :]
axins.imshow(HD1A, alpha=1.0, zorder=2)
scalebar = add_sizebar(axins, 500, HD1[13], "w")
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"HD#1",
    (1.2, 1.1),
    xytext=(0.8, 1.1),
    fontsize=13,
    c=colors[1],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        # "connectionstyle": "angle,angleA=90,angleB=0,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])

# 插入图 F
axins = ax.inset_axes((0.65, 0.05, 0.3, 0.3), zorder=1)
# axins.set_box_aspect(0.8)

FD2A = EdsMappings(FD2, colors_map=colors_map)
FD2A = ndimage.rotate(FD2A, angle=90, axes=(0, 1), reshape=True)
np.clip(FD2A, 0, 1, out=FD2A)
# FD2A = FD2A[:60, :, :]
axins.imshow(FD2A, alpha=1.0, zorder=2)
scalebar = add_sizebar(axins, 50, FD2[6], "w")
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"FD#2",
    (1.8, 0.0),
    xytext=(1.0, -0.15),
    fontsize=13,
    c=colors[1],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={"arrowstyle": "->", "ls": "-", "lw": 1.0, "color": "grey"},
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])

# 插入图 G
axins = ax.inset_axes((-0.05, 0.72, 0.4, 0.4), zorder=1)
# axins.set_box_aspect(0.8)

pristineA = EdsMappings(pristine, colors_map=colors_map)  # noqa: N816
# pristineA = ndimage.rotate(pristineA, angle=90, axes=(0, 1), reshape=True)
# np.clip(pristineA, 0, 1, out=pristineA)
pristineA = pristineA[:, :60, :]  # noqa: N816
axins.imshow(pristineA, alpha=1.0, zorder=2)
scalebar = add_sizebar(axins, 50, pristine[4], "w")
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"Pristine",
    (0.3, 1.42),
    xytext=(0.46, -0.14),
    fontsize=13,
    c=colors[2],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops=None,
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])

ax.text(0.02, 1.10, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[1, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.08, 0.1, 0.85, 0.85))
ax.set_box_aspect(0.4)

labelsa = ["K/Mn", "Mn/O", "Zn/O"]

for i in range(bulk.shape[1] - 3):
    ax.plot(
        np.arange(bulk.shape[0]),
        bulk.iloc[:, i + 2],
        c=colors[i],
        lw=1,
        ls="-",
        marker="o",
        ms=5,
        mec=colors[i],
        mfc=colors[i],
        label=labelsa[i],
        zorder=0,
    )

ax.set_ylabel(r"Relative Ratio (arb.u.)", fontsize=9)
ax.set_ylim(-0.05, 0.8)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))

ax.set_xticks(np.arange(bulk.shape[0]), labels=bulk["Samples"])
plt.setp(ax.get_xticklabels(), rotation=90, ha="right", rotation_mode="anchor", fontfamily="Arial", fontsize=9)
ax.tick_params(axis="y", which="both", direction="out", left=True, labelleft=True, right=False)

ax.text(
    -0.10,
    1.0,
    r"B",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

ax2 = ax.twinx()
ax2.set_position((0.08, 0.1, 0.85, 0.85))
ax2.set_box_aspect(0.4)

labelsb = ["K/Mn", "Mn/O", "Zn/O"]

for i in range(surface.shape[1] - 3):
    ax2.plot(
        np.arange(surface.shape[0]),
        surface.iloc[:, i + 2],
        c=colors[i],
        lw=1,
        ls="-",
        marker="D",
        ms=5,
        mec=colors[i],
        mfc=colors[i],
        alpha=0.7,
    )

ax2.set_ylabel(r"Relative Ratio (arb.u.)", fontsize=9)
ax2.set_ylim(-0.05, 1.0)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))

from matplotlib.lines import Line2D  # noqa: E402

# 图例 A：颜色代表比例类型
color_legend = [
    Line2D([0], [0], color=colors[0], lw=1, label="K/Mn"),
    Line2D([0], [0], color=colors[1], lw=1, label="Mn/O"),
    Line2D([0], [0], color=colors[2], lw=1, label="Zn/O"),
]

# 图例 B：marker 代表数据来源
marker_legend = [
    Line2D([0], [0], color="grey", marker="o", linestyle="None", markersize=4, label="Bulk"),
    Line2D([0], [0], color="grey", marker="D", linestyle="None", markersize=4, label="Surface"),
]

# 添加图例 A
legend1 = subfig.legend(handles=color_legend, loc="upper right", bbox_to_anchor=(0.7, 0.95), ncol=3, fontsize=9, frameon=False)

# 添加图例 B
legend2 = subfig.legend(handles=marker_legend, loc="upper right", bbox_to_anchor=(0.85, 0.95), ncol=1, fontsize=9, frameon=False)

# 确保两个图例都显示
subfig.add_artist(legend1)
subfig.add_artist(legend2)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_01_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_01_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_01_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_01_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### EDX, Initial Full Charged sample and others -V1

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        if len(bardata.axes_manager.navigation_shape) == 2:
            barsize = bardata.axes_manager.navigation_axes[0].scale
            unit = bardata.axes_manager.navigation_axes[0].units
        elif len(bardata.axes_manager.signal_shape) == 2:
            barsize = bardata.axes_manager.signal_axes[0].scale
            unit = bardata.axes_manager.signal_axes[0].units
        asb = AnchoredSizeBar(
            ax.transData,
            size / barsize,  # type: ignore
            "{} {}".format(size, unit),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    results = []
    for file in data:
        if len(file.axes_manager.shape) >= 2:
            if len(file.axes_manager.navigation_shape) == 2:
                if file.axes_manager.navigation_axes[0].units == r"µm":
                    file.axes_manager.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(file.axes_manager.signal_shape) == 2:  # noqa: SIM102
                if file.axes_manager.signal_axes[0].units == r"µm":
                    file.axes_manager.convert_units(axes="signal", units="nm", same_units=True, factor=1000)

        if len(file.axes_manager.shape) == 3:
            if len(file.axes_manager.navigation_shape) == 2:
                for axis in file.axes_manager.navigation_axes:
                    axis.offset = 0
            elif len(file.axes_manager.signal_shape) == 2:
                for axis in file.axes_manager.signal_axes:
                    axis.offset = 0
        results.append(file)
    return results


def element_maps(signal_list, elements=(r"HAADF", r"K", r"Mn", r"Zn"), crop_box=None):
    element_maps = {}
    for signal in signal_list:
        title = signal.metadata["General"]["title"]
        if title in elements:
            if crop_box:
                if signal.axes_manager.navigation_shape:
                    signal_crop = signal.inav[crop_box].copy()
                elif signal.axes_manager.signal_shape:
                    signal_crop = signal.isig[crop_box].copy()
            else:
                signal_crop = signal.copy()

            element_maps[title] = signal_crop
    return element_maps


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    if color0 is None:
        return LinearSegmentedColormap.from_list("", [to_rgba(color1, 0), to_rgba(color1, 1)])
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])


eds_colors = {
    "C": "#FF0000",
    "K": "#00ff00",
    "Mn": "#FFFF00",
    "Zn": "#FF8000",
    "S": "#1f3cef",
    "O": "#00ffff",
}

In [ ]:
# 读取 EDX 数据
path_file_hc1 = Path(
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EDS\Results\0005-20241211_HAADF_82000_x_EDS\Sum"
)
eds_hc1 = hs.load(path_file_hc1.joinpath(r"0005-20241211_HAADF_82000_x_EDS.emd"))
path_file_hc2 = Path(
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EDS\Results\0012-20241211_HAADF_58000_x_EDS\Sum"
)
eds_hc2 = hs.load(path_file_hc2.joinpath(r"0012-20241211_HAADF_58000_x_EDS.emd"))
path_file_fc = Path(
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\EDS\Results\0021 - 1500 Pristine MnO2 HAADF 58000 x\Sum"
)
eds_fc = hs.load(path_file_fc.joinpath(r"0021 - 1500 Pristine MnO2 HAADF 58000 x.emd"))
path_file_hd1 = Path(
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\EDS\Results\0013-HAADF_33000_x_EDS_SI\Sum"
)
eds_hd1 = hs.load(path_file_hd1.joinpath(r"0013-HAADF_33000_x_EDS_SI.emd"))
path_file_fd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\EDS\Results\0028-HAADF_67000_x_EDS_SI\Sum")
eds_fd = hs.load(path_file_fd.joinpath(r"0028-HAADF_67000_x_EDS_SI.emd"))

In [ ]:
eds_hc1 = convert_units(eds_hc1)
eds_hc2 = convert_units(eds_hc2)
eds_fc = convert_units(eds_fc)
eds_hd1 = convert_units(eds_hd1)
eds_fd = convert_units(eds_fd)

In [ ]:
eds_hc1 = element_maps(eds_hc1, elements=[r"HAADF", r"K", r"Mn", r"Zn"], crop_box=(slice(None), slice(None)))
eds_hc2 = element_maps(eds_hc2, elements=[r"HAADF", r"K", r"Mn", r"Zn"], crop_box=(slice(40, 185), slice(None)))
eds_fc = element_maps(eds_fc, elements=[r"HAADF", r"K", r"Mn", r"Zn"], crop_box=(slice(27, 160), slice(None)))
eds_hd1 = element_maps(eds_hd1, elements=[r"HAADF", r"K", r"Mn", r"Zn"], crop_box=(slice(None), slice(None)))
eds_fd = element_maps(eds_fd, elements=[r"HAADF", r"K", r"Mn", r"Zn"], crop_box=(slice(10, 60), slice(None)))

In [ ]:
# 画图
%matplotlib inline
plt.close("all")
fig = plt.figure(figsize=(7.0, 9.0))
gs = gridspec.GridSpec(5, 4, width_ratios=[1, 1, 1, 1], height_ratios=[1, 1, 1, 1, 1], wspace=0.0, hspace=0.0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hc1["HAADF"].data
ax.imshow(img, cmap="gray", alpha=1.0)
sizebar = add_sizebar(ax, 100, eds_hc1["HAADF"], "w")
sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"A",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.4, 0.0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hc1["K"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["K"]), alpha=1.0)
# sizebar = add_sizebar(ax, 100, eds_hc1['K'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"K $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"B",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.8, 0.0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hc1["Mn"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["Mn"]), alpha=1.0)
# sizebar = add_sizebar(ax, 100, eds_hc1['Mn'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"Mn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"C",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 D
subfig = fig.add_subfigure(gs[0, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-1.2, 0.0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hc1["Zn"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["Zn"]), alpha=1.0)
# sizebar = add_sizebar(ax, 100, eds_hc1['Zn'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"Zn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"D",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 E
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, -0.1, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hc2["HAADF"].data
ax.imshow(img, cmap="gray", alpha=1.0)
sizebar = add_sizebar(ax, 100, eds_hc2["HAADF"], "w")
sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"E",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 F
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.4, -0.1, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hc2["K"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["K"]), alpha=1.0)
# sizebar = add_sizebar(ax, 100, eds_hc2['K'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"K $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"F",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 G
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.8, -0.1, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hc2["Mn"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["Mn"]), alpha=1.0)
# sizebar = add_sizebar(ax, 100, eds_hc2['Mn'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"Mn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"G",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 H
subfig = fig.add_subfigure(gs[1, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-1.2, -0.1, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hc2["Zn"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["Zn"]), alpha=1.0)
# sizebar = add_sizebar(ax, 100, eds_hc2['Zn'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"Zn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"H",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 I
subfig = fig.add_subfigure(gs[2, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, -0.2, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_fc["HAADF"].data
ax.imshow(img, cmap="gray", alpha=1.0)
sizebar = add_sizebar(ax, 50, eds_fc["HAADF"], "w")
sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"I",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 J
subfig = fig.add_subfigure(gs[2, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.4, -0.2, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_fc["K"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["K"]), alpha=1.0)
# sizebar = add_sizebar(ax, 50, eds_fc['K'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"K $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"J",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 K
subfig = fig.add_subfigure(gs[2, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.8, -0.2, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_fc["Mn"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["Mn"]), alpha=1.0)
# sizebar = add_sizebar(ax, 50, eds_fc['Mn'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"Mn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"K",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 L
subfig = fig.add_subfigure(gs[2, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-1.2, -0.2, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_fc["Zn"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["Zn"]), alpha=1.0)
# sizebar = add_sizebar(ax, 50, eds_fc['Zn'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"Zn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"L",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 M
subfig = fig.add_subfigure(gs[3, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, -0.3, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hd1["HAADF"].data
ax.imshow(img, cmap="gray", alpha=1.0)
sizebar = add_sizebar(ax, 500, eds_hd1["HAADF"], "w")
sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"M",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 N
subfig = fig.add_subfigure(gs[3, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.4, -0.3, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hd1["K"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["K"]), alpha=1.0)
# sizebar = add_sizebar(ax, 500, eds_hd1['K'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"K $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"N",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 O
subfig = fig.add_subfigure(gs[3, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.8, -0.3, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hd1["Mn"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["Mn"]), alpha=1.0)
# sizebar = add_sizebar(ax, 500, eds_hd1['Mn'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"Mn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"O",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 P
subfig = fig.add_subfigure(gs[3, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-1.2, -0.3, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hd1["Zn"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["Zn"]), alpha=1.0)
# sizebar = add_sizebar(ax, 500, eds_hd1['Zn'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"Zn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"P",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 Q
subfig = fig.add_subfigure(gs[4, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, -0.4, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_fd["HAADF"].data
ax.imshow(img, cmap="gray", alpha=1.0)
sizebar = add_sizebar(ax, 100, eds_fd["HAADF"], "w")
sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"Q",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 R
subfig = fig.add_subfigure(gs[4, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.4, -0.4, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_fd["K"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["K"]), alpha=1.0)
# sizebar = add_sizebar(ax, 100, eds_fd['K'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"K $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"R",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 S
subfig = fig.add_subfigure(gs[4, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.8, -0.4, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_fd["Mn"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["Mn"]), alpha=1.0)
# sizebar = add_sizebar(ax, 100, eds_fd['Mn'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"Mn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"S",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 T
subfig = fig.add_subfigure(gs[4, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-1.2, -0.4, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_fd["Zn"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["Zn"]), alpha=1.0)
# sizebar = add_sizebar(ax, 100, eds_fd['Zn'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"Zn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"T",
#     transform=ax.transAxes,
#     fontsize=13,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_00_V1_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_00_V1_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_00_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_00_V1_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

### EDX, Initial Full Charged sample and others

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        if len(bardata.axes_manager.navigation_shape) == 2:
            barsize = bardata.axes_manager.navigation_axes[0].scale
            unit = bardata.axes_manager.navigation_axes[0].units
        elif len(bardata.axes_manager.signal_shape) == 2:
            barsize = bardata.axes_manager.signal_axes[0].scale
            unit = bardata.axes_manager.signal_axes[0].units
        asb = AnchoredSizeBar(
            ax.transData,
            size / barsize,  # type: ignore
            "{} {}".format(size, unit),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    if color0 is None:
        return LinearSegmentedColormap.from_list("", [to_rgba(color1, 0), to_rgba(color1, 1)])
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])


eds_colors = {
    "C": "#FF0000",
    "K": "#FF8000",
    "Mn": "#FFFF00",
    "Zn": "#1f3cef",
    "S": "#00ffff",
    "O": "#00ff00",
}

In [ ]:
path_file = Path(
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\EDS\Results\0021 - 1500 Pristine MnO2 HAADF 58000 x\Sum"
)
eds_fc = hs.load(path_file.joinpath(r"0021 - 1500 Pristine MnO2 HAADF 58000 x.emd"))

for file in eds_fc:
    if len(file.axes_manager.shape) >= 2:
        if len(file.axes_manager.navigation_shape) == 2:
            if file.axes_manager.navigation_axes[0].units == r"µm":
                file.axes_manager.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
        elif len(file.axes_manager.signal_shape) == 2 and file.axes_manager.signal_axes[0].units == r"µm":
            file.axes_manager.convert_units(axes="signal", units="nm", same_units=True, factor=1000)

    if len(file.axes_manager.shape) == 3:
        if len(file.axes_manager.navigation_shape) == 2:
            for axis in file.axes_manager.navigation_axes:
                axis.offset = 0
        elif len(file.axes_manager.signal_shape) == 2:
            for axis in file.axes_manager.signal_axes:
                axis.offset = 0

In [ ]:
# 画图

fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1], wspace=0.0, hspace=0.0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.0)

img = eds_fc[7].data
ax.imshow(img, cmap="gray", alpha=1.0)
sizebar = add_sizebar(ax, 50, eds_fc[7], "w")
sizebar.set_bbox_to_anchor(Bbox.from_bounds(0.15, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.35,
    0.90,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=9,
    va="bottom",
    ha="right",
    fontfamily="Arial",
    color="w",
)
ax.text(
    0.10,
    1.0,
    r"A",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.15, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.0)

img = eds_fc[4].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["Mn"]), alpha=1.0)
# sizebar = add_sizebar(ax, 50, eds_fc[4], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(0.15, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.35,
    0.90,
    r"Mn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="bottom",
    ha="right",
    fontfamily="Arial",
    color="w",
)
ax.text(
    0.10,
    1.0,
    r"B",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.3, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.0)

img = eds_fc[6].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["Zn"]), alpha=1.0)
# sizebar = add_sizebar(ax, 50, eds_fc[6], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(0.15, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.35,
    0.90,
    r"Zn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="bottom",
    ha="right",
    fontfamily="Arial",
    color="w",
)
ax.text(
    0.10,
    1.0,
    r"C",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 D
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.0)

img = eds_fc[9].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["K"]), alpha=1.0)
# sizebar = add_sizebar(ax, 50, eds_fc[9], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(0.15, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.30,
    0.90,
    r"K $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="bottom",
    ha="right",
    fontfamily="Arial",
    color="w",
)
ax.text(
    0.10,
    1.0,
    r"D",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 E
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.15, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.0)

img = eds_fc[5].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["S"]), alpha=1.0)
# sizebar = add_sizebar(ax, 50, eds_fc[5], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(0.15, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.30,
    0.90,
    r"S $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="bottom",
    ha="right",
    fontfamily="Arial",
    color="w",
)
ax.text(
    0.10,
    1.0,
    r"E",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 E
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.3, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.0)

img = eds_fc[7].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["O"]), alpha=1.0)
# sizebar = add_sizebar(ax, 50, eds_fc[7], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(0.15, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.30,
    0.90,
    r"O $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="bottom",
    ha="right",
    fontfamily="Arial",
    color="w",
)
ax.text(
    0.10,
    1.0,
    r"F",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_00_V0_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_00_V0_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_00_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_TEM_EDS_00_V0_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

## XES

In [ ]:
# XES 的数据
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\ExSitu\αMnO2\Kbeta\Results\2023-CLAESS\Results\V2")
xes_peakarea = pd.read_csv(Path.joinpath(path_file, r"peak_area_std1.csv"), comment="#", sep=r",", header=0, index_col=None)
xes_spectrum = pd.read_csv(Path.joinpath(path_file, r"all_mean_normal.csv"), comment="#", sep=r",", header=0, index_col=None)

In [ ]:
xes_peak = xes_peakarea.iloc[2:, 0:3]
xes_area = xes_peakarea.iloc[2:, [0, 5, 6]]
xes_spectrum = xes_spectrum.iloc[:, [0, *list(range(3, 12))]]
diff_spectrum = xes_spectrum.copy(deep=True)
diff_spectrum.iloc[:, 1:] = diff_spectrum.iloc[:, 1:].sub(diff_spectrum.iloc[:, 3], axis=0)

In [ ]:
# 画图
fig = plt.figure(figsize=(7, 2.5))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1], height_ratios=None, wspace=0.0, hspace=0.0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [r"$\mathrm{Ref.MnO}$", r"$\mathrm{Ref.Mn_2O_3}$", r"$\mathrm{Ref.MnO_2}$", r"Pristine", r"$1^{st} \ Discharged$", r"HC#A", r"HC#B", r"FC#C", r"HC#D", r"FD#E"]
lss = [r"--", r"--", r"--", r"-", r"-", r"-", r"-", r"-", r"-", r"-"]
colormap = ListedColormap(mpl.colormaps["sunset"](np.linspace(0.0, 0.5, xes_spectrum.shape[1] - 1)), name=r"colormap")

# 多线叠加
for i in range(xes_spectrum.shape[1] - 1):
    ax.plot(
        xes_spectrum.iloc[:, 0],
        xes_spectrum.iloc[:, 1 + i],
        lw=1,
        ls=lss[i],
        label=labels[i],
        color=colors[i % 8],
        zorder=5,
        alpha=1 - 0.01 * i,
    )
    ax.plot(
        xes_spectrum.iloc[:, 0],
        diff_spectrum.iloc[:, 1 + i] - 0.01,
        lw=1,
        ls=lss[i],
        color=colors[i % 8],
        zorder=5,
        alpha=1 - 0.01 * i,
    )

ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.set_xlim(6465, 6510)
ax.xaxis.set_major_locator(ticker.MultipleLocator(10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(5, offset=0))

ax.set_ylabel(ylabel=r"Relative Intensity (arb.u.)", fontsize=9)
ax.set_ylim(-0.04, 0.16)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.04))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.02))
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(1.0, 1.0),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    columnspacing=0.5,
)

axins = ax.inset_axes((0.68, 0.4, 0.30, 0.55))
for i in range(xes_spectrum.shape[1] - 1):
    axins.plot(
        xes_spectrum.iloc[:, 0],
        xes_spectrum.iloc[:, 1 + i],
        lw=1,
        label=labels[i],
        ls=lss[i],
        color=colors[i % 8],
        zorder=0,
        alpha=1 - 0.01 * i,
    )
axins.set_xlim(6491, 6495)
axins.set_axis_off()
axins.set(xticks=[], xlabel=None, yticks=[], ylabel=None)
ax.text(
    0.32,
    0.40,
    r"$\mathrm{K}\beta^{\prime}$",
    transform=ax.transAxes,
    fontsize=10,
    va="center",
    ha="right",
    fontfamily="Arial",
)
ax.text(
    0.65,
    0.93,
    r"$\mathrm{K}_{\beta 1,3}$",
    transform=ax.transAxes,
    fontsize=10,
    va="center",
    ha="right",
    fontfamily="Arial",
)

ax.text(
    -0.20,
    1.0,
    r"A",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.5, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.errorbar(
    x=np.arange(len(labels)),
    y=xes_peak.iloc[:, 1],
    yerr=xes_peak.iloc[:, 2],
    c=colors[0],
    fmt="o",
    linewidth=1,
    capsize=4,
)
ax.plot(np.arange(len(labels)), xes_peak.iloc[:, 1], lw=1, ls="-", color=colors[0], zorder=0)

# x 刻度线
ax.set_xticks(range(len(labels)))  # 设置 tick 的位置
ax.set_xticklabels([])
ax.tick_params(axis="x", which="minor", bottom=False, top=False, left=False, right=False)
centers = np.linspace(0, len(labels) - 1, len(labels))
for x, label in zip(centers, labels, strict=False):
    ax.text(
        x,
        -0.05,
        label,
        ha="center",
        va="top",
        transform=ax.get_xaxis_transform(),
        fontsize=9,
        rotation=90,
    )

ax.set_ylabel(r"$\mathrm{K}_{\beta 1,3} \ Position \ (eV)$", fontsize=9, labelpad=10.0)  # Total Magnetization
ax.set_ylim(6492.0, 6493.6)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=0))
formatter = ticker.ScalarFormatter(useOffset=6490)
ax.yaxis.set_major_formatter(formatter)

arrowprops = {"arrowstyle": "<-", "color": colors[0], "connectionstyle": "angle,angleA=0,angleB=90,rad=10"}
ax.annotate(r" ", xy=(0.18, 0.5), xycoords="axes fraction", xytext=(0, 0.4), textcoords="axes fraction", arrowprops=arrowprops)

ax2 = ax.twinx()
ax2.set_position((0.5, 0, 1.0, 1.0))
ax2.set_box_aspect(0.8)

ax2.errorbar(
    x=np.arange(len(labels)),
    y=xes_area.iloc[:, 1],
    yerr=xes_area.iloc[:, 2],
    c=colors[1],
    fmt="s",
    linewidth=1,
    capsize=4,
    alpha=0.5,
)
ax2.plot(np.arange(len(labels)), xes_area.iloc[:, 1], lw=1, ls="-", color=colors[1], alpha=0.5, zorder=0)

ax2.set_ylabel(r"Local Magnetic Moment ($\mathrm{\mu _B}$)", fontsize=9)  # Total Magnetization
ax2.set_ylim(2.5, 5.5)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.6, offset=0.1))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: "%.1f" % x))  # noqa: ARG005

arrowprops = {
    "arrowstyle": "<-",
    "color": colors[1],
    "alpha": 1.0,
    "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
}
ax2.annotate(r" ", xy=(0.8, 0.4), xycoords="axes fraction", xytext=(1.0, 0.47), textcoords="axes fraction", arrowprops=arrowprops)

ax.text(-0.2, 1.0, r"B", transform=ax.transAxes, fontsize=14, va="center", ha="right", fontfamily="Arial", fontweight="bold")

plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XES_00_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XES_00_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XES_00_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XES_00_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)

plt.gcf().set_facecolor("white")
plt.show()

## STXM

### Figure Clusters of STXM, Zn, Mantins

#### 子图 HAADF

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        if len(bardata.axes_manager.navigation_shape) == 2:
            barsize = bardata.axes_manager.navigation_axes[0].scale
            unit = barunits if barunits is not None else bardata.axes_manager.navigation_axes[0].units
        elif len(bardata.axes_manager.signal_shape) == 2:
            barsize = bardata.axes_manager.signal_axes[0].scale
            unit = barunits if barunits is not None else bardata.axes_manager.signal_axes[0].units
        asb = AnchoredSizeBar(
            ax.transData,
            size / barsize,  # type: ignore
            "{} {}".format(size, unit),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
# 读取数据
path_zsh = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\ZHS\Pristine\E5 ZHS\Results\20230630_E6Zn_798.3x826.9y_specnorm")
path_fd1st = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st0.9V\B6 1st0.9V\Results\20230630_B6Zn_632.7x1193.9y_specnorm")
path_hca = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st1.53V\F2 1st1.53V\Results\20230630_F2ZnB_881.9x731.1y_specnorm")
path_hcb = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st1.63V\E3 1st1.63V\Results\20230701_E3Zn_515.7x521.1y_specnorm")
path_fc1st = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st1.80V\B2 1st1.80V\Results\20230630_B2Zn_270.2x1228.4y_specnorm")
path_hd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\2nd1.30V\F4 2nd1.30V\Results\20230701_F4Zn_868.2x982.9y_specnorm")
path_fd2nd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\2nd0.9V\G1 2nd0.9V\Results\20230701_G1Zn_949.2x834.6y_specnorm")

zsh = hs.load(path_zsh.joinpath("1_data_aligned.hspy")).integrate1D(axis=-1)
fd_1st = hs.load(path_fd1st.joinpath("1_data_aligned.hspy")).integrate1D(axis=-1)
hca = hs.load(path_hca.joinpath("1_data_aligned.hspy")).integrate1D(axis=-1)
hcb = hs.load(path_hcb.joinpath("1_data_aligned.hspy")).integrate1D(axis=-1)
fc_1st = hs.load(path_fc1st.joinpath("1_data_aligned.hspy")).integrate1D(axis=-1)
hd = hs.load(path_hd.joinpath("1_data_aligned.hspy")).integrate1D(axis=-1)
fd_2nd = hs.load(path_fd2nd.joinpath("1_data_aligned.hspy")).integrate1D(axis=-1)

data = [fd_1st, hca, hcb, fc_1st, hd, fd_2nd]
# data = [zsh, fd_1st, hca, hcb, fc_1st, hd, fd_2nd]

In [ ]:
# 画图
%matplotlib inline
plt.close("all")

for i, img in enumerate(data):
    fig = plt.figure(figsize=(7, 12))
    gs = gridspec.GridSpec(5, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1, 1, 1, 1], wspace=0.0, hspace=0.0)

    # 图
    subfig = fig.add_subfigure(gs[0, 0])
    ax = subfig.add_axes((0, 0, 1.0, 1.0))
    ax.set_box_aspect(1.0)
    ax.imshow(img.data, cmap="gray")
    ax.set_axis_off()
    add_sizebar(ax, 2, img, r"w", r"$\mu m$").set_bbox_to_anchor(Bbox.from_bounds(0, 0.03, 0, 0), ax.transAxes)

    # plt.savefig(
    #     Path.joinpath(path_out, f"Chapter_IV_Figure_STXM_08_0{i}_00_V0_300.tif"),
    #     pad_inches=0.05,
    #     bbox_inches="tight",
    #     dpi=300,
    #     transparent=False,
    #     pil_kwargs={"compression": "tiff_lzw"},
    # )
    plt.savefig(
        Path.joinpath(path_out, f"Chapter_IV_Figure_STXM_08_0{i}_00_V0_600.tif"),
        pad_inches=0.05,
        bbox_inches="tight",
        dpi=600,
        transparent=False,
        pil_kwargs={"compression": "tiff_lzw"},
    )
    # plt.savefig(
    #     Path.joinpath(path_out, f"Chapter_IV_Figure_STXM_08_0{i}_00_V0_600.png"),
    #     pad_inches=0.05,
    #     bbox_inches="tight",
    #     dpi=600,
    #     transparent=False,
    # )
    # plt.savefig(
    #     Path.joinpath(path_out, f"Chapter_IV_Figure_STXM_08_0{i}_00_V0_900.svg"),
    #     pad_inches=0.05,
    #     bbox_inches="tight",
    #     dpi=900,
    #     transparent=False,
    # )
    plt.close(fig)

#### Clusters 的图

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        if len(bardata.axes_manager.navigation_shape) == 2:
            barsize = bardata.axes_manager.navigation_axes[0].scale
            unit = barunits if barunits is not None else bardata.axes_manager.navigation_axes[0].units
        elif len(bardata.axes_manager.signal_shape) == 2:
            barsize = bardata.axes_manager.signal_axes[0].scale
            unit = barunits if barunits is not None else bardata.axes_manager.signal_axes[0].units
        asb = AnchoredSizeBar(
            ax.transData,
            size / barsize,  # type: ignore
            "{} {}".format(size, unit),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])


from scipy.stats import linregress  # noqa: E402


def remove_linear_background(spectrum, energy_threshold):
    x = spectrum["Energy"].values
    y = spectrum["Absorption"].values

    # 背景拟合区域
    mask = x < energy_threshold
    if np.sum(mask) < 2:
        raise ValueError("背景区间点数不足，无法拟合。")

    x_bg = x[mask]
    y_bg = y[mask]

    # 线性拟合
    slope, intercept, *_ = linregress(x_bg, y_bg)
    background = slope * x + intercept

    return y - background

In [ ]:
# 读取数据
path_zsh = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\ZHS\Pristine\E5 ZHS\Results\20230630_E6Zn_798.3x826.9y_specnorm\MANTIS\k=3\ca")
path_fd1st = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st0.9V\B6 1st0.9V\Results\20230630_B6Zn_632.7x1193.9y_specnorm\MANTIS\k=3\ca")
path_hca = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st1.53V\F2 1st1.53V\Results\20230630_F2ZnB_881.9x731.1y_specnorm\k=3\ca")
path_hcb = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st1.63V\E3 1st1.63V\Results\20230701_E3Zn_515.7x521.1y_specnorm\k=3\ca")
path_fc1st = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st1.80V\B2 1st1.80V\Results\20230630_B2Zn_270.2x1228.4y_specnorm\k=3\ca")
path_hd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\2nd1.30V\F4 2nd1.30V\Results\20230701_F4Zn_868.2x982.9y_specnorm\k=3\ca")
path_fd2nd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\2nd0.9V\G1 2nd0.9V\Results\20230701_G1Zn_949.2x834.6y_specnorm\MANTIS\k=3\ca")

paths = [path_fd1st, path_hca, path_hcb, path_fc1st, path_hd, path_fd2nd]
# paths = [path_zsh, path_fd1st, path_hca, path_hcb, path_fc1st, path_hd, path_fd2nd]

data_dict = {}
for path in paths:
    name = path.parts[11]
    image_path = list(path.glob(r"*_CAcimg.png"))
    image = plt.imread(image_path[0])
    spectrum_path = list(path.glob(r"*.csv"))
    spectrum = []
    for file_path in spectrum_path:
        df = pd.read_csv(
            file_path,
            sep=r"\s+|\t+|,",
            engine="python",
            comment="*",
            header=None,
            names=["Energy", "Absorption"],
            skip_blank_lines=True,
        )
        df["Absorption"] = remove_linear_background(df, energy_threshold=1020.0)
        # df["Absorption"] = df["Absorption"] / df["Absorption"].max()
        spectrum.append(df)
    spectrum = pd.concat(spectrum, axis=1, ignore_index=True)
    spectrum = spectrum.iloc[:, [0, *list(range(1, spectrum.shape[1], 2))]]
    spectrum.columns = ["Energy", *[f"Spec{i}" for i in range(1, spectrum.shape[1])]]
    data_dict[name] = {"image": image, "spectrum": spectrum}

In [ ]:
# 画图
%matplotlib inline
plt.close("all")

fig = plt.figure(figsize=(7, 14))
gs = gridspec.GridSpec(7, 4, width_ratios=[1, 1, 1, 1], height_ratios=[1, 1, 1, 1, 1, 1, 1], wspace=0.0, hspace=0.0)
Letters = ["A", "B", "C", "D", "E", "F"]
# 图
for i, (_, data) in enumerate(data_dict.items()):
    subfig = fig.add_subfigure(gs[i // 2, 0]) if i % 2 == 0 else fig.add_subfigure(gs[i // 2, 2])
    ax = subfig.add_axes((0, 0, 1.0, 1.0)) if i % 2 == 0 else subfig.add_axes((0.4, 0, 1.0, 1.0))
    ax.set_box_aspect(1.0)
    ax.imshow(data["image"])
    ax.set_axis_off()
    # add_sizebar(ax, 2, 0.013000000268220901, r"w", r"$\mu m$").set_bbox_to_anchor(Bbox.from_bounds(0, 0.03, 0, 0), ax.transAxes)
    ax.text(-0.05, 0.95, f"{Letters[i]}", transform=ax.transAxes, fontsize=13, va="center", ha="center", fontfamily="Arial", fontweight="bold")

    subfig = fig.add_subfigure(gs[i // 2, 1]) if i % 2 == 0 else fig.add_subfigure(gs[i // 2, 3])
    ax = subfig.add_axes((0.2, 0.05, 1.0, 1.0)) if i % 2 == 0 else subfig.add_axes((0.6, 0.05, 1.0, 1.0))
    ax.set_box_aspect(0.8)

    for j in range(2, data["spectrum"].shape[1]):  # 如果不需要空白像素点，改为 range(2, ...)
        ax.plot(
            data["spectrum"]["Energy"],
            data["spectrum"].iloc[:, j],
            color=colors[(j - 1) % len(colors)],
            linewidth=1.0,
        )
    # 刻度线
    ax.set_xlim(1010, 1050)
    ax.set_xlabel(r"Energy (eV)", fontsize=11)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=5))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=5))

    ax.set_ylabel(r"Absorption (arb.u.)", fontsize=11)
    ax.set_yticks([])
    ax.tick_params(
        axis="both",
        which="both",
        labelsize=9,
        bottom=True,
        left=False,
        labelbottom=True,
        labelleft=False,
    )


plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_07_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_07_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_07_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_07_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure Clusters of STXM, Mn, Mantins

#### Clusters 的图 -V1

In [ ]:
from hyperspy.axes import DataAxis, UniformDataAxis
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        if len(bardata.axes_manager.navigation_shape) == 2:
            barsize = bardata.axes_manager.navigation_axes[0].scale
            unit = barunits if barunits is not None else bardata.axes_manager.navigation_axes[0].units
        elif len(bardata.axes_manager.signal_shape) == 2:
            barsize = bardata.axes_manager.signal_axes[0].scale
            unit = barunits if barunits is not None else bardata.axes_manager.signal_axes[0].units
        asb = AnchoredSizeBar(
            ax.transData,
            size / barsize,  # type: ignore
            "{} {}".format(size, unit),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])


def hs_spect(data):
    energy_axis = DataAxis(
        axis=data.iloc[:, 0].values,
        index_in_array=None,
        name="Energy",
        units="eV",
    )
    index = UniformDataAxis(
        offset=0,
        scale=1,
        size=data.shape[1] - 1,
        navigate=True,
        name=r"Index",
        units=None,
    )
    data_spc = hs.signals.Signal1D(data.iloc[:, 1:].values, axes=[energy_axis, index])
    data_spc = data_spc.inav[1:].deepcopy()
    return data_spc

In [ ]:
# 读取数据
path_pristine = Path(
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Pristine\Powder\E1 Pristine\Results\SecondTrail_ScanMn_night\20230629_E1A_749.7x177.5y_specnorm\MANTIS\k=3\ca"
)
path_fd1st = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st0.9V\B6 1st0.9V\Results\20230630_B6Mn_632.7x1193.9y_specnorm\MANTIS\k=4\ca")
path_hca = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st1.53V\F2 1st1.53V\Results\20230630_F2MnB_881.9x731.1y_specnorm\MANTIS\k=4\ca")
path_hcb = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st1.63V\E3 1st1.63V\Results\20230701_E3Mn_515.7x521.1y_specnorm\MANTIS\k=4\ca")
path_fc1st = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st1.80V\B2 1st1.80V\Results\20230630_B2Mn_270.2x1228.4y_specnorm\MANTIS\k=4\ca")
path_hd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\2nd1.30V\F4 2nd1.30V\Results\20230701_F4Mn_868.2x982.9y_specnorm\MANTIS\k=4\ca")
path_fd2nd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\2nd0.9V\G1 2nd0.9V\Results\20230701_G1Mn_949.2x834.6y_specnorm\MANTIS\k=5\ca")

paths = [path_pristine, path_hca, path_hcb, path_fc1st, path_hd, path_fd2nd]
# paths = [path_pristine, path_fd1st, path_hca, path_hcb, path_fc1st, path_hd, path_fd2nd]

data_dict = {}
for path in paths:
    name = path.parts[11]
    image_path = list(path.glob(r"*_CAcimg.png"))
    image = plt.imread(image_path[0])
    spectrum_path = list(path.glob(r"*.csv"))
    spectrum = []
    for file_path in spectrum_path:
        df = pd.read_csv(
            file_path,
            sep=r"\s+|\t+|,",
            engine="python",
            comment="*",
            header=None,
            names=["Energy", "Absorption"],
            skip_blank_lines=True,
        )
        spectrum.append(df)
    spectrum = pd.concat(spectrum, axis=1, ignore_index=True)
    spectrum = spectrum.iloc[:, [0, *list(range(1, spectrum.shape[1], 2))]]
    spectrum.columns = ["Energy", *[f"Spec{i}" for i in range(1, spectrum.shape[1])]]
    spectrum = hs_spect(spectrum)
    data_dict[name] = {"image": image, "spectrum": spectrum}

In [ ]:
# 画图
%matplotlib inline
plt.close("all")

fig = plt.figure(figsize=(7, 14))
gs = gridspec.GridSpec(7, 4, width_ratios=[1, 1, 1, 1], height_ratios=[1, 1, 1, 1, 1, 1, 1], wspace=0.0, hspace=0.0)
Letters = ["A", "B", "C", "D", "E", "F"]
# 图
for i, (_, data) in enumerate(data_dict.items()):
    subfig = fig.add_subfigure(gs[i // 2, 0]) if i % 2 == 0 else fig.add_subfigure(gs[i // 2, 2])
    ax = subfig.add_axes((0, 0, 1.0, 1.0)) if i % 2 == 0 else subfig.add_axes((0.4, 0, 1.0, 1.0))
    ax.set_box_aspect(1.0)
    ax.imshow(data["image"])
    ax.set_axis_off()
    # add_sizebar(ax, 2, 0.013000000268220901, r"w", r"$\mu m$").set_bbox_to_anchor(Bbox.from_bounds(0, 0.03, 0, 0), ax.transAxes)
    ax.text(-0.05, 0.95, f"{Letters[i]}", transform=ax.transAxes, fontsize=13, va="center", ha="center", fontfamily="Arial", fontweight="bold")

    subfig = fig.add_subfigure(gs[i // 2, 1]) if i % 2 == 0 else fig.add_subfigure(gs[i // 2, 3])
    ax = subfig.add_axes((0.2, 0.05, 1.0, 1.0)) if i % 2 == 0 else subfig.add_axes((0.6, 0.05, 1.0, 1.0))
    ax.set_box_aspect(0.8)

    hs.plot.plot_spectra(
        data["spectrum"],
        ax=ax,
        style="overlap",
        color=["#e82128", "#99cd31", "#862275", "#d98518"],
        normalise=True,
    )

    # 刻度线
    ax.set_xlim(635, 665)
    ax.set_xlabel(r"Energy (eV)", fontsize=11)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=5))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=5))

    ax.set_ylabel(r"Absorption (arb.u.)", fontsize=11)
    ax.set_ylim(-0.05, 1.05)
    ax.set_yticks([])
    ax.tick_params(
        axis="both",
        which="both",
        labelsize=9,
        bottom=True,
        left=False,
        labelbottom=True,
        labelleft=False,
    )


plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_06_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_06_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_06_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_06_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

#### Clusters 的图

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        if len(bardata.axes_manager.navigation_shape) == 2:
            barsize = bardata.axes_manager.navigation_axes[0].scale
            unit = barunits if barunits is not None else bardata.axes_manager.navigation_axes[0].units
        elif len(bardata.axes_manager.signal_shape) == 2:
            barsize = bardata.axes_manager.signal_axes[0].scale
            unit = barunits if barunits is not None else bardata.axes_manager.signal_axes[0].units
        asb = AnchoredSizeBar(
            ax.transData,
            size / barsize,  # type: ignore
            "{} {}".format(size, unit),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])


from scipy.stats import linregress  # noqa: E402


def remove_linear_background(spectrum, energy_threshold):
    x = spectrum["Energy"].values
    y = spectrum["Absorption"].values

    # 背景拟合区域
    mask = x < energy_threshold
    if np.sum(mask) < 2:
        raise ValueError("背景区间点数不足，无法拟合。")

    x_bg = x[mask]
    y_bg = y[mask]

    # 线性拟合
    slope, intercept, *_ = linregress(x_bg, y_bg)
    background = slope * x + intercept

    return y - background

In [ ]:
# 读取数据
path_pristine = Path(
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Pristine\Powder\E1 Pristine\Results\SecondTrail_ScanMn_night\20230629_E1A_749.7x177.5y_specnorm\MANTIS\k=3\ca"
)
path_fd1st = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st0.9V\B6 1st0.9V\Results\20230630_B6Mn_632.7x1193.9y_specnorm\MANTIS\k=4\ca")
path_hca = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st1.53V\F2 1st1.53V\Results\20230630_F2MnB_881.9x731.1y_specnorm\MANTIS\k=4\ca")
path_hcb = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st1.63V\E3 1st1.63V\Results\20230701_E3Mn_515.7x521.1y_specnorm\MANTIS\k=4\ca")
path_fc1st = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st1.80V\B2 1st1.80V\Results\20230630_B2Mn_270.2x1228.4y_specnorm\MANTIS\k=4\ca")
path_hd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\2nd1.30V\F4 2nd1.30V\Results\20230701_F4Mn_868.2x982.9y_specnorm\MANTIS\k=4\ca")
path_fd2nd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\2nd0.9V\G1 2nd0.9V\Results\20230701_G1Mn_949.2x834.6y_specnorm\MANTIS\k=5\ca")

paths = [path_pristine, path_hca, path_hcb, path_fc1st, path_hd, path_fd2nd]
# paths = [path_pristine, path_fd1st, path_hca, path_hcb, path_fc1st, path_hd, path_fd2nd]

data_dict = {}
for path in paths:
    name = path.parts[11]
    image_path = list(path.glob(r"*_CAcimg.png"))
    image = plt.imread(image_path[0])
    spectrum_path = list(path.glob(r"*.csv"))
    spectrum = []
    for file_path in spectrum_path:
        df = pd.read_csv(
            file_path,
            sep=r"\s+|\t+|,",
            engine="python",
            comment="*",
            header=None,
            names=["Energy", "Absorption"],
            skip_blank_lines=True,
        )
        df["Absorption"] = remove_linear_background(df, energy_threshold=635.0)
        # df["Absorption"] = df["Absorption"] / df["Absorption"].max()
        spectrum.append(df)
    spectrum = pd.concat(spectrum, axis=1, ignore_index=True)
    spectrum = spectrum.iloc[:, [0, *list(range(1, spectrum.shape[1], 2))]]
    spectrum.columns = ["Energy", *[f"Spec{i}" for i in range(1, spectrum.shape[1])]]
    data_dict[name] = {"image": image, "spectrum": spectrum}

In [ ]:
# 画图
%matplotlib inline
plt.close("all")

fig = plt.figure(figsize=(7, 14))
gs = gridspec.GridSpec(7, 4, width_ratios=[1, 1, 1, 1], height_ratios=[1, 1, 1, 1, 1, 1, 1], wspace=0.0, hspace=0.0)
Letters = ["A", "B", "C", "D", "E", "F"]
# 图
for i, (_, data) in enumerate(data_dict.items()):
    subfig = fig.add_subfigure(gs[i // 2, 0]) if i % 2 == 0 else fig.add_subfigure(gs[i // 2, 2])
    ax = subfig.add_axes((0, 0, 1.0, 1.0)) if i % 2 == 0 else subfig.add_axes((0.4, 0, 1.0, 1.0))
    ax.set_box_aspect(1.0)
    ax.imshow(data["image"])
    ax.set_axis_off()
    # add_sizebar(ax, 2, 0.013000000268220901, r"w", r"$\mu m$").set_bbox_to_anchor(Bbox.from_bounds(0, 0.03, 0, 0), ax.transAxes)
    ax.text(-0.05, 0.95, f"{Letters[i]}", transform=ax.transAxes, fontsize=13, va="center", ha="center", fontfamily="Arial", fontweight="bold")

    subfig = fig.add_subfigure(gs[i // 2, 1]) if i % 2 == 0 else fig.add_subfigure(gs[i // 2, 3])
    ax = subfig.add_axes((0.2, 0.05, 1.0, 1.0)) if i % 2 == 0 else subfig.add_axes((0.6, 0.05, 1.0, 1.0))
    ax.set_box_aspect(0.8)

    for j in range(2, data["spectrum"].shape[1]):  # 如果不需要空白像素点，改为 range(2, ...)
        ax.plot(
            data["spectrum"]["Energy"],
            data["spectrum"].iloc[:, j],
            color=colors[(j - 1) % len(colors)],
            linewidth=1.0,
        )
    # 刻度线
    ax.set_xlim(635, 665)
    ax.set_xlabel(r"Energy (eV)", fontsize=11)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=5))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=5))

    ax.set_ylabel(r"Absorption (arb.u.)", fontsize=11)
    ax.set_yticks([])
    ax.tick_params(
        axis="both",
        which="both",
        labelsize=9,
        bottom=True,
        left=False,
        labelbottom=True,
        labelleft=False,
    )


plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_06_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_06_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_06_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_06_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

#### 子图 HAADF

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        if len(bardata.axes_manager.navigation_shape) == 2:
            barsize = bardata.axes_manager.navigation_axes[0].scale
            unit = barunits if barunits is not None else bardata.axes_manager.navigation_axes[0].units
        elif len(bardata.axes_manager.signal_shape) == 2:
            barsize = bardata.axes_manager.signal_axes[0].scale
            unit = barunits if barunits is not None else bardata.axes_manager.signal_axes[0].units
        asb = AnchoredSizeBar(
            ax.transData,
            size / barsize,  # type: ignore
            "{} {}".format(size, unit),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
# 读取数据
path_pristine = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Pristine\Powder\E1 Pristine\Results\SecondTrail_ScanMn_night\20230629_E1A_749.7x177.5y_specnorm")
path_fd1st = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st0.9V\B6 1st0.9V\Results\20230630_B6Mn_632.7x1193.9y_specnorm")
path_hca = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st1.53V\F2 1st1.53V\Results\20230630_F2MnB_881.9x731.1y_specnorm")
path_hcb = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st1.63V\E3 1st1.63V\Results\20230701_E3Mn_515.7x521.1y_specnorm")
path_fc1st = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\1st1.80V\B2 1st1.80V\Results\20230630_B2Mn_270.2x1228.4y_specnorm")
path_hd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\2nd1.30V\F4 2nd1.30V\Results\20230701_F4Mn_868.2x982.9y_specnorm")
path_fd2nd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\αMnO2\Charge\Electrode\2nd0.9V\G1 2nd0.9V\Results\20230701_G1Mn_949.2x834.6y_specnorm")

pristine = hs.load(path_pristine.joinpath("1_data_aligned.hspy")).integrate1D(axis=-1)
fd_1st = hs.load(path_fd1st.joinpath("1_data_aligned.hspy")).integrate1D(axis=-1)
hca = hs.load(path_hca.joinpath("1_data_aligned.hspy")).integrate1D(axis=-1)
hcb = hs.load(path_hcb.joinpath("1_data_aligned.hspy")).integrate1D(axis=-1)
fc_1st = hs.load(path_fc1st.joinpath("1_data_aligned.hspy")).integrate1D(axis=-1)
hd = hs.load(path_hd.joinpath("1_data_aligned.hspy")).integrate1D(axis=-1)
fd_2nd = hs.load(path_fd2nd.joinpath("1_data_aligned.hspy")).integrate1D(axis=-1)

data = [pristine, fd_1st, hca, hcb, fc_1st, hd, fd_2nd]

In [ ]:
# 画图
%matplotlib inline
plt.close("all")

for i, img in enumerate(data):
    fig = plt.figure(figsize=(7, 12))
    gs = gridspec.GridSpec(5, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1, 1, 1, 1], wspace=0.0, hspace=0.0)

    # 图
    subfig = fig.add_subfigure(gs[0, 0])
    ax = subfig.add_axes((0, 0, 1.0, 1.0))
    ax.set_box_aspect(1.0)
    ax.imshow(img.data, cmap="gray")
    ax.set_axis_off()
    add_sizebar(ax, 2, img, r"w", r"$\mu m$").set_bbox_to_anchor(Bbox.from_bounds(0, 0.03, 0, 0), ax.transAxes)

    # plt.savefig(
    #     Path.joinpath(path_out, f"Chapter_IV_Figure_STXM_05_0{i}_00_V0_300.tif"),
    #     pad_inches=0.05,
    #     bbox_inches="tight",
    #     dpi=300,
    #     transparent=False,
    #     pil_kwargs={"compression": "tiff_lzw"},
    # )
    plt.savefig(
        Path.joinpath(path_out, f"Chapter_IV_Figure_STXM_05_0{i}_00_V0_600.tif"),
        pad_inches=0.05,
        bbox_inches="tight",
        dpi=600,
        transparent=False,
        pil_kwargs={"compression": "tiff_lzw"},
    )
    # plt.savefig(
    #     Path.joinpath(path_out, f"Chapter_IV_Figure_STXM_05_0{i}_00_V0_600.png"),
    #     pad_inches=0.05,
    #     bbox_inches="tight",
    #     dpi=600,
    #     transparent=False,
    # )
    # plt.savefig(
    #     Path.joinpath(path_out, f"Chapter_IV_Figure_STXM_05_0{i}_00_V0_900.svg"),
    #     pad_inches=0.05,
    #     bbox_inches="tight",
    #     dpi=900,
    #     transparent=False,
    # )
    plt.close(fig)

### Figure MCR-ALS of STXM

In [ ]:
# 读取 MCR 的数据
path_mcr = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\Andrea\SpectrumAll").joinpath(r"sTXM_MCR-ALS-4comp.xlsx")
mcr_st = pd.read_excel(path_mcr, sheet_name=1, index_col=None, header=0, comment="#", engine="openpyxl")
mcr_c = pd.read_excel(path_mcr, sheet_name=2, index_col=None, header=0, comment="#", engine="openpyxl")

In [ ]:
# 画图
%matplotlib inline
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1], height_ratios=None, wspace=0.0, hspace=0.0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.1)

ax.plot(mcr_st.iloc[:, 0], mcr_st.iloc[:, 1], ls="-", lw=1.0, c=colors[0], label=r"#1")
ax.plot(mcr_st.iloc[:, 0], mcr_st.iloc[:, 2], ls="-", lw=1.0, c=colors[1], label=r"#2")
ax.plot(mcr_st.iloc[:, 0], mcr_st.iloc[:, 3], ls="-", lw=1.0, c=colors[3], label=r"#3")
ax.plot(mcr_st.iloc[:, 0], mcr_st.iloc[:, 4], ls="-", lw=1.0, c=colors[5], label=r"#4")
ax.legend(loc="upper right", bbox_to_anchor=(0.98, 1.0), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

# 刻度线
ax.set_xlim(635, 665)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=5))
ax.set_ylim(-0.1, 3.5)
ax.set_ylabel(r"Absorption (arb.u.)", fontsize=11)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.7, offset=0.0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.35, offset=0.0))
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    left=True,
    labelbottom=True,
    labelleft=True,
)
ax.text(-0.23, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

colors_mcr = [colors[0], colors[0], colors[1], colors[3], colors[5]]
# Pristine
for i in range(1, 5):
    ax.plot(np.linspace(0, 1, num=5), mcr_c.iloc[0:5, i], label=f"#{i}", c=colors_mcr[i])
# 1st Discharge
for i in range(1, 5):
    ax.plot(np.linspace(1, 2, num=5), mcr_c.iloc[10:15, i], label=None, c=colors_mcr[i])
# HC#1
for i in range(1, 5):
    ax.plot(np.linspace(2, 3, num=10), mcr_c.iloc[20:30, i], label=None, c=colors_mcr[i])
# HC#2
for i in range(1, 5):
    ax.plot(np.linspace(3, 4, num=5), mcr_c.iloc[15:20, i], label=None, c=colors_mcr[i])
# FC
for i in range(1, 5):
    ax.plot(np.linspace(4, 5, num=5), mcr_c.iloc[5:10, i], label=None, c=colors_mcr[i])
# HD
for i in range(1, 5):
    ax.plot(np.linspace(6, 5, num=10), mcr_c.iloc[30:40, i], label=None, c=colors_mcr[i])
# FD
for i in range(1, 5):
    ax.plot(np.linspace(6, 7, num=5), mcr_c.iloc[40:45, i], label=None, c=colors_mcr[i])

ax.set_xlabel("", fontsize=11)
ax.set_xlim(0, 7)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_ylabel(r"Relative Concentrations (arb.u.)", fontsize=11)
ax.set_ylim(-0.01, 1.0)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax.legend(loc="upper left", bbox_to_anchor=(0.52, 1.0), ncols=2, frameon=False, labelcolor="linecolor", fontsize=9)

ax.vlines(
    [1, 2, 3, 4, 5, 6],
    [0, 0, 0, 0, 0, 0],
    [1, 1, 1, 1, 1, 1],
    colors=["grey", "magenta", "grey", "magenta", "grey", "magenta"],
    linestyles="--",
    lw=0.5,
    alpha=0.8,
    zorder=0,
)

for i in [0, 0.3, 0.6]:
    ax.text(
        0.06 + i,
        1.05,
        r"Border",
        transform=ax.transAxes,
        fontsize=10,
        va="top",
        ha="left",
        fontfamily="Arial",
        color="grey",
    )
    ax.text(
        0.22 + i,
        1.05,
        r"Bulk",
        transform=ax.transAxes,
        fontsize=10,
        va="top",
        ha="left",
        fontfamily="Arial",
        color="magenta",
    )

names = [r"Pristine", r"$\mathrm{1^{st} \ Discharge}$", r"HC#A", r"HC#B", r"FC#C", r"HD#D", r"FD#E"]

# x 轴
ax.set_xticks(range(len(names)))  # 设置 tick 的位置
ax.set_xticklabels([])
ax.tick_params(axis="x", which="minor", bottom=False, top=False, left=False, right=False)
centers = np.linspace(0.5, len(names) - 0.5, len(names))
for x, label in zip(centers, names, strict=False):
    ax.text(
        x,
        -0.05,
        label,
        ha="center",
        va="top",
        transform=ax.get_xaxis_transform(),
        fontsize=9,
        rotation=60,
    )

ax.text(-0.17, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_04_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_04_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_04_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_04_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### MCR - ALS - STXM 数据

In [ ]:
# 读取 MCR 的数据
path_mcr = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\Andrea\SpectrumAll").joinpath(r"sTXM_MCR-ALS-4comp.xlsx")
data = pd.read_excel(path_mcr, sheet_name=0, index_col=0, header=0, comment="#", engine="openpyxl")

# 交换顺序，使得符合放电状态
order_index = [*list(range(0, 5, 1)), *list(range(20, 25, 1)), *list(range(40, 50, 1)), *list(range(30, 35, 1)), *list(range(10, 15, 1)), *list(range(50, 60, 1)), *list(range(60, 65, 1))]
data = data.iloc[:, order_index]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(3.3, 2.5))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0.0, hspace=0.0)

# 图
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.0)

names = [r"Pristine", r"$\mathrm{1^{st} \ Discharge}$", r"HC#A", r"HC#B", r"FC#C", r"HD#D", r"FD#E"]

ax.imshow(data.T.values, aspect="auto", cmap="coolwarm", origin="lower", extent=(data.index[0], data.index[-1], 0, data.shape[1]), vmin=0, vmax=2.0)  #'jet'

# 添加 colorbar
# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.04, 0.1, 0.1, 0.8), transform=ax.transAxes),
    location="right",
    orientation="vertical",
    cmap="coolwarm",  #'jet'
    ticklocation="right",
    spacing="proportional",
    label=r"Absorption (arb.u.)",
)

colorbar.set_ticks([])
colorbar.outline.set_visible(False)  # type: ignore
colorbar.ax.text(0.5, -0.08, 0.0, ha="center", va="bottom", fontsize=9, transform=colorbar.ax.transAxes)
colorbar.ax.text(0.5, 1.06, 2.0, ha="center", va="top", fontsize=9, transform=colorbar.ax.transAxes)

# 设置刻度线等格式
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.set_xlim(630, 670)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))

groups = [range(0, 5 + 1), range(5, 10 + 1), range(10, 20 + 1), range(20, 25 + 1), range(25, 30 + 1), range(30, 40 + 1), range(40, 45 + 1)]

# === 计算上下边界和中点 ===
boundaries = []
centers = []
for g in groups:
    boundaries.extend([min(g), max(g)])
    centers.append((min(g) + max(g)) / 2)

# 设置 y 轴刻度与标签
ax.set_yticks(boundaries)
ax.set_yticklabels([])
for y, label in zip(centers, names, strict=False):
    ax.text(-0.1, y, label, ha="right", va="center", transform=ax.get_yaxis_transform(), fontsize=9)

ax.tick_params(axis="x", which="both", direction="out", labelsize=9)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_03_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_03_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_03_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_03_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure Spectrum of STXM, Mn and Zn

In [ ]:
# 读取数据 Mn
file_path = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\Andrea").glob(r"**/*Mn.txt"))
# file_path = [file_path[i] for i in [8, 6, 1, 3, 2, 0, 4, 5]]
file_path = [file_path[i] for i in [8, 6, 3, 2, 0, 4, 5]]
data_dict_mn = {}
for f in file_path:
    df = pd.read_csv(f, sep="\t", header=None, skiprows=0, index_col=None, comment="#")
    df.columns = ["Energy", "Absorption"]
    name = f.stem
    data_dict_mn[name] = df


In [ ]:
from scipy.stats import linregress


def remove_linear_background(spectrum, energy_threshold):
    x = spectrum["Energy"].values
    y = spectrum["Absorption"].values

    # 背景拟合区域
    mask = x < energy_threshold
    if np.sum(mask) < 2:
        raise ValueError("背景区间点数不足，无法拟合。")

    x_bg = x[mask]
    y_bg = y[mask]

    # 线性拟合
    slope, intercept, *_ = linregress(x_bg, y_bg)
    background = slope * x + intercept

    return y - background

In [ ]:
for _, df in data_dict_mn.items():
    data = remove_linear_background(df, energy_threshold=635.0)
    # df["Absorption"] = (data - data.min()) / (data.max() - data.min())

In [ ]:
# TXM 的数据路径, Zn
path_data = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\Andrea")


def read_data(file_path):
    df = pd.read_csv(file_path, sep=r"\s+", header=None, comment="#")
    df.columns = ["Energy", "Absorption"]
    df["Absorption"] = (df["Absorption"] - df["Absorption"].min()) / (df["Absorption"].max() - df["Absorption"].min())
    return df


# 定义所有数据集的信息
datasets = [
    (r"ZSH", r"Pristine/E5 ZHS/ref_E5_ZHS_Zn.txt"),
    # (r"1stDischarge", r"Charge/B6 1st0.9V/B6_1stDisch_Zn.txt"),
    (r"1stHalfCharge#2", r"Charge/E3 1st1.63V/E3_1stHCh_1p63V_Zn.txt"),
    (r"1stFullCharge", r"Charge/B2 1st1.80V/B2_1stCh_Zn.txt"),
    (r"2ndHalfDischarge", r"Charge/F4 2nd1.30V/F4_2ndDisch_1p3V_Zn.txt"),
    (r"2ndFullDischarge", r"Charge/G1 2nd0.9V/G1_2ndDisch_Zn.txt"),
]

# 加载和处理数据
data_dict_zn = {name: read_data(path_data.joinpath(path)) for name, path in datasets}

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1], height_ratios=None, wspace=0.0, hspace=0.0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.2)

# names = [r'Ref. MnOOH', r'Pristine', r'$\mathrm{1^{st} \ Discharge}$', r"HC#A", r"HC#B", r"FC#C", r"HD#D", r"FD#E"]
names = [r"Ref. MnOOH", r"Pristine", r"HC#A", r"HC#B", r"FC#C", r"HD#D", r"FD#E"]

for i, (_, data) in enumerate(data_dict_mn.items()):
    ax.plot(data["Energy"], data["Absorption"] + 0.5 * i, label=names[i], color=colors_opt2[i], lw=1.0)

line_paras = [(644.2, 0.25, 0.40, "grey"), (644.0, 0.45, 0.6, "grey"), (644.0, 0.6, 0.7, "grey"), (644.0, 0.7, 0.8, "grey"), (644.0, 0.8, 0.9, "grey"), (644.0, 0.9, 0.98, "grey")]
# line_paras = [(644.2, 0.23, 0.33, 'grey'), (644.2, 0.35, 0.45, 'grey'), (644.0, 0.5, 0.6, 'grey'), (644.0, 0.6, 0.7, 'grey'), (644.0, 0.7, 0.8, 'grey'), (644.0, 0.8, 0.9, 'grey'), (644.0, 0.9, 0.98, 'grey')]
for x, ymin, ymax, color in line_paras:
    ax.axvline(x=x, ymin=ymin, ymax=ymax, ls="--", color=color, lw=1.0)
ax.plot([644.2, 644.0], [1.5, 1.75], ls="dotted", color="grey", lw=1.0)

# 设置刻度线等格式
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.set_xlim(635, 665.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=5))

ax.set_ylabel(r"Absorption (arb.u.)", fontsize=11)
ax.set_yticks([])
ax.tick_params(axis="x", which="both", direction="out", labelsize=9)
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.04), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(-0.13, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.1, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.2)

# names = ["Ref. ZSH", r"$\mathrm{1^{st}\ Discharge}$", r"HC#B", r"FC#C", r"HD#D", r"FD#E"]
names = ["Ref. ZSH", r"HC#B", r"FC#C", r"HD#D", r"FD#E"]

for i, (_, data) in enumerate(data_dict_zn.items()):
    if i == 0:
        ax.plot(data["Energy"], data["Absorption"] + 0.5 * i, label=names[i], color=colors_opt2[i], lw=1.0)
    if i > 0:
        ax.plot(data["Energy"], data["Absorption"] + 0.5 * i, label=names[i], color=colors_opt2[i + 2], lw=1.0)


# 设置刻度线等格式
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.set_xlim(1010.0, 1050.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))

ax.set_ylabel(r"Absorption (arb.u.)", fontsize=11)
ax.set_yticks([])
ax.tick_params(axis="x", which="both", direction="out", labelsize=9)
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.04), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(-0.13, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_02_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_02_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_02_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_02_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure STXM Mn Mappings

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        if len(bardata.axes_manager.navigation_shape) == 2:
            barsize = bardata.axes_manager.navigation_axes[0].scale
            unit = bardata.axes_manager.navigation_axes[0].units
        elif len(bardata.axes_manager.signal_shape) == 2:
            barsize = bardata.axes_manager.signal_axes[0].scale
            unit = bardata.axes_manager.signal_axes[0].units
        asb = AnchoredSizeBar(
            ax.transData,
            size / barsize,  # type: ignore
            "{} {}".format(size, unit),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
# 电化学上的时间
path_file = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\1M ZnSO4 + 0.2M MnSO4")

echem = pd.read_excel(path_file.joinpath(r"LC-0.1C-Zn-1M Zn+0.2M Mn-alpha MnO2-80992-5.9mg_20230316111305.xlsx"), sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(
    axis=1, how="all"
)
# 转换数据格式
echem[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = echem[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")

echem["TestTime"] = echem["TestTime"].apply(pd.to_timedelta, errors="coerce")
echem["Cycle"] = echem["Cycle"].astype(np.int16)

echem = echem[echem.iloc[:, 0].isin([0, 1, 2])]
echem = echem[echem.iloc[:, 2] >= 0]

In [ ]:
# 读取 TXM 的数据
def read_data(file_path, name, low_cut=0.20, high_cut=0.551):
    df = pd.read_csv(file_path, sep=",", header=None, comment="#")
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)
    if name in [r"Pristine", r"MnOOH"]:
        return df
    else:
        return df.where((df >= low_cut) & (df <= high_cut), 0)


# TXM 的数据路径
path_data = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\STXM\ExSitu\Andrea")
scale = 0.013000000268220901  # 像素比例尺 (um/px)
cutoff = (0.20, 0.551)

# 定义所有数据集的信息
datasets = [
    (r"Pristine", r"Pristine/E1 Pristine/Pristine_Im_ratio2.txt"),
    (r"MnOOH", r"Pristine/F8 MnOOH/MnOOH_Imratio2.txt"),
    (r"1stDischarge", r"Charge/B6 1st0.9V/B6_Imratio2.txt"),
    (r"1stHalfCharge#1", r"Charge/F2 1st1.53V/F2_Im_ratio2.txt"),
    (r"1stHalfCharge#2", r"Charge/E3 1st1.63V/E3_Im_ratio2.txt"),
    (r"1stFullCharge", r"Charge/B2 1st1.80V/B2_Im_ratio2.txt"),
    (r"2ndHalfDischarge", r"Charge/F4 2nd1.30V/F4_Im_ratio2.txt"),
    (r"2ndFullDischarge", r"Charge/G1 2nd0.9V/G1_Im_ratio2.txt"),
]

# 加载和处理数据
data_dict = {name: read_data(path_data.joinpath(path), name, *cutoff) for name, path in datasets}

In [ ]:
# 对比分析：包含0值 vs 排除0值的平均值
print("\n" + "=" * 80)
print("对比分析：包含0值 vs 排除0值的平均值差异")
print("=" * 80)
print(f"{'数据集':<20} {'包含0值':<12} {'排除0值':<12} {'差异倍数':<10} {'零值比例'}")
print("-" * 80)

comparison_data = []
for key in data_dict.keys():
    avg_with_zero = averages[key]
    avg_without_zero = averages_nonzero[key]

    if avg_with_zero > 0:
        ratio = avg_without_zero / avg_with_zero
    else:
        ratio = float("inf") if avg_without_zero > 0 else 1.0

    # 计算零值比例
    total_points = data_dict[key].size
    zero_points = total_points - len(data_dict[key][data_dict[key] != 0])
    zero_percentage = (zero_points / total_points) * 100

    comparison_data.append((key, avg_with_zero, avg_without_zero, ratio, zero_percentage))

    print(f"{key:<20} {avg_with_zero:<12.6f} {avg_without_zero:<12.6f} {ratio:<10.1f}x {zero_percentage:<6.1f}%")

# 按排除0值后的平均值排序
sorted_nonzero = sorted(averages_nonzero.items(), key=lambda x: x[1], reverse=True)
print(f"\n按排除0值后的平均值从高到低排序:")
for i, (key, value) in enumerate(sorted_nonzero, 1):
    print(f"{i:2d}. {key:<20}: {value:.6f}")

print(f"\n最大提升倍数: {max([x[3] for x in comparison_data if x[3] != float('inf')]):.1f}x")
print(f"平均零值比例: {sum([x[4] for x in comparison_data]) / len(comparison_data):.1f}%")

In [ ]:
def create_subplot(subfig_pos, ax_pos, data_key, title, cax_pos, caption):
    """创建标准化子图"""
    subfig = fig.add_subfigure(subfig_pos)
    ax = subfig.add_axes(ax_pos)
    ax.set_box_aspect(1)

    data = data_dict[data_key]
    im = ax.imshow(data, vmin=data.min().min(), vmax=data.max().max(), cmap="hot")

    # 添加比例尺
    add_sizebar(ax, 2, scale, "w", r"$\mu m$").set_bbox_to_anchor(Bbox.from_bounds(0, 0.03, 0, 0), ax.transAxes)

    # 添加颜色条
    if cax_pos:
        cax = subfig.add_axes(cax_pos)
        cbar = subfig.colorbar(im, cax=cax, ticks=np.linspace(0, 1, 21), format="%.2f")
        cbar.ax.set_ylim(cutoff[0], cutoff[1])

    # 添加标题
    ax.set_title(title, loc="left", fontdict={"fontsize": 12, "fontfamily": "Arial"}, transform=ax.transAxes, y=1.0, color=colors[1])
    ax.axis("off")

    # labels
    if caption:
        ax.text(
            -0.05,
            1.05,
            f"{caption}",
            transform=ax.transAxes,
            fontsize=13,
            va="center",
            ha="right",
            fontfamily="Arial",
            fontweight="bold",
        )
    return ax

In [ ]:
# 画图
%matplotlib inline
plt.close("all")

fig = plt.figure(figsize=(7, 9))
gs = gridspec.GridSpec(3, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1, 1], wspace=0.0, hspace=0.0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0:2])
ax = subfig.add_axes((0, 0, 0.9, 1.0))
ax.set_box_aspect(0.6)

for _, idx in enumerate(echem.iloc[:, 0].unique()):
    temp = echem[echem.iloc[:, 0] == idx].reset_index(drop=True)

    # 找到电压最小值的索引
    idx_min = temp["Voltage/V"].idxmin()

    if idx == 1:
        ax.plot(temp.loc[idx_min:, "SpeCap/mAh/g"], temp.loc[idx_min:, "Voltage/V"], c=colors[0], label=None)

    if idx == 2:
        ax.plot(temp.loc[:idx_min, "SpeCap/mAh/g"], temp.loc[:idx_min, "Voltage/V"], c=colors[0])

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-5, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
# ax.legend(loc='upper right', bbox_to_anchor=(1.0, 0.8), ncols=1, frameon=False, labelcolor='linecolor', fontsize=9)

ax.text(130, 1.62, r"HC#A", ha="left", va="top", transform=ax.transData, fontsize=11, c=colors[1])
ax.plot(170, 1.53, marker="*", markersize=8, color=colors[1], alpha=0.5)
ax.text(260, 1.57, r"HC#B", ha="left", va="top", transform=ax.transData, fontsize=11, c=colors[1])
ax.plot(285, 1.63, marker="*", markersize=8, color=colors[1], alpha=0.5)
ax.text(340, 1.75, r"FC#C", ha="left", va="top", transform=ax.transData, fontsize=11, c=colors[1])
ax.plot(355, 1.80, marker="*", markersize=8, color=colors[1], alpha=0.5)
ax.text(110, 1.25, r"HD#D", ha="left", va="top", transform=ax.transData, fontsize=11, c=colors[1])
ax.plot(123, 1.30, marker="*", markersize=8, color=colors[1], alpha=0.5)
ax.text(305, 0.92, r"FD#E", ha="left", va="top", transform=ax.transData, fontsize=11, c=colors[1])
ax.plot(295, 0.9, marker="*", markersize=8, color=colors[1], alpha=0.5)

ax.text(0.02, 0.10, r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")

ax.text(-0.15, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B-G
names = [r"Pristine", r"Ref. MnOOH", r"$\mathrm{1^{st} \ Discharge}$", r"HC#A", r"HC#B", r"FC#C", r"HD#D", r"FD#E"]
axes_scale = 1.1
# 子图布局参数
subplot_params = [
    (gs[0, 2], [0.0, -0.1, axes_scale * 1.0, axes_scale * 1.0], "1stHalfCharge#1", names[3], [1.12, 0.05, 0.08, 0.8], "B"),
    (gs[1, 0], [-0.25, -0.20, axes_scale * 1.0, axes_scale * 1.0], "1stHalfCharge#2", names[4], None, "C"),
    (gs[1, 1], [0.0, -0.20, axes_scale * 1.0, axes_scale * 1.0], "1stFullCharge", names[5], None, "D"),
    (gs[2, 0], [-0.25, -0.15, axes_scale * 1.0, axes_scale * 1.0], "2ndHalfDischarge", names[6], None, "E"),
    (gs[2, 1], [0.0, -0.15, axes_scale * 1.0, axes_scale * 1.0], "2ndFullDischarge", names[7], None, "F"),
]

for params in subplot_params:
    create_subplot(*params)

# 图 I
subfig = fig.add_subfigure(gs[1:, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.3, -0.01, 1.0, 1.0))
ax.set_box_aspect(2.15)
line_paras = [(0.29, 0.1, 0.95, "r"), (0.74, 0.1, 0.25, "r"), (0.39, 0.40, 0.50, "w"), (0.45, 0.6, 0.7, "w"), (0.44, 0.68, 0.78, "k"), (0.42, 0.75, 0.85, "w"), (0.35, 0.88, 0.95, "w")]

data_dict.pop("1stDischarge", None)
names = [r"Pristine", r"Ref. MnOOH", r"HC#A", r"HC#B", r"FC#C", r"HD#D", r"FD#E"]

for i, (key, value) in enumerate(data_dict.items()):
    if key in [r"Pristine", r"MnOOH"]:
        ax.hist(value.values.flatten(), bins=100, density=True, color=colors_opt2[i], align="mid", range=(0.2, 1.0), zorder=0, label=names[i], bottom=0, alpha=1.0, stacked=True)
        ax.axvline(x=line_paras[i][0], ymax=line_paras[i][1], ymin=line_paras[i][2], color=line_paras[i][3], linestyle="--", linewidth=2, zorder=10)
    else:
        ax.hist(value.values.flatten(), bins=100, density=True, color=colors_opt2[i], align="mid", range=cutoff, zorder=len(data_dict) - i + 1, label=names[i], bottom=2.0 + 3.0 * i, alpha=1.0)
        ax.axvline(x=line_paras[i][0], ymax=line_paras[i][1], ymin=line_paras[i][2], color=line_paras[i][3], linestyle="--", linewidth=2, zorder=10)

ax.set_ylabel(r"Normalized Frequency", fontsize=11)
ax.set_xlabel(
    r"Shape Index",
    fontsize=11,
)
ax.set_xlim(0.2, 0.9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))

# ax.set_ylim(0, 8.5)
# ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
# ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))

ax.tick_params(
    axis="y",
    which="both",
    left=False,
    labelbottom=False,
    labelleft=False,
)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.48, 1.0),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    handletextpad=0.3,
    labelspacing=0.3,
)
ax.text(-0.11, 1.0, r"G", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_01_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_01_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_01_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_01_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

## XPS + UPS

### XPS ratio

In [ ]:
# 读取数据
xps_path = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XPS+UPS\ExSitu\αMnO2")

xps = pd.read_csv(xps_path.joinpath(r"XPS_Ratio.csv"), sep=",", header=0, index_col=0, comment="#")
xps = xps.apply(pd.to_numeric, errors="coerce")

weight_counts = {
    r"Mn (IV)": xps.iloc[0, :].values,
    r"Mn (III)": xps.iloc[1, :].values,
    r"Mn (II)": xps.iloc[2, :].values,
}

In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(3.3, 2.5))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

species = [r"Pristine", r"HC#A", r"HC#B", r"FC#C", r"HD#D", r"FD#E"]

width = 0.5
bottom = np.zeros(len(species))
i = 0
for boolean, weight_count in weight_counts.items():
    p = ax.bar(species, weight_count, width, label=boolean, bottom=bottom, color=colors[i])
    bottom += weight_count
    i += 1

ax.set_ylabel(r"Relative Percentage (%)", fontsize=11)
ax.set_ylim(0, 120)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))
ax.tick_params(axis="both", which="both", labelsize=9)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.3, 1.0),
    ncols=3,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.8,
    handletextpad=0.5,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_07_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_07_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_07_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_07_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure XPS， 2nd0.9V

In [ ]:
# 读取数据
path_xps = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XPS+UPS\ExSitu\αMnO2\Charge\Electrode\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\Results")
Mn_xps = pd.read_csv(
    Path.joinpath(path_xps, r"LC_Mn.txt"),
    sep="\t",
    skiprows=6,
    index_col=None,
    header=1,
    comment="#",
).dropna(axis=1, how="all")
Mn_xps = Mn_xps.iloc[:, 28:]

O_xps = pd.read_csv(Path.joinpath(path_xps, "LC_O.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
O_xps = O_xps.iloc[:, 7:]

Zn_xps = pd.read_csv(Path.joinpath(path_xps, "LC_Zn.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
Zn_xps = Zn_xps.iloc[:, 2:]

In [ ]:
# 数据处理, Mn
envelope_Mn_IV = Mn_xps.iloc[:, 2:10].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_III = Mn_xps.iloc[:, 10:18].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_II = Mn_xps.iloc[:, 18:26].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816

In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 Mn
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 27], label=r"Fit", ls="--", color=colors[1])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 26], label=r"Background", color=colors[2], zorder=2)  # background

ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_IV, color=colors[3], linestyle="--", label=r"Mn(IV)")  # envelope_Mn_IV
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_III, color=colors[5], linestyle="--", label=r"Mn(III)")  # envelope_Mn_III
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_II, color=colors[4], linestyle="--", label=r"Mn(II)", zorder=3)  # envelope_Mn_II

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 2:10], color=colors[3], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 2], y2=Mn_xps.iloc[:, 26], facecolor=colors[3], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 10:18], color=colors[5], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 10], y2=Mn_xps.iloc[:, 26], facecolor=colors[5], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 18:26], color=colors[4], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 18], y2=Mn_xps.iloc[:, 26], facecolor=colors[4], alpha=0.3)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, 1.0),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.8,
    handletextpad=0.5,
)

ax.text(
    0.55,
    0.95,
    r"Mn(III): 31.2(9) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[5],
)
ax.text(
    0.55,
    0.85,
    r"Mn(IV): 65.2(7) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)
ax.text(
    0.55,
    0.75,
    r"Mn(II): 3.4(3) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)

ax.text(
    1.0,
    1.11,
    r"Mn $\mathit{2p_{3/2}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=12,
)

ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=11,
)
ax.set_xlim(648.5, 638.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0.5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0.5))

ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=11,
)
ax.set_ylim(3000, 25000)
ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500, offset=0))
ax.tick_params(axis="both", which="both", labelsize=9)

# 图 O
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.4, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 1], ls="-", lw=1.0, c=colors[0], label="Data")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 6], ls="--", lw=1.0, c=colors[1], label="Fit")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 5], ls="-", lw=1.0, c=colors[2], label="Background", zorder=6)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 2], ls="-", lw=1.0, c=colors[3], label=r"Lattice Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 2], y2=O_xps.iloc[:, 5], facecolor=colors[3], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 3], ls="-", lw=1.0, c=colors[4], label="Hydroxide, Hydrated\n or Defective Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 3], y2=O_xps.iloc[:, 5], facecolor=colors[4], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 4], ls="-", lw=1.0, c=colors[5], label=r"Water")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 4], y2=O_xps.iloc[:, 5], facecolor=colors[5], alpha=0.5)

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(538.0, 527.0)
ax.set_ylim(1000, 25000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=11,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=11,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=1))
ax.tick_params(axis="both", which="both", labelsize=9)

ax.annotate(
    r"530.09 eV",
    (0.70, 0.82),
    xytext=(0.4, 0.90),
    fontsize=9,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[3],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[3], "ls": "--"},
)

ax.annotate(
    r"531.51 eV",
    (0.6, 0.20),
    xytext=(0.35, 0.3),
    fontsize=9,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[4],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[4], "ls": "--"},
)
ax.annotate(
    r"533.81 eV",
    (0.38, 0.12),
    xytext=(0.05, 0.20),
    fontsize=9,
    c=colors[5],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="bold",
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[5], "ls": "--"},
)
ax.text(
    1.0,
    1.10,
    r"O $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=12,
)

# 图 Zn
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.8, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Zn_xps.iloc[:, 0], Zn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(1018, 1050)
ax.set_ylim(11000, 15000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=11,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=11,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=1000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=500))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.tick_params(axis="both", which="both", labelsize=9)

ax.text(
    1.0,
    1.10,
    r"Zn $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=12,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_06_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_06_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_06_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_06_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure XPS， 2nd1.30V

In [ ]:
# 读取数据
path_xps = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XPS+UPS\ExSitu\αMnO2\Charge\Electrode\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\Results")
Mn_xps = pd.read_csv(
    Path.joinpath(path_xps, r"LC_Mn.txt"),
    sep="\t",
    skiprows=6,
    index_col=None,
    header=1,
    comment="#",
).dropna(axis=1, how="all")
Mn_xps = Mn_xps.iloc[:, 28:]

O_xps = pd.read_csv(Path.joinpath(path_xps, "LC_O.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
O_xps = O_xps.iloc[:, 7:]

Zn_xps = pd.read_csv(Path.joinpath(path_xps, "LC_Zn.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
Zn_xps = Zn_xps.iloc[:, 2:]

In [ ]:
# 数据处理, Mn
envelope_Mn_IV = Mn_xps.iloc[:, 2:10].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_III = Mn_xps.iloc[:, 10:18].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_II = Mn_xps.iloc[:, 18:26].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816

In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 Mn
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 27], label=r"Fit", ls="--", color=colors[1])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 26], label=r"Background", color=colors[2], zorder=2)  # background

ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_IV, color=colors[3], linestyle="--", label=r"Mn(IV)")  # envelope_Mn_IV
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_III, color=colors[5], linestyle="--", label=r"Mn(III)")  # envelope_Mn_III
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_II, color=colors[4], linestyle="--", label=r"Mn(II)", zorder=3)  # envelope_Mn_II

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 2:10], color=colors[3], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 2], y2=Mn_xps.iloc[:, 26], facecolor=colors[3], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 10:18], color=colors[5], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 10], y2=Mn_xps.iloc[:, 26], facecolor=colors[5], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 18:26], color=colors[4], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 18], y2=Mn_xps.iloc[:, 26], facecolor=colors[4], alpha=0.3)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, 1.0),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.8,
    handletextpad=0.5,
)

ax.text(
    0.55,
    0.95,
    r"Mn(III): 27.2(2) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[5],
)
ax.text(
    0.55,
    0.85,
    r"Mn(IV): 72.7(0) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)
ax.text(
    0.55,
    0.75,
    r"Mn(II): 0.7(1) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)

ax.text(
    1.0,
    1.11,
    r"Mn $\mathit{2p_{3/2}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=12,
)

ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=11,
)
ax.set_xlim(648.5, 638.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0.5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0.5))

ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=11,
)
ax.set_ylim(2000, 15000)
ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=3000, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1500, offset=0))
ax.tick_params(axis="both", which="both", labelsize=9)

# 图 O
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.4, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 1], ls="-", lw=1.0, c=colors[0], label="Data")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 6], ls="--", lw=1.0, c=colors[1], label="Fit")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 5], ls="-", lw=1.0, c=colors[2], label="Background", zorder=6)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 2], ls="-", lw=1.0, c=colors[3], label=r"Lattice Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 2], y2=O_xps.iloc[:, 5], facecolor=colors[3], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 3], ls="-", lw=1.0, c=colors[4], label="Hydroxide, Hydrated\n or Defective Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 3], y2=O_xps.iloc[:, 5], facecolor=colors[4], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 4], ls="-", lw=1.0, c=colors[5], label=r"Water")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 4], y2=O_xps.iloc[:, 5], facecolor=colors[5], alpha=0.5)

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(535.0, 526.0)
ax.set_ylim(1000, 25000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=11,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=11,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=1))
ax.tick_params(axis="both", which="both", labelsize=9)

# ax.annotate(
#     r"530.08 eV",
#     (0.65, 0.78),
#     xytext=(0.7, 0.90),
#     fontsize=9,
#     fontweight="bold",
#     xycoords="axes fraction",
#     textcoords="axes fraction",
#     c=colors[3],
#     arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[3], "ls": "--"},
# )

# ax.annotate(
#     r"531.04 eV",
#     (0.55, 0.20),
#     xytext=(0.35, 0.35),
#     fontsize=9,
#     fontweight="bold",
#     xycoords="axes fraction",
#     textcoords="axes fraction",
#     c=colors[4],
#     arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[4], "ls": "--"},
# )
# ax.annotate(
#     r"532.77 eV",
#     (0.28, 0.12),
#     xytext=(0.05, 0.25),
#     fontsize=9,
#     c=colors[5],
#     xycoords="axes fraction",
#     textcoords="axes fraction",
#     fontweight="bold",
#     arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[5], "ls": "--"},
# )
ax.text(
    1.0,
    1.10,
    r"O $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=12,
)

# 图 Zn
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.8, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Zn_xps.iloc[:, 0], Zn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(1018, 1050)
ax.set_ylim(7000, 9000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=11,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=11,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=500))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=250))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.tick_params(axis="both", which="both", labelsize=9)

ax.text(
    1.0,
    1.10,
    r"Zn $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=12,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_05_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_05_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_05_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_05_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure XPS， 1st1.80V

In [ ]:
# 读取数据
path_xps = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XPS+UPS\ExSitu\αMnO2\Charge\Electrode\1st1.80V\1M ZnSO4 + 0.2M MnSO4\Results")
Mn_xps = pd.read_csv(
    Path.joinpath(path_xps, r"LC_Mn.txt"),
    sep="\t",
    skiprows=6,
    index_col=None,
    header=1,
    comment="#",
).dropna(axis=1, how="all")
Mn_xps = Mn_xps.iloc[:, 28:]

O_xps = pd.read_csv(Path.joinpath(path_xps, "LC_O.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
O_xps = O_xps.iloc[:, 7:]

Zn_xps = pd.read_csv(Path.joinpath(path_xps, "LC_Zn.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
Zn_xps = Zn_xps.iloc[:, 2:]

In [ ]:
# 数据处理, Mn
envelope_Mn_IV = Mn_xps.iloc[:, 2:10].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_III = Mn_xps.iloc[:, 10:18].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_II = Mn_xps.iloc[:, 18:26].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816

In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 Mn
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 27], label=r"Fit", ls="--", color=colors[1])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 26], label=r"Background", color=colors[2], zorder=2)  # background

ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_IV, color=colors[3], linestyle="--", label=r"Mn(IV)")  # envelope_Mn_IV
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_III, color=colors[5], linestyle="--", label=r"Mn(III)")  # envelope_Mn_III
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_II, color=colors[4], linestyle="--", label=r"Mn(II)", zorder=3)  # envelope_Mn_II

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 2:10], color=colors[3], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 2], y2=Mn_xps.iloc[:, 26], facecolor=colors[3], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 10:18], color=colors[5], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 10], y2=Mn_xps.iloc[:, 26], facecolor=colors[5], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 18:26], color=colors[4], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 18], y2=Mn_xps.iloc[:, 26], facecolor=colors[4], alpha=0.3)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, 1.0),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.8,
    handletextpad=0.5,
)

ax.text(
    0.55,
    0.95,
    r"Mn(III): 33.0(4) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[5],
)
ax.text(
    0.55,
    0.85,
    r"Mn(IV): 58.4(5) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)
ax.text(
    0.55,
    0.75,
    r"Mn(II): 8.4(9) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)

ax.text(
    1.0,
    1.11,
    r"Mn $\mathit{2p_{3/2}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=12,
)

ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=11,
)
ax.set_xlim(648.5, 638.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0.5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0.5))

ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=11,
)
ax.set_ylim(3000, 23000)
ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000, offset=3000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500, offset=3000))
ax.tick_params(axis="both", which="both", labelsize=9)

# 图 O
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.4, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 1], ls="-", lw=1.0, c=colors[0], label="Data")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 6], ls="--", lw=1.0, c=colors[1], label="Fit")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 5], ls="-", lw=1.0, c=colors[2], label="Background", zorder=6)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 2], ls="-", lw=1.0, c=colors[3], label=r"Lattice Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 2], y2=O_xps.iloc[:, 5], facecolor=colors[3], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 3], ls="-", lw=1.0, c=colors[4], label="Hydroxide, Hydrated\n or Defective Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 3], y2=O_xps.iloc[:, 5], facecolor=colors[4], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 4], ls="-", lw=1.0, c=colors[5], label=r"Water")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 4], y2=O_xps.iloc[:, 5], facecolor=colors[5], alpha=0.5)

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(538.0, 526.0)
ax.set_ylim(1000, 25000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=11,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=11,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))
ax.tick_params(axis="both", which="both", labelsize=9)

ax.annotate(
    r"530.00 eV",
    (0.65, 0.78),
    xytext=(0.7, 0.90),
    fontsize=9,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[3],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[3], "ls": "--"},
)

ax.annotate(
    r"531.28 eV",
    (0.55, 0.20),
    xytext=(0.35, 0.35),
    fontsize=9,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[4],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[4], "ls": "--"},
)
ax.annotate(
    r"534.29 eV",
    (0.28, 0.12),
    xytext=(0.05, 0.25),
    fontsize=9,
    c=colors[5],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="bold",
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[5], "ls": "--"},
)
ax.text(
    1.0,
    1.10,
    r"O $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=12,
)

# 图 Zn
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.8, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Zn_xps.iloc[:, 0], Zn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(1018, 1050)
ax.set_ylim(10000, 13000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=11,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=11,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=1000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=500))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.tick_params(axis="both", which="both", labelsize=9)

ax.text(
    1.0,
    1.10,
    r"Zn $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=12,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_04_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_04_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_04_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_04_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure XPS， 1st1.63V

In [ ]:
# 读取数据
path_xps = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XPS+UPS\ExSitu\αMnO2\Charge\Electrode\1st1.63V\1M ZnSO4 + 0.2M MnSO4\Results")
Mn_xps = pd.read_csv(
    Path.joinpath(path_xps, r"LC_Mn.txt"),
    sep="\t",
    skiprows=6,
    index_col=None,
    header=1,
    comment="#",
).dropna(axis=1, how="all")
Mn_xps = Mn_xps.iloc[:, 28:]

O_xps = pd.read_csv(Path.joinpath(path_xps, "LC_O.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
O_xps = O_xps.iloc[:, 7:]

Zn_xps = pd.read_csv(Path.joinpath(path_xps, "LC_Zn.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
Zn_xps = Zn_xps.iloc[:, 2:]

In [ ]:
# 数据处理, Mn
envelope_Mn_IV = Mn_xps.iloc[:, 2:10].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_III = Mn_xps.iloc[:, 10:18].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_II = Mn_xps.iloc[:, 18:26].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816

In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 Mn
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 27], label=r"Fit", ls="--", color=colors[1])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 26], label=r"Background", color=colors[2], zorder=2)  # background

ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_IV, color=colors[3], linestyle="--", label=r"Mn(IV)")  # envelope_Mn_IV
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_III, color=colors[5], linestyle="--", label=r"Mn(III)")  # envelope_Mn_III
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_II, color=colors[4], linestyle="--", label=r"Mn(II)", zorder=3)  # envelope_Mn_II

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 2:10], color=colors[3], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 2], y2=Mn_xps.iloc[:, 26], facecolor=colors[3], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 10:18], color=colors[5], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 10], y2=Mn_xps.iloc[:, 26], facecolor=colors[5], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 18:26], color=colors[4], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 18], y2=Mn_xps.iloc[:, 26], facecolor=colors[4], alpha=0.3)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, 1.0),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.8,
    handletextpad=0.5,
)

ax.text(
    0.55,
    0.95,
    r"Mn(III): 12.1(9) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[5],
)
ax.text(
    0.55,
    0.85,
    r"Mn(IV): 76.8(4) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)
ax.text(
    0.55,
    0.75,
    r"Mn(II): 10.9(5) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)

ax.text(
    1.0,
    1.11,
    r"Mn $\mathit{2p_{3/2}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=12,
)

ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=11,
)
ax.set_xlim(648.5, 638.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0.5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0.5))

ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=11,
)
ax.set_ylim(3000, 24000)
ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000, offset=3000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500, offset=3000))
ax.tick_params(axis="both", which="both", labelsize=9)

# 图 O
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.4, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 1], ls="-", lw=1.0, c=colors[0], label="Data")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 6], ls="--", lw=1.0, c=colors[1], label="Fit")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 5], ls="-", lw=1.0, c=colors[2], label="Background", zorder=6)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 2], ls="-", lw=1.0, c=colors[3], label=r"Lattice Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 2], y2=O_xps.iloc[:, 5], facecolor=colors[3], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 3], ls="-", lw=1.0, c=colors[4], label="Hydroxide, Hydrated\n or Defective Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 3], y2=O_xps.iloc[:, 5], facecolor=colors[4], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 4], ls="-", lw=1.0, c=colors[5], label=r"Water")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 4], y2=O_xps.iloc[:, 5], facecolor=colors[5], alpha=0.5)

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(536.0, 528.0)
ax.set_ylim(1000, 25000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=11,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=11,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))
ax.tick_params(axis="both", which="both", labelsize=9)

ax.annotate(
    r"530.00 eV",
    (0.74, 0.78),
    xytext=(0.7, 0.90),
    fontsize=9,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[3],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[3], "ls": "--"},
)

ax.annotate(
    r"531.91 eV",
    (0.63, 0.20),
    xytext=(0.38, 0.35),
    fontsize=9,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[4],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[4], "ls": "--"},
)
ax.annotate(
    r"533.13 eV",
    (0.28, 0.14),
    xytext=(0.05, 0.30),
    fontsize=9,
    c=colors[5],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="bold",
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[5], "ls": "--"},
)
ax.text(
    1.0,
    1.10,
    r"O $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=12,
)

# 图 Zn
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.8, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Zn_xps.iloc[:, 0], Zn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(1018, 1050)
ax.set_ylim(11000, 14000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=11,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=11,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=1000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=500))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.tick_params(axis="both", which="both", labelsize=9)

ax.text(
    1.0,
    1.10,
    r"Zn $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=12,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_03_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_03_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_03_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_03_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure XPS， 1st1.53V

In [ ]:
# 读取数据
path_xps = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XPS+UPS\ExSitu\αMnO2\Charge\Electrode\1st1.53V\1M ZnSO4 + 0.2M MnSO4\Results")
Mn_xps = pd.read_csv(
    Path.joinpath(path_xps, r"LC_Mn.txt"),
    sep="\t",
    skiprows=6,
    index_col=None,
    header=1,
    comment="#",
).dropna(axis=1, how="all")
Mn_xps = Mn_xps.iloc[:, 28:]

O_xps = pd.read_csv(Path.joinpath(path_xps, "LC_O.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
O_xps = O_xps.iloc[:, 7:]

Zn_xps = pd.read_csv(Path.joinpath(path_xps, "LC_Zn.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
Zn_xps = Zn_xps.iloc[:, 2:]

In [ ]:
# 数据处理, Mn
envelope_Mn_IV = Mn_xps.iloc[:, 2:10].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_III = Mn_xps.iloc[:, 10:18].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_II = Mn_xps.iloc[:, 18:26].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816

In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 Mn
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 27], label=r"Fit", ls="--", color=colors[1])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 26], label=r"Background", color=colors[2], zorder=2)  # background

ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_IV, color=colors[3], linestyle="--", label=r"Mn(IV)")  # envelope_Mn_IV
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_III, color=colors[5], linestyle="--", label=r"Mn(III)")  # envelope_Mn_III
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_II, color=colors[4], linestyle="--", label=r"Mn(II)", zorder=3)  # envelope_Mn_II

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 2:10], color=colors[3], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 2], y2=Mn_xps.iloc[:, 26], facecolor=colors[3], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 10:18], color=colors[5], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 10], y2=Mn_xps.iloc[:, 26], facecolor=colors[5], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 18:26], color=colors[4], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 18], y2=Mn_xps.iloc[:, 26], facecolor=colors[4], alpha=0.3)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, 1.0),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.8,
    handletextpad=0.5,
)

ax.text(
    0.55,
    0.95,
    r"Mn(III): 36.8(1) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[5],
)
ax.text(
    0.55,
    0.85,
    r"Mn(IV): 59.2(6) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)
ax.text(
    0.55,
    0.75,
    r"Mn(II): 3.9(0) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)

ax.text(
    1.0,
    1.11,
    r"Mn $\mathit{2p_{3/2}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=12,
)

ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=11,
)
ax.set_xlim(648.5, 638.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0.5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0.5))

ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=11,
)
ax.set_ylim(3000, 23000)
ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000, offset=3000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500, offset=3000))
ax.tick_params(axis="both", which="both", labelsize=9)

# 图 O
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.4, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 1], ls="-", lw=1.0, c=colors[0], label="Data")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 6], ls="--", lw=1.0, c=colors[1], label="Fit")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 5], ls="-", lw=1.0, c=colors[2], label="Background", zorder=6)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 2], ls="-", lw=1.0, c=colors[3], label=r"Lattice Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 2], y2=O_xps.iloc[:, 5], facecolor=colors[3], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 3], ls="-", lw=1.0, c=colors[4], label="Hydroxide, Hydrated\n or Defective Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 3], y2=O_xps.iloc[:, 5], facecolor=colors[4], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 4], ls="-", lw=1.0, c=colors[5], label=r"Water")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 4], y2=O_xps.iloc[:, 5], facecolor=colors[5], alpha=0.5)

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(536.0, 528.0)
ax.set_ylim(1000, 25000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=11,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=11,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))
ax.tick_params(axis="both", which="both", labelsize=9)

ax.annotate(
    r"530.10 eV",
    (0.74, 0.78),
    xytext=(0.7, 0.90),
    fontsize=9,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[3],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[3], "ls": "--"},
)

ax.annotate(
    r"531.08 eV",
    (0.56, 0.20),
    xytext=(0.38, 0.35),
    fontsize=9,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[4],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[4], "ls": "--"},
)
ax.annotate(
    r"533.11 eV",
    (0.28, 0.14),
    xytext=(0.05, 0.30),
    fontsize=9,
    c=colors[5],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="bold",
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[5], "ls": "--"},
)
ax.text(
    1.0,
    1.10,
    r"O $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=12,
)

# 图 Zn
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.8, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Zn_xps.iloc[:, 0], Zn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(1018, 1050)
ax.set_ylim(11000, 13000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=11,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=11,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=500))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=250))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.tick_params(axis="both", which="both", labelsize=9)

ax.text(
    1.0,
    1.10,
    r"Zn $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=12,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_02_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_02_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_02_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_02_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure XPS, Pristine

In [ ]:
# 读取数据
path_xps = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XPS+UPS\ExSitu\αMnO2\Pristine\Powder\Results\2025-ICN2\V3")
Mn_xps = pd.read_csv(
    Path.joinpath(path_xps, r"2025_090_sample4_LC_Mn.txt"),
    sep="\t",
    skiprows=6,
    index_col=None,
    header=1,
    comment="#",
).dropna(axis=1, how="all")
Mn_xps = Mn_xps.iloc[:, 28:]

O_xps = pd.read_csv(Path.joinpath(path_xps, "2025_090_sample4_LC_O.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
O_xps = O_xps.iloc[:, 7:]

In [ ]:
# 数据处理, Mn
envelope_Mn_IV = Mn_xps.iloc[:, 2:10].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_III = Mn_xps.iloc[:, 10:18].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_II = Mn_xps.iloc[:, 18:26].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816

In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 Mn
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 27], label=r"Fit", ls="--", color=colors[1])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 26], label=r"Background", color=colors[2], zorder=2)  # background

ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_IV, color=colors[3], linestyle="--", label=r"Mn(IV)")  # envelope_Mn_IV
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_III, color=colors[5], linestyle="--", label=r"Mn(III)")  # envelope_Mn_III
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_II, color=colors[4], linestyle="--", label=r"Mn(II)", zorder=3)  # envelope_Mn_II

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 2:10], color=colors[3], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 2], y2=Mn_xps.iloc[:, 26], facecolor=colors[3], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 10:18], color=colors[5], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 10], y2=Mn_xps.iloc[:, 26], facecolor=colors[5], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 18:26], color=colors[4], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 18], y2=Mn_xps.iloc[:, 26], facecolor=colors[4], alpha=0.3)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, 1.0),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.8,
    handletextpad=0.5,
)

ax.text(
    0.55,
    0.95,
    r"Mn(III): 41.9(3) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[5],
)
ax.text(
    0.55,
    0.85,
    r"Mn(IV): 57.9(6) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)
ax.text(
    0.55,
    0.75,
    r"Mn(II): 0.1(3) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)

ax.text(
    1.0,
    1.11,
    r"Mn $\mathit{2p_{3/2}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=12,
)

ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=11,
)
ax.set_xlim(648.5, 638.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0.5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0.5))

ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=11,
)
ax.set_ylim(1000, 5000)
ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=1000, offset=1000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=500, offset=1000))
ax.tick_params(axis="both", which="both", labelsize=9)

# 图 O
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.4, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 1], ls="-", lw=1.0, c=colors[0], label="Data")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 6], ls="--", lw=1.0, c=colors[1], label="Fit")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 5], ls="-", lw=1.0, c=colors[2], label="Background", zorder=6)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 2], ls="-", lw=1.0, c=colors[3], label=r"Lattice Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 2], y2=O_xps.iloc[:, 5], facecolor=colors[3], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 3], ls="-", lw=1.0, c=colors[4], label="Hydroxide, Hydrated\n or Defective Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 3], y2=O_xps.iloc[:, 5], facecolor=colors[4], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 4], ls="-", lw=1.0, c=colors[5], label=r"Water")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 4], y2=O_xps.iloc[:, 5], facecolor=colors[5], alpha=0.5)

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(536.5, 527.5)
ax.set_ylim(500, 6000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=11,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=11,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=1000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=500))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0.5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0.5))
ax.tick_params(axis="both", which="both", labelsize=9)

ax.annotate(
    r"530.03 eV",
    (0.70, 0.75),
    xytext=(0.65, 0.90),
    fontsize=9,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[3],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[3], "ls": "--"},
)

ax.annotate(
    r"531.39 eV",
    (0.56, 0.20),
    xytext=(0.38, 0.35),
    fontsize=9,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[4],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[4], "ls": "--"},
)
ax.annotate(
    r"533.96 eV",
    (0.25, 0.14),
    xytext=(0.05, 0.30),
    fontsize=9,
    c=colors[5],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="bold",
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[5], "ls": "--"},
)
ax.text(
    1.0,
    1.11,
    r"O $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=12,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_01_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_01_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_01_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_01_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure UPS -V1

In [ ]:
# 读取数据
all_ups_paths = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XPS+UPS\ExSitu\αMnO2").glob(r"**\Data\*_UPS.txt"))

path_ups = [p for p in all_ups_paths if "1M ZnSO4 + 0.2M MnSO4" in str(p) and "BulkCell" not in str(p)]

path_ups = [path_ups[i] for i in [1, 2, 3, 5, 4]]
# path_ups = [path_ups[i] for i in [0, 1, 2, 3, 5, 4]]

# 读取数据
df_ups = [
    pd.read_csv(
        p,
        sep=r"\s+",
        skiprows=7,
        index_col=None,
        names=[r"B.E.1", r"Overview:vb", r"B.E.2", r"Overview:wf-10v"],
        header=None,
        comment="#",
    )
    for p in path_ups
]

In [ ]:
%matplotlib inline
plt.close("all")
fig = plt.figure(figsize=(3.3, 2.5))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

sample_name = [
    r"HC#A",
    r"HC#B",
    r"FC#C",
    r"HD#D",
    r"FD#E",
]

# sample_name = [
#     r"$\mathrm{1^{st}\ Discharge}$",
#     r"$\mathrm{1^{st}\ HalfCharge\#1}$",
#     r"$\mathrm{1^{st}\ HalfCharge\#2}$",
#     r"$\mathrm{1^{st}\ FullCharge}$",
#     r"$\mathrm{2^{nd}\ HalfDischarge}$",
#     r"$\mathrm{2^{nd}\ FullDischarge}$",
# ]

for i, (name, ups) in enumerate(zip(sample_name, df_ups, strict=False)):
    ax.plot(ups["B.E.2"] + 10, ups["Overview:wf-10v"], c=colors_opt2[i + 2], label=f"{name}", zorder=10 - i)

ax.set_xlabel(r"Binding Energy (eV)", fontsize=11)
ax.set_xlim(0, 25)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.5))

ax.set_ylabel(r"Intensity (counts)", fontsize=11)
ax.set_ylim(-1000, 200000)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=50000, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=25000, offset=0))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
formatter.set_scientific(True)
ax.yaxis.set_major_formatter(formatter)

ax.legend(loc="upper right", bbox_to_anchor=(0.3, 0.9), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    0.7,
    0.95,
    r"Bias - 10 eV",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=11,
    color="k",
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_00_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_00_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_00_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_00_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure UPS

In [ ]:
# 读取数据
all_ups_paths = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XPS+UPS\ExSitu\αMnO2").glob(r"**\Data\*_UPS.txt"))

path_ups = [p for p in all_ups_paths if "1M ZnSO4 + 0.2M MnSO4" in str(p) and "BulkCell" not in str(p)]

path_ups = [path_ups[i] for i in [0, 1, 2, 3, 5, 4]]

# 读取数据
df_ups = [
    pd.read_csv(
        p,
        sep=r"\s+",
        skiprows=7,
        index_col=None,
        names=[r"B.E.1", r"Overview:vb", r"B.E.2", r"Overview:wf-10v"],
        header=None,
        comment="#",
    )
    for p in path_ups
]

In [ ]:
%matplotlib inline
plt.close("all")
fig = plt.figure(figsize=(3.3, 2.5))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

sample_name = [
    r"$\mathrm{1^{st}\ Discharge}$",
    r"$\mathrm{1^{st}\ HalfCharge\#1}$",
    r"$\mathrm{1^{st}\ HalfCharge\#2}$",
    r"$\mathrm{1^{st}\ FullCharge}$",
    r"$\mathrm{2^{nd}\ HalfDischarge}$",
    r"$\mathrm{2^{nd}\ FullDischarge}$",
]

for i, (name, ups) in enumerate(zip(sample_name, df_ups, strict=False)):
    ax.plot(ups["B.E.2"] + 10, ups["Overview:wf-10v"], c=colors[i], label=f"{name}", zorder=10 - i)

ax.set_xlabel(r"Binding Energy (eV)", fontsize=11)
ax.set_xlim(0, 25)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.5))

ax.set_ylabel(r"Intensity (counts)", fontsize=11)
ax.set_ylim(-1000, 400000)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=100000, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=50000, offset=0))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
formatter.set_scientific(True)
ax.yaxis.set_major_formatter(formatter)

ax.legend(loc="upper right", bbox_to_anchor=(0.5, 1.02), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    0.7,
    0.95,
    r"Bias - 10 eV",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=11,
    color="k",
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_00_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_00_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_00_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XPS+UPS_00_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

## XRD

### Operando XRD, EMD, being from No pH Buffer, 1 M + 0.2 M, 30uA -V2

In [ ]:
# 读取电化学数据
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\EMD\2025-MSPD\Results\IS17_D\Echem\B").glob(r"**\*.txt"))
echem_all = []
for path_file in path_filelist[0:1]:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break
    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    df["Voltage/V"] = df["Ewe/V"] - df["Ece/V"]
    echem_all.append(df)
# 合并所有电化学数据为一个二维表格
echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)


# 谱线上的时间
# 读取文件中时间戳
filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\EMD\2025-MSPD\Results\IS17_D\Data\IS17_D").glob(r"*.xye"))
range_index, wave_length, time_processed = [], [], []
for path_file in filelist:
    with open(path_file, "r") as file:
        lines = file.readlines()
        file_id = path_file.stem.split("_")[-1].split(".")[0][-5:]
        range_index.append(file_id)
        for line in lines:
            if line.startswith("# Wave"):
                wave_value = float(line.split()[3])
                wave_length.append(wave_value)
            elif line.startswith("# Date"):
                recored_time = str(line.split()[3])
                time_processed.append(recored_time)
spectrum_time_all = pd.DataFrame({
    "Range_Index": range_index,
    "time/s": time_processed,
    "Wave_Length": wave_length,
})
spectrum_time_all["time/s"] = pd.to_datetime(spectrum_time_all["time/s"], format=r"%Y-%m-%d_%H:%M:%S")
spectrum_time_all["Range_Index"] = pd.to_numeric(spectrum_time_all["Range_Index"])
spectrum_time_all["Wave_Length"] = pd.to_numeric(spectrum_time_all["Wave_Length"])

# 读取 XRD 的数据
path_xrd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\EMD\2025-MSPD\Results\IS17_D\Data")

spectrum_all = pd.read_csv(path_xrd.joinpath(r"spectrum_all_d_spacing.csv"), index_col=0, header=0)
spectrum_all.index = pd.to_numeric(spectrum_all.index)
spectrum_all.columns = pd.to_numeric(spectrum_all.columns)

In [ ]:
# 只保留第一圈的充放电与第二圈的充电数据
selected_echem = echem_all[echem_all["cycle number"].isin([1, 2])]
selected_echem = selected_echem[selected_echem["Voltage/V"] >= 0.8]

# selected_echem = selected_echem.iloc[:-5, :].copy()
selected_echem = selected_echem.copy()
selected_echem["charge_time"] = (selected_echem["time/s"] - selected_echem["time/s"].iloc[0]).dt.total_seconds() / 3600

# 匹配谱线和电化学上的时间
selected_spectrum_time = (
    pd.merge_asof(
        selected_echem.sort_values(by="time/s"),
        spectrum_time_all.sort_values(by="time/s"),
        on="time/s",
        direction="nearest",  # 找最近的时间点
        tolerance=pd.Timedelta("5s"),  # 可设定允许的最大偏差
    )
    .dropna(subset=["Range_Index"], inplace=False)
    .drop_duplicates(subset=["Range_Index"], keep="first", inplace=False)
    .reset_index(drop=False, inplace=False)
)

# 选择谱线的区间
d_spacing_range = (0.5, 15.0)
spectrum_all = spectrum_all.loc[(spectrum_all.index >= d_spacing_range[0]) & (spectrum_all.index <= d_spacing_range[1])]

# 修复数据类型匹配问题：直接使用整数列表来匹配 spectrum_all 的列名
range_index_list = selected_spectrum_time["Range_Index"].tolist()
selected_spectrum = spectrum_all.loc[:, spectrum_all.columns.isin(range_index_list)]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 4, 4], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Voltage/V"], selected_echem["charge_time"], ls="-", lw=1.0, c=colors[0], label=r"Voltage")

# 添加索引文本标注
index_labels = [15, 25, 37, 46, 50, 61, 70, 75, 87, 99, 109, 115, 126, 137]

for i, idx in enumerate(selected_spectrum_time["Range_Index"].tolist()):
    if idx in index_labels:
        value = selected_spectrum_time.loc[selected_spectrum_time["Range_Index"] == idx, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c=colors[0],
            edgecolors="face",
            s=50,
            zorder=5,
        )
    else:
        value = selected_spectrum_time.loc[selected_spectrum_time.index == i, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c="lightgrey",
            edgecolors="face",
            s=30,
            zorder=1,
        )
ax.set_xlabel(
    r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$",
    fontsize=11,
)
ax.set_xlim(0.8, 2.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=0))

ax.set_ylabel(r"Duration Time (hour)", fontsize=11)
ax.set_ylim(-0.5, 36)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))


ax2 = ax.twiny()
ax2.set_position((0, 0, 1, 1))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"] * 1000, selected_echem["charge_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(
    r"$\mathrm{Current  \ (\mu A)}$",
    fontsize=11,
)
ax2.set_xlim(-60, 60)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=60, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=30, offset=0))
ax2.tick_params(axis="both", which="both", labelsize=9, right=True, labelright=True)
ax.text(-0.6, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.05, 0.035, 0.94, 0.94))
ax.set_box_aspect(1.0)

ax.imshow(
    selected_spectrum.T,
    cmap="coolwarm",
    aspect="auto",
    vmin=6300,
    vmax=31700,
    origin="lower",
    extent=[selected_spectrum.index.min(), selected_spectrum.index.max(), selected_spectrum.columns.min(), selected_spectrum.columns.max()],
)

ax.set_yticks([])
ax.set_xlim(0.5, 8.0)
ax.set_xlabel(
    r"$\mathrm{d \ Spacing \ (\AA)}$",
    fontsize=11,
)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))
ax.tick_params(axis="x", which="both", direction="out", labelsize=9, top=False)

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.03, 0.1, 0.08, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.text(
    1.08,
    0.97,
    "High",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)
ax.text(
    1.08,
    0.08,
    "Low",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)

ax.text(-0.05, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)

opxrd_ref_d_spacing = [1.414916, 2.44884, 3.365342, 5.967425, 8.042391]
vmin_vmax = [
    (12000, 15500),   # d=1.41Å
    (22416, 29000),   # d=2.45Å
    (30348, 40000),   # d=3.37Å
    (24678, 28961),
    (22549, 27786),
]
peak_width = [
    (0.05, 0.05),   # d=1.41Å
    (0.05, 0.05),   # d=2.45Å
    (0.4, 0.4),   # d=3.37Å
    (0.4, 0.4),
    (0.4, 0.4),
]
for i in range(len(opxrd_ref_d_spacing)):
    ax = subfig.add_axes((0.1 + i * 0.14, 0.035, 0.12, 0.94), zorder=0)
    d_spacing_value = opxrd_ref_d_spacing[i]
    temp = selected_spectrum.loc[(selected_spectrum.index >= d_spacing_value - peak_width[i][0]) & (selected_spectrum.index <= d_spacing_value + peak_width[i][1])]
    vmin = vmin_vmax[i][0] if vmin_vmax[i][0] is not None else temp.min().min()
    vmax = vmin_vmax[i][1] if vmin_vmax[i][1] is not None else temp.max().max()
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin, vmax=vmax, extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(
        d_spacing_value,
        ls="--",
        lw=1.0,
        color="grey",
        alpha=0.5,
    )

    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([d_spacing_value]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor="k",
        bottom=True,
        left=True,
        labelbottom=True,
        labelleft=True,
    )
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[1], alpha=0.5)

ax.text(-5.0, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_05_V2_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_05_V2_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_05_V2_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_05_V2_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

In [ ]:
%matplotlib inline
plt.close("all")
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 0.3, 0.3))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Voltage/V"], selected_echem["charge_time"], ls="-", lw=1.0, c=colors[0], label=r"Voltage")

# 添加索引文本标注
index_labels = [15, 25, 37, 46, 50, 61, 70, 75, 87, 99, 109, 115, 126, 137]

for i, idx in enumerate(selected_spectrum_time["Range_Index"].tolist()):
    if idx in index_labels:
        value = selected_spectrum_time.loc[selected_spectrum_time["Range_Index"] == idx, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c=colors[0],
            edgecolors="face",
            s=50,
            zorder=5,
        )
    else:
        value = selected_spectrum_time.loc[selected_spectrum_time.index == i, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c="lightgrey",
            edgecolors="face",
            s=30,
            zorder=1,
        )
ax.set_xlabel(
    r"Voltage (V)",
    fontsize=9,
)
ax.set_xlim(0.8, 2.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.2, offset=-0.4))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.6, offset=-0.4))

ax.set_ylabel(r"Duration Time (hour)", fontsize=9)
ax.set_ylim(-1, 36)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))


ax2 = ax.twiny()
ax2.set_position((0, 0, 0.3, 0.3))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"] * 1000, selected_echem["charge_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(
    r"$\mathrm{Current  \ (\mu A)}$",
    fontsize=9,
)
ax2.set_xlim(-60, 60)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=60, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=30, offset=0))
ax2.tick_params(axis="both", which="both", labelsize=9, right=True, labelright=True)

ax.text(-1.0, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 B
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
opxrd_ref_d_spacing = [1.578523, 1.821636, 2.32571, 2.473, 2.536224, 2.671445, 2.714108, 3.128303, 3.230361, 3.408803, 3.51733, 3.644164, 4.172575, 5.465435, 10.9373]
vmin_vmax = [
    (None, None), (None, None), (None, None), (None, None), (None, None), (None, None),
    (None, None), (None, None), (None, None), (None, None), (None, None), (None, None),
    (27800, 34000), (None, None), (None, None)
]
peak_width = [
    (0.1, 0.1), (0.1, 0.1), (0.1, 0.1), (0.1, 0.1), (0.1, 0.1), (0.1, 0.1),
    (0.1, 0.1), (0.1, 0.1), (0.1, 0.1), (0.1, 0.1), (0.1, 0.1), (0.1, 0.1),
    (0.1, 0.1), (0.1, 0.1), (0.3, 0.3)
]

for i in range(0, len(opxrd_ref_d_spacing) // 2 - 1):
    # print(i)
    ax = subfig.add_axes((0.22 + i * 0.055, 0.0, 0.05, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_d_spacing[i] - peak_width[i][0]) & (selected_spectrum.index <= opxrd_ref_d_spacing[i] + peak_width[i][1])]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_d_spacing[i], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_d_spacing[i]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, labelbottom=True)
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[1], alpha=0.5)

for i in range(len(opxrd_ref_d_spacing) // 2 - 1, len(opxrd_ref_d_spacing)):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - len(opxrd_ref_d_spacing) // 2 + 1) * 0.055, -0.43, 0.05, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_d_spacing[i] - peak_width[i][0]) & (selected_spectrum.index <= opxrd_ref_d_spacing[i] + peak_width[i][1])]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_d_spacing[i], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_d_spacing[i]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, labelbottom=True)
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[1], alpha=0.5)

ax.text(-5.5, 2.45, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_06_V2_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_06_V2_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_06_V2_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_06_V2_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Operando XRD, EMD, being from No pH Buffer, 1 M + 0.2 M, 30uA -V1

In [ ]:
# 处理 XRD 的 xye 文件
# 1) 读取单个文件，提取波长与数据
def extract_wave_and_data(file_path: Path):
    wave_value = np.nan
    recorded_time = None  # 用 str 保存也可以，下面再转 datetime
    data_rows = []

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue

            # 头部信息
            if line.startswith("#"):
                # Wave: 允许 "1.5406", "1.5406A", "1.5406 Å" 等
                if "wave" in line.lower():
                    m = re.search(r"([-+]?\d*\.?\d+)", line)
                    if m:
                        wave_value = float(m.group(1))

                # Date: 抓取整行，后面统一解析
                elif "date" in line.lower():
                    # 直接保留原字符串；也可以只取某个片段
                    recorded_time = line.replace("#", "").strip()
                continue

            # 数据行：空白分隔，容错不同列数
            parts = re.split(r"\s+", line)
            nums = []
            for p in parts:
                try:
                    nums.append(float(p))
                except ValueError:
                    nums.append(np.nan)

            # 只要首列 2THETA 能转成数就收
            if len(nums) >= 1 and np.isfinite(nums[0]):
                data_rows.append(nums)

    # 统一成至少三列 [2THETA, Int1, Int2]，不足的补 NaN
    cleaned = []
    for row in data_rows:
        # 至少有 2THETA 与一个强度
        tth = row[0]
        i1 = row[1] if len(row) > 1 else np.nan
        i2 = row[2] if len(row) > 2 else np.nan  # 有些 xye 第三列是 error；你可按需改成忽略
        cleaned.append([tth, i1, i2])

    df = pd.DataFrame(cleaned, columns=["2THETA", "Intensity1", "Intensity2"])
    return recorded_time, wave_value, df


# 2) 处理目录下所有文件，生成长表
def process_files(directory: Path):
    rows = []  # 长表
    for file_path in sorted(directory.glob("*.xye")):
        recorded_time, wave_value, df = extract_wave_and_data(file_path)

        # 从文件名末尾提取数字（更健壮，如 ..._f00024 / ..._123 / ..._idx-007）
        m = re.search(r"(\d+)$", file_path.stem)
        file_id = float(m.group(1)) if m else np.nan

        # 清洗 & 叠加到长表
        df = df.dropna(subset=["2THETA"]).copy()
        for _, r in df.iterrows():
            rows.append([
                file_id,
                recorded_time,
                wave_value,
                float(r["2THETA"]),
                float(r["Intensity1"]) if pd.notna(r["Intensity1"]) else np.nan,
                float(r["Intensity2"]) if pd.notna(r["Intensity2"]) else np.nan,
            ])

    spectrum_all = pd.DataFrame(
        rows,
        columns=["Index", "recorded_time", "wave_value", "2THETA", "Intensity1", "Intensity2"],
    )
    # 数据类型转换与清洗
    spectrum_all["Index"] = pd.to_numeric(spectrum_all["Index"], errors="coerce")
    spectrum_all["2THETA"] = pd.to_numeric(spectrum_all["2THETA"], errors="coerce")
    spectrum_all["Cnt2_D1"] = spectrum_all[["Intensity1", "Intensity2"]].sum(axis=1, min_count=1)

    spectrum_all = spectrum_all.dropna(subset=["Index", "2THETA", "Cnt2_D1"])
    spectrum_all = spectrum_all.sort_values(["Index", "2THETA"]).reset_index(drop=True)
    return spectrum_all


In [ ]:
# 读取电化学数据
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\EMD\2025-MSPD\Results\IS17_D\Echem\B").glob(r"**\*.txt"))
echem_all = []
for path_file in path_filelist[0:1]:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break
    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    df["Voltage/V"] = df["Ewe/V"] - df["Ece/V"]
    echem_all.append(df)
# 合并所有电化学数据为一个二维表格
echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 谱线上的时间
# 读取文件中时间戳
filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\EMD\2025-MSPD\Results\IS17_D\Data\IS17_D").glob(r"*.xye"))
range_index, wave_length, time_processed = [], [], []
for path_file in filelist:
    with open(path_file, "r") as file:
        lines = file.readlines()
        file_id = path_file.stem.split("_")[-1].split(".")[0][-5:]
        range_index.append(file_id)
        for line in lines:
            if line.startswith("# Wave"):
                wave_value = float(line.split()[3])
                wave_length.append(wave_value)
            elif line.startswith("# Date"):
                recored_time = str(line.split()[3])
                time_processed.append(recored_time)
spectrum_time_all = pd.DataFrame({
    "Range_Index": range_index,
    "time/s": time_processed,
    "Wave_Length": wave_length,
})
spectrum_time_all["time/s"] = pd.to_datetime(spectrum_time_all["time/s"], format=r"%Y-%m-%d_%H:%M:%S")
spectrum_time_all["Range_Index"] = pd.to_numeric(spectrum_time_all["Range_Index"])
spectrum_time_all["Wave_Length"] = pd.to_numeric(spectrum_time_all["Wave_Length"])

# 读取 XRD 的数据
file_list = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\EMD\2025-MSPD\Results\IS17_D\Data\IS17_D")
spectrum_all = process_files(file_list)
spectrum_all = spectrum_all.groupby(["Index", "2THETA"], as_index=False)["Cnt2_D1"].mean().pivot_table(index="2THETA", columns="Index", values="Cnt2_D1").sort_index()

# # 可选：保存
# spectrum_all.to_csv(file_list.joinpath("spectrum_all.csv"), index=True)

In [ ]:
# 只保留第一圈的充放电与第二圈的充电数据
selected_echem = echem_all[echem_all["cycle number"].isin([1, 2])]
selected_echem = selected_echem[selected_echem["Voltage/V"] >= 0.8]

# selected_echem = selected_echem.iloc[:-5, :].copy()
selected_echem = selected_echem.copy()
selected_echem["charge_time"] = (selected_echem["time/s"] - selected_echem["time/s"].iloc[0]).dt.total_seconds() / 3600

# 匹配谱线和电化学上的时间
selected_spectrum_time = (
    pd.merge_asof(
        selected_echem.sort_values(by="time/s"),
        spectrum_time_all.sort_values(by="time/s"),
        on="time/s",
        direction="nearest",  # 找最近的时间点
        tolerance=pd.Timedelta("5s"),  # 可设定允许的最大偏差
    )
    .dropna(subset=["Range_Index"], inplace=False)
    .drop_duplicates(subset=["Range_Index"], keep="first", inplace=False)
    .reset_index(drop=False, inplace=False)
)

# 选择 谱线的区间
spectrum_all = spectrum_all[(spectrum_all.index >= 2.0) & (spectrum_all.index <= 27.0)]
selected_spectrum = spectrum_all.loc[:, spectrum_all.columns.isin(selected_spectrum_time["Range_Index"].to_list())]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 4, 4], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Voltage/V"], selected_echem["charge_time"], ls="-", lw=1.0, c=colors[0], label=r"Voltage")

# 添加索引文本标注
index_labels = [15, 25, 37, 46, 50, 61, 70, 75, 87, 99, 109, 115, 126, 137]

for i, idx in enumerate(selected_spectrum_time["Range_Index"].tolist()):
    if idx in index_labels:
        value = selected_spectrum_time.loc[selected_spectrum_time["Range_Index"] == idx, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c=colors[0],
            edgecolors="face",
            s=50,
            zorder=5,
        )
    else:
        value = selected_spectrum_time.loc[selected_spectrum_time.index == i, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c="lightgrey",
            edgecolors="face",
            s=30,
            zorder=1,
        )
ax.set_xlabel(
    r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$",
    fontsize=11,
)
ax.set_xlim(0.8, 2.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=0))

ax.set_ylabel(r"Duration Time (hour)", fontsize=11)
ax.set_ylim(-0.5, 36)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))


ax2 = ax.twiny()
ax2.set_position((0, 0, 1, 1))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"] * 1000, selected_echem["charge_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(
    r"$\mathrm{Current  \ (\mu A)}$",
    fontsize=11,
)
ax2.set_xlim(-60, 60)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=60, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=30, offset=0))
ax2.tick_params(axis="both", which="both", labelsize=9, right=True, labelright=True)
ax.text(-0.6, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.05, 0.035, 0.94, 0.94))
ax.set_box_aspect(1.0)

ax.imshow(
    selected_spectrum.T,
    cmap="coolwarm",
    aspect="auto",
    vmin=10000,
    vmax=40000,
    origin="lower",
    extent=[selected_spectrum.index.min(), selected_spectrum.index.max(), selected_spectrum.columns.min(), selected_spectrum.columns.max()],
)

ax.set_yticks([])
ax.set_xlim(2, 27)
ax.set_xlabel(
    r"2-Theta (degree)",
    fontsize=11,
)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=-3))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.5, offset=-3))
ax.tick_params(axis="x", which="both", direction="out", labelsize=9, top=False)

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.03, 0.1, 0.08, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.text(
    1.08,
    0.97,
    "High",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)
ax.text(
    1.08,
    0.08,
    "Low",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)

ax.text(-0.05, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C

subfig = fig.add_subfigure(gs[0, 2], zorder=0)

opxrd_ref_peak = [[3.55], [4.75], [6.15], [11.5], [20.0], [23.35]]
opxrd_ref_d_spacing = [[8.01], [5.98], [4.62], [2.48], [1.43], [1.22]]
opxrd_colors = ["blue", "blue", "blue", "blue", "blue", "blue"]
vmin_vmax = [(None, None), (None, None), (None, None), (23500, 29000), (None, None), (None, None)]
for i in range(len(opxrd_ref_peak)):
    ax = subfig.add_axes((0.1 + i * 0.14, 0.035, 0.12, 0.94), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.4) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.4)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(
        opxrd_ref_peak[i][0],
        ls="--",
        lw=1.0,
        color="grey",
        alpha=0.5,
    )
    ax.text(0.9 + 0.01 * i, -0.15, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")

    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor="k",
        bottom=True,
        left=True,
        labelbottom=True,
        labelleft=True,
    )
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)
ax.text(-6.0, -0.14, r"$\mathrm{d \ Spacing\ (\AA)}$", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold", c="blue")

ax.text(-6.0, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_05_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_05_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_05_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_05_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

In [ ]:
%matplotlib inline
plt.close("all")
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 0.3, 0.3))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Voltage/V"], selected_echem["charge_time"], ls="-", lw=1.0, c=colors[0], label=r"Voltage")

# 添加索引文本标注
index_labels = [15, 25, 37, 46, 50, 61, 70, 75, 87, 99, 109, 115, 126, 137]

for i, idx in enumerate(selected_spectrum_time["Range_Index"].tolist()):
    if idx in index_labels:
        value = selected_spectrum_time.loc[selected_spectrum_time["Range_Index"] == idx, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c=colors[0],
            edgecolors="face",
            s=50,
            zorder=5,
        )
    else:
        value = selected_spectrum_time.loc[selected_spectrum_time.index == i, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c="lightgrey",
            edgecolors="face",
            s=30,
            zorder=1,
        )
ax.set_xlabel(
    r"Voltage (V)",
    fontsize=9,
)
ax.set_xlim(0.8, 2.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.2, offset=-0.4))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.6, offset=-0.4))

ax.set_ylabel(r"Duration Time (hour)", fontsize=9)
ax.set_ylim(-1, 36)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))


ax2 = ax.twiny()
ax2.set_position((0, 0, 0.3, 0.3))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"] * 1000, selected_echem["charge_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(
    r"$\mathrm{Current  \ (\mu A)}$",
    fontsize=9,
)
ax2.set_xlim(-60, 60)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=60, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=30, offset=0))
ax2.tick_params(axis="both", which="both", labelsize=9, right=True, labelright=True)

ax.text(-1.0, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 B
subfig = fig.add_subfigure(gs[0, 0], zorder=0)

opxrd_ref_peak = [
    [2.6],
    [5.20],
    [6.82],
    [8.10],
    [8.90],
    [10.0],
    [10.4],
    [10.53],
    [11.00],
    [11.52],
    [12.25],
    [12.86],
    [13.81],
    [14.57],
    [15.31],
    [15.72],
    [16.00],
    [16.80],
    [17.24],
    [17.82],
    [18.33],
    [19.384],
    [21.96],
    [22.13],
    [24.8],
]
opxrd_ref_d_spacing = [
    [10.93],
    [5.47],
    [4.17],
    [3.51],
    [3.20],
    [2.85],
    [2.74],
    [2.70],
    [2.59],
    [2.47],
    [2.33],
    [2.21],
    [2.06],
    [1.96],
    [1.86],
    [1.81],
    [1.78],
    [1.70],
    [1.65],
    [1.60],
    [1.56],
    [1.47],
    [1.30],
    [1.29],
    [1.15],
]

vmin_vmax = [
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
]
for i in range(0, len(opxrd_ref_peak) // 3 - 1):
    # print(i)
    ax = subfig.add_axes((0.23 + i * 0.09, 0.0, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.4) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.4)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.text(0.75 + 0.001 * i, -0.32, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, labelbottom=True)
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(len(opxrd_ref_peak) // 3 - 1, 2 * len(opxrd_ref_peak) // 3):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - len(opxrd_ref_peak) // 3 + 1) * 0.09, -0.43, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.4) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.4)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.text(0.75 + 0.001 * i, -0.23, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, labelbottom=True)
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(2 * len(opxrd_ref_peak) // 3, len(opxrd_ref_peak)):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - 2 * len(opxrd_ref_peak) // 3) * 0.09, -0.82, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.4) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.4)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.text(0.75 + 0.001 * i, -0.23, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, labelbottom=True)
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=1.0)
ax.text(-6.7, 2.42, r"$\mathrm{d \ Spacing\ (\AA)}$", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold", c="blue")
ax.text(-6.8, 3.65, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_06_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_06_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_06_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_06_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Operando XRD, EMD, being from No pH Buffer, 1 M + 0.2 M, 30uA

In [ ]:
# 处理 XRD 的 xye 文件
# 1) 读取单个文件，提取波长与数据
def extract_wave_and_data(file_path: Path):
    wave_value = np.nan
    recorded_time = None  # 用 str 保存也可以，下面再转 datetime
    data_rows = []

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue

            # 头部信息
            if line.startswith("#"):
                # Wave: 允许 "1.5406", "1.5406A", "1.5406 Å" 等
                if "wave" in line.lower():
                    m = re.search(r"([-+]?\d*\.?\d+)", line)
                    if m:
                        wave_value = float(m.group(1))

                # Date: 抓取整行，后面统一解析
                elif "date" in line.lower():
                    # 直接保留原字符串；也可以只取某个片段
                    recorded_time = line.replace("#", "").strip()
                continue

            # 数据行：空白分隔，容错不同列数
            parts = re.split(r"\s+", line)
            nums = []
            for p in parts:
                try:
                    nums.append(float(p))
                except ValueError:
                    nums.append(np.nan)

            # 只要首列 2THETA 能转成数就收
            if len(nums) >= 1 and np.isfinite(nums[0]):
                data_rows.append(nums)

    # 统一成至少三列 [2THETA, Int1, Int2]，不足的补 NaN
    cleaned = []
    for row in data_rows:
        # 至少有 2THETA 与一个强度
        tth = row[0]
        i1 = row[1] if len(row) > 1 else np.nan
        i2 = row[2] if len(row) > 2 else np.nan  # 有些 xye 第三列是 error；你可按需改成忽略
        cleaned.append([tth, i1, i2])

    df = pd.DataFrame(cleaned, columns=["2THETA", "Intensity1", "Intensity2"])
    return recorded_time, wave_value, df


# 2) 处理目录下所有文件，生成长表
def process_files(directory: Path):
    rows = []  # 长表
    for file_path in sorted(directory.glob("*.xye")):
        recorded_time, wave_value, df = extract_wave_and_data(file_path)

        # 从文件名末尾提取数字（更健壮，如 ..._f00024 / ..._123 / ..._idx-007）
        m = re.search(r"(\d+)$", file_path.stem)
        file_id = float(m.group(1)) if m else np.nan

        # 清洗 & 叠加到长表
        df = df.dropna(subset=["2THETA"]).copy()
        for _, r in df.iterrows():
            rows.append([
                file_id,
                recorded_time,
                wave_value,
                float(r["2THETA"]),
                float(r["Intensity1"]) if pd.notna(r["Intensity1"]) else np.nan,
                float(r["Intensity2"]) if pd.notna(r["Intensity2"]) else np.nan,
            ])

    spectrum_all = pd.DataFrame(
        rows,
        columns=["Index", "recorded_time", "wave_value", "2THETA", "Intensity1", "Intensity2"],
    )
    # 数据类型转换与清洗
    spectrum_all["Index"] = pd.to_numeric(spectrum_all["Index"], errors="coerce")
    spectrum_all["2THETA"] = pd.to_numeric(spectrum_all["2THETA"], errors="coerce")
    spectrum_all["Cnt2_D1"] = spectrum_all[["Intensity1", "Intensity2"]].sum(axis=1, min_count=1)

    spectrum_all = spectrum_all.dropna(subset=["Index", "2THETA", "Cnt2_D1"])
    spectrum_all = spectrum_all.sort_values(["Index", "2THETA"]).reset_index(drop=True)
    return spectrum_all


In [ ]:
# 读取电化学数据
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\EMD\2025-MSPD\Results\IS17_D\Echem\B").glob(r"**\*.txt"))
echem_all = []
for path_file in path_filelist[0:1]:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break
    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    df["Voltage/V"] = df["Ewe/V"] - df["Ece/V"]
    echem_all.append(df)
# 合并所有电化学数据为一个二维表格
echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 谱线上的时间
# 读取文件中时间戳
filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\EMD\2025-MSPD\Results\IS17_D\Data\IS17_D").glob(r"*.xye"))
range_index, wave_length, time_processed = [], [], []
for path_file in filelist:
    with open(path_file, "r") as file:
        lines = file.readlines()
        file_id = path_file.stem.split("_")[-1].split(".")[0][-5:]
        range_index.append(file_id)
        for line in lines:
            if line.startswith("# Wave"):
                wave_value = float(line.split()[3])
                wave_length.append(wave_value)
            elif line.startswith("# Date"):
                recored_time = str(line.split()[3])
                time_processed.append(recored_time)
spectrum_time_all = pd.DataFrame({
    "Range_Index": range_index,
    "time/s": time_processed,
    "Wave_Length": wave_length,
})
spectrum_time_all["time/s"] = pd.to_datetime(spectrum_time_all["time/s"], format=r"%Y-%m-%d_%H:%M:%S")
spectrum_time_all["Range_Index"] = pd.to_numeric(spectrum_time_all["Range_Index"])
spectrum_time_all["Wave_Length"] = pd.to_numeric(spectrum_time_all["Wave_Length"])

# 读取 XRD 的数据
file_list = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\EMD\2025-MSPD\Results\IS17_D\Data\IS17_D")
spectrum_all = process_files(file_list)
spectrum_all = spectrum_all.groupby(["Index", "2THETA"], as_index=False)["Cnt2_D1"].mean().pivot_table(index="2THETA", columns="Index", values="Cnt2_D1").sort_index()

# # 可选：保存
# spectrum_all.to_csv(file_list.joinpath("spectrum_all.csv"), index=True)

In [ ]:
# 只保留第一圈的充放电与第二圈的充电数据
selected_echem = echem_all[echem_all["cycle number"].isin([1, 2])]
selected_echem = selected_echem[selected_echem["Voltage/V"] >= 0.8]

# selected_echem = selected_echem.iloc[:-5, :].copy()
selected_echem = selected_echem.copy()
selected_echem["charge_time"] = (selected_echem["time/s"] - selected_echem["time/s"].iloc[0]).dt.total_seconds() / 3600

# 匹配谱线和电化学上的时间
selected_spectrum_time = (
    pd.merge_asof(
        selected_echem.sort_values(by="time/s"),
        spectrum_time_all.sort_values(by="time/s"),
        on="time/s",
        direction="nearest",  # 找最近的时间点
        tolerance=pd.Timedelta("5s"),  # 可设定允许的最大偏差
    )
    .dropna(subset=["Range_Index"], inplace=False)
    .drop_duplicates(subset=["Range_Index"], keep="first", inplace=False)
    .reset_index(drop=False, inplace=False)
)

# 选择 谱线的区间
spectrum_all = spectrum_all[(spectrum_all.index >= 2.0) & (spectrum_all.index <= 27.0)]
selected_spectrum = spectrum_all.loc[:, spectrum_all.columns.isin(selected_spectrum_time["Range_Index"].to_list())]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 4, 4], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Voltage/V"], selected_echem["charge_time"], ls="-", lw=1.0, c=colors[0], label=r"Voltage")

# 添加索引文本标注
index_labels = [15, 27, 37, 46, 61, 75, 89, 99, 109, 126, 137]

for i, idx in enumerate(selected_spectrum_time["Range_Index"].tolist()):
    if idx in index_labels:
        value = selected_spectrum_time.loc[selected_spectrum_time["Range_Index"] == idx, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c=colors[0],
            edgecolors="face",
            s=50,
            zorder=5,
        )
    else:
        value = selected_spectrum_time.loc[selected_spectrum_time.index == i, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c="lightgrey",
            edgecolors="face",
            s=30,
            zorder=1,
        )
ax.set_xlabel(
    r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$",
    fontsize=11,
)
ax.set_xlim(0.8, 2.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=0))

ax.set_ylabel(r"Duration Time (hour)", fontsize=11)
ax.set_ylim(-0.5, 36)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))


ax2 = ax.twiny()
ax2.set_position((0, 0, 1, 1))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"] * 1000, selected_echem["charge_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(
    r"$\mathrm{Current  \ (\mu A)}$",
    fontsize=11,
)
ax2.set_xlim(-60, 60)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=60, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=30, offset=0))
ax2.tick_params(axis="both", which="both", labelsize=9, right=True, labelright=True)
ax.text(-0.6, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.05, 0.035, 0.94, 0.94))
ax.set_box_aspect(1.0)

ax.imshow(
    selected_spectrum.T,
    cmap="coolwarm",
    aspect="auto",
    vmin=10000,
    vmax=40000,
    origin="lower",
    extent=[selected_spectrum.index.min(), selected_spectrum.index.max(), selected_spectrum.columns.min(), selected_spectrum.columns.max()],
)

ax.set_yticks([])
ax.set_xlim(2, 27)
ax.set_xlabel(
    r"2-Theta (degree)",
    fontsize=11,
)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=-3))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.5, offset=-3))
ax.tick_params(axis="x", which="both", direction="out", labelsize=9, top=False)

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.03, 0.1, 0.08, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.text(
    1.08,
    0.97,
    "High",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)
ax.text(
    1.08,
    0.08,
    "Low",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)

ax.text(-0.05, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C

subfig = fig.add_subfigure(gs[0, 2], zorder=0)

opxrd_ref_peak = [[3.55], [4.75], [6.15], [11.5], [20.0], [23.35]]
opxrd_colors = ["blue", "blue", "blue", "blue", "blue", "blue"]
vmin_vmax = [(None, None), (None, None), (None, None), (23500, 29000), (None, None), (None, None)]
for i in range(len(opxrd_ref_peak)):
    ax = subfig.add_axes((0.1 + i * 0.14, 0.035, 0.12, 0.94), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.4) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.4)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor=opxrd_colors[i],
        bottom=True,
        left=True,
        labelbottom=True,
        labelleft=True,
    )
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

ax.text(-6.0, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_05_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_05_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_05_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_05_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

In [ ]:
%matplotlib inline
plt.close("all")
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 0.3, 0.3))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Voltage/V"], selected_echem["charge_time"], ls="-", lw=1.0, c=colors[0], label=r"Voltage")

# 添加索引文本标注
index_labels = [15, 27, 37, 46, 61, 75, 89, 99, 109, 126, 137]

for i, idx in enumerate(selected_spectrum_time["Range_Index"].tolist()):
    if idx in index_labels:
        value = selected_spectrum_time.loc[selected_spectrum_time["Range_Index"] == idx, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c=colors[0],
            edgecolors="face",
            s=50,
            zorder=5,
        )
    else:
        value = selected_spectrum_time.loc[selected_spectrum_time.index == i, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c="lightgrey",
            edgecolors="face",
            s=30,
            zorder=1,
        )
ax.set_xlabel(
    r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$",
    fontsize=11,
)
ax.set_xlim(0.8, 2.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.2, offset=-0.4))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.6, offset=-0.4))

ax.set_ylabel(r"Duration Time (hour)", fontsize=11)
ax.set_ylim(-1, 36)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))


ax2 = ax.twiny()
ax2.set_position((0, 0, 0.3, 0.3))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"] * 1000, selected_echem["charge_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(
    r"$\mathrm{Current  \ (\mu A)}$",
    fontsize=11,
)
ax2.set_xlim(-60, 60)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=60, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=30, offset=0))
ax2.tick_params(axis="both", which="both", labelsize=9, right=True, labelright=True)

ax.text(-1.0, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 B
subfig = fig.add_subfigure(gs[0, 0], zorder=0)

opxrd_ref_peak = [
    [2.6],
    [5.20],
    [6.82],
    [8.10],
    [8.90],
    [10.0],
    [10.4],
    [10.53],
    [11.00],
    [11.52],
    [12.25],
    [12.86],
    [13.81],
    [14.57],
    [15.31],
    [15.72],
    [16.00],
    [16.80],
    [17.24],
    [17.82],
    [18.33],
    [19.384],
    [21.96],
    [22.13],
    [24.8],
]
vmin_vmax = [
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
]
for i in range(0, len(opxrd_ref_peak) // 3 - 1):
    # print(i)
    ax = subfig.add_axes((0.23 + i * 0.09, 0.0, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.4) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.4)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, labelbottom=True)
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(len(opxrd_ref_peak) // 3 - 1, 2 * len(opxrd_ref_peak) // 3):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - len(opxrd_ref_peak) // 3 + 1) * 0.09, -0.43, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.1) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.1)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, labelbottom=True)
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(2 * len(opxrd_ref_peak) // 3, len(opxrd_ref_peak)):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - 2 * len(opxrd_ref_peak) // 3) * 0.09, -0.8, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.1) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.1)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, labelbottom=True)
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=1.0)

ax.text(-6.8, 3.65, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_06_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_06_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_06_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_06_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Operando XRD, EMD, being from pH Buffer, 1 M + 0.2 M, 30uA - V1

In [ ]:
# 处理 XRD 的 xye 文件
# 1) 读取单个文件，提取波长与数据
def extract_wave_and_data(file_path: Path):
    wave_value = np.nan
    recorded_time = None  # 用 str 保存也可以，下面再转 datetime
    data_rows = []

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue

            # 头部信息
            if line.startswith("#"):
                # Wave: 允许 "1.5406", "1.5406A", "1.5406 Å" 等
                if "wave" in line.lower():
                    m = re.search(r"([-+]?\d*\.?\d+)", line)
                    if m:
                        wave_value = float(m.group(1))

                # Date: 抓取整行，后面统一解析
                elif "date" in line.lower():
                    # 直接保留原字符串；也可以只取某个片段
                    recorded_time = line.replace("#", "").strip()
                continue

            # 数据行：空白分隔，容错不同列数
            parts = re.split(r"\s+", line)
            nums = []
            for p in parts:
                try:
                    nums.append(float(p))
                except ValueError:
                    nums.append(np.nan)

            # 只要首列 2THETA 能转成数就收
            if len(nums) >= 1 and np.isfinite(nums[0]):
                data_rows.append(nums)

    # 统一成至少三列 [2THETA, Int1, Int2]，不足的补 NaN
    cleaned = []
    for row in data_rows:
        # 至少有 2THETA 与一个强度
        tth = row[0]
        i1 = row[1] if len(row) > 1 else np.nan
        i2 = row[2] if len(row) > 2 else np.nan  # 有些 xye 第三列是 error；你可按需改成忽略
        cleaned.append([tth, i1, i2])

    df = pd.DataFrame(cleaned, columns=["2THETA", "Intensity1", "Intensity2"])
    return recorded_time, wave_value, df


# 2) 处理目录下所有文件，生成长表
def process_files(directory: Path):
    rows = []  # 长表
    for file_path in sorted(directory.glob("*.xye")):
        recorded_time, wave_value, df = extract_wave_and_data(file_path)

        # 从文件名末尾提取数字（更健壮，如 ..._f00024 / ..._123 / ..._idx-007）
        m = re.search(r"(\d+)$", file_path.stem)
        file_id = float(m.group(1)) if m else np.nan

        # 清洗 & 叠加到长表
        df = df.dropna(subset=["2THETA"]).copy()
        for _, r in df.iterrows():
            rows.append([
                file_id,
                recorded_time,
                wave_value,
                float(r["2THETA"]),
                float(r["Intensity1"]) if pd.notna(r["Intensity1"]) else np.nan,
                float(r["Intensity2"]) if pd.notna(r["Intensity2"]) else np.nan,
            ])

    spectrum_all = pd.DataFrame(
        rows,
        columns=["Index", "recorded_time", "wave_value", "2THETA", "Intensity1", "Intensity2"],
    )
    # 数据类型转换与清洗
    spectrum_all["Index"] = pd.to_numeric(spectrum_all["Index"], errors="coerce")
    spectrum_all["2THETA"] = pd.to_numeric(spectrum_all["2THETA"], errors="coerce")
    spectrum_all["Cnt2_D1"] = spectrum_all[["Intensity1", "Intensity2"]].sum(axis=1, min_count=1)

    spectrum_all = spectrum_all.dropna(subset=["Index", "2THETA", "Cnt2_D1"])
    spectrum_all = spectrum_all.sort_values(["Index", "2THETA"]).reset_index(drop=True)
    return spectrum_all


In [ ]:
# 读取电化学数据
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\EMD\2025-MSPD\Results\IS16_C\Echem\A").glob(r"**\*.txt"))
echem_all = []
for path_file in path_filelist[1:]:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break
    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    df["Voltage/V"] = df["Ewe/V"] - df["Ece/V"]
    echem_all.append(df)
# 合并所有电化学数据为一个二维表格
echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 谱线上的时间
# 读取文件中时间戳
filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\EMD\2025-MSPD\Results\IS16_C\Data\IS16_C").glob(r"*.xye"))
range_index, wave_length, time_processed = [], [], []
for path_file in filelist:
    with open(path_file, "r") as file:
        lines = file.readlines()
        file_id = path_file.stem.split("_")[-1].split(".")[0][-5:]
        range_index.append(file_id)
        for line in lines:
            if line.startswith("# Wave"):
                wave_value = float(line.split()[3])
                wave_length.append(wave_value)
            elif line.startswith("# Date"):
                recored_time = str(line.split()[3])
                time_processed.append(recored_time)
spectrum_time_all = pd.DataFrame({
    "Range_Index": range_index,
    "time/s": time_processed,
    "Wave_Length": wave_length,
})
spectrum_time_all["time/s"] = pd.to_datetime(spectrum_time_all["time/s"], format=r"%Y-%m-%d_%H:%M:%S")
spectrum_time_all["Range_Index"] = pd.to_numeric(spectrum_time_all["Range_Index"])
spectrum_time_all["Wave_Length"] = pd.to_numeric(spectrum_time_all["Wave_Length"])

# 读取 XRD 的数据
file_list = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\EMD\2025-MSPD\Results\IS16_C\Data\IS16_C")
spectrum_all = process_files(file_list)
spectrum_all = spectrum_all.groupby(["Index", "2THETA"], as_index=False)["Cnt2_D1"].mean().pivot_table(index="2THETA", columns="Index", values="Cnt2_D1").sort_index()

# # 可选：保存
# spectrum_all.to_csv(file_list.joinpath("spectrum_all.csv"), index=True)

In [ ]:
# 只保留第一圈的充放电与第二圈的充电数据
selected_echem = echem_all[echem_all["cycle number"].isin([1, 2])]
selected_echem = selected_echem[selected_echem["Voltage/V"] >= 0.8]

selected_echem = selected_echem.iloc[:-5, :].copy()
selected_echem["charge_time"] = (selected_echem["time/s"] - selected_echem["time/s"].iloc[0]).dt.total_seconds() / 3600

# 匹配谱线和电化学上的时间
selected_spectrum_time = (
    pd.merge_asof(
        selected_echem.sort_values(by="time/s"),
        spectrum_time_all.sort_values(by="time/s"),
        on="time/s",
        direction="nearest",  # 找最近的时间点
        tolerance=pd.Timedelta("5s"),  # 可设定允许的最大偏差
    )
    .dropna(subset=["Range_Index"], inplace=False)
    .drop_duplicates(subset=["Range_Index"], keep="first", inplace=False)
    .reset_index(drop=False, inplace=False)
)

# 选择 谱线的区间
spectrum_all = spectrum_all[(spectrum_all.index >= 2.0) & (spectrum_all.index <= 27.0)]
selected_spectrum = spectrum_all.loc[:, spectrum_all.columns.isin(selected_spectrum_time["Range_Index"].to_list())]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 4, 4], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Voltage/V"], selected_echem["charge_time"], ls="-", lw=1.0, c=colors[0], label=r"Voltage")

# 添加索引文本标注
index_labels = [19, 31, 40, 54, 60, 72, 82, 89, 106, 115, 126, 130, 142, 153]

for i, idx in enumerate(selected_spectrum_time["Range_Index"].tolist()):
    if idx in index_labels:
        value = selected_spectrum_time.loc[selected_spectrum_time["Range_Index"] == idx, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c=colors[0],
            edgecolors="face",
            s=50,
            zorder=5,
        )
    else:
        value = selected_spectrum_time.loc[selected_spectrum_time.index == i, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c="lightgrey",
            edgecolors="face",
            s=30,
            zorder=1,
        )
ax.set_xlabel(
    r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$",
    fontsize=11,
)
ax.set_xlim(0.95, 1.65)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.35, offset=-0.1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.175, offset=-0.1))

ax.set_ylabel(r"Duration Time (hour)", fontsize=11)
ax.set_ylim(-0.5, 38.5)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))


ax2 = ax.twiny()
ax2.set_position((0, 0, 1, 1))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"] * 1000, selected_echem["charge_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(
    r"$\mathrm{Current  \ (\mu A)}$",
    fontsize=11,
)
ax2.set_xlim(-60, 60)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=30, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=15, offset=0))
ax2.tick_params(axis="both", which="both", labelsize=9, right=True, labelright=True)
ax.text(-0.6, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.05, 0.035, 0.94, 0.94))
ax.set_box_aspect(1.0)

ax.imshow(
    selected_spectrum.T,
    cmap="coolwarm",
    aspect="auto",
    vmin=5000,
    vmax=40000,
    origin="lower",
    extent=[selected_spectrum.index.min(), selected_spectrum.index.max(), selected_spectrum.columns.min(), selected_spectrum.columns.max()],
)

ax.set_yticks([])
ax.set_xlim(2, 27)
ax.set_xlabel(
    r"2-Theta (degree)",
    fontsize=11,
)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=-3))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.5, offset=-3))
ax.tick_params(axis="x", which="both", direction="out", labelsize=9, top=False)

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.03, 0.1, 0.08, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.text(
    1.08,
    0.97,
    "High",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)
ax.text(
    1.08,
    0.08,
    "Low",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)

ax.text(-0.05, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C

subfig = fig.add_subfigure(gs[0, 2], zorder=0)

opxrd_ref_peak = [[3.55], [4.75], [6.15], [11.5], [20.0], [23.35]]
opxrd_ref_d_spacing = [[8.01], [5.98], [4.62], [2.48], [1.43], [1.22]]
opxrd_colors = ["blue", "blue", "blue", "blue", "blue", "blue"]
vmin_vmax = [(None, None), (None, None), (None, None), (21000, 26000), (12000, 14000), (None, None)]
for i in range(len(opxrd_ref_peak)):
    ax = subfig.add_axes((0.1 + i * 0.14, 0.035, 0.12, 0.94), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.4) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.4)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.text(0.9 + 0.01 * i, -0.15, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor="k",
        bottom=True,
        left=True,
        labelbottom=True,
        labelleft=True,
    )
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)
ax.text(-6.0, -0.14, r"$\mathrm{d \ Spacing\ (\AA)}$", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold", c="blue")

ax.text(-6.0, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_03_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_03_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_03_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_03_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

In [ ]:
%matplotlib inline
plt.close("all")
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 0.3, 0.3))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Voltage/V"], selected_echem["charge_time"], ls="-", lw=1.0, c=colors[0], label=r"Voltage")

# 添加索引文本标注
index_labels = [19, 31, 40, 54, 60, 72, 82, 89, 106, 115, 126, 130, 142, 153]

for i, idx in enumerate(selected_spectrum_time["Range_Index"].tolist()):
    if idx in index_labels:
        value = selected_spectrum_time.loc[selected_spectrum_time["Range_Index"] == idx, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c=colors[0],
            edgecolors="face",
            s=50,
            zorder=5,
        )
    else:
        value = selected_spectrum_time.loc[selected_spectrum_time.index == i, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c="lightgrey",
            edgecolors="face",
            s=30,
            zorder=1,
        )
ax.set_xlabel(
    r"Voltage (V)",
    fontsize=9,
)
ax.set_xlim(0.95, 1.65)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.7, offset=0.25))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.35, offset=0.25))

ax.set_ylabel(r"Duration Time (hour)", fontsize=9)
ax.set_ylim(-1, 39)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))


ax2 = ax.twiny()
ax2.set_position((0, 0, 0.3, 0.3))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"] * 1000, selected_echem["charge_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(
    r"$\mathrm{Current  \ (\mu A)}$",
    fontsize=9,
)
ax2.set_xlim(-60, 60)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=60, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=30, offset=0))
ax2.tick_params(axis="both", which="both", labelsize=9, right=True, labelright=True)
ax.text(-1.0, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 B
subfig = fig.add_subfigure(gs[0, 0], zorder=0)

opxrd_ref_peak = [
    [2.6],
    [5.20],
    [6.82],
    [8.10],
    [8.90],
    [10.0],
    [10.4],
    [10.53],
    [11.00],
    [11.52],
    [12.25],
    [12.86],
    [13.81],
    [14.57],
    [15.31],
    [15.72],
    [16.00],
    [16.80],
    [17.24],
    [17.82],
    [18.33],
    [19.384],
    [21.96],
    [22.13],
    [24.8],
]
opxrd_ref_d_spacing = [
    [10.93],
    [5.47],
    [4.17],
    [3.51],
    [3.20],
    [2.85],
    [2.74],
    [2.70],
    [2.59],
    [2.47],
    [2.33],
    [2.21],
    [2.06],
    [1.96],
    [1.86],
    [1.81],
    [1.78],
    [1.70],
    [1.65],
    [1.60],
    [1.56],
    [1.47],
    [1.30],
    [1.29],
    [1.15],
]
vmin_vmax = [
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
]
for i in range(0, len(opxrd_ref_peak) // 3 - 1):
    # print(i)
    ax = subfig.add_axes((0.23 + i * 0.09, 0.0, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.4) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.4)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.text(0.75 + 0.001 * i, -0.32, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, labelbottom=True)
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(len(opxrd_ref_peak) // 3 - 1, 2 * len(opxrd_ref_peak) // 3):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - len(opxrd_ref_peak) // 3 + 1) * 0.09, -0.43, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.4) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.4)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.text(0.75 + 0.001 * i, -0.23, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, labelbottom=True)
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(2 * len(opxrd_ref_peak) // 3, len(opxrd_ref_peak)):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - 2 * len(opxrd_ref_peak) // 3) * 0.09, -0.82, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.4) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.4)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.text(0.75 + 0.001 * i, -0.23, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, labelbottom=True)
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=1.0)
ax.text(-6.7, 2.42, r"$\mathrm{d \ Spacing\ (\AA)}$", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold", c="blue")
ax.text(-6.8, 3.65, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_04_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_04_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_04_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_04_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Operando XRD, EMD, being from pH Buffer, 1 M + 0.2 M, 30uA

In [ ]:
# 处理 XRD 的 xye 文件
# 1) 读取单个文件，提取波长与数据
def extract_wave_and_data(file_path: Path):
    wave_value = np.nan
    recorded_time = None  # 用 str 保存也可以，下面再转 datetime
    data_rows = []

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue

            # 头部信息
            if line.startswith("#"):
                # Wave: 允许 "1.5406", "1.5406A", "1.5406 Å" 等
                if "wave" in line.lower():
                    m = re.search(r"([-+]?\d*\.?\d+)", line)
                    if m:
                        wave_value = float(m.group(1))

                # Date: 抓取整行，后面统一解析
                elif "date" in line.lower():
                    # 直接保留原字符串；也可以只取某个片段
                    recorded_time = line.replace("#", "").strip()
                continue

            # 数据行：空白分隔，容错不同列数
            parts = re.split(r"\s+", line)
            nums = []
            for p in parts:
                try:
                    nums.append(float(p))
                except ValueError:
                    nums.append(np.nan)

            # 只要首列 2THETA 能转成数就收
            if len(nums) >= 1 and np.isfinite(nums[0]):
                data_rows.append(nums)

    # 统一成至少三列 [2THETA, Int1, Int2]，不足的补 NaN
    cleaned = []
    for row in data_rows:
        # 至少有 2THETA 与一个强度
        tth = row[0]
        i1 = row[1] if len(row) > 1 else np.nan
        i2 = row[2] if len(row) > 2 else np.nan  # 有些 xye 第三列是 error；你可按需改成忽略
        cleaned.append([tth, i1, i2])

    df = pd.DataFrame(cleaned, columns=["2THETA", "Intensity1", "Intensity2"])
    return recorded_time, wave_value, df


# 2) 处理目录下所有文件，生成长表
def process_files(directory: Path):
    rows = []  # 长表
    for file_path in sorted(directory.glob("*.xye")):
        recorded_time, wave_value, df = extract_wave_and_data(file_path)

        # 从文件名末尾提取数字（更健壮，如 ..._f00024 / ..._123 / ..._idx-007）
        m = re.search(r"(\d+)$", file_path.stem)
        file_id = float(m.group(1)) if m else np.nan

        # 清洗 & 叠加到长表
        df = df.dropna(subset=["2THETA"]).copy()
        for _, r in df.iterrows():
            rows.append([
                file_id,
                recorded_time,
                wave_value,
                float(r["2THETA"]),
                float(r["Intensity1"]) if pd.notna(r["Intensity1"]) else np.nan,
                float(r["Intensity2"]) if pd.notna(r["Intensity2"]) else np.nan,
            ])

    spectrum_all = pd.DataFrame(
        rows,
        columns=["Index", "recorded_time", "wave_value", "2THETA", "Intensity1", "Intensity2"],
    )
    # 数据类型转换与清洗
    spectrum_all["Index"] = pd.to_numeric(spectrum_all["Index"], errors="coerce")
    spectrum_all["2THETA"] = pd.to_numeric(spectrum_all["2THETA"], errors="coerce")
    spectrum_all["Cnt2_D1"] = spectrum_all[["Intensity1", "Intensity2"]].sum(axis=1, min_count=1)

    spectrum_all = spectrum_all.dropna(subset=["Index", "2THETA", "Cnt2_D1"])
    spectrum_all = spectrum_all.sort_values(["Index", "2THETA"]).reset_index(drop=True)
    return spectrum_all


In [ ]:
# 读取电化学数据
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\EMD\2025-MSPD\Results\IS16_C\Echem\A").glob(r"**\*.txt"))
echem_all = []
for path_file in path_filelist[1:]:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break
    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    df["Voltage/V"] = df["Ewe/V"] - df["Ece/V"]
    echem_all.append(df)
# 合并所有电化学数据为一个二维表格
echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 谱线上的时间
# 读取文件中时间戳
filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\EMD\2025-MSPD\Results\IS16_C\Data\IS16_C").glob(r"*.xye"))
range_index, wave_length, time_processed = [], [], []
for path_file in filelist:
    with open(path_file, "r") as file:
        lines = file.readlines()
        file_id = path_file.stem.split("_")[-1].split(".")[0][-5:]
        range_index.append(file_id)
        for line in lines:
            if line.startswith("# Wave"):
                wave_value = float(line.split()[3])
                wave_length.append(wave_value)
            elif line.startswith("# Date"):
                recored_time = str(line.split()[3])
                time_processed.append(recored_time)
spectrum_time_all = pd.DataFrame({
    "Range_Index": range_index,
    "time/s": time_processed,
    "Wave_Length": wave_length,
})
spectrum_time_all["time/s"] = pd.to_datetime(spectrum_time_all["time/s"], format=r"%Y-%m-%d_%H:%M:%S")
spectrum_time_all["Range_Index"] = pd.to_numeric(spectrum_time_all["Range_Index"])
spectrum_time_all["Wave_Length"] = pd.to_numeric(spectrum_time_all["Wave_Length"])

# 读取 XRD 的数据
file_list = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\EMD\2025-MSPD\Results\IS16_C\Data\IS16_C")
spectrum_all = process_files(file_list)
spectrum_all = spectrum_all.groupby(["Index", "2THETA"], as_index=False)["Cnt2_D1"].mean().pivot_table(index="2THETA", columns="Index", values="Cnt2_D1").sort_index()

# # 可选：保存
# spectrum_all.to_csv(file_list.joinpath("spectrum_all.csv"), index=True)

In [ ]:
# 只保留第一圈的充放电与第二圈的充电数据
selected_echem = echem_all[echem_all["cycle number"].isin([1, 2])]
selected_echem = selected_echem[selected_echem["Voltage/V"] >= 0.8]

selected_echem = selected_echem.iloc[:-5, :].copy()
selected_echem["charge_time"] = (selected_echem["time/s"] - selected_echem["time/s"].iloc[0]).dt.total_seconds() / 3600

# 匹配谱线和电化学上的时间
selected_spectrum_time = (
    pd.merge_asof(
        selected_echem.sort_values(by="time/s"),
        spectrum_time_all.sort_values(by="time/s"),
        on="time/s",
        direction="nearest",  # 找最近的时间点
        tolerance=pd.Timedelta("5s"),  # 可设定允许的最大偏差
    )
    .dropna(subset=["Range_Index"], inplace=False)
    .drop_duplicates(subset=["Range_Index"], keep="first", inplace=False)
    .reset_index(drop=False, inplace=False)
)

# 选择 谱线的区间
spectrum_all = spectrum_all[(spectrum_all.index >= 2.0) & (spectrum_all.index <= 27.0)]
selected_spectrum = spectrum_all.loc[:, spectrum_all.columns.isin(selected_spectrum_time["Range_Index"].to_list())]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 4, 4], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Voltage/V"], selected_echem["charge_time"], ls="-", lw=1.0, c=colors[0], label=r"Voltage")

# 添加索引文本标注
index_labels = [19, 34, 54, 72, 89, 109, 126, 142, 152]

for i, idx in enumerate(selected_spectrum_time["Range_Index"].tolist()):
    if idx in index_labels:
        value = selected_spectrum_time.loc[selected_spectrum_time["Range_Index"] == idx, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c=colors[0],
            edgecolors="face",
            s=50,
            zorder=5,
        )
    else:
        value = selected_spectrum_time.loc[selected_spectrum_time.index == i, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c="lightgrey",
            edgecolors="face",
            s=30,
            zorder=1,
        )
ax.set_xlabel(
    r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$",
    fontsize=11,
)
ax.set_xlim(0.95, 1.65)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.35, offset=-0.1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.175, offset=-0.1))

ax.set_ylabel(r"Duration Time (hour)", fontsize=11)
ax.set_ylim(-0.5, 38.5)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))


ax2 = ax.twiny()
ax2.set_position((0, 0, 1, 1))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"] * 1000, selected_echem["charge_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(
    r"$\mathrm{Current  \ (\mu A)}$",
    fontsize=11,
)
ax2.set_xlim(-60, 60)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=30, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=15, offset=0))
ax2.tick_params(axis="both", which="both", labelsize=9, right=True, labelright=True)
ax.text(-0.6, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.05, 0.035, 0.94, 0.94))
ax.set_box_aspect(1.0)

ax.imshow(
    selected_spectrum.T,
    cmap="coolwarm",
    aspect="auto",
    vmin=5000,
    vmax=40000,
    origin="lower",
    extent=[selected_spectrum.index.min(), selected_spectrum.index.max(), selected_spectrum.columns.min(), selected_spectrum.columns.max()],
)

ax.set_yticks([])
ax.set_xlim(2, 27)
ax.set_xlabel(
    r"2-Theta (degree)",
    fontsize=11,
)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=-3))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.5, offset=-3))
ax.tick_params(axis="x", which="both", direction="out", labelsize=9, top=False)

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.03, 0.1, 0.08, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.text(
    1.08,
    0.97,
    "High",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)
ax.text(
    1.08,
    0.08,
    "Low",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)

ax.text(-0.05, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C

subfig = fig.add_subfigure(gs[0, 2], zorder=0)

opxrd_ref_peak = [[3.55], [4.75], [6.15], [11.5], [20.0], [23.35]]
opxrd_colors = ["blue", "blue", "blue", "blue", "blue", "blue"]
vmin_vmax = [(None, None), (None, None), (None, None), (21000, 26000), (12000, 14000), (None, None)]
for i in range(len(opxrd_ref_peak)):
    ax = subfig.add_axes((0.1 + i * 0.14, 0.035, 0.12, 0.94), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.4) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.4)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor=opxrd_colors[i],
        bottom=True,
        left=True,
        labelbottom=True,
        labelleft=True,
    )
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

ax.text(-6.0, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_03_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_03_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_03_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_03_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

In [ ]:
%matplotlib inline
plt.close("all")
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 0.3, 0.3))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Voltage/V"], selected_echem["charge_time"], ls="-", lw=1.0, c=colors[0], label=r"Voltage")

# 添加索引文本标注
index_labels = [19, 34, 54, 72, 89, 109, 126, 142, 152]

for i, idx in enumerate(selected_spectrum_time["Range_Index"].tolist()):
    if idx in index_labels:
        value = selected_spectrum_time.loc[selected_spectrum_time["Range_Index"] == idx, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c=colors[0],
            edgecolors="face",
            s=50,
            zorder=5,
        )
    else:
        value = selected_spectrum_time.loc[selected_spectrum_time.index == i, ["Voltage/V", "charge_time"]]
        ax.scatter(
            value["Voltage/V"],
            value["charge_time"],
            c="lightgrey",
            edgecolors="face",
            s=30,
            zorder=1,
        )
ax.set_xlabel(
    r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$",
    fontsize=11,
)
ax.set_xlim(0.95, 1.65)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.7, offset=0.25))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.35, offset=0.25))

ax.set_ylabel(r"Duration Time (hour)", fontsize=11)
ax.set_ylim(-1, 39)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))


ax2 = ax.twiny()
ax2.set_position((0, 0, 0.3, 0.3))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"] * 1000, selected_echem["charge_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(
    r"$\mathrm{Current  \ (\mu A)}$",
    fontsize=11,
)
ax2.set_xlim(-60, 60)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=60, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=30, offset=0))
ax2.tick_params(axis="both", which="both", labelsize=9, right=True, labelright=True)
ax.text(-1.0, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 B
subfig = fig.add_subfigure(gs[0, 0], zorder=0)

opxrd_ref_peak = [
    [2.6],
    [5.20],
    [6.82],
    [8.10],
    [8.90],
    [10.0],
    [10.4],
    [10.53],
    [11.00],
    [11.52],
    [12.25],
    [12.86],
    [13.81],
    [14.57],
    [15.31],
    [15.72],
    [16.00],
    [16.80],
    [17.24],
    [17.82],
    [18.33],
    [19.384],
    [21.96],
    [22.13],
    [24.8],
]
vmin_vmax = [
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
]
for i in range(0, len(opxrd_ref_peak) // 3 - 1):
    # print(i)
    ax = subfig.add_axes((0.23 + i * 0.09, 0.0, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.4) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.4)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, labelbottom=True)
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(len(opxrd_ref_peak) // 3 - 1, 2 * len(opxrd_ref_peak) // 3):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - len(opxrd_ref_peak) // 3 + 1) * 0.09, -0.43, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.1) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.1)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, labelbottom=True)
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(2 * len(opxrd_ref_peak) // 3, len(opxrd_ref_peak)):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - 2 * len(opxrd_ref_peak) // 3) * 0.09, -0.8, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.1) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.1)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, labelbottom=True)
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=1.0)

ax.text(-6.8, 3.65, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_04_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_04_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_04_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_04_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Operando XRD, αMnO2 -V4

In [ ]:
path_xrd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\αMnO2\2022-ICMAB\Results")

# 读取 PDF card 的数据
pdfmo = pd.read_csv(
    Path.joinpath(path_xrd, r"PDFCards", "PDF_1stDischarge-d-spacing.csv"),
    sep=",",
    index_col=None,
    header=0,
    skiprows=1,
    comment="#",
)

# 读取 spectrum 的数据
spectrum_all = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", "Spectum_all_d spacing.csv"),
    sep=r",",
    index_col=None,
    header=0,
    comment="#",
)
spectrum_all.drop(columns=['2THETA'], inplace=True)
spectrum_all.set_index(keys=["d_spacing"], inplace=True)
spectrum_all.index = pd.to_numeric(spectrum_all.index, errors="coerce")
spectrum_all.columns = pd.to_numeric(spectrum_all.columns, errors="coerce")

# 电化学上的时间
with open(Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", r"EchemOperando1_c10_1544mg_C02.txt"), "r") as file:
    lines = file.readlines()
for line in lines:
    if line.startswith("Nb header lines"):
        line_skip = int(line.split(":")[1].strip())

echem_all = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", r"EchemOperando1_c10_1544mg_C02.txt"),
    sep="\t",
    index_col=None,
    header=0,
    comment="#",
    skiprows=line_skip - 1,
    encoding="latin_1",
    date_format="%m/%d/%y %H:%M:%S.%f",
    parse_dates=[1, 2],
).dropna(axis=1, how="all", inplace=False)
echem_all["time/s"] = pd.to_datetime(
    echem_all["time/s"],
)
echem_all["Ewe/V"] = pd.to_numeric(
    echem_all["Ewe/V"],
)
echem_all["<I>/mA"] = pd.to_numeric(
    echem_all["<I>/mA"],
)
# echem_all.info()

# 谱线上的时间
spectrum_time_all = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", r"Time_index_spectrum.csv"),
    sep=",",
    index_col=0,
    header=0,
    comment="#",
    date_format="%m/%d/%y %H:%M:%S.%f",
    parse_dates=[1],
)
spectrum_time_all["Time"] = pd.to_datetime(spectrum_time_all["Time"])
# spectrum_time_all.info()

# 匹配谱线和电化学上的时间
spectrum_time = [abs(echem_all["time/s"] - t).idxmin() for t in spectrum_time_all["Time"]]
spectrum_time = (echem_all.loc[spectrum_time, [r"Ewe/V", r"<I>/mA"]].reset_index(drop=False).sort_values(by="Ewe/V", ascending=True)).reset_index(drop=False, inplace=False)

In [ ]:
# 选择需要的电化学数据以及对应的谱线
index_labels = [17, 26, 31, 37, 40, 45, 50, 54]
selected_spectrum_time = (
    spectrum_time[(spectrum_time["level_0"] <= index_labels[-1] + 1) & (spectrum_time["level_0"] >= index_labels[0])].sort_values(by="level_0", ascending=True).reset_index(drop=True)
)

selected_echem = echem_all[(echem_all.index <= selected_spectrum_time["index"].iloc[-1]) & (echem_all.index >= selected_spectrum_time["index"].iloc[0])]
selected_echem = selected_echem.copy()
selected_echem["FD1st_time"] = (selected_echem["time/s"] - selected_echem["time/s"].iloc[0]).dt.total_seconds() / 3600

# 修复：使用正确的列选择方式
# spectrum_all 的列是数字1-88，对应不同的谱线编号
column_range_start = selected_spectrum_time["level_0"].iloc[0] + 1
column_range_end = selected_spectrum_time["level_0"].iloc[-1] + 1

# 选择对应范围的列
selected_columns = [col for col in spectrum_all.columns if col >= column_range_start and col <= column_range_end]
selected_spectrum = spectrum_all[selected_columns]

In [ ]:
%matplotlib inline

plt.close("all")

# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 4, 4], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Ewe/V"], selected_echem["FD1st_time"], ls="-", lw=1.0, c=colors[0], label=r"opCoin_1", zorder=5)

for i, idx in enumerate(selected_spectrum_time["level_0"]):
    if idx in index_labels:
        index_value = selected_spectrum_time.loc[selected_spectrum_time["level_0"] == idx, "index"].iloc[0]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c=colors[0],
            edgecolors="face",
            s=30,
        )
    else:
        index_value = selected_spectrum_time.loc[i, "index"]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c="lightgrey",
            edgecolors="face",
            s=15,
        )

ax.set_xlabel(r"Voltage (V)", fontsize=11)
ax.set_xlim(0.85, 1.85)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=-0.15))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=-0.15))

ax.set_ylabel(r"Duration Time (hour)", fontsize=11)
ax.set_ylim(-0.5, 30.5)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))

# ax.legend(loc='upper right', bbox_to_anchor=(0.9, 1.0), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
# ax.text(-0.1, -0.06, r'$\mathrm{3.288 mg\ cm^{-2}}$', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)
# ax.text(-0.1, 0, r'C/10, 1C=250 mA/g', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)

ax2 = ax.twiny()
ax2.set_position((0, 0, 1, 1))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"], selected_echem["FD1st_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(r"Current (mA)", fontsize=11)
ax2.set_xlim(-0.2, 0.2)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax2.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=False,
    top=True,
    labeltop=True,
    labelright=False,
)
# ax2.legend(loc='upper left', bbox_to_anchor=(0.15, 1.05), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
ax.text(-0.6, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.05, 0.035, 0.94, 0.94))
ax.set_box_aspect(1.0)

ax.imshow(
    selected_spectrum.T,
    cmap="coolwarm",
    aspect="auto",  # vmin=5000, vmax=40000,
    origin="lower",
    extent=[selected_spectrum.index.min(), selected_spectrum.index.max(), selected_spectrum.columns.min(), selected_spectrum.columns.max()],
)
ax.set_yticks([])
ax.set_xlim(2.0, 8.0)  # d_spacing范围通常在1.5-8Å之间
ax.set_xlabel(
    r"d Spacing (Å)",  # 修改X轴标签为d_spacing
    fontsize=11,
)
# # 修改刻度设置以适配d_spacing
# ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=0))
# ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.03, 0.1, 0.08, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.text(
    1.08,
    0.97,
    "High",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)
ax.text(
    1.08,
    0.08,
    "Low",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)

pdf_scale = [0.001, 0.006, 0.001, 0.03, 0.002, 0.001, 0.0001, 0.0005]
XRD_y = [0, 0.15, 0.15, 0.15, 0.15, 0.07, 0.0, 0.09]
XRDcolors = ["blue", "k", "k", "k", "k", "orange", "purple", "green"]

for i in range(pdfmo.shape[0]):
    temp = pdfmo.iloc[:, 2 * i : 2 * i + 2].dropna(axis="index", how="all")
    for j in range((temp.shape[0])):
        ax.axvline(x=temp.iloc[j, 0], ymin=XRD_y[i], ymax=(XRD_y[i] + temp.iloc[j, 1] * pdf_scale[i]), lw=1, c=XRDcolors[i])

ax.text(
    0,
    1.09,
    r"$\alpha$-MnO$\mathrm{_2}$ #00-044-0141",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="blue",
)
ax.text(
    0.55,
    1.09,
    r"Zn(HSO$_{4}$)$_{2}$ #00-052-0258",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="orange",
)
ax.text(0, 1.02, r"ZSH", horizontalalignment="left", verticalalignment="bottom", transform=ax.transAxes, fontsize=9, c="k")
ax.text(
    0.13,
    1.01,
    r"K$_{0.66}$Mn$_4$O$_{8}$ #04-023-7544",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="purple",
)
ax.text(
    0.78,
    1.02,
    r"Ti #00-044-1294",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="green",
)

ax.text(-0.05, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)

opxrd_ref_d_spacing = [2.150451, 2.390493, 3.099262, 4.895775, 6.93]
peak_widths = [(0.02, 0.02), (0.02, 0.02), (0.05, 0.05), (0.05, 0.05), (0.1, 0.1)]
vmin_vmax = [(None, None), (None, None), (None, None), (None, None), (None, None)]

for i in range(len(opxrd_ref_d_spacing)):
    ax = subfig.add_axes((0.1 + i * 0.14, 0.035, 0.12, 0.94), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_d_spacing[i] - peak_widths[i][0]) & (selected_spectrum.index <= opxrd_ref_d_spacing[i] + peak_widths[i][1])]
    vmin = vmin_vmax[i][0] if vmin_vmax[i][0] is not None else temp.min().min()
    vmax = vmin_vmax[i][1] if vmin_vmax[i][1] is not None else temp.max().max()
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin, vmax=vmax, extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_d_spacing[i], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_d_spacing[i]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor="k",
        bottom=True,
        left=True,
        labelbottom=True,
        labelleft=True,
    )
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[1], alpha=0.5)

ax.text(-4.9, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V4_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
# plt.savefig(
#     Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V4_600.tif"),
#     pad_inches=0.05,
#     bbox_inches="tight",
#     dpi=600,
#     transparent=False,
#     pil_kwargs={"compression": "tiff_lzw"},
# )
# plt.savefig(
#     Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V4_600.png"),
#     pad_inches=0.05,
#     bbox_inches="tight",
#     dpi=600,
#     transparent=False,
# )
# plt.savefig(
#     Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V4_900.svg"),
#     pad_inches=0.05,
#     bbox_inches="tight",
#     dpi=900,
#     transparent=False,
# )

plt.gcf().set_facecolor("white")
plt.show()

In [ ]:
%matplotlib inline

# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 0.3, 0.3))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Ewe/V"], selected_echem["FD1st_time"], ls="-", lw=1.0, c=colors[0], label=r"opCoin_1", zorder=5)


for i, idx in enumerate(selected_spectrum_time["level_0"]):
    if idx in index_labels:
        index_value = selected_spectrum_time.loc[selected_spectrum_time["level_0"] == idx, "index"].iloc[0]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c=colors[0],
            edgecolors="face",
            s=30,
            zorder=10,
        )
    else:
        index_value = selected_spectrum_time.loc[i, "index"]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c="lightgrey",
            edgecolors="face",
            s=5,
            zorder=1,
        )

ax.set_xlabel(r"Voltage (V)", fontsize=9)
ax.set_xlim(0.85, 1.85)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=-0.15))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=-0.15))

ax.set_ylabel(r"Duration Time (hour)", fontsize=9)
ax.set_ylim(-0.5, 30.5)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))

# ax.legend(loc='upper right', bbox_to_anchor=(0.9, 1.0), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
# ax.text(-0.1, -0.06, r'$\mathrm{3.288 mg\ cm^{-2}}$', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)
# ax.text(-0.1, 0, r'C/10, 1C=250 mA/g', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)

ax2 = ax.twiny()
ax2.set_position((0, 0, 0.3, 0.3))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"], selected_echem["FD1st_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(r"Current (mA)", fontsize=9)
ax2.set_xlim(-0.2, 0.2)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax2.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=False,
    top=True,
    labeltop=True,
    labelright=False,
)
# ax2.legend(loc='upper left', bbox_to_anchor=(0.15, 1.05), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
ax.text(-1.0, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 B
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.046, -0.56, 0.625, 0.08))

K1 = "#295488"
K2 = "k"
K3 = "purple"
K4 = "orange"

ax.text(
    0,
    0.55,
    r"$\mathrm{Zn_4SO_4(OH)_6 \cdot 5H_2O}$ #00-060-0655, #04-012-8189",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K1,
)
ax.text(
    0,
    0.02,
    r"$\mathrm{Zn_4SO_4(OH)_6 \cdot 3H_2O}$ #01-082-3605, #00-039-0689",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K2,
)
ax.text(
    0.66,
    0.55,
    r"$\mathrm{K_{0.66}Mn_4O_8}$ #04-023-7544",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K3,
)
ax.text(
    0.67,
    0.02,
    r"$\mathrm{Zn(HSO_4)_2}$ #00-052-0258",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K4,
)
ax.spines[:].set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
ax.set_facecolor("gainsboro")

opxrd_ref_d_spacing = [2.212224, 2.282709, 2.31839, 2.46728, 2.530539, 2.558665, 2.663994, 2.705785, 2.73078, 3.190244, 3.235243, 3.40052, 3.510488, 3.634241, 4.162956, 5.454092]

# KMnO
opxrd_colors = [K1, K1, K2, K1, K1, K1, K1, K1, K3, K2, K4, K1, K2, K2, K1, K1, K3, K4, K1, K1, K4]
vmin_vmax = [
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
]
peak_widths = [(0.1, 0.1)] * len(opxrd_ref_d_spacing)
for i in range(0, len(opxrd_ref_d_spacing) // 2):
    # print(i)
    ax = subfig.add_axes((0.23 + i * 0.055, 0.0, 0.05, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_d_spacing[i] - peak_widths[i][0]) & (selected_spectrum.index <= opxrd_ref_d_spacing[i] + peak_widths[i][1])]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_d_spacing[i], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_d_spacing[i]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    # ax.tick_params(axis='both', which='both', labelcolor='k', bottom=True, labelbottom=True)
    ax.tick_params(axis="both", which="both", labelcolor=opxrd_colors[i], bottom=True, labelbottom=True)
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(len(opxrd_ref_d_spacing) // 2, len(opxrd_ref_d_spacing)):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - len(opxrd_ref_d_spacing) // 2) * 0.055, -0.42, 0.05, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_d_spacing[i] - peak_widths[i][0]) & (selected_spectrum.index <= opxrd_ref_d_spacing[i] + peak_widths[i][1])]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_d_spacing[i], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_d_spacing[i]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    # ax.tick_params(axis='both', which='both', labelcolor='k', bottom=True, labelbottom=True)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor=opxrd_colors[i],
        bottom=True,
        labelbottom=True,
    )
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

# ax.text(-4.6, 3.75, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V4_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
# plt.savefig(
#     Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V4_600.tif"),
#     pad_inches=0.05,
#     bbox_inches="tight",
#     dpi=600,
#     transparent=False,
#     pil_kwargs={"compression": "tiff_lzw"},
# )
# plt.savefig(
#     Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V4_600.png"),
#     pad_inches=0.05,
#     bbox_inches="tight",
#     dpi=600,
#     transparent=False,
# )
# plt.savefig(
#     Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V4_900.svg"),
#     pad_inches=0.05,
#     bbox_inches="tight",
#     dpi=900,
#     transparent=False,
# )

plt.gcf().set_facecolor("white")
plt.show()

### Operando XRD, αMnO2 -V3

In [ ]:
path_xrd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\αMnO2\2022-ICMAB\Results")
# 读取 alfaMnO2capillary 数据
xrd = pd.read_csv(Path.joinpath(path_xrd, r"Ex situ", r"alfaMnO2capillary.uxd"), sep=r"\s+", index_col=None, header=0, comment="#")

# 读取 PDF card 的数据
pdf = pd.read_csv(Path.joinpath(path_xrd, r"PDFCards", "PDF-00-044-0141-Mo.csv"), sep=r",", index_col=None, header=0, comment="#")
pdfmo = pd.read_csv(
    Path.joinpath(path_xrd, r"PDFCards", "PDF_1stDischarge.csv"),
    sep=",",
    index_col=None,
    header=0,
    skiprows=1,
    comment="#",
)

# 读取 spectrum 的数据
spectrum_all = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", "Spectum_all.csv"),
    sep=r",",
    index_col=None,
    header=0,
    comment="#",
)

# 电化学上的时间
with open(Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", r"EchemOperando1_c10_1544mg_C02.txt"), "r") as file:
    lines = file.readlines()
for line in lines:
    if line.startswith("Nb header lines"):
        line_skip = int(line.split(":")[1].strip())

echem_all = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", r"EchemOperando1_c10_1544mg_C02.txt"),
    sep="\t",
    index_col=None,
    header=0,
    comment="#",
    skiprows=line_skip - 1,
    encoding="latin_1",
    date_format="%m/%d/%y %H:%M:%S.%f",
    parse_dates=[1, 2],
).dropna(axis=1, how="all", inplace=False)
echem_all["time/s"] = pd.to_datetime(
    echem_all["time/s"],
)
echem_all["Ewe/V"] = pd.to_numeric(
    echem_all["Ewe/V"],
)
echem_all["<I>/mA"] = pd.to_numeric(
    echem_all["<I>/mA"],
)
# echem_all.info()

# 谱线上的时间
spectrum_time_all = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", r"Time_index_spectrum.csv"),
    sep=",",
    index_col=0,
    header=0,
    comment="#",
    date_format="%m/%d/%y %H:%M:%S.%f",
    parse_dates=[1],
)
spectrum_time_all["Time"] = pd.to_datetime(spectrum_time_all["Time"])
# spectrum_time_all.info()

# 匹配谱线和电化学上的时间
spectrum_time = [abs(echem_all["time/s"] - t).idxmin() for t in spectrum_time_all["Time"]]
spectrum_time = (echem_all.loc[spectrum_time, [r"Ewe/V", r"<I>/mA"]].reset_index(drop=False).sort_values(by="Ewe/V", ascending=True)).reset_index(drop=False, inplace=False)

In [ ]:
# 选择需要的电化学数据以及对应的谱线
index_labels = [17, 26, 31, 37, 40, 45, 50, 54]
selected_spectrum_time = (
    spectrum_time[(spectrum_time["level_0"] <= index_labels[-1] + 1) & (spectrum_time["level_0"] >= index_labels[0])].sort_values(by="level_0", ascending=True).reset_index(drop=True)
)

selected_echem = echem_all[(echem_all.index <= selected_spectrum_time["index"].iloc[-1]) & (echem_all.index >= selected_spectrum_time["index"].iloc[0])]
selected_echem = selected_echem.copy()
selected_echem["FD1st_time"] = (selected_echem["time/s"] - selected_echem["time/s"].iloc[0]).dt.total_seconds() / 3600

selected_spectrum = spectrum_all[(spectrum_all["range"] <= selected_spectrum_time["level_0"].iloc[-1] + 1) & (spectrum_all["range"] >= selected_spectrum_time["level_0"].iloc[0] + 1)].pivot(
    index="2THETA", columns="range", values="Cnt2_D1"
)

In [ ]:
%matplotlib inline

plt.close("all")

# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 4, 4], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Ewe/V"], selected_echem["FD1st_time"], ls="-", lw=1.0, c=colors[0], label=r"opCoin_1", zorder=5)


for i, idx in enumerate(selected_spectrum_time["level_0"]):
    if idx in index_labels:
        index_value = selected_spectrum_time.loc[selected_spectrum_time["level_0"] == idx, "index"].iloc[0]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c=colors[0],
            edgecolors="face",
            s=30,
        )
    else:
        index_value = selected_spectrum_time.loc[i, "index"]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c="lightgrey",
            edgecolors="face",
            s=15,
        )

ax.set_xlabel(r"Voltage (V)", fontsize=11)
ax.set_xlim(0.85, 1.85)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=-0.15))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=-0.15))

ax.set_ylabel(r"Duration Time (hour)", fontsize=11)
ax.set_ylim(-0.5, 30.5)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))

# ax.legend(loc='upper right', bbox_to_anchor=(0.9, 1.0), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
# ax.text(-0.1, -0.06, r'$\mathrm{3.288 mg\ cm^{-2}}$', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)
# ax.text(-0.1, 0, r'C/10, 1C=250 mA/g', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)

ax2 = ax.twiny()
ax2.set_position((0, 0, 1, 1))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"], selected_echem["FD1st_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(r"Current (mA)", fontsize=11)
ax2.set_xlim(-0.2, 0.2)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax2.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=False,
    top=True,
    labeltop=True,
    labelright=False,
)
# ax2.legend(loc='upper left', bbox_to_anchor=(0.15, 1.05), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
ax.text(-0.6, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.05, 0.035, 0.94, 0.94))
ax.set_box_aspect(1.0)

# ax.contourf(selected_spectrum.index, selected_spectrum.columns, selected_spectrum.T, cmap="coolwarm", levels=100)
ax.imshow(
    selected_spectrum.T,
    cmap="coolwarm",
    aspect="auto",  # vmin=5000, vmax=40000,
    origin="lower",
    extent=[selected_spectrum.index.min(), selected_spectrum.index.max(), selected_spectrum.columns.min(), selected_spectrum.columns.max()],
)
ax.set_yticks([])
ax.set_xlim(5, 20)
ax.set_xlabel(
    r"2-Theta$\mathrm{_{Mo}}$ (degree)",
    fontsize=11,
)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=-5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=-5))

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.03, 0.1, 0.08, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.text(
    1.08,
    0.97,
    "High",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)
ax.text(
    1.08,
    0.08,
    "Low",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)

pdf_scale = [0.001, 0.006, 0.001, 0.03, 0.002, 0.001, 0.0001, 0.0005]
XRD_y = [0, 0.15, 0.15, 0.15, 0.15, 0.07, 0.0, 0.09]
XRDcolors = ["blue", "k", "k", "k", "k", "orange", "purple", "green"]
for i in range(pdfmo.shape[0]):
    temp = pdfmo.iloc[:, 2 * i : 2 * i + 2].dropna(axis="index", how="all")
    for j in range((temp.shape[0])):
        ax.axvline(x=temp.iloc[j, 0], ymin=XRD_y[i], ymax=(XRD_y[i] + temp.iloc[j, 1] * pdf_scale[i]), lw=1, c=XRDcolors[i])

ax.text(
    0,
    1.09,
    r"$\alpha$-MnO$\mathrm{_2}$ #00-044-0141",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="blue",
)
ax.text(
    0.55,
    1.09,
    r"Zn(HSO$_{4}$)$_{2}$ #00-052-0258",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="orange",
)
ax.text(0, 1.02, r"ZSH", horizontalalignment="left", verticalalignment="bottom", transform=ax.transAxes, fontsize=9, c="k")
ax.text(
    0.13,
    1.01,
    r"K$_{0.66}$Mn$_4$O$_{8}$ #04-023-7544",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="purple",
)
ax.text(
    0.78,
    1.02,
    r"Ti #00-044-1294",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="green",
)

ax.text(-0.05, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)

# opxrd_ref_peak = [[5.87], [8.31], [13.15], [17.045], [18.19], [18.98]]
# opxrd_colors = ["blue", "blue", "blue", "blue", "green", "blue"]

opxrd_ref_peak = [[5.87], [8.31], [13.12], [17.045], [18.98]]
opxrd_ref_d_spacing = [[6.93], [4.89], [3.10], [2.39], [2.15]]
vmin_vmax = [(None, None), (None, None), (None, None), (800, 1400), (None, None)]
for i in range(len(opxrd_ref_peak)):
    ax = subfig.add_axes((0.1 + i * 0.14, 0.035, 0.12, 0.94), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.1) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.1)]
    # ax.contourf(temp.index, temp.columns, temp.T, cmap="coolwarm", levels=100)
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.text(0.9 + 0.005 * i, -0.15, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor="k",
        bottom=True,
        left=True,
        labelbottom=True,
        labelleft=True,
    )
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

    if i == 3:
        ax.axvline(opxrd_ref_peak[i][0] + 0.02, ymin=0.25, ymax=0.8, ls="--", lw=1.0, color="grey", alpha=0.5)
ax.text(-5.0, -0.14, r"$\mathrm{d \ Spacing\ (\AA)}$", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold", c="blue")
ax.text(-4.9, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V3_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V3_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V3_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V3_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

In [ ]:
%matplotlib inline

# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 0.3, 0.3))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Ewe/V"], selected_echem["FD1st_time"], ls="-", lw=1.0, c=colors[0], label=r"opCoin_1", zorder=5)


for i, idx in enumerate(selected_spectrum_time["level_0"]):
    if idx in index_labels:
        index_value = selected_spectrum_time.loc[selected_spectrum_time["level_0"] == idx, "index"].iloc[0]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c=colors[0],
            edgecolors="face",
            s=30,
            zorder=10,
        )
    else:
        index_value = selected_spectrum_time.loc[i, "index"]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c="lightgrey",
            edgecolors="face",
            s=5,
            zorder=1,
        )

ax.set_xlabel(r"Voltage (V)", fontsize=9)
ax.set_xlim(0.85, 1.85)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=-0.15))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=-0.15))

ax.set_ylabel(r"Duration Time (hour)", fontsize=9)
ax.set_ylim(-0.5, 30.5)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))

# ax.legend(loc='upper right', bbox_to_anchor=(0.9, 1.0), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
# ax.text(-0.1, -0.06, r'$\mathrm{3.288 mg\ cm^{-2}}$', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)
# ax.text(-0.1, 0, r'C/10, 1C=250 mA/g', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)

ax2 = ax.twiny()
ax2.set_position((0, 0, 0.3, 0.3))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"], selected_echem["FD1st_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(r"Current (mA)", fontsize=9)
ax2.set_xlim(-0.2, 0.2)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax2.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=False,
    top=True,
    labeltop=True,
    labelright=False,
)
# ax2.legend(loc='upper left', bbox_to_anchor=(0.15, 1.05), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
ax.text(-1.0, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 B
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.046, -1.0, 0.625, 0.08))

K1 = "#295488"
K2 = "k"
K3 = "purple"
K4 = "orange"

ax.text(
    0,
    0.55,
    r"$\mathrm{Zn_4SO_4(OH)_6 \cdot 5H_2O}$ #00-060-0655, #04-012-8189",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K1,
)
ax.text(
    0,
    0.02,
    r"$\mathrm{Zn_4SO_4(OH)_6 \cdot 3H_2O}$ #01-082-3605, #00-039-0689",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K2,
)
ax.text(
    0.66,
    0.55,
    r"$\mathrm{K_{0.66}Mn_4O_8}$ #04-023-7544",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K3,
)
ax.text(
    0.67,
    0.02,
    r"$\mathrm{Zn(HSO_4)_2}$ #00-052-0258",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K4,
)
ax.spines[:].set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
ax.set_facecolor("gainsboro")

# opxrd_ref_peak = [
#     [7.451],[9.775],[11.194],[11.591],[11.837],[11.945],[12.611],[12.755],
#     [13.060],[14.918],[15.055],[15.276],[15.452],[15.802],[15.914],
#     [16.103],[16.541],[17.585],[18.457],
# ]

opxrd_ref_peak = [
    [7.46],
    [9.78],
    [11.20],
    [11.60],
    [11.84],
    [11.95],
    [12.60],
    [12.76],
    [13.06],
    [14.92],
    [15.055],
    [15.28],
    [15.45],
    [15.80],
    [15.92],
    [16.103],
    [16.54],
    [17.585],
    [18.457],
]
opxrd_ref_d_spacing = [[5.45], [4.16], [3.63], [3.51], [3.44], [3.40], [3.24], [3.19], [3.12], [2.73], [2.71], [2.67], [2.64], [2.58], [2.56], [2.53], [2.47], [2.32], [2.21]]
# KMnO
opxrd_colors = [K1, K1, K2, K1, K1, K1, K1, K1, K3, K2, K4, K1, K2, K2, K1, K1, K3, K4, K1]
vmin_vmax = [
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
]
for i in range(0, len(opxrd_ref_peak) // 3 - 1):
    # print(i)
    ax = subfig.add_axes((0.23 + i * 0.09, 0.0, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.1) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.1)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.text(0.75 + 0.001 * i, -0.30, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    # ax.tick_params(axis='both', which='both', labelcolor='k', bottom=True, labelbottom=True)
    ax.tick_params(axis="both", which="both", labelcolor=opxrd_colors[i], bottom=True, labelbottom=True)
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(len(opxrd_ref_peak) // 3 - 1, 2 * len(opxrd_ref_peak) // 3):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - len(opxrd_ref_peak) // 3 + 1) * 0.09, -0.42, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.1) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.1)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.text(0.75 + 0.001 * i, -0.23, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    # ax.tick_params(axis='both', which='both', labelcolor='k', bottom=True, labelbottom=True)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor=opxrd_colors[i],
        bottom=True,
        labelbottom=True,
    )
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(2 * len(opxrd_ref_peak) // 3, len(opxrd_ref_peak)):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - 2 * len(opxrd_ref_peak) // 3) * 0.09, -0.82, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.1) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.1)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.text(0.75 + 0.001 * i, -0.23, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    # ax.tick_params(axis='both', which='both', labelcolor='k', bottom=True, labelbottom=True)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor=opxrd_colors[i],
        bottom=True,
        labelbottom=True,
    )
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=1.0)
ax.text(-4.5, 2.42, r"$\mathrm{d \ Spacing\ (\AA)}$", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold", c="blue")
ax.text(-4.6, 3.75, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V3_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V3_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V3_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V3_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Operando XRD, αMnO2 -V2

In [ ]:
path_xrd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\αMnO2\2022-ICMAB\Results")
# 读取 alfaMnO2capillary 数据
xrd = pd.read_csv(Path.joinpath(path_xrd, r"Ex situ", r"alfaMnO2capillary.uxd"), sep=r"\s+", index_col=None, header=0, comment="#")

# 读取 PDF card 的数据
pdf = pd.read_csv(Path.joinpath(path_xrd, r"PDFCards", "PDF-00-044-0141-Mo.csv"), sep=r",", index_col=None, header=0, comment="#")
pdfmo = pd.read_csv(
    Path.joinpath(path_xrd, r"PDFCards", "PDF_1stDischarge.csv"),
    sep=",",
    index_col=None,
    header=0,
    skiprows=1,
    comment="#",
)

# 读取 spectrum 的数据
spectrum_all = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", "Spectum_all.csv"),
    sep=r",",
    index_col=None,
    header=0,
    comment="#",
)

# 电化学上的时间
with open(Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", r"EchemOperando1_c10_1544mg_C02.txt"), "r") as file:
    lines = file.readlines()
for line in lines:
    if line.startswith("Nb header lines"):
        line_skip = int(line.split(":")[1].strip())

echem_all = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", r"EchemOperando1_c10_1544mg_C02.txt"),
    sep="\t",
    index_col=None,
    header=0,
    comment="#",
    skiprows=line_skip - 1,
    encoding="latin_1",
    date_format="%m/%d/%y %H:%M:%S.%f",
    parse_dates=[1, 2],
).dropna(axis=1, how="all", inplace=False)
echem_all["time/s"] = pd.to_datetime(
    echem_all["time/s"],
)
echem_all["Ewe/V"] = pd.to_numeric(
    echem_all["Ewe/V"],
)
echem_all["<I>/mA"] = pd.to_numeric(
    echem_all["<I>/mA"],
)
# echem_all.info()

# 谱线上的时间
spectrum_time_all = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", r"Time_index_spectrum.csv"),
    sep=",",
    index_col=0,
    header=0,
    comment="#",
    date_format="%m/%d/%y %H:%M:%S.%f",
    parse_dates=[1],
)
spectrum_time_all["Time"] = pd.to_datetime(spectrum_time_all["Time"])
# spectrum_time_all.info()

# 匹配谱线和电化学上的时间
spectrum_time = [abs(echem_all["time/s"] - t).idxmin() for t in spectrum_time_all["Time"]]
spectrum_time = (echem_all.loc[spectrum_time, [r"Ewe/V", r"<I>/mA"]].reset_index(drop=False).sort_values(by="Ewe/V", ascending=True)).reset_index(drop=False, inplace=False)

In [ ]:
# 选择需要的电化学数据以及对应的谱线
index_labels = [17, 27, 37, 45, 54]
selected_spectrum_time = (
    spectrum_time[(spectrum_time["level_0"] <= index_labels[-1] + 1) & (spectrum_time["level_0"] >= index_labels[0])].sort_values(by="level_0", ascending=True).reset_index(drop=True)
)

selected_echem = echem_all[(echem_all.index <= selected_spectrum_time["index"].iloc[-1]) & (echem_all.index >= selected_spectrum_time["index"].iloc[0])]
selected_echem = selected_echem.copy()
selected_echem["FD1st_time"] = (selected_echem["time/s"] - selected_echem["time/s"].iloc[0]).dt.total_seconds() / 3600

selected_spectrum = spectrum_all[(spectrum_all["range"] <= selected_spectrum_time["level_0"].iloc[-1] + 1) & (spectrum_all["range"] >= selected_spectrum_time["level_0"].iloc[0] + 1)].pivot(
    index="2THETA", columns="range", values="Cnt2_D1"
)

In [ ]:
%matplotlib inline

plt.close("all")

# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 4, 4], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Ewe/V"], selected_echem["FD1st_time"], ls="-", lw=1.0, c=colors[0], label=r"opCoin_1", zorder=5)


for i, idx in enumerate(selected_spectrum_time["level_0"]):
    if idx in index_labels:
        index_value = selected_spectrum_time.loc[selected_spectrum_time["level_0"] == idx, "index"].iloc[0]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c=colors[0],
            edgecolors="face",
            s=30,
        )
    else:
        index_value = selected_spectrum_time.loc[i, "index"]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c="lightgrey",
            edgecolors="face",
            s=15,
        )

ax.set_xlabel(r"Voltage (V)", fontsize=11)
ax.set_xlim(0.85, 1.85)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=-0.15))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=-0.15))

ax.set_ylabel(r"Duration Time (hour)", fontsize=11)
ax.set_ylim(-0.5, 30.5)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))

# ax.legend(loc='upper right', bbox_to_anchor=(0.9, 1.0), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
# ax.text(-0.1, -0.06, r'$\mathrm{3.288 mg\ cm^{-2}}$', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)
# ax.text(-0.1, 0, r'C/10, 1C=250 mA/g', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)

ax2 = ax.twiny()
ax2.set_position((0, 0, 1, 1))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"], selected_echem["FD1st_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(r"Current (mA)", fontsize=11)
ax2.set_xlim(-0.2, 0.2)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax2.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=False,
    top=True,
    labeltop=True,
    labelright=False,
)
# ax2.legend(loc='upper left', bbox_to_anchor=(0.15, 1.05), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
ax.text(-0.6, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.05, 0.035, 0.94, 0.94))
ax.set_box_aspect(1.0)

# ax.contourf(selected_spectrum.index, selected_spectrum.columns, selected_spectrum.T, cmap="coolwarm", levels=100)
ax.imshow(
    selected_spectrum.T,
    cmap="coolwarm",
    aspect="auto",  # vmin=5000, vmax=40000,
    origin="lower",
    extent=[selected_spectrum.index.min(), selected_spectrum.index.max(), selected_spectrum.columns.min(), selected_spectrum.columns.max()],
)
ax.set_yticks([])
ax.set_xlim(5, 20)
ax.set_xlabel(
    r"2-Theta$\mathrm{_{Mo}}$ (degree)",
    fontsize=11,
)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=-5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=-5))

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.03, 0.1, 0.08, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.text(
    1.08,
    0.97,
    "High",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)
ax.text(
    1.08,
    0.08,
    "Low",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)

pdf_scale = [0.001, 0.006, 0.001, 0.03, 0.002, 0.001, 0.0001, 0.0005]
XRD_y = [0, 0.15, 0.15, 0.15, 0.15, 0.07, 0.0, 0.09]
XRDcolors = ["blue", "k", "k", "k", "k", "orange", "purple", "green"]
for i in range(pdfmo.shape[0]):
    temp = pdfmo.iloc[:, 2 * i : 2 * i + 2].dropna(axis="index", how="all")
    for j in range((temp.shape[0])):
        ax.axvline(x=temp.iloc[j, 0], ymin=XRD_y[i], ymax=(XRD_y[i] + temp.iloc[j, 1] * pdf_scale[i]), lw=1, c=XRDcolors[i])

ax.text(
    0,
    1.09,
    r"$\alpha$-MnO$\mathrm{_2}$ #00-044-0141",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="blue",
)
ax.text(
    0.55,
    1.09,
    r"Zn(HSO$_{4}$)$_{2}$ #00-052-0258",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="orange",
)
ax.text(0, 1.02, r"ZSH", horizontalalignment="left", verticalalignment="bottom", transform=ax.transAxes, fontsize=9, c="k")
ax.text(
    0.13,
    1.01,
    r"K$_{0.66}$Mn$_4$O$_{8}$ #04-023-7544",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="purple",
)
ax.text(
    0.78,
    1.02,
    r"Ti #00-044-1294",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="green",
)

ax.text(-0.05, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)

# opxrd_ref_peak = [[5.87], [8.31], [13.15], [17.045], [18.19], [18.98]]
# opxrd_colors = ["blue", "blue", "blue", "blue", "green", "blue"]

opxrd_ref_peak = [[5.87], [8.31], [13.15], [17.045], [18.98]]
opxrd_ref_d_spacing = [[6.93], [4.89], [3.10], [2.39], [2.15]]
vmin_vmax = [(None, None), (None, None), (None, None), (800, 1400), (None, None)]
for i in range(len(opxrd_ref_peak)):
    ax = subfig.add_axes((0.1 + i * 0.14, 0.035, 0.12, 0.94), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.1) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.1)]
    # ax.contourf(temp.index, temp.columns, temp.T, cmap="coolwarm", levels=100)
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.text(0.9 + 0.005 * i, -0.15, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor="k",
        bottom=True,
        left=True,
        labelbottom=True,
        labelleft=True,
    )
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

    if i == 3:
        ax.axvline(opxrd_ref_peak[i][0] + 0.02, ymin=0.25, ymax=0.8, ls="--", lw=1.0, color="grey", alpha=0.5)
ax.text(-5.0, -0.14, r"$\mathrm{Spacing\ (\AA)}$", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold", c="blue")
ax.text(-4.9, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V2_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V2_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V2_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V2_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

In [ ]:
%matplotlib inline

# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 0.3, 0.3))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Ewe/V"], selected_echem["FD1st_time"], ls="-", lw=1.0, c=colors[0], label=r"opCoin_1", zorder=5)


for i, idx in enumerate(selected_spectrum_time["level_0"]):
    if idx in index_labels:
        index_value = selected_spectrum_time.loc[selected_spectrum_time["level_0"] == idx, "index"].iloc[0]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c=colors[0],
            edgecolors="face",
            s=30,
            zorder=10,
        )
    else:
        index_value = selected_spectrum_time.loc[i, "index"]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c="lightgrey",
            edgecolors="face",
            s=5,
            zorder=1,
        )

ax.set_xlabel(r"Voltage (V)", fontsize=9)
ax.set_xlim(0.85, 1.85)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=-0.15))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=-0.15))

ax.set_ylabel(r"Duration Time (hour)", fontsize=9)
ax.set_ylim(-0.5, 30.5)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))

# ax.legend(loc='upper right', bbox_to_anchor=(0.9, 1.0), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
# ax.text(-0.1, -0.06, r'$\mathrm{3.288 mg\ cm^{-2}}$', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)
# ax.text(-0.1, 0, r'C/10, 1C=250 mA/g', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)

ax2 = ax.twiny()
ax2.set_position((0, 0, 0.3, 0.3))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"], selected_echem["FD1st_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(r"Current (mA)", fontsize=9)
ax2.set_xlim(-0.2, 0.2)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax2.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=False,
    top=True,
    labeltop=True,
    labelright=False,
)
# ax2.legend(loc='upper left', bbox_to_anchor=(0.15, 1.05), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
ax.text(-1.0, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 B
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.046, -1.0, 0.625, 0.08))

K1 = "#295488"
K2 = "k"
K3 = "purple"
K4 = "orange"

ax.text(
    0,
    0.55,
    r"$\mathrm{Zn_4SO_4(OH)_6 \cdot 5H_2O}$ #00-060-0655, #04-012-8189",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K1,
)
ax.text(
    0,
    0.02,
    r"$\mathrm{Zn_4SO_4(OH)_6 \cdot 3H_2O}$ #01-082-3605, #00-039-0689",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K2,
)
ax.text(
    0.66,
    0.55,
    r"$\mathrm{K_{0.66}Mn_4O_8}$ #04-023-7544",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K3,
)
ax.text(
    0.67,
    0.02,
    r"$\mathrm{Zn(HSO_4)_2}$ #00-052-0258",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K4,
)
ax.spines[:].set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
ax.set_facecolor("gainsboro")

# opxrd_ref_peak = [
#     [7.451],[9.775],[11.194],[11.591],[11.837],[11.945],[12.611],[12.755],
#     [13.060],[14.918],[15.055],[15.276],[15.452],[15.802],[15.914],
#     [16.103],[16.541],[17.585],[18.457],
# ]

opxrd_ref_peak = [
    [7.46],
    [9.78],
    [11.20],
    [11.60],
    [11.84],
    [11.95],
    [12.60],
    [12.76],
    [13.06],
    [14.92],
    [15.055],
    [15.28],
    [15.45],
    [15.80],
    [15.92],
    [16.103],
    [16.54],
    [17.585],
    [18.457],
]
opxrd_ref_d_spacing = [[5.45], [4.16], [3.63], [3.51], [3.44], [3.40], [3.24], [3.19], [3.12], [2.73], [2.71], [2.67], [2.64], [2.58], [2.56], [2.53], [2.47], [2.32], [2.21]]
# KMnO
opxrd_colors = [K1, K1, K2, K1, K1, K1, K1, K1, K3, K2, K4, K1, K2, K2, K1, K1, K3, K4, K1]
vmin_vmax = [
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
]
for i in range(0, len(opxrd_ref_peak) // 3 - 1):
    # print(i)
    ax = subfig.add_axes((0.23 + i * 0.09, 0.0, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.1) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.1)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.text(0.75 + 0.001 * i, -0.30, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    # ax.tick_params(axis='both', which='both', labelcolor='k', bottom=True, labelbottom=True)
    ax.tick_params(axis="both", which="both", labelcolor=opxrd_colors[i], bottom=True, labelbottom=True)
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(len(opxrd_ref_peak) // 3 - 1, 2 * len(opxrd_ref_peak) // 3):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - len(opxrd_ref_peak) // 3 + 1) * 0.09, -0.42, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.1) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.1)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.text(0.75 + 0.001 * i, -0.23, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    # ax.tick_params(axis='both', which='both', labelcolor='k', bottom=True, labelbottom=True)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor=opxrd_colors[i],
        bottom=True,
        labelbottom=True,
    )
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(2 * len(opxrd_ref_peak) // 3, len(opxrd_ref_peak)):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - 2 * len(opxrd_ref_peak) // 3) * 0.09, -0.82, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.1) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.1)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.text(0.75 + 0.001 * i, -0.23, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    # ax.tick_params(axis='both', which='both', labelcolor='k', bottom=True, labelbottom=True)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor=opxrd_colors[i],
        bottom=True,
        labelbottom=True,
    )
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=1.0)
ax.text(-4.5, 2.42, r"$\mathrm{Spacing\ (\AA)}$", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold", c="blue")
ax.text(-4.6, 3.75, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V2_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V2_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V2_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V2_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Operando XRD, αMnO2 -V1

In [ ]:
path_xrd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\αMnO2\2022-ICMAB\Results")
# 读取 alfaMnO2capillary 数据
xrd = pd.read_csv(Path.joinpath(path_xrd, r"Ex situ", r"alfaMnO2capillary.uxd"), sep=r"\s+", index_col=None, header=0, comment="#")

# 读取 PDF card 的数据
pdf = pd.read_csv(Path.joinpath(path_xrd, r"PDFCards", "PDF-00-044-0141-Mo.csv"), sep=r",", index_col=None, header=0, comment="#")
pdfmo = pd.read_csv(
    Path.joinpath(path_xrd, r"PDFCards", "PDF_1stDischarge.csv"),
    sep=",",
    index_col=None,
    header=0,
    skiprows=1,
    comment="#",
)

# 读取 spectrum 的数据
spectrum_all = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", "Spectum_all.csv"),
    sep=r",",
    index_col=None,
    header=0,
    comment="#",
)

# 电化学上的时间
with open(Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", r"EchemOperando1_c10_1544mg_C02.txt"), "r") as file:
    lines = file.readlines()
for line in lines:
    if line.startswith("Nb header lines"):
        line_skip = int(line.split(":")[1].strip())

echem_all = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", r"EchemOperando1_c10_1544mg_C02.txt"),
    sep="\t",
    index_col=None,
    header=0,
    comment="#",
    skiprows=line_skip - 1,
    encoding="latin_1",
    date_format="%m/%d/%y %H:%M:%S.%f",
    parse_dates=[1, 2],
).dropna(axis=1, how="all", inplace=False)
echem_all["time/s"] = pd.to_datetime(
    echem_all["time/s"],
)
echem_all["Ewe/V"] = pd.to_numeric(
    echem_all["Ewe/V"],
)
echem_all["<I>/mA"] = pd.to_numeric(
    echem_all["<I>/mA"],
)
# echem_all.info()

# 谱线上的时间
spectrum_time_all = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", r"Time_index_spectrum.csv"),
    sep=",",
    index_col=0,
    header=0,
    comment="#",
    date_format="%m/%d/%y %H:%M:%S.%f",
    parse_dates=[1],
)
spectrum_time_all["Time"] = pd.to_datetime(spectrum_time_all["Time"])
# spectrum_time_all.info()

# 匹配谱线和电化学上的时间
spectrum_time = [abs(echem_all["time/s"] - t).idxmin() for t in spectrum_time_all["Time"]]
spectrum_time = (echem_all.loc[spectrum_time, [r"Ewe/V", r"<I>/mA"]].reset_index(drop=False).sort_values(by="Ewe/V", ascending=True)).reset_index(drop=False, inplace=False)

In [ ]:
# 选择需要的电化学数据以及对应的谱线
index_labels = [17, 27, 37, 45, 54]
selected_spectrum_time = (
    spectrum_time[(spectrum_time["level_0"] <= index_labels[-1] + 1) & (spectrum_time["level_0"] >= index_labels[0])].sort_values(by="level_0", ascending=True).reset_index(drop=True)
)

selected_echem = echem_all[(echem_all.index <= selected_spectrum_time["index"].iloc[-1]) & (echem_all.index >= selected_spectrum_time["index"].iloc[0])]
selected_echem = selected_echem.copy()
selected_echem["FD1st_time"] = (selected_echem["time/s"] - selected_echem["time/s"].iloc[0]).dt.total_seconds() / 3600

selected_spectrum = spectrum_all[(spectrum_all["range"] <= selected_spectrum_time["level_0"].iloc[-1] + 1) & (spectrum_all["range"] >= selected_spectrum_time["level_0"].iloc[0] + 1)].pivot(
    index="2THETA", columns="range", values="Cnt2_D1"
)

In [ ]:
%matplotlib inline

plt.close("all")

# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 4, 4], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Ewe/V"], selected_echem["FD1st_time"], ls="-", lw=1.0, c=colors[0], label=r"opCoin_1", zorder=5)


for i, idx in enumerate(selected_spectrum_time["level_0"]):
    if idx in index_labels:
        index_value = selected_spectrum_time.loc[selected_spectrum_time["level_0"] == idx, "index"].iloc[0]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c=colors[0],
            edgecolors="face",
            s=30,
        )
    else:
        index_value = selected_spectrum_time.loc[i, "index"]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c="lightgrey",
            edgecolors="face",
            s=15,
        )

ax.set_xlabel(r"Voltage (V)", fontsize=11)
ax.set_xlim(0.85, 1.85)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=-0.15))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=-0.15))

ax.set_ylabel(r"Duration Time (hour)", fontsize=11)
ax.set_ylim(-0.5, 30.5)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))

# ax.legend(loc='upper right', bbox_to_anchor=(0.9, 1.0), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
# ax.text(-0.1, -0.06, r'$\mathrm{3.288 mg\ cm^{-2}}$', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)
# ax.text(-0.1, 0, r'C/10, 1C=250 mA/g', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)

ax2 = ax.twiny()
ax2.set_position((0, 0, 1, 1))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"], selected_echem["FD1st_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(r"Current (mA)", fontsize=11)
ax2.set_xlim(-0.2, 0.2)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax2.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=False,
    top=True,
    labeltop=True,
    labelright=False,
)
# ax2.legend(loc='upper left', bbox_to_anchor=(0.15, 1.05), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
ax.text(-0.6, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.05, 0.035, 0.94, 0.94))
ax.set_box_aspect(1.0)

# ax.contourf(selected_spectrum.index, selected_spectrum.columns, selected_spectrum.T, cmap="coolwarm", levels=100)
ax.imshow(
    selected_spectrum.T,
    cmap="coolwarm",
    aspect="auto",  # vmin=5000, vmax=40000,
    origin="lower",
    extent=[selected_spectrum.index.min(), selected_spectrum.index.max(), selected_spectrum.columns.min(), selected_spectrum.columns.max()],
)
ax.set_yticks([])
ax.set_xlim(5, 20)
ax.set_xlabel(
    r"2-Theta$\mathrm{_{Mo}}$ (degree)",
    fontsize=11,
)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=-5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=-5))

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.03, 0.1, 0.08, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.text(
    1.08,
    0.97,
    "High",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)
ax.text(
    1.08,
    0.08,
    "Low",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)

pdf_scale = [0.001, 0.006, 0.001, 0.03, 0.002, 0.001, 0.0001, 0.0005]
XRD_y = [0, 0.15, 0.15, 0.15, 0.15, 0.07, 0.0, 0.09]
XRDcolors = ["blue", "k", "k", "k", "k", "orange", "purple", "green"]
for i in range(pdfmo.shape[0]):
    temp = pdfmo.iloc[:, 2 * i : 2 * i + 2].dropna(axis="index", how="all")
    for j in range((temp.shape[0])):
        ax.axvline(x=temp.iloc[j, 0], ymin=XRD_y[i], ymax=(XRD_y[i] + temp.iloc[j, 1] * pdf_scale[i]), lw=1, c=XRDcolors[i])

ax.text(
    0,
    1.09,
    r"$\alpha$-MnO$\mathrm{_2}$ #00-044-0141",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="blue",
)
ax.text(
    0.55,
    1.09,
    r"Zn(HSO$_{4}$)$_{2}$ #00-052-0258",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="orange",
)
ax.text(0, 1.02, r"ZSH", horizontalalignment="left", verticalalignment="bottom", transform=ax.transAxes, fontsize=9, c="k")
ax.text(
    0.13,
    1.01,
    r"K$_{0.66}$Mn$_4$O$_{8}$ #04-023-7544",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="purple",
)
ax.text(
    0.78,
    1.02,
    r"Ti #00-044-1294",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="green",
)

ax.text(-0.05, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)

# opxrd_ref_peak = [[5.87], [8.31], [13.15], [17.045], [18.19], [18.98]]
# opxrd_colors = ["blue", "blue", "blue", "blue", "green", "blue"]

opxrd_ref_peak = [[5.87], [8.31], [13.15], [17.045], [18.98]]
opxrd_colors = ["blue", "blue", "blue", "blue", "blue"]
vmin_vmax = [(None, None), (None, None), (None, None), (800, 1400), (None, None)]
for i in range(len(opxrd_ref_peak)):
    ax = subfig.add_axes((0.1 + i * 0.14, 0.035, 0.12, 0.94), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.1) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.1)]
    # ax.contourf(temp.index, temp.columns, temp.T, cmap="coolwarm", levels=100)
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor=opxrd_colors[i],
        bottom=True,
        left=True,
        labelbottom=True,
        labelleft=True,
    )
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

    if i == 3:
        ax.axvline(opxrd_ref_peak[i][0] + 0.02, ymin=0.25, ymax=0.8, ls="--", lw=1.0, color="grey", alpha=0.5)
ax.text(-4.9, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

In [ ]:
%matplotlib inline

# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 0.3, 0.3))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Ewe/V"], selected_echem["FD1st_time"], ls="-", lw=1.0, c=colors[0], label=r"opCoin_1", zorder=5)


for i, idx in enumerate(selected_spectrum_time["level_0"]):
    if idx in index_labels:
        index_value = selected_spectrum_time.loc[selected_spectrum_time["level_0"] == idx, "index"].iloc[0]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c=colors[0],
            edgecolors="face",
            s=30,
            zorder=10,
        )
    else:
        index_value = selected_spectrum_time.loc[i, "index"]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c="lightgrey",
            edgecolors="face",
            s=5,
            zorder=1,
        )

ax.set_xlabel(r"Voltage (V)", fontsize=11)
ax.set_xlim(0.85, 1.85)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=-0.15))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=-0.15))

ax.set_ylabel(r"Duration Time (hour)", fontsize=11)
ax.set_ylim(-0.5, 30.5)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))

# ax.legend(loc='upper right', bbox_to_anchor=(0.9, 1.0), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
# ax.text(-0.1, -0.06, r'$\mathrm{3.288 mg\ cm^{-2}}$', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)
# ax.text(-0.1, 0, r'C/10, 1C=250 mA/g', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)

ax2 = ax.twiny()
ax2.set_position((0, 0, 0.3, 0.3))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"], selected_echem["FD1st_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(r"Current (mA)", fontsize=11)
ax2.set_xlim(-0.2, 0.2)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax2.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=False,
    top=True,
    labeltop=True,
    labelright=False,
)
# ax2.legend(loc='upper left', bbox_to_anchor=(0.15, 1.05), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
ax.text(-1.0, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 B
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.046, -0.94, 0.625, 0.08))

K1 = "#295488"
K2 = "k"
K3 = "purple"
K4 = "orange"

ax.text(
    0,
    0.55,
    r"$\mathrm{Zn_4SO_4(OH)_6 \cdot 5H_2O}$ #00-060-0655, #04-012-8189",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K1,
)
ax.text(
    0,
    0.02,
    r"$\mathrm{Zn_4SO_4(OH)_6 \cdot 3H_2O}$ #01-082-3605, #00-039-0689",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K2,
)
ax.text(
    0.66,
    0.55,
    r"$\mathrm{K_{0.66}Mn_4O_8}$ #04-023-7544",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K3,
)
ax.text(
    0.67,
    0.02,
    r"$\mathrm{Zn(HSO_4)_2}$ #00-052-0258",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K4,
)
ax.spines[:].set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
ax.set_facecolor("gainsboro")

# opxrd_ref_peak = [
#     [7.451],[9.775],[11.194],[11.591],[11.837],[11.945],[12.611],[12.755],
#     [13.060],[14.918],[15.055],[15.276],[15.452],[15.802],[15.914],
#     [16.103],[16.541],[17.585],[18.457],
# ]

opxrd_ref_peak = [
    [7.46],
    [9.78],
    [11.20],
    [11.60],
    [11.84],
    [11.95],
    [12.60],
    [12.76],
    [13.06],
    [14.92],
    [15.055],
    [15.28],
    [15.45],
    [15.80],
    [15.92],
    [16.103],
    [16.54],
    [17.585],
    [18.457],
]
# KMnO
opxrd_colors = [K1, K1, K2, K1, K1, K1, K1, K1, K3, K2, K4, K1, K2, K2, K1, K1, K3, K4, K1]
vmin_vmax = [
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
]
for i in range(0, len(opxrd_ref_peak) // 3 - 1):
    # print(i)
    ax = subfig.add_axes((0.23 + i * 0.09, 0.0, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.1) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.1)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    # ax.tick_params(axis='both', which='both', labelcolor='k', bottom=True, labelbottom=True)
    ax.tick_params(axis="both", which="both", labelcolor=opxrd_colors[i], bottom=True, labelbottom=True)
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(len(opxrd_ref_peak) // 3 - 1, 2 * len(opxrd_ref_peak) // 3):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - len(opxrd_ref_peak) // 3 + 1) * 0.09, -0.40, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.1) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.1)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    # ax.tick_params(axis='both', which='both', labelcolor='k', bottom=True, labelbottom=True)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor=opxrd_colors[i],
        bottom=True,
        labelbottom=True,
    )
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(2 * len(opxrd_ref_peak) // 3, len(opxrd_ref_peak)):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - 2 * len(opxrd_ref_peak) // 3) * 0.09, -0.8, 0.08, 0.3), zorder=0)
    temp = selected_spectrum.loc[(selected_spectrum.index >= opxrd_ref_peak[i][0] - 0.1) & (selected_spectrum.index <= opxrd_ref_peak[i][0] + 0.1)]
    ax.imshow(temp.T, cmap="coolwarm", aspect="auto", origin="lower", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1], extent=[temp.index.min(), temp.index.max(), temp.columns.min(), temp.columns.max()])
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    # ax.tick_params(axis='both', which='both', labelcolor='k', bottom=True, labelbottom=True)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor=opxrd_colors[i],
        bottom=True,
        labelbottom=True,
    )
    for idx in index_labels:
        ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=1.0)

ax.text(-4.6, 3.65, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Operando XRD, αMnO2 - V0

In [ ]:
path_xrd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\Operando\αMnO2\2022-ICMAB\Results")
# 读取 alfaMnO2capillary 数据
xrd = pd.read_csv(Path.joinpath(path_xrd, r"Ex situ", r"alfaMnO2capillary.uxd"), sep=r"\s+", index_col=None, header=0, comment="#")

# 读取 PDF card 的数据
pdf = pd.read_csv(Path.joinpath(path_xrd, r"PDFCards", "PDF-00-044-0141-Mo.csv"), sep=r",", index_col=None, header=0, comment="#")
pdfmo = pd.read_csv(
    Path.joinpath(path_xrd, r"PDFCards", "PDF_1stDischarge.csv"),
    sep=",",
    index_col=None,
    header=0,
    skiprows=1,
    comment="#",
)

# 读取 spectrum 的数据
spectum_all = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", "Spectum_all.csv"),
    sep=r",",
    index_col=None,
    header=0,
    comment="#",
)

# 电化学上的时间
with open(Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", r"EchemOperando1_c10_1544mg_C02.txt"), "r") as file:
    lines = file.readlines()
for line in lines:
    if line.startswith("Nb header lines"):
        line_skip = int(line.split(":")[1].strip())

echem = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", r"EchemOperando1_c10_1544mg_C02.txt"),
    sep="\t",
    index_col=None,
    header=0,
    comment="#",
    skiprows=line_skip - 1,
    encoding="latin_1",
    date_format="%m/%d/%y %H:%M:%S.%f",
    parse_dates=[1, 2],
).dropna(axis=1, how="all", inplace=False)
echem["time/s"] = pd.to_datetime(
    echem["time/s"],
)
echem["Ewe/V"] = pd.to_numeric(
    echem["Ewe/V"],
)
echem["<I>/mA"] = pd.to_numeric(
    echem["<I>/mA"],
)
# echem.info()

# 谱线上的时间
time_spectrum = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", r"Time_index_spectrum.csv"),
    sep=",",
    index_col=0,
    header=0,
    comment="#",
    date_format="%m/%d/%y %H:%M:%S.%f",
    parse_dates=[1],
)
time_spectrum["Time"] = pd.to_datetime(time_spectrum["Time"])
# time_spectrum.info()

# 匹配谱线和电化学上的时间
index_spectrum = [abs(echem["time/s"] - t).idxmin() for t in time_spectrum["Time"]]
index_voltage = echem.loc[index_spectrum, [r"Ewe/V", r"<I>/mA"]].reset_index(drop=False).sort_values(by="Ewe/V", ascending=True)
index_voltage.reset_index(drop=False, inplace=True)

In [ ]:
# 选择需要的电化学数据以及对应的谱线
index = [17, 22, 27, 32, 37, 40, 45, 50, 54]
selected = index_voltage[index_voltage["level_0"].isin(index)].sort_values(by="level_0", ascending=True).reset_index(drop=True)
selected_echem = echem[(echem.index <= selected["index"].iloc[-1]) & (echem.index >= selected["index"].iloc[0])]
mapping_spectum = spectum_all[(spectum_all["range"] <= selected["level_0"].iloc[-1] + 1) & (spectum_all["range"] >= selected["level_0"].iloc[0] + 1)].pivot(
    index="2THETA", columns="range", values="Cnt2_D1"
)
FD1st_time = (selected_echem["time/s"] - selected_echem["time/s"].iloc[0]).dt.total_seconds() / 3600

In [ ]:
%matplotlib inline

plt.close("all")

# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 4, 4], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Ewe/V"], FD1st_time, ls="-", lw=1.0, c=colors[0], label=r"opCoin_1", zorder=5)
for i, idx in enumerate(selected["index"]):
    if i % 2 == 0:
        ax.scatter(
            selected_echem["Ewe/V"].iloc[selected_echem.index == idx],
            FD1st_time.iloc[FD1st_time.index == idx],
            c=colors[0],
            edgecolors="face",
        )
    else:
        ax.scatter(
            selected_echem["Ewe/V"].iloc[selected_echem.index == idx],
            FD1st_time.iloc[FD1st_time.index == idx],
            c=colors[1],
            edgecolors="face",
        )

# # 添加索引文本标注
# for i in range(selected.shape[0]):
#     row_idx = selected_echem.index == selected["index"].iloc[i]  # 提取索引值
#     ax.text(
#         selected_echem.loc[row_idx, "Ewe/V"] + 0.02,
#         FD1st_time.iloc[row_idx],
#         str(selected["level_0"].iloc[i]),
#         fontsize=10,
#         verticalalignment="bottom",
#         horizontalalignment="right",
#     )

ax.set_xlabel(r"Voltage (V)", fontsize=11)
ax.set_xlim(0.85, 1.85)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=-0.15))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=-0.15))

ax.set_ylabel(r"Duration Time (hour)", fontsize=11)
ax.set_ylim(-0.5, 30.5)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))

# ax.legend(loc='upper right', bbox_to_anchor=(0.9, 1.0), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
# ax.text(-0.1, -0.06, r'$\mathrm{3.288 mg\ cm^{-2}}$', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)
# ax.text(-0.1, 0, r'C/10, 1C=250 mA/g', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)

ax2 = ax.twiny()
ax2.set_position((0, 0, 1, 1))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"], FD1st_time, ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(r"Current (mA)", fontsize=11)
ax2.set_xlim(-0.2, 0.2)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax2.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=False,
    top=True,
    labeltop=True,
    labelright=False,
)
# ax2.legend(loc='upper left', bbox_to_anchor=(0.15, 1.05), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
ax.text(-0.6, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.05, 0.035, 0.94, 0.94))
ax.set_box_aspect(1.0)

ax.contourf(mapping_spectum.index, mapping_spectum.columns, mapping_spectum.T, cmap="coolwarm", levels=100)

ax.set_yticks([])
ax.set_xlim(5, 20)
ax.set_xlabel(
    r"2-Theta$\mathrm{_{Mo}}$ (degree)",
    fontsize=11,
)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=-5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=-5))

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.03, 0.1, 0.08, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.text(
    1.08,
    0.97,
    "High",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)
ax.text(
    1.08,
    0.08,
    "Low",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)

pdf_scale = [0.001, 0.006, 0.001, 0.03, 0.002, 0.001, 0.0001, 0.0005]
XRD_y = [0, 0.15, 0.15, 0.15, 0.15, 0.07, 0.0, 0.09]
XRDcolors = ["blue", "k", "k", "k", "k", "orange", "purple", "green"]
for i in range(pdfmo.shape[0]):
    temp = pdfmo.iloc[:, 2 * i : 2 * i + 2].dropna(axis="index", how="all")
    for j in range((temp.shape[0])):
        ax.axvline(x=temp.iloc[j, 0], ymin=XRD_y[i], ymax=(XRD_y[i] + temp.iloc[j, 1] * pdf_scale[i]), lw=1, c=XRDcolors[i])

ax.text(
    0,
    1.09,
    r"$\alpha$-MnO$\mathrm{_2}$ #00-044-0141",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="blue",
)
ax.text(
    0.55,
    1.09,
    r"Zn(HSO$_{4}$)$_{2}$ #00-052-0258",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="orange",
)
ax.text(0, 1.02, r"ZSH", horizontalalignment="left", verticalalignment="bottom", transform=ax.transAxes, fontsize=9, c="k")
ax.text(
    0.13,
    1.01,
    r"K$_{0.66}$Mn$_4$O$_{8}$ #04-023-7544",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="purple",
)
ax.text(
    0.78,
    1.02,
    r"Ti #00-044-1294",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="green",
)

ax.text(-0.05, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)

# opxrd_ref_peak = [[5.87], [8.31], [13.15], [17.045], [18.19], [18.98]]
# opxrd_colors = ["blue", "blue", "blue", "blue", "green", "blue"]

opxrd_ref_peak = [[5.87], [8.31], [13.15], [17.045], [18.98]]
opxrd_colors = ["blue", "blue", "blue", "blue", "blue"]

for i in range(len(opxrd_ref_peak)):
    ax = subfig.add_axes((0.1 + i * 0.14, 0.035, 0.12, 0.94), zorder=0)
    temp = mapping_spectum.loc[(mapping_spectum.index >= opxrd_ref_peak[i][0] - 0.1) & (mapping_spectum.index <= opxrd_ref_peak[i][0] + 0.1)]
    # if i != 3:
    ax.contourf(temp.index, temp.columns, temp.T, cmap="coolwarm", levels=100)
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor=opxrd_colors[i],
        bottom=True,
        left=True,
        labelbottom=True,
        labelleft=True,
    )
    for j, idx in enumerate(index):
        if j % 2 == 0:
            ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

    if i == 3:
        ax.axvline(opxrd_ref_peak[i][0] + 0.02, ymin=0.25, ymax=0.8, ls="--", lw=1.0, color="grey", alpha=0.5)
ax.text(-4.9, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_01_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

In [ ]:
%matplotlib inline

# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 0.3, 0.3))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Ewe/V"], FD1st_time, ls="-", lw=1.0, c=colors[0], label=r"opCoin_1", zorder=5)
for i, idx in enumerate(selected["index"]):
    if i % 2 == 0:
        ax.scatter(
            selected_echem["Ewe/V"].iloc[selected_echem.index == idx],
            FD1st_time.iloc[FD1st_time.index == idx],
            c=colors[0],
            edgecolors="face",
        )
    else:
        ax.scatter(
            selected_echem["Ewe/V"].iloc[selected_echem.index == idx],
            FD1st_time.iloc[FD1st_time.index == idx],
            c=colors[1],
            edgecolors="face",
        )

# # 添加索引文本标注
# for i in range(selected.shape[0]):
#     row_idx = selected_echem.index == selected["index"].iloc[i]  # 提取索引值
#     ax.text(
#         selected_echem.loc[row_idx, "Ewe/V"] + 0.02,
#         FD1st_time.iloc[row_idx],
#         str(selected["level_0"].iloc[i]),
#         fontsize=10,
#         verticalalignment="bottom",
#         horizontalalignment="right",
#     )

ax.set_xlabel(r"Voltage (V)", fontsize=11)
ax.set_xlim(0.85, 1.85)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=-0.15))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=-0.15))

ax.set_ylabel(r"Duration Time (hour)", fontsize=11)
ax.set_ylim(-0.5, 30.5)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))

# ax.legend(loc='upper right', bbox_to_anchor=(0.9, 1.0), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
# ax.text(-0.1, -0.06, r'$\mathrm{3.288 mg\ cm^{-2}}$', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)
# ax.text(-0.1, 0, r'C/10, 1C=250 mA/g', transform=ax.transAxes, fontsize=9, color=colors[3], va='top', ha='right', fontfamily='Arial',)

ax2 = ax.twiny()
ax2.set_position((0, 0, 0.3, 0.3))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"], FD1st_time, ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(r"Current (mA)", fontsize=11)
ax2.set_xlim(-0.2, 0.2)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax2.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=False,
    top=True,
    labeltop=True,
    labelright=False,
)
# ax2.legend(loc='upper left', bbox_to_anchor=(0.15, 1.05), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)
ax.text(-1.0, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 B
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.046, -0.94, 0.625, 0.08))

K1 = "#295488"
K2 = "k"
K3 = "purple"
K4 = "orange"

ax.text(
    0,
    0.55,
    r"$\mathrm{Zn_4SO_4(OH)_6 \cdot 5H_2O}$ #00-060-0655, #04-012-8189",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K1,
)
ax.text(
    0,
    0.02,
    r"$\mathrm{Zn_4SO_4(OH)_6 \cdot 3H_2O}$ #01-082-3605, #00-039-0689",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K2,
)
ax.text(
    0.66,
    0.55,
    r"$\mathrm{K_{0.66}Mn_4O_8}$ #04-023-7544",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K3,
)
ax.text(
    0.67,
    0.02,
    r"$\mathrm{Zn(HSO_4)_2}$ #00-052-0258",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c=K4,
)
ax.spines[:].set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
ax.set_facecolor("gainsboro")

# opxrd_ref_peak = [
#     [7.451],[9.775],[11.194],[11.591],[11.837],[11.945],[12.611],[12.755],
#     [13.060],[14.918],[15.055],[15.276],[15.452],[15.802],[15.914],
#     [16.103],[16.541],[17.585],[18.457],
# ]

opxrd_ref_peak = [
    [7.46],
    [9.78],
    [11.20],
    [11.60],
    [11.84],
    [11.95],
    [12.60],
    [12.76],
    [13.06],
    [14.92],
    [15.055],
    [15.28],
    [15.45],
    [15.80],
    [15.92],
    [16.103],
    [16.54],
    [17.585],
    [18.457],
]
# KMnO
opxrd_colors = [K1, K1, K2, K1, K1, K1, K1, K1, K3, K2, K4, K1, K2, K2, K1, K1, K3, K4, K1]

for i in range(0, len(opxrd_ref_peak) // 3 - 1):
    # print(i)
    ax = subfig.add_axes((0.23 + i * 0.09, 0.0, 0.08, 0.3), zorder=0)
    temp = mapping_spectum.loc[(mapping_spectum.index >= opxrd_ref_peak[i][0] - 0.1) & (mapping_spectum.index <= opxrd_ref_peak[i][0] + 0.1)]
    ax.contourf(temp.index, temp.columns, temp.T, cmap="coolwarm", levels=100)
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.6)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    # ax.tick_params(axis='both', which='both', labelcolor='k', bottom=True, labelbottom=True)
    ax.tick_params(axis="both", which="both", labelcolor=opxrd_colors[i], bottom=True, labelbottom=True)
    for j, idx in enumerate(index):
        if j % 2 == 0:
            ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(len(opxrd_ref_peak) // 3 - 1, 2 * len(opxrd_ref_peak) // 3):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - len(opxrd_ref_peak) // 3 + 1) * 0.09, -0.40, 0.08, 0.3), zorder=0)
    temp = mapping_spectum.loc[(mapping_spectum.index >= opxrd_ref_peak[i][0] - 0.1) & (mapping_spectum.index <= opxrd_ref_peak[i][0] + 0.1)]
    ax.contourf(temp.index, temp.columns, temp.T, cmap="coolwarm", levels=100)
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.6)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    # ax.tick_params(axis='both', which='both', labelcolor='k', bottom=True, labelbottom=True)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor=opxrd_colors[i],
        bottom=True,
        labelbottom=True,
    )
    for j, idx in enumerate(index):
        if j % 2 == 0:
            ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(2 * len(opxrd_ref_peak) // 3, len(opxrd_ref_peak)):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - 2 * len(opxrd_ref_peak) // 3) * 0.09, -0.8, 0.08, 0.3), zorder=0)
    temp = mapping_spectum.loc[(mapping_spectum.index >= opxrd_ref_peak[i][0] - 0.1) & (mapping_spectum.index <= opxrd_ref_peak[i][0] + 0.1)]
    ax.contourf(temp.index, temp.columns, temp.T, cmap="coolwarm", levels=100)
    ax.axvline(opxrd_ref_peak[i][0], ls="--", lw=1.0, color="grey", alpha=0.6)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_peak[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    # ax.tick_params(axis='both', which='both', labelcolor='k', bottom=True, labelbottom=True)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor=opxrd_colors[i],
        bottom=True,
        labelbottom=True,
    )
    for j, idx in enumerate(index):
        if j % 2 == 0:
            ax.axhline(y=idx, lw=0.6, ls="--", color=colors[0], alpha=1.0)

ax.text(-4.6, 3.65, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_02_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure XRD of pristine αMnO2 powder + EMD - V2

In [ ]:
# 读取数据
path_xrd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\ExSitu\αMnO2\Pristine\Results\2024-ICMAB")
xrd = pd.read_csv(Path.joinpath(path_xrd, r"αMnO2_Pristine_2024.UXD"), sep=r"\s+", index_col=None, header=0, comment="#")
pdf = pd.read_csv(Path.joinpath(path_xrd, r"PDF-00-044-0141.csv"), sep=r",", index_col=None, header=0, comment="#")
path_img = Path.joinpath(path_xrd, r"PDF Card - 00-044-0141_withK.tif")

xrd["Spacing"] = 1.5406 / (2 * np.sin(np.radians(xrd["2THETA"] / 2)))

In [ ]:
# 读取数据
path_xrd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\ExSitu\EMD\Pristine\Results\EMD-1.60V-2mAh_cm-2-1MZnAC2+02M-pH5.0_1M+0.2M")
xrd_emd = pd.read_csv(Path.joinpath(path_xrd, r"#1EMD_CP.uxd"), sep=r"\s+", index_col=None, header=0, comment="#")
xrd_cp = pd.read_csv(Path.joinpath(path_xrd, r"#2CP.uxd"), sep=r"\s+", index_col=None, header=0, comment="#")
pdf2 = pd.read_csv(Path.joinpath(path_xrd, r"PDF Card - 00-030-0820.csv"), sep=r",", index_col=None, header=0, comment="#")
xrd_emd["Spacing"] = 1.5406 / (2 * np.sin(np.radians(xrd_emd["2THETA"] / 2)))
xrd_cp["Spacing"] = 1.5406 / (2 * np.sin(np.radians(xrd_cp["2THETA"] / 2)))

In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

line2d = ax.plot(xrd.iloc[:, 2], xrd.iloc[:, 1] + 50, ls="-", lw=1.0, c=colors[0], label=r"$\alpha$-MnO${_2}$")
stem = ax.stem(
    pdf.iloc[:, 1],
    pdf.iloc[:, 2] * 0.8,
    linefmt=colors[3],
    markerfmt="none",
    bottom=0,
    orientation="vertical",
    label=r"PDF #00-044-0141",
)

ax.set_xlim(1, 9)
ax.set_xlabel(r"$\mathrm{d \ Spacing \ (\AA)}$", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1, offset=-1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=-1))

ax.set_ylim(0, 700)
ax.set_ylabel("Relative Intensity (arb.u.)", fontsize=9)
ax.set(yticks=[])
ax.tick_params(axis="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.legend(
    handles=line2d,
    loc="upper right",
    bbox_to_anchor=(1.02, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)
ax.text(
    0.98,
    0.88,
    r"PDF#00-044-0141",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],  # type: ignore
    va="top",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    zorder=5,
)

axins = ax.inset_axes((0.28, 0.5, 0.45, 0.45))
AA = plt.imread(path_img, format="tif")
axins.imshow(AA, origin="upper", zorder=2)
axins.set_axis_off()
ax.text(-0.07, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.1, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

line2d_emd = ax.plot(xrd_emd.iloc[:, 2], xrd_emd.iloc[:, 1] + 50, ls="-", lw=1.0, c=colors[0], label=r"$\mathrm{EMD, \epsilon -MnO_2}$")
line2d_cp = ax.plot(xrd_cp.iloc[:, 2], xrd_cp.iloc[:, 1] * 0.1 + 150, ls="-", lw=1.0, c=colors[1], label=r"Carbon Paper")
stem = ax.stem(
    pdf2.iloc[:, 1],
    pdf2.iloc[:, 2] * 2.0,
    linefmt=colors[3],
    markerfmt="none",
    bottom=0,
    orientation="vertical",
    label=r"PDF #00-030-0820",
)

ax.set_xlim(1, 9)
ax.set_xlabel(r"$\mathrm{d \ Spacing \ (\AA)}$", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1, offset=-1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=-1))

ax.set_ylim(0, 1800)
ax.set_ylabel("Relative Intensity (arb.u.)", fontsize=9)
ax.set(yticks=[])
ax.tick_params(axis="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.legend(
    handles=line2d_emd + line2d_cp,
    loc="upper right",
    bbox_to_anchor=(1.02, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)

ax.text(
    0.97,
    0.8,
    r"PDF #00-030-0820",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],  # type: ignore
    va="top",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    zorder=5,
)
ax.text(-0.07, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_00_V2_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_00_V2_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_00_V2_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_00_V2_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure XRD of pristine αMnO2 powder + EMD - V1

In [ ]:
# 读取数据
path_xrd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\ExSitu\αMnO2\Pristine\Results\2024-ICMAB")
xrd = pd.read_csv(Path.joinpath(path_xrd, r"αMnO2_Pristine_2024.UXD"), sep=r"\s+", index_col=None, header=0, comment="#")
pdf = pd.read_csv(Path.joinpath(path_xrd, r"PDF-00-044-0141.csv"), sep=r",", index_col=None, header=0, comment="#")
path_img = Path.joinpath(path_xrd, r"PDF Card - 00-044-0141_withK.tif")

xrd["Spacing"] = 1.5406 / (2 * np.sin(np.radians(xrd["2THETA"] / 2)))

In [ ]:
# 读取数据
path_xrd = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XRD\ExSitu\EMD\Pristine\Results\EMD-1.60V-2mAh_cm-2-1MZnAC2+02M-pH5.0_1M+0.2M")
xrd_emd = pd.read_csv(Path.joinpath(path_xrd, r"#1EMD_CP.uxd"), sep=r"\s+", index_col=None, header=0, comment="#")
xrd_cp = pd.read_csv(Path.joinpath(path_xrd, r"#2CP.uxd"), sep=r"\s+", index_col=None, header=0, comment="#")
pdf2 = pd.read_csv(Path.joinpath(path_xrd, r"PDF Card - 00-030-0820.csv"), sep=r",", index_col=None, header=0, comment="#")
xrd_emd["Spacing"] = 1.5406 / (2 * np.sin(np.radians(xrd_emd["2THETA"] / 2)))
xrd_cp["Spacing"] = 1.5406 / (2 * np.sin(np.radians(xrd_cp["2THETA"] / 2)))

In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

line2d = ax.plot(xrd.iloc[:, 2], xrd.iloc[:, 1] + 50, ls="-", lw=1.0, c=colors[0], label=r"$\alpha$-MnO${_2}$")
stem = ax.stem(
    pdf.iloc[:, 1],
    pdf.iloc[:, 2] * 0.8,
    linefmt=colors[3],
    markerfmt="none",
    bottom=0,
    orientation="vertical",
    label=r"PDF #00-044-0141",
)

ax.set_xlim(1, 9)
ax.set_xlabel(r"$\mathrm{Lattice \ Spacing \ (\AA)}$", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1, offset=-1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=-1))

ax.set_ylim(0, 700)
ax.set_ylabel("Relative Intensity (arb.u.)", fontsize=9)
ax.set(yticks=[])
ax.tick_params(axis="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.legend(
    handles=line2d,
    loc="upper right",
    bbox_to_anchor=(1.02, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)
ax.text(
    0.98,
    0.85,
    r"PDF#00-044-0141",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],  # type: ignore
    va="top",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    zorder=5,
)

axins = ax.inset_axes((0.28, 0.5, 0.45, 0.45))
AA = plt.imread(path_img, format="tif")
axins.imshow(AA, origin="upper", zorder=2)
axins.set_axis_off()
ax.text(-0.07, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.1, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

line2d_emd = ax.plot(xrd_emd.iloc[:, 2], xrd_emd.iloc[:, 1] + 50, ls="-", lw=1.0, c=colors[0], label=r"$\mathrm{Buffered \ EMD, \epsilon -MnO_2}$")
line2d_cp = ax.plot(xrd_cp.iloc[:, 2], xrd_cp.iloc[:, 1] * 0.1 + 150, ls="-", lw=1.0, c=colors[1], label=r"Carbon Paper")
stem = ax.stem(
    pdf2.iloc[:, 1],
    pdf2.iloc[:, 2] * 2.0,
    linefmt=colors[3],
    markerfmt="none",
    bottom=0,
    orientation="vertical",
    label=r"PDF #00-030-0820",
)

ax.set_xlim(1, 9)
ax.set_xlabel(r"$\mathrm{Lattice \ Spacing \ (\AA)}$", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1, offset=-1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=-1))

ax.set_ylim(0, 1800)
ax.set_ylabel("Relative Intensity (arb.u.)", fontsize=9)
ax.set(yticks=[])
ax.tick_params(axis="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.legend(
    handles=line2d_emd + line2d_cp,
    loc="upper right",
    bbox_to_anchor=(1.02, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)

ax.text(
    0.86,
    0.8,
    r"PDF #00-030-0820",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],  # type: ignore
    va="top",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    zorder=5,
)
ax.text(-0.07, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_00_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_00_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_00_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_00_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure XRD of pristine αMnO2 powder + EMD

In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

line2d = ax.plot(xrd.iloc[:, 0], xrd.iloc[:, 1] + 50, ls="-", lw=1.0, c=colors[0], label=r"$\alpha$-MnO${_2}$")
stem = ax.stem(
    pdf.iloc[:, 0],
    pdf.iloc[:, 2] * 0.8,
    linefmt=colors[3],
    markerfmt="none",
    bottom=0,
    orientation="vertical",
    label=r"PDF #00-044-0141",
)

ax.set_xlim(5, 80)
ax.set_xlabel(r"2-Theta$\mathrm{\ _{Cu}}$ (degree)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=15, offset=5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=5))

ax.set_ylim(0, 700)
ax.set_ylabel("Relative Intensity (arb.u.)", fontsize=9)
ax.set(yticks=[])
ax.tick_params(axis="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.legend(
    handles=line2d,
    loc="upper right",
    bbox_to_anchor=(1.02, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)
ax.text(
    0.98,
    0.85,
    r"PDF#00-044-0141",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],  # type: ignore
    va="top",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    zorder=5,
)

axins = ax.inset_axes((0.28, 0.5, 0.45, 0.45))
AA = plt.imread(path_img, format="tif")
axins.imshow(AA, origin="upper", zorder=2)
axins.set_axis_off()
ax.text(-0.07, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.1, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

line2d_emd = ax.plot(xrd_emd.iloc[:, 0], xrd_emd.iloc[:, 1] + 50, ls="-", lw=1.0, c=colors[0], label=r"$\mathrm{Buffered \ EMD, \epsilon -MnO_2}$")
line2d_cp = ax.plot(xrd_cp.iloc[:, 0], xrd_cp.iloc[:, 1] * 0.1 + 150, ls="-", lw=1.0, c=colors[1], label=r"Carbon Paper")
stem = ax.stem(
    pdf2.iloc[:, 0],
    pdf2.iloc[:, 2] * 2.0,
    linefmt=colors[3],
    markerfmt="none",
    bottom=0,
    orientation="vertical",
    label=r"PDF #00-030-0820",
)

ax.set_xlim(5, 80)
ax.set_xlabel(r"2-Theta$\mathrm{\ _{Cu}}$ (degree)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=15, offset=5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=5))

ax.set_ylim(0, 1800)
ax.set_ylabel("Relative Intensity (arb.u.)", fontsize=9)
ax.set(yticks=[])
ax.tick_params(axis="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.legend(
    handles=line2d_emd + line2d_cp,
    loc="upper right",
    bbox_to_anchor=(1.02, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)

ax.text(
    0.86,
    0.8,
    r"PDF #00-030-0820",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],  # type: ignore
    va="top",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    zorder=5,
)
ax.text(-0.07, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_00_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_00_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_00_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_XRD_00_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

## Echem

### αMnO2 cycled in 1M Na2SO4 + 0.2M ZnSO4 in Swagelok cell 

In [ ]:
# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\SpecialTest\DifferentElectrolytes\1M Na2SO4+02M MnSO4\Zn-αMnO2\Results\2025-10").glob(r"LC*.xlsx"))
echema = []
path_filelist = [path_filelist[i] for i in [0, 3]]

# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echema.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echema)):
    echema[i] = echema[i][echema[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echema[i] = echema[i][echema[i].iloc[:, 2] >= 0]

In [ ]:
# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\SpecialTest\DifferentElectrolytes\1M Na2SO4+02M MnSO4\Zn-αMnO2\Results\2025-10").glob(r"**/*.txt"))

# 读取电化学数据
echemb = []
for path_file in path_filelist:
    line_skip = None
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=",").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echemb.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemb)):
    echemb[i] = echemb[i][echemb[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echemb[i] = echemb[i].copy()
    echemb[i]["Voltage"] = echemb[i]["Ewe/V"] - echemb[i]["Ece/V"]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(3.3, 2.5))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"#1", None, None, None], [r"#2", None, None, None], [r"#3", None, None, None], [r"#4", None, None, None]]
# for i, data in enumerate(echema):
#     capacity_max = data['SpeCap/mAh/g'].max()
#     for j, idx in enumerate(data.iloc[:, 0].unique()):
#         temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)

#         # 找到电压最小值的索引
#         idx_min = temp['Voltage/V'].idxmin()
#         ax.plot(temp.loc[:idx_min-1, 'SpeCap/mAh/g']/capacity_max*100, temp.loc[:idx_min-1, 'Voltage/V'], ls='-', lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
#         ax.plot(temp.loc[idx_min+1:, 'SpeCap/mAh/g']/capacity_max*100, temp.loc[idx_min+1:, 'Voltage/V'], ls='-', lw=1.0, c=colors[i], label=None, zorder=0)

for i, data in enumerate(echemb):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
        capacity_max = temp["Capacity/mA.h"].max()
        # 找到最大电压的索引
        idx_max = temp["Voltage"].idxmax()
        ax.plot(temp.loc[:idx_max, "Capacity/mA.h"] / capacity_max * 100, temp.loc[:idx_max, "Voltage"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_max + 2 :, "Capacity/mA.h"] / capacity_max * 100, temp.loc[idx_max + 2 :, "Voltage"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)


# 设置刻度线等格式
ax.set_xlabel(r"Full Charge Percentage (%)", fontsize=11)
ax.set_xlim(0, 105)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.1))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.8), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.15, r"$\mathrm{C/10}$" + "\n" + r"$\mathrm{1 \ M \ Na_2SO_4 + 0.2 \ M \ MnSO_4}$", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_10_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_10_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_10_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_10_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Organic Echem - 2 Zn + Mn

In [ ]:
# 1st1.80V - 1M ZnSO4 + 0.2M MnSO4 - ZHS

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\SpecialTest\OrganicEchem\1st1.80V\ZHS\1M ZnSO4 + 0.2M MnSO4\Results\15#-03-2023-2").glob(r"LC*.xlsx"))
echema = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echema.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echema)):
    if i == 2:
        echema[i] = echema[i][echema[i].iloc[:, 0].isin([0, 1, 3])]
    else:
        echema[i] = echema[i][echema[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echema[i] = echema[i][echema[i].iloc[:, 2] >= 0]

# 调整电化学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
indices = [0, 1, 2]  # 指定新顺序
echema = [echema[i] for i in indices]


# 2nd0.9V - 1M ZnSO4 + 0.2M MnSO4 - ZHS

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\SpecialTest\OrganicEchem\2nd0.9V\ZHS\1M ZnSO4 + 0.2M MnSO4\Results\10#-03-2023-B").glob(r"LC*.xlsx"))
echemb = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemb.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemb)):
    echemb[i] = echemb[i][echemb[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echemb[i] = echemb[i][echemb[i].iloc[:, 2] >= 0]

# 调整电化学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
indices = [0, 1, 2]  # 指定新顺序
echemb = [echemb[i] for i in indices]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(0.8)

labels = [
    [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2M \ MnSO_4}$", None, None, None],
    [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO}$", None, None, None],
    [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2M \ MnSO_4}$", None, None, None],
]
for i, data in enumerate(echema):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(320, 1.80, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11)
ax.set_xlim(-0.1, 450)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=90, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=45, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1.3), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.6, 0.07, r"No Acid-treated, C/10", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.21, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.15, 0, 1, 1))
ax.set_box_aspect(0.8)

labels = [
    [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2M \ MnSO_4}$", None, None, None],
    [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO}$", None, None, None],
    [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2M \ MnSO_4}$", None, None, None],
]
for i, data in enumerate(echemb):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(328, 0.93, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11)
ax.set_xlim(-1, 500)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=100, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=50, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1.3), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.6, 0.07, r"No Acid-treated, C/10", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.21, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_09_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_09_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_09_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_09_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Organic Echem - 1 Zn(Otf)2

In [ ]:
# 1st0.9V - 1M ZnSO4 + 0.2M MnSO4 - ZHS

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\SpecialTest\OrganicEchem\1st0.9V\ZHS\1M ZnSO4 + 0.2M MnSO4\Results\15#-03-2023").glob(r"LC*.xlsx"))
echema = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echema.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echema)):
    echema[i] = echema[i][echema[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echema[i] = echema[i][echema[i].iloc[:, 2] >= 0]

# 调整电化学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
indices = [0, 1, 2]  # 指定新顺序
echema = [echema[i] for i in indices]


# 1st0.9V - ZHS - 1M ZnSO4

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\SpecialTest\OrganicEchem\1st0.9V\ZHS\1M ZnSO4\Results\15#-03-2023").glob(r"LC*.xlsx"))
echemb = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemb.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemb)):
    echemb[i] = echemb[i][echemb[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echemb[i] = echemb[i][echemb[i].iloc[:, 2] >= 0]

# 调整电化学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
indices = [0, 1, 2]  # 指定新顺序
echemb = [echemb[i] for i in indices]

# 1st1.80V - ZHS - 1M ZnSO4 + 0.2M MnSO4

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\SpecialTest\OrganicEchem\1st1.80V\ZHS\1M ZnSO4 + 0.2M MnSO4\Results\15#-03-2023").glob(r"LC*.xlsx"))
echemc = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemc.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemc)):
    echemc[i] = echemc[i][echemc[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echemc[i] = echemc[i][echemc[i].iloc[:, 2] >= 0]

# 调整电化学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
indices = [0, 1, 2]  # 指定新顺序
echemc = [echemc[i] for i in indices]


# 1st1.80V - ZHS - 1M ZnSO4

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\SpecialTest\OrganicEchem\1st1.80V\ZHS\1M ZnSO4\Results\15#-03-2023").glob(r"LC*.xlsx"))
echemd = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemd.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemd)):
    echemd[i] = echemd[i][echemd[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echemd[i] = echemd[i][echemd[i].iloc[:, 2] >= 0]

# 调整电化学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
indices = [0, 1, 2]  # 指定新顺序
echemd = [echemd[i] for i in indices]

# 1st1.80V - No ZHS - 1M ZnSO4

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\SpecialTest\OrganicEchem\1st1.80V\No ZHS\1M ZnSO4 + 0.2M MnSO4\Results\15#-03-2023").glob(r"LC*.xlsx"))
echeme = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echeme.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echeme)):
    echeme[i] = echeme[i][echeme[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echeme[i] = echeme[i][echeme[i].iloc[:, 2] >= 0]

# 调整电化学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
indices = [0, 1, 2]  # 指定新顺序
echeme = [echeme[i] for i in indices]

# 1st1.80V - No ZHS - 1M ZnSO4

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\SpecialTest\OrganicEchem\1st1.80V\No ZHS\1M ZnSO4\Results\15#-03-2023").glob(r"LC*.xlsx"))
echemf = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemf.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemf)):
    echemf[i] = echemf[i][echemf[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echemf[i] = echemf[i][echemf[i].iloc[:, 2] >= 0]

# 调整电化学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
indices = [0, 1, 2]  # 指定新顺序
echemf = [echemf[i] for i in indices]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 7.5))
gs = gridspec.GridSpec(3, 2, width_ratios=[1, 1], height_ratios=[1, 1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(0.8)

labels = [
    [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", None, None, None],
    [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO}$", None, None, None],
    [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO + H_2O}$", None, None, None],
]
for i, data in enumerate(echema):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(300, 0.98, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11)
ax.set_xlim(-0.1, 320)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1.0), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.6, 0.07, r"No Acid-treated, C/10", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.21, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.2, 0, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{1 \ M \ ZnSO_4}$", None, None, None], [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO}$", None, None, None], [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO + H_2O}$", None, None, None]]
for i, data in enumerate(echemb):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(415, 0.98, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11)
ax.set_xlim(-0.1, 450)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=90, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=45, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1.0), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.6, 0.07, r"No Acid-treated, C/10", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.21, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[1, 0])
ax = subfig.add_axes((0, -0.3, 1, 1))
ax.set_box_aspect(0.8)

labels = [
    [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", None, None, None],
    [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO}$", None, None, None],
    [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO + H_2O}$", None, None, None],
]
for i, data in enumerate(echemc):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(433, 1.80, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11)
ax.set_xlim(-1, 450)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=90, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=45, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.65), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.6, 0.07, r"No Acid-treated, C/10", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.21, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
subfig = fig.add_subfigure(gs[1, 1])
ax = subfig.add_axes((0.2, -0.3, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{1 \ M \ ZnSO_4}$", None, None, None], [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO}$", None, None, None], [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO + H_2O}$", None, None, None]]
for i, data in enumerate(echemd):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(155, 1.80, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11)
ax.set_xlim(-0.1, 300)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=60, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=30, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.65), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.6, 0.07, r"No Acid-treated, C/10", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.21, 1.0, r"D", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 E
subfig = fig.add_subfigure(gs[2, 0])
ax = subfig.add_axes((0.0, -0.6, 1, 1))
ax.set_box_aspect(0.8)

labels = [
    [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2M \ MnSO_4}$", None, None, None],
    [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO}$", None, None, None],
    [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO + H_2O}$", None, None, None],
]
for i, data in enumerate(echeme):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(330, 1.80, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11)
ax.set_xlim(-0.1, 350)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=70, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=35, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.65), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.65, 0.07, r"Acid-treated, C/10", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.21, 1.0, r"E", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 F
subfig = fig.add_subfigure(gs[2, 1])
ax = subfig.add_axes((0.2, -0.6, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{1 \ M \ ZnSO_4}$", None, None, None], [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO}$", None, None, None], [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO + H_2O}$", None, None, None]]
for i, data in enumerate(echemf):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(263, 1.80, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11)
ax.set_xlim(-0.1, 450)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=90, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=45, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.67), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.65, 0.07, r"Acid-treated, C/10", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.21, 1.0, r"F", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_08_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_08_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_08_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_08_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### 0.2 M MnSO4 + 1stCharge -v1

In [ ]:
# 1st1.80V - 1M ZnSO4 + 0.2M MnSO4 - ZHS

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st1.80V\1M ZnSO4 + 0.2M MnSO4\Results\ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echema = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echema.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echema)):
    echema[i] = echema[i][echema[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echema[i] = echema[i][echema[i].iloc[:, 2] >= 0]

# 1st1.80V - 1M ZnSO4 - ZHS

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st1.80V\1M ZnSO4\Results\ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echemb = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemb.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemb)):
    echemb[i] = echemb[i][echemb[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echemb[i] = echemb[i][echemb[i].iloc[:, 2] >= 0]

# 1st1.80V - 1M ZnSO4 + 0.2M MnSO4 - No ZHS

# 电化学上的时间
path_filelist = list(
    Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st1.80V\1M ZnSO4 + 0.2M MnSO4\Results\No ZHS\17#-03-2023").glob(r"**\*.xlsx")
)
echemc = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemc.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemc)):
    echemc[i] = echemc[i][echemc[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echemc[i] = echemc[i][echemc[i].iloc[:, 2] >= 0]

# 1st1.80V - 1M ZnSO4 - No ZHS

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st1.80V\1M ZnSO4\Results\No ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echemd = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemd.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemd)):
    echemd[i] = echemd[i][echemd[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echemd[i] = echemd[i][echemd[i].iloc[:, 2] >= 0]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$" + "\n" + r"$\mathrm{1^{st} \ Charge}$", None, None]]
for i, data in enumerate(echema):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[:idx_min, "SpeCap/mAh/g"], temp.loc[:idx_min, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 2 :, "SpeCap/mAh/g"], temp.loc[idx_min + 2 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(285, 1.80, marker="*", markersize=10, color=colors[1], alpha=0.5)


# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-0.1, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.07, r"No Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.15, 0, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4, 1^{st} \ Charge}$", None, None]]
for i, data in enumerate(echemb):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(138, 1.80, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-0.1, 300)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=50, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=25, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.07, r"No Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[1, 0])
ax = subfig.add_axes((0, -0.3, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$" + "\n" + r"$\mathrm{1^{st} \ Charge}$", None, None]]
for i, data in enumerate(echemc):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(320, 1.80, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11)
ax.set_xlim(-0.1, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.07, r"Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
subfig = fig.add_subfigure(gs[1, 1])
ax = subfig.add_axes((0.15, -0.3, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4, 1^{st} \ Charge}$", None, None]]
for i, data in enumerate(echemd):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 2 :, "SpeCap/mAh/g"], temp.loc[idx_min + 2 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(200, 1.80, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-0.1, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.07, r"Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"D", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_07_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_07_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_07_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_07_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### 0.2 M MnSO4 + 1stCharge

In [ ]:
# 1st1.80V - 1M ZnSO4 + 0.2M MnSO4 - ZHS

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st1.80V\1M ZnSO4 + 0.2M MnSO4\Results\ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echema = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echema.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echema)):
    echema[i] = echema[i][echema[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echema[i] = echema[i][echema[i].iloc[:, 2] >= 0]

# 1st1.80V - 1M ZnSO4 - ZHS

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st1.80V\1M ZnSO4\Results\ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echemb = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemb.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemb)):
    echemb[i] = echemb[i][echemb[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echemb[i] = echemb[i][echemb[i].iloc[:, 2] >= 0]

# 1st1.80V - 1M ZnSO4 + 0.2M MnSO4 - No ZHS

# 电化学上的时间
path_filelist = list(
    Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st1.80V\1M ZnSO4 + 0.2M MnSO4\Results\No ZHS\17#-03-2023").glob(r"**\*.xlsx")
)
echemc = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemc.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemc)):
    echemc[i] = echemc[i][echemc[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echemc[i] = echemc[i][echemc[i].iloc[:, 2] >= 0]

# 1st1.80V - 1M ZnSO4 - No ZHS

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st1.80V\1M ZnSO4\Results\No ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echemd = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemd.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemd)):
    echemd[i] = echemd[i][echemd[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echemd[i] = echemd[i][echemd[i].iloc[:, 2] >= 0]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$" + "\n" + r"$\mathrm{1^{st} \ Charge}$", None, None]]
for i, data in enumerate(echema):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[:idx_min, "SpeCap/mAh/g"], temp.loc[:idx_min, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 2 :, "SpeCap/mAh/g"], temp.loc[idx_min + 2 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)


# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-0.1, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.07, r"No Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.15, 0, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4, 1^{st} \ Charge}$", None, None]]
for i, data in enumerate(echemb):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-0.1, 300)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=50, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=25, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.07, r"No Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[1, 0])
ax = subfig.add_axes((0, -0.3, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$" + "\n" + r"$\mathrm{1^{st} \ Charge}$", None, None]]
for i, data in enumerate(echemc):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)


# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11)
ax.set_xlim(-0.1, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.07, r"Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
subfig = fig.add_subfigure(gs[1, 1])
ax = subfig.add_axes((0.15, -0.3, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4, 1^{st} \ Charge}$", None, None]]
for i, data in enumerate(echemd):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 2 :, "SpeCap/mAh/g"], temp.loc[idx_min + 2 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)


# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-0.1, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.07, r"Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"D", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_07_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_07_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_07_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_07_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### 0.2 M MnSO4 + 1stDischarge -V1

In [ ]:
# 1st0.9V - 1M ZnSO4 + 0.2M MnSO4 - ZHS

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st0.9V\1M ZnSO4 + 0.2M MnSO4\Results\ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echema = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echema.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echema)):
    echema[i] = echema[i][echema[i].iloc[:, 0].isin([0, 1, 2, 3])]

# 1st0.9V - 1M ZnSO4 - ZHS

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st0.9V\1M ZnSO4\Results\ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echemb = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemb.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemb)):
    echemb[i] = echemb[i][echemb[i].iloc[:, 0].isin([0, 1, 2, 3])]

# 1st0.9V - 1M ZnSO4 + 0.2M MnSO4 - No ZHS

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st0.9V\1M ZnSO4 + 0.2M MnSO4\Results\No ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echemc = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemc.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemc)):
    echemc[i] = echemc[i][echemc[i].iloc[:, 0].isin([0, 1, 2, 3])]

# 1st0.9V - 1M ZnSO4 - No ZHS

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st0.9V\1M ZnSO4\Results\No ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echemd = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemd.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemd)):
    echemd[i] = echemd[i][echemd[i].iloc[:, 0].isin([0, 1, 2, 3])]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$" + "\n" + r"$\mathrm{1^{st} \ Discharge}$", None, None]]
for i, data in enumerate(echema):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[:idx_min, "SpeCap/mAh/g"], temp.loc[:idx_min, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 2 :, "SpeCap/mAh/g"], temp.loc[idx_min + 2 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(305, 0.9, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-0.1, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.05, 0.07, r"No Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.15, 0, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4, 1^{st} \ Discharge}$", None, None]]
for i, data in enumerate(echemb):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[:idx_min, "SpeCap/mAh/g"], temp.loc[:idx_min, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 2 :, "SpeCap/mAh/g"], temp.loc[idx_min + 2 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(360, 0.9, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-0.1, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.05, 0.07, r"No Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[1, 0])
ax = subfig.add_axes((0, -0.3, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$" + "\n" + r"$\mathrm{1^{st} \ Discharge}$", None, None]]
for i, data in enumerate(echemc):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[:idx_min, "SpeCap/mAh/g"], temp.loc[:idx_min, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 2 :, "SpeCap/mAh/g"], temp.loc[idx_min + 2 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(348, 0.9, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-0.1, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.05, 0.07, r"Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
subfig = fig.add_subfigure(gs[1, 1])
ax = subfig.add_axes((0.15, -0.3, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4, 1^{st} \ Discharge}$", None, None]]
for i, data in enumerate(echemd):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[:idx_min, "SpeCap/mAh/g"], temp.loc[:idx_min, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 2 :, "SpeCap/mAh/g"], temp.loc[idx_min + 2 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(290, 0.93, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-0.1, 300)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=50, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=25, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.05, 0.07, r"Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"D", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_06_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_06_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_06_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_06_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### 0.2 M MnSO4 + 1stDischarge

In [ ]:
# 1st0.9V - 1M ZnSO4 + 0.2M MnSO4 - ZHS

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st0.9V\1M ZnSO4 + 0.2M MnSO4\Results\ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echema = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echema.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echema)):
    echema[i] = echema[i][echema[i].iloc[:, 0].isin([0, 1, 2, 3])]

# 1st0.9V - 1M ZnSO4 - ZHS

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st0.9V\1M ZnSO4\Results\ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echemb = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemb.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemb)):
    echemb[i] = echemb[i][echemb[i].iloc[:, 0].isin([0, 1, 2, 3])]

# 1st0.9V - 1M ZnSO4 + 0.2M MnSO4 - No ZHS

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st0.9V\1M ZnSO4 + 0.2M MnSO4\Results\No ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echemc = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemc.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemc)):
    echemc[i] = echemc[i][echemc[i].iloc[:, 0].isin([0, 1, 2, 3])]

# 1st0.9V - 1M ZnSO4 - No ZHS

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st0.9V\1M ZnSO4\Results\No ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echemd = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemd.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemd)):
    echemd[i] = echemd[i][echemd[i].iloc[:, 0].isin([0, 1, 2, 3])]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$" + "\n" + r"$\mathrm{1^{st} \ Discharge}$", None, None]]
for i, data in enumerate(echema):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[:idx_min, "SpeCap/mAh/g"], temp.loc[:idx_min, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 2 :, "SpeCap/mAh/g"], temp.loc[idx_min + 2 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)


# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-0.1, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.05, 0.07, r"No Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.15, 0, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4, 1^{st} \ Discharge}$", None, None]]
for i, data in enumerate(echemb):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[:idx_min, "SpeCap/mAh/g"], temp.loc[:idx_min, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 2 :, "SpeCap/mAh/g"], temp.loc[idx_min + 2 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)


# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-0.1, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.05, 0.07, r"No Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[1, 0])
ax = subfig.add_axes((0, -0.3, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$" + "\n" + r"$\mathrm{1^{st} \ Discharge}$", None, None]]
for i, data in enumerate(echemc):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[:idx_min, "SpeCap/mAh/g"], temp.loc[:idx_min, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 2 :, "SpeCap/mAh/g"], temp.loc[idx_min + 2 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)


# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-0.1, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.05, 0.07, r"Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
subfig = fig.add_subfigure(gs[1, 1])
ax = subfig.add_axes((0.15, -0.3, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4, 1^{st} \ Discharge}$", None, None]]
for i, data in enumerate(echemd):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[:idx_min, "SpeCap/mAh/g"], temp.loc[:idx_min, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 2 :, "SpeCap/mAh/g"], temp.loc[idx_min + 2 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)


# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-0.1, 300)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=50, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=25, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.05, 0.07, r"Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"D", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_06_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_06_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_06_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_06_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Mn2+ additive and comparison to normal α-MnO2 -V1

In [ ]:
# Carbon Paper in Swagelok Cells, 1M ZnSO4 + 0.2M MnSO4

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\Substrates\CarbonPaper_H23\GCD\DifferentConcentration\Results").glob(r"**\*009*.xlsx"))

# 选取需要的数据
path_filelist = [path_filelist[i] for i in [0, 1, 3, 4]]

# 读取电化学数据
echemb = []
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[["Voltage/V", "Current/uA", "Capacity/uAh"]] = df[["Voltage/V", "Current/uA", "Capacity/uAh"]].apply(pd.to_numeric, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemb.append(df)

# 选取每个数据的第一到第三圈
for i in range(len(echemb)):
    echemb[i] = echemb[i][echemb[i]["Cycle"].isin([1, 2, 3])]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(3.3, 2.5))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0.0, hspace=0.0)

# 图 B
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [
    [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", None, None],
    [r"$\mathrm{0 \ M \ ZnSO_4 + 1 \ M \ MnSO_4}$", None, None],
    [r"$\mathrm{1 \ M \ ZnSO_4 + 0 \ M \ MnSO_4}$", None, None],
    [r"$\mathrm{1 \ M \ ZnSO_4 + 1 \ M \ MnSO_4}$", None, None],
]
for i, data in enumerate(echemb):
    if i == 0:
        for j, idx in enumerate(data.iloc[:, 0].unique()):
            temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
            # 找到电压最小值的索引
            idx_min = temp["Voltage/V"].idxmin()
            ax.plot(temp.loc[: idx_min - 1, "Capacity/uAh"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
            ax.plot(temp.loc[idx_min + 1 :, "Capacity/uAh"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
    if i != 0:
        for j, idx in enumerate(data.iloc[:, 0].unique()):
            temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
            # 找到电压最大值的索引
            idx_max = temp["Voltage/V"].idxmax()
            ax.plot(temp.loc[: idx_max - 1, "Capacity/uAh"] / 0.503, temp.loc[: idx_max - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
            ax.plot(temp.loc[idx_max + 1 :, "Capacity/uAh"] / 0.503, temp.loc[idx_max + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (\mu Ah \ cm^{-2})}$", fontsize=11)
ax.set_xlim(-0.2, 15)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=3, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1.5, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.8, 2.0)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.2))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.8), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.08, r"C/10", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
# ax.text(-0.18, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_05_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_05_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_05_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_05_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Mn2+ additive and comparison to normal α-MnO2

In [ ]:
# αMnO2 in Swagelok Cells, 1M ZnSO4 + 0.2M MnSO4

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\1M ZnSO4 + 0.2M MnSO4").glob(r"LC*.xlsx"))
echema = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echema.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echema)):
    echema[i] = echema[i][echema[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echema[i] = echema[i][echema[i].iloc[:, 2] >= 0]


# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\Substrates\CarbonPaper_H23\GCD\DifferentConcentration\Results").glob(r"**\*009*.xlsx"))

# 选取需要的数据
path_filelist = [path_filelist[i] for i in [0, 1, 3, 4]]

# 读取电化学数据
echemb = []
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[["Voltage/V", "Current/uA", "Capacity/uAh"]] = df[["Voltage/V", "Current/uA", "Capacity/uAh"]].apply(pd.to_numeric, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemb.append(df)

# 选取每个数据的第一到第三圈
for i in range(len(echemb)):
    echemb[i] = echemb[i][echemb[i]["Cycle"].isin([1, 2, 3])]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1], height_ratios=None, wspace=0.0, hspace=0.0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [[r"#1", None, None, None], [r"#2", None, None, None]]
for i, data in enumerate(echema):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)

        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        ax.plot(temp.loc[: idx_min - 1, "Capacity/uAh"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "Capacity/uAh"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)


# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (\mu Ah_{MnO_2})}$", fontsize=11)
ax.set_xlim(-5, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=100, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=50, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.1))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.7), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.15, r"$\mathrm{C/10}$" + "\n" + r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.3, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [
    [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", None, None],
    [r"$\mathrm{0 \ M \ ZnSO_4 + 1 \ M \ MnSO_4}$", None, None],
    [r"$\mathrm{1 \ M \ ZnSO_4 + 0 \ M \ MnSO_4}$", None, None],
    [r"$\mathrm{1 \ M \ ZnSO_4 + 1 \ M \ MnSO_4}$", None, None],
]
for i, data in enumerate(echemb):
    if i == 0:
        for j, idx in enumerate(data.iloc[:, 0].unique()):
            temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
            # 找到电压最小值的索引
            idx_min = temp["Voltage/V"].idxmin()
            ax.plot(temp.loc[: idx_min - 1, "Capacity/uAh"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
            ax.plot(temp.loc[idx_min + 1 :, "Capacity/uAh"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
    if i != 0:
        for j, idx in enumerate(data.iloc[:, 0].unique()):
            temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
            # 找到电压最大值的索引
            idx_max = temp["Voltage/V"].idxmax()
            ax.plot(temp.loc[: idx_max - 1, "Capacity/uAh"], temp.loc[: idx_max - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
            ax.plot(temp.loc[idx_max + 1 :, "Capacity/uAh"], temp.loc[idx_max + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (\mu Ah_{MnO_2})}$", fontsize=11)
ax.set_xlim(-0.2, 16)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.8, 2.0)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.2))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.5), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.08, r"C/10", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_05_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_05_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_05_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_05_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### αMnO2 in 1 M ZnSO4

In [ ]:
# α-MnO2 in 1 M ZnSO4

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\1M ZnSO4").glob(r"LC*.xlsx"))
echem = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echem.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echem)):
    echem[i] = echem[i][echem[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echem[i] = echem[i][echem[i].iloc[:, 2] >= 0]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(3.3, 2.5))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"#1", None, None, None], [r"#2", None, None, None]]
for i, data in enumerate(echem):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)

        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)


# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-3, 350)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=70, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=35, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.1))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1.0), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.15, r"$\mathrm{C/10}$" + "\n" + r"$\mathrm{1 \ M \ ZnSO_4}$", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_04_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_04_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_04_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_04_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### SEM - Echem of SEM, Discharged samples

In [ ]:
# Echem of Operando XAS, Cell 3

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell3\Echem").glob(r"*.txt"))
# 读取电化学数据
echem = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem.append(df)

In [ ]:
%matplotlib inline
plt.close("all")
fig = plt.figure(figsize=(3.3, 2.5))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1.0, 1))
ax.set_box_aspect(0.8)

labels = [[r"#1", None], [r"#2", None]]
mass_loadings = [1.064]
for _, data in enumerate(echem):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
        # 找到电压最小的时候的索引
        idx_min = temp["Ewe/V"].idxmin()
        ax.plot(temp.loc[:idx_min, "Capacity/mA.h"] * 1000 / mass_loadings[0], temp.loc[:idx_min, "Ewe/V"], ls="-", lw=1.0, c=colors[j], label=None, zorder=0)
        ax.plot(temp.loc[idx_min + 2 :, "Capacity/mA.h"] * 1000 / mass_loadings[0], temp.loc[idx_min + 2 :, "Ewe/V"], ls="-", lw=1.0, c=colors[j], label=None, zorder=0)

ax.plot(365, 1.80, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"Capacity (mAh g$\mathrm{^{-1}_{MnO_2}}$)", fontsize=9)
ax.set_xlim(-3, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=100))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=50))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}}\!)$", fontsize=9)
ax.set_ylim(0.8, 2.0)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1))

ax.tick_params(axis="both", which="both", bottom=True, left=True, labelbottom=True, labelleft=True)
# ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

ax.text(
    0.03,
    0.15,
    r"C/10" + "\n" + r"0.5 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=9,
    c=colors[0],  # type: ignore
)


plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_03_SEM_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_03_SEM_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_03_SEM_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_03_SEM_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### alpha MnO2 and EMD cycled in Bulk Cell without Buffering - V1

In [ ]:
# αMnO2 in Bulk Cells, 1M ZnSO4 + 0.2M MnSO4, no buffering

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\SpecialTest\BulkCell\NopHBufffer\1M ZnSO4 + 0.2M MnSO4\Results\02C-aMnO2-1M+02M-1").glob(r"*.txt"))

# 读取电化学数据
echema = []
for path_file in path_filelist:
    line_skip = None
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=",").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echema.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echema)):
    echema[i] = echema[i][echema[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echema[i] = echema[i].copy()
    echema[i]["Voltage"] = echema[i]["Ewe/V"] - echema[i]["Ece/V"]


# αMnO2 in Bulk Cells, 1M ZnSO4 + 0.2M MnSO4, no buffering => extended windows

# 电化学上的时间
path_filelist = list(
    Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\SpecialTest\BulkCell\NopHBufffer\1M ZnSO4 + 0.2M MnSO4\Results\02C-aMnO2-1M+02M-2-extendedwindows").glob(r"*.txt")
)

# 读取电化学数据
echemb = []
for path_file in path_filelist:
    line_skip = None
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=",").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echemb.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemb)):
    echemb[i] = echemb[i][echemb[i].iloc[:, 0].isin([0, 1, 2])]
    echemb[i] = echemb[i].copy()
    echemb[i]["Voltage"] = echemb[i]["Ewe/V"] - echemb[i]["Ece/V"]


# EMD in Bulk Cells, 1M ZnSO4 + 0.2M MnSO4, no buffering, 估计也是 C/5

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\EMD\GCD\1M ZnSO4 + 0.2M MnSO4\Results\BulkCell-2V-2mAh-1M+02M\01-02").glob(r"*.txt"))

# 读取电化学数据
echemc = []
for path_file in path_filelist:
    line_skip = None
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=",").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echemc.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemc)):
    echemc[i] = echemc[i][echemc[i].iloc[:, 0].isin([0, 1, 2])]
    echemc[i] = echemc[i].copy()
    echemc[i]["Voltage"] = echemc[i]["Ewe/V"] - echemc[i]["Ece/V"]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0.0, hspace=0.0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [[r"#1", None, None, None], [r"#2", None, None, None]]
mass_loading = [
    0.56,
]
for i, data in enumerate(echema):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
        # 找到电压的最大值的索引
        idx_max = temp["Voltage"].idxmax()
        if j == 0:
            ax.plot(temp.loc[idx_max + 2 :, "Capacity/mA.h"] * 1000 / 0.503, temp.loc[idx_max + 2 :, "Voltage"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (\mu Ah \ cm^{-2})}$", fontsize=11)
# ax.set_xlim(-0.05, 3.0)
# ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
# ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.1))

ax.tick_params(axis="both", direction="out", labelsize=9)
# ax.legend(loc='upper right', bbox_to_anchor=(1.0, 0.7), ncols=1, frameon=False, labelcolor='linecolor', fontsize=9)
ax.text(0.05, 0.15, r"$\mathrm{BulkCell, \, C/5}$" + "\n" + r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.15, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [[r"#1", None, None, None], [r"#2", None, None, None]]
mass_loading = [
    0.56,
]
for i, data in enumerate(echema):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
        # 找到电压的最大值的索引
        idx_max = temp["Voltage"].idxmax()
        if j == 0:
            continue
            # ax.plot(temp.loc[idx_max+2:, 'Capacity/mA.h']*1000/mass_loading[i]/60, temp.loc[idx_max+2:, 'Voltage'], ls='-', lw=1.0, c=colors[i], label=None, zorder=0)
        else:
            ax.plot(temp.loc[idx_max + 2 :, "Capacity/mA.h"] * 1000 / 0.503, temp.loc[idx_max + 2 :, "Voltage"], ls="-", lw=1.0, c=colors[j], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[:idx_max, "Capacity/mA.h"] * 1000 / 0.503, temp.loc[:idx_max, "Voltage"], ls="-", lw=1.0, c=colors[j], label=labels[i][j], zorder=0)


# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (\mu Ah \ cm^{-2})}$", fontsize=11)
ax.set_xlim(-0.05, 6.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.1))

ax.tick_params(axis="both", direction="out", labelsize=9)
# ax.legend(loc='upper right', bbox_to_anchor=(1.0, 0.7), ncols=1, frameon=False, labelcolor='linecolor', fontsize=9)
ax.text(
    0.4,
    0.9,
    r"$\mathrm{BulkCell, \, C/5}$" + "\n" + r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$" + "\n" + r"$\mathrm{Voltage \ Cut-off: \ 1.80 - 0.80 \ V}$",
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=9,
    c="k",
)
ax.text(-0.18, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, -0.25, 1.0, 1.0))
ax.set_box_aspect(0.8)
labels = [[r"#1", None, None, None], [r"#2", None, None, None]]
mass_loadings = [0.4, 1.36]
for _, data in enumerate(echemb):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
        # 找到容量最大的时候的索引
        idx_max = temp["Capacity/mA.h"].idxmax()
        ax.plot(temp.loc[:idx_max, "Capacity/mA.h"], temp.loc[:idx_max, "Voltage"], ls="-", lw=1.0, c=colors[j], label=None, zorder=0)
        ax.plot(temp.loc[idx_max + 1 :, "Capacity/mA.h"], temp.loc[idx_max + 1 :, "Voltage"], ls="-", lw=1.0, c=colors[j], label=None, zorder=0)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh_{MnO_2})}$", fontsize=11)
ax.set_xlim(-0.005, 0.6)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.05, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.9, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0.1))

ax.tick_params(axis="both", direction="out", labelsize=9)
# ax.legend(loc='upper right', bbox_to_anchor=(1.0, 1.03), ncols=1, frameon=False, labelcolor='linecolor', fontsize=9)
ax.text(
    0.03,
    0.2,
    r"$\mathrm{Bulk Cell, C/5}$" + "\n" + r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$" + "\n" + r"$\mathrm{Voltage \ Cut-off: \ 2.0 - 0.95 \ V}$",
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=9,
    c="k",
)
ax.text(-0.18, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.15, -0.25, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [[r"#1", None, None, None], [r"#2", None, None, None]]
for _, data in enumerate(echemc):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
        capacity_max = temp["Capacity/mA.h"].max()
        # 找到最小电压的索引
        idx_max = temp["Voltage"].idxmin()
        ax.plot(temp.loc[:idx_max, "Capacity/mA.h"] / capacity_max * 100, temp.loc[:idx_max, "Voltage"], ls="-", lw=1.0, c=colors[j], label=None, zorder=0)
        ax.plot(temp.loc[idx_max + 1 :, "Capacity/mA.h"] / capacity_max * 100, temp.loc[idx_max + 1 :, "Voltage"], ls="-", lw=1.0, c=colors[j], label=None, zorder=0)


# 设置刻度线等格式
ax.set_xlabel(r"Full Charge Percentage (%)", fontsize=11)
ax.set_xlim(0, 105)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))

ax.set_ylabel(
    r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$",
    fontsize=11,
)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.1))

ax.tick_params(axis="both", direction="out", labelsize=9)
# ax.legend(loc='upper right', bbox_to_anchor=(1.0, 0.7), ncols=1, frameon=False, labelcolor='linecolor', fontsize=9)
ax.text(0.03, 0.15, r"$\mathrm{BulkCell}$" + "\n" + r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"D", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_02_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_02_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_02_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_02_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### alpha MnO2 and EMD cycled in Bulk Cell without Buffering

In [ ]:
# αMnO2 in Bulk Cells, 1M ZnSO4 + 0.2M MnSO4, no buffering

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\SpecialTest\BulkCell\NopHBufffer\1M ZnSO4 + 0.2M MnSO4\Results\02C-aMnO2-1M+02M-1").glob(r"*.txt"))

# 读取电化学数据
echema = []
for path_file in path_filelist:
    line_skip = None
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=",").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echema.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echema)):
    echema[i] = echema[i][echema[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echema[i] = echema[i].copy()
    echema[i]["Voltage"] = echema[i]["Ewe/V"] - echema[i]["Ece/V"]


# αMnO2 in Bulk Cells, 1M ZnSO4 + 0.2M MnSO4, no buffering => extended windows

# 电化学上的时间
path_filelist = list(
    Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\SpecialTest\BulkCell\NopHBufffer\1M ZnSO4 + 0.2M MnSO4\Results\02C-aMnO2-1M+02M-2-extendedwindows").glob(r"*.txt")
)

# 读取电化学数据
echemb = []
for path_file in path_filelist:
    line_skip = None
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=",").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echemb.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemb)):
    echemb[i] = echemb[i][echemb[i].iloc[:, 0].isin([0, 1, 2])]
    echemb[i] = echemb[i].copy()
    echemb[i]["Voltage"] = echemb[i]["Ewe/V"] - echemb[i]["Ece/V"]


# EMD in Bulk Cells, 1M ZnSO4 + 0.2M MnSO4, no buffering, 估计也是 C/5

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\EMD\GCD\1M ZnSO4 + 0.2M MnSO4\Results\BulkCell-2V-2mAh-1M+02M\01-02").glob(r"*.txt"))

# 读取电化学数据
echemc = []
for path_file in path_filelist:
    line_skip = None
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=",").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echemc.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemc)):
    echemc[i] = echemc[i][echemc[i].iloc[:, 0].isin([0, 1, 2])]
    echemc[i] = echemc[i].copy()
    echemc[i]["Voltage"] = echemc[i]["Ewe/V"] - echemc[i]["Ece/V"]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0.0, hspace=0.0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [[r"#1", None, None, None], [r"#2", None, None, None]]
mass_loading = [
    0.56,
]
for i, data in enumerate(echema):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
        # 找到电压的最大值的索引
        idx_max = temp["Voltage"].idxmax()
        if j == 0:
            continue
            # ax.plot(temp.loc[idx_max+2:, 'Capacity/mA.h']*1000/mass_loading[i]/60, temp.loc[idx_max+2:, 'Voltage'], ls='-', lw=1.0, c=colors[i], label=None, zorder=0)
        else:
            ax.plot(temp.loc[idx_max + 2 :, "Capacity/mA.h"] * 1000, temp.loc[idx_max + 2 :, "Voltage"], ls="-", lw=1.0, c=colors[j], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[:idx_max, "Capacity/mA.h"] * 1000, temp.loc[:idx_max, "Voltage"], ls="-", lw=1.0, c=colors[j], label=labels[i][j], zorder=0)


# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (\mu Ah_{MnO_2})}$", fontsize=11)
ax.set_xlim(-0.05, 3.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.1))

ax.tick_params(axis="both", direction="out", labelsize=9)
# ax.legend(loc='upper right', bbox_to_anchor=(1.0, 0.7), ncols=1, frameon=False, labelcolor='linecolor', fontsize=9)
ax.text(0.3, 0.9, r"$\mathrm{BulkCell, \, C/5}$" + "\n" + r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.23, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.4, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)
labels = [[r"#1", None, None, None], [r"#2", None, None, None]]
mass_loadings = [0.4, 1.36]
for _, data in enumerate(echemb):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
        # 找到容量最大的时候的索引
        idx_max = temp["Capacity/mA.h"].idxmax()
        ax.plot(temp.loc[:idx_max, "Capacity/mA.h"], temp.loc[:idx_max, "Voltage"], ls="-", lw=1.0, c=colors[j], label=None, zorder=0)
        ax.plot(temp.loc[idx_max + 1 :, "Capacity/mA.h"], temp.loc[idx_max + 1 :, "Voltage"], ls="-", lw=1.0, c=colors[j], label=None, zorder=0)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh_{MnO_2})}$", fontsize=11)
ax.set_xlim(-0.005, 0.6)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.05, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.9, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0.1))

ax.tick_params(axis="both", direction="out", labelsize=9)
# ax.legend(loc='upper right', bbox_to_anchor=(1.0, 1.03), ncols=1, frameon=False, labelcolor='linecolor', fontsize=9)
ax.text(0.03, 0.15, r"$\mathrm{Bulk Cell, C/5}$" + "\n" + r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.23, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.8, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [[r"#1", None, None, None], [r"#2", None, None, None]]
for _, data in enumerate(echemc):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
        capacity_max = temp["Capacity/mA.h"].max()
        # 找到最小电压的索引
        idx_max = temp["Voltage"].idxmin()
        ax.plot(temp.loc[:idx_max, "Capacity/mA.h"] / capacity_max * 100, temp.loc[:idx_max, "Voltage"], ls="-", lw=1.0, c=colors[j], label=None, zorder=0)
        ax.plot(temp.loc[idx_max + 1 :, "Capacity/mA.h"] / capacity_max * 100, temp.loc[idx_max + 1 :, "Voltage"], ls="-", lw=1.0, c=colors[j], label=None, zorder=0)


# 设置刻度线等格式
ax.set_xlabel(r"Full Charge Percentage (%)", fontsize=11)
ax.set_xlim(0, 105)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))

ax.set_ylabel(
    r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$",
    fontsize=11,
)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.1))

ax.tick_params(axis="both", direction="out", labelsize=9)
# ax.legend(loc='upper right', bbox_to_anchor=(1.0, 0.7), ncols=1, frameon=False, labelcolor='linecolor', fontsize=9)
ax.text(0.03, 0.15, r"$\mathrm{BulkCell}$" + "\n" + r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.23, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_02_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_02_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_02_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_02_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure Comparasions of alpha MnO2 and EMD in Swagelok cell and Bulk Cell with buffering -V1

In [ ]:
# αMnO2 in Swagelok Cells, 1M ZnSO4 + 0.2M MnSO4

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\1M ZnSO4 + 0.2M MnSO4").glob(r"LC*.xlsx"))
echema = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echema.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echema)):
    echema[i] = echema[i][echema[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echema[i] = echema[i][echema[i].iloc[:, 2] >= 0]


# αMnO2 in Bulk Cells, 1M ZnSO4 + 0.2M MnSO4 + Buffering

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\SpecialTest\BulkCell\pHBuffer").glob(r"Bulk-Buffer-Siavash*/**/*.txt"))
# 读取电化学数据
echemb = []
for path_file in path_filelist:
    line_skip = None
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=",").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echemb.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemb)):
    echemb[i] = echemb[i][echemb[i].iloc[:, 0].isin([0, 1])]
    echemb[i] = echemb[i].copy()
    echemb[i].loc[:, "Voltage"] = echemb[i].loc[:, "Ewe/V"] - echemb[i].loc[:, "Ece/V"]


# EMD in Swagelok Cells, 1M ZnSO4 + 0.2M MnSO4

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\EMD\GCD\1M ZnSO4 + 0.2M MnSO4\Results\1.60V-2mAh_cm-2-1MZnAC2+02M-pH5.0_1M+0.2M").glob(r"**/LC*.xlsx"))
echemc = []
path_filelist = path_filelist[0:2]
# 读取电化学数据
for path in path_filelist:
    if r"2V" not in path.stem:
        df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
        # 转换数据格式
        df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
        df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
        df["Cycle"] = df["Cycle"].astype(np.int16)
        echemc.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemc)):
    echemc[i] = echemc[i][echemc[i].iloc[:, 0].isin([0, 1, 2])]
    echemc[i] = echemc[i][echemc[i].iloc[:, 2] >= 0]

# EMD in Bulk Cells, 1M ZnSO4 + 0.2M MnSO4 + Buffering

# 电化学上的时间
path_filelist = list(
    Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\EMD\GCD\1M ZnSO4 + 0.2M MnSO4\Results\BulkCell_Buffer_SK-pH Buffer-Zn-1M ZnSO4 + 0.4M MnSO4-HAC-NaAC-CarbonFelt").glob(r"*.txt")
)
# 读取电化学数据
echemd = []
for path_file in path_filelist:
    line_skip = None
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率
    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=",").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echemd.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemd)):
    echemd[i] = echemd[i][echemd[i].iloc[:, 0].isin([0, 1, 2])]
    echemd[i] = echemd[i].copy()
    echemd[i].loc[:, "Voltage"] = echemd[i].loc[:, "Ewe/V"] - echemd[i].loc[:, "Ece/V"]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0.0, hspace=0.0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [[r"#1", None, None, None], [r"#2", None, None, None]]
for i, data in enumerate(echema):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)

        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        ax.plot(temp.loc[: idx_min - 1, "Capacity/uAh"] / 1000 / 0.503, temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "Capacity/uAh"] / 1000 / 0.503, temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)


# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \ cm^{-2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-0.01, 0.8)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.1))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.7), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.15, r"$\mathrm{C/10}$" + "\n" + r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.15, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)
labels = [[r"#1", None, None, None], [r"#2", None, None, None]]
mass_loadings = [0.4, 1.36]
for i, data in enumerate(echemb):
    data_capacity_max = data["Capacity/mA.h"].max()
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
        # 找到容量最大的时候的索引
        idx_max = temp["Capacity/mA.h"].idxmax()
        ax.plot(temp.loc[:idx_max, "Capacity/mA.h"] / data_capacity_max * 100, temp.loc[:idx_max, "Voltage"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_max + 1 :, "Capacity/mA.h"] / data_capacity_max * 100, temp.loc[idx_max + 1 :, "Voltage"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)

# 设置刻度线等格式
ax.set_xlabel(r"Full Charge Percentage (%)", fontsize=11)
ax.set_xlim(0, 105)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1.0), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.15, r"$\mathrm{Bulk Cell, Buffered}$" + "\n" + r"$\mathrm{0.5 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, -0.3, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [[r"#1", None, None, None], [r"#2", None, None, None]]
for i, data in enumerate(echemc):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
        # 找到最小电压的索引
        idx_max = temp["Voltage/V"].idxmin()
        ax.plot(temp.loc[:idx_max, "Capacity/uAh"] / 1000 / 0.503, temp.loc[:idx_max, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_max + 1 :, "Capacity/uAh"] / 1000 / 0.503, temp.loc[idx_max + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)


# 设置刻度线等格式
ax.set_xlabel(
    r"$\mathrm{Capacity \ (mAh \ cm^{-2})}$",
    fontsize=11,
)
ax.set_xlim(-0.01, 1.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))

ax.set_ylabel(
    r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$",
    fontsize=11,
)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.1))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.7), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.15, r"$\mathrm{Totally \  1 mAh, \, C/10}$" + "\n" + r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.15, -0.3, 1.0, 1.0))
ax.set_box_aspect(0.8)
labels = [[r"#1", None, None, None], [r"#2", None, None, None]]
mass_loadings = [0.4, 1.36]
for i, data in enumerate(echemd):
    data_capacity_max = data["Capacity/mA.h"].max()
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
        # 找到容量最大的时候的索引
        idx_max = temp["Capacity/mA.h"].idxmax()
        ax.plot(temp.loc[:idx_max, "Capacity/mA.h"] / data_capacity_max * 100, temp.loc[:idx_max, "Voltage"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_max + 1 :, "Capacity/mA.h"] / data_capacity_max * 100, temp.loc[idx_max + 1 :, "Voltage"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)

# 设置刻度线等格式
ax.set_xlabel(r"Full Charge Percentage (%)", fontsize=11)
ax.set_xlim(0, 105)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.9), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.15, r"$\mathrm{Bulk Cell, Buffered}$" + "\n" + r"$\mathrm{0.5 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"D", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_01_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_01_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_01_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_01_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

### Figure Comparasions of alpha MnO2 and EMD in Swagelok cell and Bulk Cell with buffering

In [ ]:
# αMnO2 in Swagelok Cells, 1M ZnSO4 + 0.2M MnSO4

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\GCD\1M ZnSO4 + 0.2M MnSO4").glob(r"LC*.xlsx"))
echema = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echema.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echema)):
    echema[i] = echema[i][echema[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echema[i] = echema[i][echema[i].iloc[:, 2] >= 0]


# αMnO2 in Bulk Cells, 1M ZnSO4 + 0.2M MnSO4 + Buffering

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\αMnO2\SpecialTest\BulkCell\pHBuffer").glob(r"Bulk-Buffer-Siavash*/**/*.txt"))
# 读取电化学数据
echemb = []
for path_file in path_filelist:
    line_skip = None
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=",").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echemb.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemb)):
    echemb[i] = echemb[i][echemb[i].iloc[:, 0].isin([0, 1])]
    echemb[i] = echemb[i].copy()
    echemb[i].loc[:, "Voltage"] = echemb[i].loc[:, "Ewe/V"] - echemb[i].loc[:, "Ece/V"]


# EMD in Swagelok Cells, 1M ZnSO4 + 0.2M MnSO4

# 电化学上的时间
path_filelist = list(Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\EMD\GCD\1M ZnSO4 + 0.2M MnSO4\Results\1.60V-2mAh_cm-2-1MZnAC2+02M-pH5.0_1M+0.2M").glob(r"**/LC*.xlsx"))
echemc = []
path_filelist = path_filelist[0:2]
# 读取电化学数据
for path in path_filelist:
    if r"2V" not in path.stem:
        df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
        # 转换数据格式
        df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
        df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
        df["Cycle"] = df["Cycle"].astype(np.int16)
        echemc.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemc)):
    echemc[i] = echemc[i][echemc[i].iloc[:, 0].isin([0, 1, 2])]
    echemc[i] = echemc[i][echemc[i].iloc[:, 2] >= 0]

# EMD in Bulk Cells, 1M ZnSO4 + 0.2M MnSO4 + Buffering

# 电化学上的时间
path_filelist = list(
    Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperDos\Echem\EMD\GCD\1M ZnSO4 + 0.2M MnSO4\Results\BulkCell_Buffer_SK-pH Buffer-Zn-1M ZnSO4 + 0.4M MnSO4-HAC-NaAC-CarbonFelt").glob(r"*.txt")
)
# 读取电化学数据
echemd = []
for path_file in path_filelist:
    line_skip = None
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率
    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=",").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echemd.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemd)):
    echemd[i] = echemd[i][echemd[i].iloc[:, 0].isin([0, 1, 2])]
    echemd[i] = echemd[i].copy()
    echemd[i].loc[:, "Voltage"] = echemd[i].loc[:, "Ewe/V"] - echemd[i].loc[:, "Ece/V"]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0.0, hspace=0.0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [[r"#1", None, None, None], [r"#2", None, None, None]]
for i, data in enumerate(echema):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)

        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)


# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-5, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=100, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=50, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.1))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.7), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.15, r"$\mathrm{C/10}$" + "\n" + r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.15, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)
labels = [[r"#1", None, None, None], [r"#2", None, None, None]]
mass_loadings = [0.4, 1.36]
for i, data in enumerate(echemb):
    data_capacity_max = data["Capacity/mA.h"].max()
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
        # 找到容量最大的时候的索引
        idx_max = temp["Capacity/mA.h"].idxmax()
        ax.plot(temp.loc[:idx_max, "Capacity/mA.h"] / data_capacity_max * 100, temp.loc[:idx_max, "Voltage"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_max + 1 :, "Capacity/mA.h"] / data_capacity_max * 100, temp.loc[idx_max + 1 :, "Voltage"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)

# 设置刻度线等格式
ax.set_xlabel(r"Full Charge Percentage (%)", fontsize=11)
ax.set_xlim(0, 105)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1.0), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.15, r"$\mathrm{Bulk Cell, Buffered}$" + "\n" + r"$\mathrm{0.5 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, -0.3, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [[r"#1", None, None, None], [r"#2", None, None, None]]
for i, data in enumerate(echemc):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
        # 找到最小电压的索引
        idx_max = temp["Voltage/V"].idxmin()
        ax.plot(temp.loc[:idx_max, "Capacity/uAh"] / 1000, temp.loc[:idx_max, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_max + 1 :, "Capacity/uAh"] / 1000, temp.loc[idx_max + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)


# 设置刻度线等格式
ax.set_xlabel(
    r"$\mathrm{Capacity \ (mAh_{MnO_2})}$",
    fontsize=11,
)
# ax.set_xlim(-0.005, 0.15)
# ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.05, offset=0))
# ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.025, offset=0))

ax.set_ylabel(
    r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$",
    fontsize=11,
)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.1))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.7), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.15, r"$\mathrm{Totally \  1 mAh, \, C/10}$" + "\n" + r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"C", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.15, -0.3, 1.0, 1.0))
ax.set_box_aspect(0.8)
labels = [[r"#1", None, None, None], [r"#2", None, None, None]]
mass_loadings = [0.4, 1.36]
for i, data in enumerate(echemd):
    data_capacity_max = data["Capacity/mA.h"].max()
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
        # 找到容量最大的时候的索引
        idx_max = temp["Capacity/mA.h"].idxmax()
        ax.plot(temp.loc[:idx_max, "Capacity/mA.h"] / data_capacity_max * 100, temp.loc[:idx_max, "Voltage"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_max + 1 :, "Capacity/mA.h"] / data_capacity_max * 100, temp.loc[idx_max + 1 :, "Voltage"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)

# 设置刻度线等格式
ax.set_xlabel(r"Full Charge Percentage (%)", fontsize=11)
ax.set_xlim(0, 105)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.9), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.15, r"$\mathrm{Bulk Cell, Buffered}$" + "\n" + r"$\mathrm{0.5 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(-0.18, 1.0, r"D", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_01_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_01_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_01_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_Echem_01_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()